# Modelo de Pronóstico Electoral Midterms 2026 — versión 17

Paquete reproducible del modelo nacional, el módulo estatal del Senado y la
capa distrital de la Cámara de Representantes. La arquitectura estadística se
mantiene congelada; esta revisión final integra la identidad visual Núcleo 42,
la geometría oficial CD120 y mejoras exclusivamente informativas en los hovers
del mapa del Senado.

El proyecto usa `Model.xlsx` como fuente canónica, escribe los reportes
auditables en `outputs/` y genera un único dashboard HTML autocontenido.


## Capa visual final — identidad Núcleo 42

La versión 17 conserva intactos los Bloques 1–7 y modifica solamente la capa de
presentación del Bloque 8:

- logo oficial Núcleo 42 integrado en el HTML offline;
- mapa geográfico House con los 435 distritos CD120 oficiales;
- activos GIS externos auditables con respaldo embebido en el notebook;
- hovers del Senado diferenciados por métrica (probabilidad, rating, margen,
  forecast y holds/flips);
- dashboard responsive, autocontenido y preparado para publicación.


In [ ]:
# ============================================================
# BLOQUE 1 — CARGA, LIMPIEZA Y DETECCIÓN DE VARIABLES P
# ============================================================

import re
import hashlib
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------
# 1. Ruta del archivo
# -----------------------------

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_FILE = BASE_DIR / "Model.xlsx"
FINAL_REPORT_PATH = OUTPUT_DIR / "Election_Model_Final_Report_v17.xlsx"
HTML_OUTPUT_PATH = BASE_DIR / "Election_Model_2026_Dashboard_v17.html"

# Un único nombre canónico evita mezclar Model.xlsx con una copia vieja llamada Model .xlsx.
if not MODEL_FILE.exists():
    raise FileNotFoundError(f"No encontré el archivo: {MODEL_FILE}")

# Huella de la fuente: viajará hasta el reporte y el HTML.
MODEL_SHA256 = hashlib.sha256(MODEL_FILE.read_bytes()).hexdigest()
MODEL_MODIFIED_UTC = datetime.fromtimestamp(
    MODEL_FILE.stat().st_mtime, tz=timezone.utc
).isoformat()
RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
RUN_ID = f"2026-{MODEL_SHA256[:12]}-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"

# -----------------------------
# 2. Cargar Excel sin asumir encabezados
# -----------------------------

raw = pd.read_excel(MODEL_FILE, sheet_name="Model", header=None, dtype=object)

header_idx = None

for i in range(len(raw)):
    row = raw.iloc[i].astype(str).str.strip().tolist()
    if "Midterm Year" in row:
        header_idx = i
        break

if header_idx is None:
    raise ValueError("No encontré la fila con 'Midterm Year'.")

section_idx = header_idx - 1

section_row = raw.iloc[section_idx].ffill()
header_row = raw.iloc[header_idx].astype(str).str.strip()

df = raw.iloc[header_idx + 1:].copy()
df.columns = header_row

# Quitar filas sin año
df = df[df["Midterm Year"].notna()].copy()
df["Midterm Year"] = df["Midterm Year"].astype(int)

# -----------------------------
# 3. Detectar P en 2026
# -----------------------------

rows_2026 = df[df["Midterm Year"] == 2026]
if len(rows_2026) != 1:
    raise ValueError("La hoja Model debe contener exactamente una fila para 2026.")
row_2026 = rows_2026.iloc[0]

p_cols = [
    c for c in df.columns
    if str(row_2026[c]).strip().upper() == "P"
]

w_cols = [
    c for c in df.columns
    if str(row_2026[c]).strip().upper() == "W"
]

feature_cols = [
    c for c in df.columns
    if c not in p_cols
]

# -----------------------------
# 4. Identificar categóricas conocidas
# -----------------------------

categorical_features = [
    "Incumbent Party PRES",
    "Incumbent Party Senate",
    "Incumbent Party House"
]

categorical_features = [
    c for c in categorical_features
    if c in feature_cols
]

numeric_features = [
    c for c in feature_cols
    if c not in categorical_features
]

# -----------------------------
# 5. Funciones de limpieza
# -----------------------------

def clean_numeric_value(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, str):
        x = x.strip()
        x = x.replace("%", "")
        x = x.replace(",", "")

        if x.upper() in ["P", "W", "NAN", "NONE", ""]:
            return np.nan

    return pd.to_numeric(x, errors="coerce")


def clean_model_dataframe(data):
    out = data.copy()

    for col in numeric_features:
        out[col] = out[col].apply(clean_numeric_value)

    for col in categorical_features:
        out[col] = out[col].astype(str).str.strip()

    for col in p_cols:
        out[col] = out[col].apply(clean_numeric_value)

    return out

model_df = clean_model_dataframe(df)


# -----------------------------
# 5B. Contrato de unidades v12
# -----------------------------
#
# La base combina proporciones (0–1) y puntos porcentuales (0–100) en unas
# pocas baterías. Esa incompatibilidad no afecta la validación histórica si
# todos los ciclos pasados comparten escala, pero sí vuelve 2026 artificialmente
# fuera de distribución. La corrección es semántica y anterior al modelado:
# todas estas respuestas se expresan como proporciones.

proportion_batteries = {
    "Economic conditions": [
        "ECC Excellent", "ECC Good", "ECC Only Fair", "ECC Poor"
    ],
    "Economy direction": [
        "ECO GBetter", "ECO GWorse", "ECO Same"
    ],
    "Job market": [
        "Job Good Time", "Job Bad Time"
    ],
}

unit_audit_rows = []

for battery_name, columns in proportion_batteries.items():
    missing_columns = [column for column in columns if column not in model_df.columns]
    if missing_columns:
        raise ValueError(
            f"Faltan columnas del contrato de unidades {battery_name}: {missing_columns}"
        )

    for row_index in model_df.index:
        year = int(model_df.at[row_index, "Midterm Year"])
        before = model_df.loc[row_index, columns].astype(float)
        before_sum = float(before.sum())
        conversion = "none"

        if before_sum > 1.5:
            model_df.loc[row_index, columns] = before / 100.0
            conversion = "percentage points -> proportion"

        after = model_df.loc[row_index, columns].astype(float)
        after_sum = float(after.sum())
        unit_audit_rows.append({
            "Year": year,
            "Battery": battery_name,
            "Columns": ", ".join(columns),
            "Before Sum": before_sum,
            "After Sum": after_sum,
            "Action": conversion,
            "Passed": bool(0.90 <= after_sum <= 1.10),
        })

# Las columnas netas históricas estaban en puntos porcentuales, mientras las
# favorabilidades y desfavorabilidades ya estaban en proporciones. Recalcular
# el neto desde sus componentes evita inferencias frágiles basadas en magnitud.
favorability_specs = [
    ("Democratic favorability", "DEM FAVORABLE", "DEM UNFAVORABLE", "DEM NET FAVORABILITY"),
    ("Republican favorability", "REP FAVORABLE", "REP UNFAVORABLE", "REP NET FAVORABILITY"),
]

for label, favorable, unfavorable, net in favorability_specs:
    missing_columns = [
        column for column in [favorable, unfavorable, net]
        if column not in model_df.columns
    ]
    if missing_columns:
        raise ValueError(f"Faltan columnas de {label}: {missing_columns}")

    for row_index in model_df.index:
        year = int(model_df.at[row_index, "Midterm Year"])
        before = float(model_df.at[row_index, net])
        after = float(
            model_df.at[row_index, favorable] - model_df.at[row_index, unfavorable]
        )
        model_df.at[row_index, net] = after
        unit_audit_rows.append({
            "Year": year,
            "Battery": label,
            "Columns": f"{favorable} - {unfavorable}",
            "Before Sum": before,
            "After Sum": after,
            "Action": "recomputed from proportion components",
            "Passed": bool(-1.0 <= after <= 1.0),
        })

unit_normalization_audit = pd.DataFrame(unit_audit_rows)
if not unit_normalization_audit["Passed"].all():
    raise AssertionError(
        "Falló el contrato de unidades:\n"
        + unit_normalization_audit.loc[
            ~unit_normalization_audit["Passed"]
        ].to_string(index=False)
    )

# -----------------------------
# 6. Separar histórico y 2026
# -----------------------------

train_df = model_df[model_df["Midterm Year"] < 2026].copy()
pred_df = model_df[model_df["Midterm Year"] == 2026].copy()

X_train = train_df[feature_cols].copy()
Y_train = train_df[p_cols].copy()
X_2026 = pred_df[feature_cols].copy()

# -----------------------------
# 7. Reporte inicial
# -----------------------------

if len(p_cols) != 42 or len(feature_cols) != 71:
    raise ValueError(
        f"Estructura inesperada: {len(feature_cols)} features y {len(p_cols)} P; "
        "se esperaban 71 y 42."
    )

missing_2026_features = model_df.loc[model_df["Midterm Year"] == 2026, feature_cols].isna().sum()
missing_2026_features = missing_2026_features[missing_2026_features > 0]
if not missing_2026_features.empty:
    raise ValueError("Inputs 2026 faltantes:\n" + missing_2026_features.to_string())

print("Modelo cargado correctamente")
print("Run ID:", RUN_ID)
print("SHA-256:", MODEL_SHA256)
print("Archivo:", MODEL_FILE)
print("Filas:", df.shape[0])
print("Columnas totales:", df.shape[1])
print("Años:", df["Midterm Year"].tolist())
print("Features conocidas:", len(feature_cols))
print("Targets P:", len(p_cols))
print("W detectadas:", len(w_cols))
print("Numéricas:", len(numeric_features))
print("Categóricas:", len(categorical_features))
print("Controles de unidades aprobados:", int(unit_normalization_audit["Passed"].sum()), "/", len(unit_normalization_audit))

print("\nCategóricas:")
for c in categorical_features:
    print("-", c)

print("\nVariables P:")
for c in p_cols:
    print("-", c)

# Verificación de faltantes históricos en P
missing_p = Y_train.isna().sum().sort_values(ascending=False)

print("\nMáximo de faltantes históricos en P:", missing_p.max())

if missing_p.max() > 0:
    print("Advertencia: hay P históricas con faltantes.")
    display(missing_p[missing_p > 0])
else:
    print("OK: las P históricas están completas.")

## BLOQUE 2 — TWO-EXAM TIME MACHINE v12

Examen 1: selección completamente anidada y predicción del ciclo exterior. Examen 2: paso de esas mismas predicciones selladas por las restricciones e identidades del pipeline. Los cinco pronósticos parciales de 2026 se conservan como análisis de estabilidad; el ajuste con todo 2006–2022 es la única predicción de producción.


In [ ]:
# ============================================================
# BLOQUE 2 — NATIONAL FORECAST ENGINE (v12 TIME MACHINE)
# Nested leave-one-election-year-out validation, prudent tuning,
# simple baselines and a complexity guardrail.
# ============================================================

from copy import deepcopy
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
import warnings
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. CLASIFICAR VARIABLES P
# ------------------------------------------------------------

derived_cols = [
    "D Senate Seats LOST",
    "R Senate Seats LOST",
    "D House Seats LOST",
    "R House Seats Lost",
    "D Senate After",
    "R Senate After",
    "D House After",
    "R House After",
]
derived_cols = [c for c in derived_cols if c in p_cols]

popular_vote_cols = [c for c in p_cols if c in ["DPP", "RPP"]]
house_bucket_cols = [c for c in p_cols if c.startswith("DHou") or c.startswith("RHou")]
senate_bucket_cols = [c for c in p_cols if c.startswith("DSen") or c.startswith("RSen")]
ml_target_cols = popular_vote_cols + house_bucket_cols + senate_bucket_cols
ml_target_cols = [c for c in ml_target_cols if c not in derived_cols]


def get_group(col):
    if col in popular_vote_cols:
        return "Popular Vote"
    if col in house_bucket_cols:
        return "House Buckets"
    if col in senate_bucket_cols:
        return "Senate Buckets"
    if col in derived_cols:
        return "Derived Later"
    return "Other"


target_groups = {col: get_group(col) for col in p_cols}
modeling_groups = ["Popular Vote", "House Buckets", "Senate Buckets"]

print("Targets P totales:", len(p_cols))
print("Targets ML:", len(ml_target_cols))
print("Derived later:", len(derived_cols))

# ------------------------------------------------------------
# 2. CONFIGURACIÓN PRUDENTE
# ------------------------------------------------------------

NATIONAL_RANDOM_STATE = 42
NATIONAL_TREE_COUNT = 250
NATIONAL_COMPLEXITY_TOLERANCE = 0.02

# La cuadrícula es deliberadamente pequeña. Con cinco ciclos, una búsqueda
# extensa aprendería el ruido histórico en vez de generalizar.
national_parameter_grid = {
    "ExtraTrees": [
        {"max_features": "sqrt", "min_samples_leaf": 1, "max_depth": None},
        {"max_features": 0.50, "min_samples_leaf": 1, "max_depth": 3},
    ],
    "RandomForest": [
        {"max_features": "sqrt", "min_samples_leaf": 1, "max_depth": None},
        {"max_features": 0.50, "min_samples_leaf": 1, "max_depth": 3},
    ],
    "Ridge": [
        {"alpha": 1.0, "solver": "lsqr"},
        {"alpha": 10.0, "solver": "lsqr"},
        {"alpha": 100.0, "solver": "lsqr"},
    ],
}


def make_tree_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", SimpleImputer(strategy="median"), numeric_features),
        ]
    )


def make_ridge_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                    dtype=np.float64,
                ),
                categorical_features,
            ),
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]),
                numeric_features,
            ),
        ]
    )


def make_national_model(family, parameters):
    parameters = deepcopy(parameters)
    if family == "ExtraTrees":
        estimator = ExtraTreesRegressor(
            n_estimators=NATIONAL_TREE_COUNT,
            random_state=NATIONAL_RANDOM_STATE,
            n_jobs=-1,
            **parameters,
        )
        return Pipeline([("prep", make_tree_preprocessor()), ("model", estimator)])
    if family == "RandomForest":
        estimator = RandomForestRegressor(
            n_estimators=NATIONAL_TREE_COUNT,
            bootstrap=True,
            random_state=NATIONAL_RANDOM_STATE,
            n_jobs=-1,
            **parameters,
        )
        return Pipeline([("prep", make_tree_preprocessor()), ("model", estimator)])
    if family == "Ridge":
        # Cada target se ajusta por separado y en escala estandarizada.
        # Esto evita que un diseño extremadamente ancho (muy pocos ciclos y
        # muchas variables) produzca coeficientes o productos matriciales
        # desbordados en algunas combinaciones de NumPy/SciPy/Scikit-learn.
        base_ridge = Ridge(max_iter=10_000, tol=1e-8, **parameters)
        independent_targets = MultiOutputRegressor(base_ridge, n_jobs=1)
        estimator = TransformedTargetRegressor(
            regressor=independent_targets,
            transformer=StandardScaler(),
            check_inverse=True,
        )
        return Pipeline([("prep", make_ridge_preprocessor()), ("model", estimator)])
    raise ValueError(f"Familia nacional no reconocida: {family}")


def parameter_label(parameters):
    return "; ".join(f"{key}={parameters[key]}" for key in sorted(parameters))


def fit_predict_family(family, parameters, fit_frame, predict_frame):
    target_values = fit_frame[ml_target_cols].to_numpy(dtype=np.float64)
    if not np.isfinite(target_values).all():
        raise FloatingPointError(
            f"{family} recibió targets con NaN o infinitos."
        )

    # SimpleImputer puede resolver NaN, pero no debe recibir infinitos.
    fit_numeric = fit_frame[numeric_features].to_numpy(dtype=np.float64)
    predict_numeric = predict_frame[numeric_features].to_numpy(dtype=np.float64)
    if np.isinf(fit_numeric).any() or np.isinf(predict_numeric).any():
        raise FloatingPointError(
            f"{family} recibió features numéricas con infinitos."
        )

    fitted = make_national_model(family, parameters)
    # No ocultar inestabilidad numérica: cualquier RuntimeWarning producido
    # durante el ajuste o la predicción se convierte en un error explícito.
    with warnings.catch_warnings(record=True) as caught_warnings:
        warnings.simplefilter("always", RuntimeWarning)
        fitted.fit(fit_frame[feature_cols], target_values)
        prediction = np.asarray(
            fitted.predict(predict_frame[feature_cols]), dtype=np.float64
        )

    numerical_warnings = [
        warning
        for warning in caught_warnings
        if issubclass(warning.category, RuntimeWarning)
    ]
    if numerical_warnings:
        warning_messages = sorted({
            str(warning.message) for warning in numerical_warnings
        })
        raise FloatingPointError(
            f"{family} produjo advertencias numéricas: "
            + " | ".join(warning_messages)
        )

    if family == "Ridge":
        target_transformer = fitted.named_steps["model"]
        multioutput_model = target_transformer.regressor_
        ridge_estimators = multioutput_model.estimators_
        ridge_parameters = [
            values
            for estimator in ridge_estimators
            for values in (
                np.asarray(estimator.coef_, dtype=np.float64),
                np.asarray(estimator.intercept_, dtype=np.float64),
            )
        ]
        target_scale = np.asarray(
            target_transformer.transformer_.scale_, dtype=np.float64
        )
        if (
            not all(np.isfinite(values).all() for values in ridge_parameters)
            or not np.isfinite(target_scale).all()
        ):
            raise FloatingPointError(
                "Ridge produjo parámetros o escalas de target no finitos."
            )

    if prediction.ndim == 1:
        prediction = prediction.reshape(1, -1)
    if prediction.shape != (len(predict_frame), len(ml_target_cols)):
        raise AssertionError(
            f"Forma inesperada para {family}: {prediction.shape}"
        )
    if not np.isfinite(prediction).all():
        raise FloatingPointError(f"{family} produjo NaN o infinitos.")
    return fitted, prediction


def historical_mean_predictions(fit_frame, predict_frame):
    mean_vector = fit_frame[ml_target_cols].mean(axis=0).to_numpy(float)
    return np.repeat(mean_vector.reshape(1, -1), len(predict_frame), axis=0)


def build_inner_oof(available_frame, family=None, parameters=None):
    """Predicciones internas sin usar el año pronosticado."""
    predictions = pd.DataFrame(
        np.nan, index=available_frame.index, columns=ml_target_cols, dtype=float
    )
    for inner_year in sorted(available_frame["Midterm Year"].astype(int).unique()):
        inner_train = available_frame[
            available_frame["Midterm Year"].astype(int) != int(inner_year)
        ]
        inner_test = available_frame[
            available_frame["Midterm Year"].astype(int) == int(inner_year)
        ]
        if len(inner_train) < 2:
            raise ValueError("Se requieren al menos dos ciclos para el ajuste interno.")
        if family == "HistoricalMean":
            fold_prediction = historical_mean_predictions(inner_train, inner_test)
        else:
            _, fold_prediction = fit_predict_family(
                family, parameters, inner_train, inner_test
            )
        predictions.loc[inner_test.index, :] = fold_prediction
    if predictions.isna().any().any():
        raise AssertionError(f"OOF interno incompleto para {family}.")
    return predictions


def group_mae(actual_frame, prediction_frame, group):
    columns = [c for c in ml_target_cols if target_groups[c] == group]
    actual = actual_frame.loc[prediction_frame.index, columns].to_numpy(float)
    predicted = prediction_frame[columns].to_numpy(float)
    return float(np.mean(np.abs(actual - predicted)))


def inverse_mae_weights(extra_mae, forest_mae):
    inverse_extra = 1.0 / (float(extra_mae) + 1e-9)
    inverse_forest = 1.0 / (float(forest_mae) + 1e-9)
    total = inverse_extra + inverse_forest
    return {
        "ExtraTrees": inverse_extra / total,
        "RandomForest": inverse_forest / total,
    }


def choose_strategy(strategy_mae):
    """Prefiere el método más simple cuando está dentro de 2% del mejor MAE."""
    complexity_rank = {
        "HistoricalMean": 0,
        "ExpectedVoteAnchor": 0,
        "ExpectedVoteErrorCorrected": 1,
        "Ridge": 1,
        "TreeEnsemble": 2,
    }
    best_mae = min(strategy_mae.values())
    tolerance = max(abs(best_mae) * NATIONAL_COMPLEXITY_TOLERANCE, 1e-9)
    eligible = [
        strategy for strategy, mae in strategy_mae.items()
        if mae <= best_mae + tolerance
    ]
    return min(eligible, key=lambda name: complexity_rank[name])



# ------------------------------------------------------------
# 2B. SUBMODELO ESPECÍFICO DEL VOTO POPULAR
# ------------------------------------------------------------
#
# EDPP and ERPP are the contemporaneous pre-election national expectations.
# They are compared with the broad 71-feature models; they are not forced to
# win. Other/independent vote is estimated only from the available historical
# cycles, and the correction to the two-party margin is selected inside each
# outer fold.

POPULAR_VOTE_CORRECTION_GRID = [
    0.0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.75, 1.0
]
POPULAR_VOTE_INPUT_COLUMNS = ["EDPP", "ERPP"]

if not set(POPULAR_VOTE_INPUT_COLUMNS).issubset(feature_cols):
    raise ValueError(
        "El submodelo de voto popular requiere EDPP y ERPP como inputs."
    )


def popular_vote_raw_margin_pp(frame):
    return (
        frame["DPP"].to_numpy(float)
        - frame["RPP"].to_numpy(float)
    ) * 100.0


def popular_vote_two_party_margin_pp(frame, d_column, r_column):
    d_values = frame[d_column].to_numpy(float)
    r_values = frame[r_column].to_numpy(float)
    total = d_values + r_values
    if np.any(total <= 0):
        raise ValueError(
            f"Total bipartidista no positivo en {d_column}/{r_column}."
        )
    return 100.0 * (d_values - r_values) / total


def predict_popular_vote_expectation(
    fit_frame,
    predict_frame,
    correction_weight=0.0,
):
    """Predict DPP/RPP from the national expectation without Senate inputs."""
    correction_weight = float(correction_weight)
    if not 0.0 <= correction_weight <= 1.0:
        raise ValueError("El peso de corrección popular debe estar entre 0 y 1.")

    historical_other = (
        1.0
        - fit_frame["DPP"].to_numpy(float)
        - fit_frame["RPP"].to_numpy(float)
    )
    other_share = float(np.clip(np.mean(historical_other), 0.0, 0.15))

    expected_margin = popular_vote_two_party_margin_pp(
        predict_frame, "EDPP", "ERPP"
    )
    actual_margin_fit = popular_vote_two_party_margin_pp(
        fit_frame, "DPP", "RPP"
    )
    expected_margin_fit = popular_vote_two_party_margin_pp(
        fit_frame, "EDPP", "ERPP"
    )
    mean_historical_miss = float(np.mean(
        actual_margin_fit - expected_margin_fit
    ))
    corrected_margin = np.clip(
        expected_margin + correction_weight * mean_historical_miss,
        -100.0,
        100.0,
    )

    d_two_party = (corrected_margin + 100.0) / 200.0
    allocated_share = 1.0 - other_share
    result = pd.DataFrame(
        {
            "DPP": allocated_share * d_two_party,
            "RPP": allocated_share * (1.0 - d_two_party),
        },
        index=predict_frame.index,
        dtype=float,
    )
    if not np.isfinite(result.to_numpy(float)).all():
        raise FloatingPointError(
            "El submodelo de voto popular produjo valores no finitos."
        )
    return result


def build_popular_vote_expectation_oof(
    available_frame,
    correction_weight=0.0,
):
    predictions = pd.DataFrame(
        np.nan,
        index=available_frame.index,
        columns=popular_vote_cols,
        dtype=float,
    )
    for inner_year in sorted(
        available_frame["Midterm Year"].astype(int).unique()
    ):
        inner_train = available_frame[
            available_frame["Midterm Year"].astype(int) != int(inner_year)
        ]
        inner_test = available_frame[
            available_frame["Midterm Year"].astype(int) == int(inner_year)
        ]
        fold_prediction = predict_popular_vote_expectation(
            inner_train,
            inner_test,
            correction_weight=correction_weight,
        )
        predictions.loc[inner_test.index, popular_vote_cols] = (
            fold_prediction[popular_vote_cols]
        )
    if predictions.isna().any().any():
        raise AssertionError("OOF popular incompleto.")
    return predictions


def popular_vote_margin_mae(actual_frame, prediction_frame):
    actual_margin = popular_vote_raw_margin_pp(
        actual_frame.loc[prediction_frame.index]
    )
    predicted_margin = popular_vote_raw_margin_pp(prediction_frame)
    return float(np.mean(np.abs(actual_margin - predicted_margin)))


def select_popular_vote_correction(available_frame):
    candidates = []
    oof_by_weight = {}
    for correction_weight in POPULAR_VOTE_CORRECTION_GRID:
        candidate_oof = build_popular_vote_expectation_oof(
            available_frame,
            correction_weight=correction_weight,
        )
        candidate_mae = popular_vote_margin_mae(
            available_frame[popular_vote_cols],
            candidate_oof,
        )
        oof_by_weight[float(correction_weight)] = candidate_oof
        candidates.append({
            "Correction Weight": float(correction_weight),
            "Popular Margin Inner OOF MAE PP": candidate_mae,
        })

    candidate_table = pd.DataFrame(candidates).sort_values(
        ["Popular Margin Inner OOF MAE PP", "Correction Weight"],
        kind="mergesort",
    )
    best_mae = float(
        candidate_table["Popular Margin Inner OOF MAE PP"].min()
    )
    tolerance = max(
        abs(best_mae) * NATIONAL_COMPLEXITY_TOLERANCE,
        1e-9,
    )
    eligible = candidate_table[
        candidate_table["Popular Margin Inner OOF MAE PP"]
        <= best_mae + tolerance
    ]
    selected_weight = float(eligible["Correction Weight"].min())
    return {
        "selected_weight": selected_weight,
        "selected_oof": oof_by_weight[selected_weight],
        "anchor_oof": oof_by_weight[0.0],
        "candidate_table": candidate_table,
    }


def tune_on_available_cycles(available_frame, stage_label):
    """Selecciona hiperparámetros, pesos y estrategia usando solo estos ciclos."""
    oof_cache = {}
    tuning_rows = []

    mean_oof = build_inner_oof(available_frame, family="HistoricalMean")
    oof_cache[("HistoricalMean", "mean")] = mean_oof

    for family, configurations in national_parameter_grid.items():
        for config_number, parameters in enumerate(configurations, start=1):
            key = (family, config_number)
            candidate_oof = build_inner_oof(
                available_frame, family=family, parameters=parameters
            )
            oof_cache[key] = candidate_oof
            for group in modeling_groups:
                tuning_rows.append({
                    "Stage": stage_label,
                    "Family": family,
                    "Configuration": config_number,
                    "Parameters": parameter_label(parameters),
                    "Group": group,
                    "Inner OOF MAE": group_mae(
                        available_frame[ml_target_cols], candidate_oof, group
                    ),
                })

    tuning_table = pd.DataFrame(tuning_rows)
    selection = {}

    for group in modeling_groups:
        selected_configs = {}
        selected_oof = {}
        selected_mae = {}

        for family in ["ExtraTrees", "RandomForest", "Ridge"]:
            family_rows = tuning_table[
                (tuning_table["Group"] == group)
                & (tuning_table["Family"] == family)
            ].sort_values(
                ["Inner OOF MAE", "Configuration"],
                kind="mergesort",
            )
            winner = family_rows.iloc[0]
            config_number = int(winner["Configuration"])
            selected_configs[family] = {
                "number": config_number,
                "parameters": deepcopy(
                    national_parameter_grid[family][config_number - 1]
                ),
            }
            selected_oof[family] = oof_cache[(family, config_number)]
            selected_mae[family] = float(winner["Inner OOF MAE"])

        tree_weights = inverse_mae_weights(
            selected_mae["ExtraTrees"], selected_mae["RandomForest"]
        )
        group_columns = [
            c for c in ml_target_cols if target_groups[c] == group
        ]
        tree_oof = (
            tree_weights["ExtraTrees"] * selected_oof["ExtraTrees"][group_columns]
            + tree_weights["RandomForest"] * selected_oof["RandomForest"][group_columns]
        )

        strategy_mae = {
            "HistoricalMean": group_mae(
                available_frame[ml_target_cols], mean_oof, group
            ),
            "Ridge": selected_mae["Ridge"],
            "TreeEnsemble": group_mae(
                available_frame[ml_target_cols], tree_oof, group
            ),
        }
        selection[group] = {
            "selected_configs": selected_configs,
            "tree_weights": tree_weights,
            "strategy_mae": strategy_mae,
            "selected_strategy": choose_strategy(strategy_mae),
            "selected_oof": selected_oof,
            "mean_oof": mean_oof,
            "tree_oof": tree_oof,
        }


    # Popular vote is selected on the headline D-R margin, not on the average
    # of two separate party-share losses. This keeps the objective aligned with
    # the dashboard and makes the direct national expectation a mandatory
    # baseline rather than an omitted comparator.
    popular_choice = select_popular_vote_correction(available_frame)
    popular_existing = selection["Popular Vote"]
    popular_strategy_oof = {
        "HistoricalMean": popular_existing["mean_oof"][popular_vote_cols],
        "Ridge": popular_existing["selected_oof"]["Ridge"][popular_vote_cols],
        "TreeEnsemble": popular_existing["tree_oof"][popular_vote_cols],
        "ExpectedVoteAnchor": popular_choice["anchor_oof"],
        "ExpectedVoteErrorCorrected": popular_choice["selected_oof"],
    }
    popular_strategy_mae = {
        strategy: popular_vote_margin_mae(
            available_frame[popular_vote_cols],
            predictions,
        )
        for strategy, predictions in popular_strategy_oof.items()
    }
    popular_existing["strategy_mae"] = popular_strategy_mae
    popular_existing["selected_strategy"] = choose_strategy(
        popular_strategy_mae
    )
    popular_existing["popular_vote_correction_weight"] = (
        popular_choice["selected_weight"]
    )
    popular_existing["popular_vote_strategy_oof"] = popular_strategy_oof
    popular_existing["popular_vote_correction_table"] = (
        popular_choice["candidate_table"]
    )
    popular_existing["selection_metric"] = (
        "D-R popular vote margin MAE (percentage points)"
    )

    popular_tuning_rows = []
    for strategy, score in popular_strategy_mae.items():
        popular_tuning_rows.append({
            "Stage": stage_label,
            "Family": strategy,
            "Configuration": 0,
            "Parameters": (
                f"correction_weight="
                f"{popular_choice['selected_weight']:.2f}"
                if strategy == "ExpectedVoteErrorCorrected"
                else "popular-vote candidate"
            ),
            "Group": "Popular Vote",
            "Inner OOF MAE": float(score),
            "Selection Metric": (
                "D-R popular vote margin MAE (percentage points)"
            ),
        })
    tuning_table["Selection Metric"] = tuning_table.get(
        "Selection Metric",
        "Mean target MAE in native units",
    )
    tuning_table = pd.concat(
        [tuning_table, pd.DataFrame(popular_tuning_rows)],
        ignore_index=True,
    )

    return selection, tuning_table


def predictions_for_selected_configs(selection, fit_frame, predict_frame):
    """Ajusta cada configuración seleccionada una sola vez y reutiliza el resultado."""
    prediction_cache = {}
    fitted_cache = {}
    for group in modeling_groups:
        for family in ["ExtraTrees", "RandomForest", "Ridge"]:
            selected = selection[group]["selected_configs"][family]
            key = (
                family,
                selected["number"],
                parameter_label(selected["parameters"]),
            )
            if key not in prediction_cache:
                fitted, prediction = fit_predict_family(
                    family, selected["parameters"], fit_frame, predict_frame
                )
                fitted_cache[key] = fitted
                prediction_cache[key] = prediction
    return fitted_cache, prediction_cache


# ------------------------------------------------------------
# 3. VALIDACIÓN EXTERIOR COMPLETAMENTE ANIDADA
# ------------------------------------------------------------

loocv_records = []
national_nested_rows = []
national_nested_fold_rows = []
outer_tuning_tables = []
fold_2026_raw_records = []

historical_years = sorted(train_df["Midterm Year"].astype(int).unique())
if len(historical_years) < 5:
    raise ValueError(
        "La validación v12 requiere al menos cinco ciclos electorales históricos."
    )

for outer_year in historical_years:
    print(f"Nested LOEO — ciclo exterior dejado fuera: {outer_year}")
    outer_train = train_df[
        train_df["Midterm Year"].astype(int) != int(outer_year)
    ].copy()
    outer_test = train_df[
        train_df["Midterm Year"].astype(int) == int(outer_year)
    ].copy()

    outer_selection, outer_tuning = tune_on_available_cycles(
        outer_train, stage_label=f"Outer train excluding {outer_year}"
    )
    outer_tuning["Outer Test Year"] = int(outer_year)
    outer_tuning_tables.append(outer_tuning)

    # Predict the held-out historical cycle and 2026 from the same model fit.
    # Row 0 is the genuine outer test; row 1 is the 2026 jackknife diagnostic.
    outer_prediction_frame = pd.concat(
        [outer_test, pred_df], axis=0, ignore_index=True
    )
    _, outer_prediction_cache = predictions_for_selected_configs(
        outer_selection, outer_train, outer_prediction_frame
    )
    mean_prediction_matrix = historical_mean_predictions(
        outer_train, outer_prediction_frame
    )
    mean_prediction = mean_prediction_matrix[0]
    mean_prediction_2026 = mean_prediction_matrix[1]
    actual_vector = outer_test[ml_target_cols].iloc[0].to_numpy(float)
    training_years_label = ", ".join(
        map(str, sorted(outer_train["Midterm Year"].astype(int).unique()))
    )

    for group in modeling_groups:
        columns = [c for c in ml_target_cols if target_groups[c] == group]
        positions = [ml_target_cols.index(c) for c in columns]
        selection = outer_selection[group]
        family_predictions = {}
        popular_candidate_predictions = {}
        popular_candidate_predictions_2026 = {}
        if group == "Popular Vote":
            popular_anchor_matrix = predict_popular_vote_expectation(
                outer_train, outer_prediction_frame, correction_weight=0.0
            )
            popular_corrected_matrix = predict_popular_vote_expectation(
                outer_train,
                outer_prediction_frame,
                correction_weight=(
                    selection["popular_vote_correction_weight"]
                ),
            )
            for strategy_name, matrix in {
                "ExpectedVoteAnchor": popular_anchor_matrix,
                "ExpectedVoteErrorCorrected": popular_corrected_matrix,
            }.items():
                current_vector = np.full(len(ml_target_cols), np.nan)
                forecast_vector = np.full(len(ml_target_cols), np.nan)
                current_vector[positions] = (
                    matrix.iloc[0][columns].to_numpy(float)
                )
                forecast_vector[positions] = (
                    matrix.iloc[1][columns].to_numpy(float)
                )
                popular_candidate_predictions[strategy_name] = current_vector
                popular_candidate_predictions_2026[strategy_name] = forecast_vector
        family_predictions_2026 = {}

        for family in ["ExtraTrees", "RandomForest", "Ridge"]:
            selected = selection["selected_configs"][family]
            key = (
                family,
                selected["number"],
                parameter_label(selected["parameters"]),
            )
            family_predictions[family] = outer_prediction_cache[key][0]
            family_predictions_2026[family] = outer_prediction_cache[key][1]

        tree_prediction = (
            selection["tree_weights"]["ExtraTrees"]
            * family_predictions["ExtraTrees"]
            + selection["tree_weights"]["RandomForest"]
            * family_predictions["RandomForest"]
        )

        tree_prediction_2026 = (
            selection["tree_weights"]["ExtraTrees"]
            * family_predictions_2026["ExtraTrees"]
            + selection["tree_weights"]["RandomForest"]
            * family_predictions_2026["RandomForest"]
        )
        strategy_vectors = {
            "HistoricalMean": mean_prediction,
            "Ridge": family_predictions["Ridge"],
            "TreeEnsemble": tree_prediction,
            **popular_candidate_predictions,
        }
        strategy_vectors_2026 = {
            "HistoricalMean": mean_prediction_2026,
            "Ridge": family_predictions_2026["Ridge"],
            "TreeEnsemble": tree_prediction_2026,
            **popular_candidate_predictions_2026,
        }
        selected_strategy = selection["selected_strategy"]
        selected_prediction = strategy_vectors[selected_strategy]
        selected_prediction_2026 = strategy_vectors_2026[selected_strategy]

        for column, position in zip(columns, positions):
            fold_2026_raw_records.append({
                "Omitted Historical Cycle": int(outer_year),
                "Training Cycles": training_years_label,
                "Variable": column,
                "Group": group,
                "Selected Strategy": selected_strategy,
                "Raw Prediction": float(selected_prediction_2026[position]),
            })

        outer_absolute_errors = []
        for column, position in zip(columns, positions):
            actual = float(actual_vector[position])
            model_values = {
                "HistoricalMean": float(mean_prediction[position]),
                "Ridge": float(family_predictions["Ridge"][position]),
                "ExtraTrees": float(family_predictions["ExtraTrees"][position]),
                "RandomForest": float(family_predictions["RandomForest"][position]),
                "TreeEnsemble": float(tree_prediction[position]),
                "NestedSelected": float(selected_prediction[position]),
            }
            if group == "Popular Vote":
                model_values.update({
                    strategy_name: float(values[position])
                    for strategy_name, values
                    in popular_candidate_predictions.items()
                })
            for model_name, predicted in model_values.items():
                loocv_records.append({
                    "Year": int(outer_year),
                    "Model": model_name,
                    "Variable": column,
                    "Group": group,
                    "Actual": actual,
                    "Predicted": predicted,
                    "Absolute Error": abs(actual - predicted),
                })

            final_predicted = float(selected_prediction[position])
            outer_absolute_errors.append(abs(actual - final_predicted))
            national_nested_rows.append({
                "Year": int(outer_year),
                "Variable": column,
                "Group": group,
                "Actual": actual,
                "Predicted": final_predicted,
                "Signed Error": final_predicted - actual,
                "Absolute Error": abs(final_predicted - actual),
                "Selected Strategy": selected_strategy,
                "ExtraTrees Prediction": float(
                    family_predictions["ExtraTrees"][position]
                ),
                "RandomForest Prediction": float(
                    family_predictions["RandomForest"][position]
                ),
                "Ridge Prediction": float(family_predictions["Ridge"][position]),
                "Historical Mean Prediction": float(mean_prediction[position]),
            })

        configs = selection["selected_configs"]
        national_nested_fold_rows.append({
            "Outer Test Year": int(outer_year),
            "Group": group,
            "Selected Strategy": selected_strategy,
            "Outer MAE": float(np.mean(outer_absolute_errors)),
            "Inner HistoricalMean MAE": selection["strategy_mae"]["HistoricalMean"],
            "Inner Ridge MAE": selection["strategy_mae"]["Ridge"],
            "Inner TreeEnsemble MAE": selection["strategy_mae"]["TreeEnsemble"],
            "ExtraTrees Weight": selection["tree_weights"]["ExtraTrees"],
            "RandomForest Weight": selection["tree_weights"]["RandomForest"],
            "ExtraTrees Parameters": parameter_label(
                configs["ExtraTrees"]["parameters"]
            ),
            "RandomForest Parameters": parameter_label(
                configs["RandomForest"]["parameters"]
            ),
            "Ridge Parameters": parameter_label(configs["Ridge"]["parameters"]),
            "Selection Metric": selection.get(
                "selection_metric", "Mean target MAE in native units"
            ),
            "Inner ExpectedVoteAnchor MAE": selection["strategy_mae"].get(
                "ExpectedVoteAnchor", np.nan
            ),
            "Inner ExpectedVoteErrorCorrected MAE": selection[
                "strategy_mae"
            ].get("ExpectedVoteErrorCorrected", np.nan),
            "Popular Vote Correction Weight": selection.get(
                "popular_vote_correction_weight", np.nan
            ),
        })

loocv_df = pd.DataFrame(loocv_records)
national_nested_oof = pd.DataFrame(national_nested_rows).sort_values(
    ["Year", "Variable"]
).reset_index(drop=True)
national_nested_fold_diagnostics = pd.DataFrame(national_nested_fold_rows)
national_outer_tuning = pd.concat(outer_tuning_tables, ignore_index=True)

loocv_summary = (
    loocv_df.groupby(["Model", "Group"])["Absolute Error"]
    .mean()
    .reset_index()
    .rename(columns={"Absolute Error": "MAE"})
)

# Reconstruct the popular-vote candidates at election grain. These are genuine
# outer predictions; no row uses its own result during fitting or selection.
popular_vote_validation_rows = []
popular_oof = loocv_df.loc[
    loocv_df["Group"].eq("Popular Vote")
]
for (year, model_name), fold in popular_oof.groupby(["Year", "Model"]):
    values = fold.set_index("Variable")
    if not {"DPP", "RPP"}.issubset(values.index):
        continue
    actual_d = float(values.at["DPP", "Actual"])
    actual_r = float(values.at["RPP", "Actual"])
    predicted_d = float(values.at["DPP", "Predicted"])
    predicted_r = float(values.at["RPP", "Predicted"])
    actual_two_party = 100.0 * (actual_d - actual_r) / (actual_d + actual_r)
    predicted_two_party = (
        100.0 * (predicted_d - predicted_r) / (predicted_d + predicted_r)
    )
    popular_vote_validation_rows.append({
        "Year": int(year),
        "Model": model_name,
        "Actual D (%)": actual_d * 100.0,
        "Predicted D (%)": predicted_d * 100.0,
        "Actual R (%)": actual_r * 100.0,
        "Predicted R (%)": predicted_r * 100.0,
        "Actual Other (%)": (1.0 - actual_d - actual_r) * 100.0,
        "Predicted Other (%)": (
            1.0 - predicted_d - predicted_r
        ) * 100.0,
        "Actual Raw D-R Margin PP": (actual_d - actual_r) * 100.0,
        "Predicted Raw D-R Margin PP": (
            predicted_d - predicted_r
        ) * 100.0,
        "Raw Margin Absolute Error PP": abs(
            (predicted_d - predicted_r - actual_d + actual_r) * 100.0
        ),
        "Actual Two-Party Margin PP": actual_two_party,
        "Predicted Two-Party Margin PP": predicted_two_party,
        "Two-Party Margin Absolute Error PP": abs(
            predicted_two_party - actual_two_party
        ),
    })

national_popular_vote_validation = pd.DataFrame(
    popular_vote_validation_rows
).sort_values(["Model", "Year"]).reset_index(drop=True)
national_popular_vote_method_summary = (
    national_popular_vote_validation.groupby("Model")
    .agg(
        Raw_Margin_MAE_PP=("Raw Margin Absolute Error PP", "mean"),
        Two_Party_Margin_MAE_PP=(
            "Two-Party Margin Absolute Error PP", "mean"
        ),
        Mean_Actual_Other_PP=("Actual Other (%)", "mean"),
        Mean_Predicted_Other_PP=("Predicted Other (%)", "mean"),
        Outer_Tests=("Year", "nunique"),
    )
    .reset_index()
    .sort_values(
        ["Raw_Margin_MAE_PP", "Two_Party_Margin_MAE_PP", "Model"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3B. TIME-MACHINE TARGET CONTRACT: 34 LEARNED + 8 DERIVED
# ------------------------------------------------------------
#
# The historical fold is judged only on a cycle that the fitted model never saw.
# Raw bucket predictions pass through the same non-negativity, normalization and
# Hamilton constraints used by the 2026 production path. This produces all 42
# expected-versus-actual target comparisons without pretending that derived
# identities are separately learned observations.

def _loss_target_columns(columns, party):
    marker = "ResLR" if party == "D" else "ResLD"
    return [column for column in columns if marker in column]


def _hamilton_from_raw(raw_predictions, columns, required_total):
    values = np.array(
        [max(float(raw_predictions.get(column, 0.0)), 0.0) for column in columns],
        dtype=float,
    )
    required_total = int(round(float(required_total)))
    raw_total = float(values.sum())
    if not np.isfinite(raw_total) or raw_total <= 0:
        raise ValueError(f"Time Machine recibió un total no positivo: {columns}")

    quotas = values / raw_total * required_total
    allocation = np.floor(quotas).astype(int)
    seats_left = required_total - int(allocation.sum())
    if seats_left < 0 or seats_left > len(columns):
        raise RuntimeError("Time Machine recibió cuotas Hamilton incoherentes.")
    if seats_left:
        remainders = quotas - allocation
        order = np.argsort(-remainders, kind="stable")
        allocation[order[:seats_left]] += 1
    return dict(zip(columns, allocation))


def project_all_42_targets(raw_predictions, election_row):
    projected = {}
    for column in popular_vote_cols:
        projected[column] = float(np.clip(raw_predictions[column], 0.0, 1.0))

    group_specs = [
        (house_bucket_cols, "DHou", election_row["DH Before"]),
        (house_bucket_cols, "RHou", election_row["RH Before"]),
        (senate_bucket_cols, "DSen", election_row["DSS UP"]),
        (senate_bucket_cols, "RSen", election_row["RSS UP"]),
    ]
    for all_columns, prefix, required_total in group_specs:
        columns = [column for column in all_columns if column.startswith(prefix)]
        projected.update(_hamilton_from_raw(raw_predictions, columns, required_total))

    d_house_loss = sum(
        projected[column]
        for column in _loss_target_columns(
            [c for c in house_bucket_cols if c.startswith("DHou")], "D"
        )
    )
    r_house_loss = sum(
        projected[column]
        for column in _loss_target_columns(
            [c for c in house_bucket_cols if c.startswith("RHou")], "R"
        )
    )
    d_senate_loss = sum(
        projected[column]
        for column in _loss_target_columns(
            [c for c in senate_bucket_cols if c.startswith("DSen")], "D"
        )
    )
    r_senate_loss = sum(
        projected[column]
        for column in _loss_target_columns(
            [c for c in senate_bucket_cols if c.startswith("RSen")], "R"
        )
    )

    projected.update({
        "D House Seats LOST": int(d_house_loss),
        "R House Seats Lost": int(r_house_loss),
        "D Senate Seats LOST": int(d_senate_loss),
        "R Senate Seats LOST": int(r_senate_loss),
        "D House After": int(election_row["DH Before"] - d_house_loss + r_house_loss),
        "R House After": int(election_row["RH Before"] + d_house_loss - r_house_loss),
        "D Senate After": int(election_row["DS before"] - d_senate_loss + r_senate_loss),
        "R Senate After": int(election_row["RS Before"] + d_senate_loss - r_senate_loss),
    })

    missing = [column for column in p_cols if column not in projected]
    if missing:
        raise AssertionError(f"Time Machine no pudo proyectar estos targets: {missing}")
    return projected



def _sum_historical_losses(row, columns, party):
    return int(sum(
        float(row[column])
        for column in _loss_target_columns(columns, party)
    ))


# Reconcile historical derived loss totals against their detailed buckets and
# the authoritative post-election chamber totals. This does not rewrite the
# source workbook; it records the exception and chooses the representation
# that reconstructs both final party totals.
historical_identity_rows = []
canonical_actual_overrides = {}
identity_specs = [
    {
        "chamber": "House",
        "d_bucket_cols": [
            c for c in house_bucket_cols if c.startswith("DHou")
        ],
        "r_bucket_cols": [
            c for c in house_bucket_cols if c.startswith("RHou")
        ],
        "d_loss_col": "D House Seats LOST",
        "r_loss_col": "R House Seats Lost",
        "d_before_col": "DH Before",
        "r_before_col": "RH Before",
        "d_after_col": "D House After",
        "r_after_col": "R House After",
    },
    {
        "chamber": "Senate",
        "d_bucket_cols": [
            c for c in senate_bucket_cols if c.startswith("DSen")
        ],
        "r_bucket_cols": [
            c for c in senate_bucket_cols if c.startswith("RSen")
        ],
        "d_loss_col": "D Senate Seats LOST",
        "r_loss_col": "R Senate Seats LOST",
        "d_before_col": "DS before",
        "r_before_col": "RS Before",
        "d_after_col": "D Senate After",
        "r_after_col": "R Senate After",
    },
]

for _, election_row in train_df.iterrows():
    year = int(election_row["Midterm Year"])
    for spec in identity_specs:
        aggregate_d_loss = int(float(election_row[spec["d_loss_col"]]))
        aggregate_r_loss = int(float(election_row[spec["r_loss_col"]]))
        bucket_d_loss = _sum_historical_losses(
            election_row, spec["d_bucket_cols"], "D"
        )
        bucket_r_loss = _sum_historical_losses(
            election_row, spec["r_bucket_cols"], "R"
        )

        d_before = int(float(election_row[spec["d_before_col"]]))
        r_before = int(float(election_row[spec["r_before_col"]]))
        actual_d_after = int(float(election_row[spec["d_after_col"]]))
        actual_r_after = int(float(election_row[spec["r_after_col"]]))

        aggregate_d_after = d_before - aggregate_d_loss + aggregate_r_loss
        aggregate_r_after = r_before + aggregate_d_loss - aggregate_r_loss
        bucket_d_after = d_before - bucket_d_loss + bucket_r_loss
        bucket_r_after = r_before + bucket_d_loss - bucket_r_loss

        aggregate_matches = (
            aggregate_d_after == actual_d_after
            and aggregate_r_after == actual_r_after
        )
        bucket_matches = (
            bucket_d_after == actual_d_after
            and bucket_r_after == actual_r_after
        )

        if bucket_matches and not aggregate_matches:
            canonical_source = "Detailed result buckets"
            canonical_d_loss, canonical_r_loss = (
                bucket_d_loss, bucket_r_loss
            )
            resolution_status = "Resolved"
        elif aggregate_matches:
            canonical_source = "Aggregate loss fields"
            canonical_d_loss, canonical_r_loss = (
                aggregate_d_loss, aggregate_r_loss
            )
            resolution_status = "Resolved"
        elif (d_before + r_before) != (actual_d_after + actual_r_after):
            # Some historical chambers include independents/caucus changes that
            # make a two-party loss identity non-comparable. Preserve the source
            # loss fields and document the structural composition exception.
            canonical_source = (
                "Aggregate loss fields — non-two-party composition exception"
            )
            canonical_d_loss, canonical_r_loss = (
                aggregate_d_loss, aggregate_r_loss
            )
            resolution_status = "Resolved"
        else:
            canonical_source = "Unresolved"
            canonical_d_loss, canonical_r_loss = (
                aggregate_d_loss, aggregate_r_loss
            )
            resolution_status = "Unresolved"

        canonical_actual_overrides[(year, spec["d_loss_col"])] = float(
            canonical_d_loss
        )
        canonical_actual_overrides[(year, spec["r_loss_col"])] = float(
            canonical_r_loss
        )
        historical_identity_rows.append({
            "Year": year,
            "Chamber": spec["chamber"],
            "Aggregate D Loss": aggregate_d_loss,
            "Aggregate R Loss": aggregate_r_loss,
            "Bucket D Loss": bucket_d_loss,
            "Bucket R Loss": bucket_r_loss,
            "Observed D After": actual_d_after,
            "Observed R After": actual_r_after,
            "Aggregate Reconstructs Final": aggregate_matches,
            "Buckets Reconstruct Final": bucket_matches,
            "Canonical Source": canonical_source,
            "Canonical D Loss": canonical_d_loss,
            "Canonical R Loss": canonical_r_loss,
            "Resolution Status": resolution_status,
            "Source Exception": bool(
                aggregate_d_loss != bucket_d_loss
                or aggregate_r_loss != bucket_r_loss
                or not aggregate_matches
                or not bucket_matches
            ),
        })

historical_input_reconciliation = pd.DataFrame(historical_identity_rows)

time_machine_rows = []
time_machine_model_names = [
    "ExtraTrees", "RandomForest", "Ridge", "HistoricalMean",
    "TreeEnsemble", "NestedSelected",
]
time_machine_source = loocv_df.loc[
    loocv_df["Model"].isin(time_machine_model_names)
]
for (year, model_name), fold in time_machine_source.groupby(["Year", "Model"]):
    raw_predictions = fold.set_index("Variable")["Predicted"].to_dict()
    election_row = train_df.loc[
        train_df["Midterm Year"].astype(int) == int(year)
    ].iloc[0]
    projected = project_all_42_targets(raw_predictions, election_row)

    for column in p_cols:
        actual = canonical_actual_overrides.get(
            (int(year), column), float(election_row[column])
        )
        predicted = float(projected[column])
        time_machine_rows.append({
            "Year": int(year),
            "Training Cycles": ", ".join(
                map(str, [y for y in historical_years if int(y) != int(year)])
            ),
            "Model": model_name,
            "Variable": column,
            "Group": target_groups[column],
            "Target Type": "Learned" if column in ml_target_cols else "Derived identity",
            "Actual": actual,
            "Predicted": predicted,
            "Signed Error": predicted - actual,
            "Absolute Error": abs(predicted - actual),
        })

national_time_machine_targets = pd.DataFrame(time_machine_rows)
national_time_machine_selected = (
    national_time_machine_targets.loc[
        national_time_machine_targets["Model"].eq("NestedSelected")
    ]
    .sort_values(["Year", "Variable"])
    .reset_index(drop=True)
)
national_time_machine_target_summary = (
    national_time_machine_targets.groupby(["Model", "Group", "Target Type"])
    .agg(
        MAE=("Absolute Error", "mean"),
        Bias=("Signed Error", "mean"),
        Comparisons=("Variable", "size"),
        Cycles=("Year", "nunique"),
    )
    .reset_index()
)

headline_specs = [
    ("Popular Vote Margin (pp)", "DPP", "RPP", 100.0),
    ("Democratic House Seats", "D House After", None, 1.0),
    ("Republican House Seats", "R House After", None, 1.0),
    ("House Seat Margin (D-R)", "D House After", "R House After", 1.0),
    ("Democratic Senate Seats (national prior)", "D Senate After", None, 1.0),
    ("Republican Senate Seats (national prior)", "R Senate After", None, 1.0),
    ("Senate Seat Margin (D-R)", "D Senate After", "R Senate After", 1.0),
]
national_outcome_rows = []
for (year, model_name), fold in national_time_machine_targets.groupby(
    ["Year", "Model"]
):
    predicted = fold.set_index("Variable")["Predicted"].to_dict()
    actual = fold.set_index("Variable")["Actual"].to_dict()
    for outcome, primary, secondary, scale in headline_specs:
        if secondary is None:
            predicted_value = predicted[primary] * scale
            actual_value = actual[primary] * scale
        else:
            predicted_value = (predicted[primary] - predicted[secondary]) * scale
            actual_value = (actual[primary] - actual[secondary]) * scale
        national_outcome_rows.append({
            "Year": int(year),
            "Model": model_name,
            "Outcome": outcome,
            "Actual": float(actual_value),
            "Predicted": float(predicted_value),
            "Signed Error": float(predicted_value - actual_value),
            "Absolute Error": float(abs(predicted_value - actual_value)),
        })

national_outcome_oof = pd.DataFrame(national_outcome_rows)
national_outcome_summary = (
    national_outcome_oof.groupby(["Model", "Outcome"])
    .agg(
        MAE=("Absolute Error", "mean"),
        Bias=("Signed Error", "mean"),
        Cycles=("Year", "nunique"),
    )
    .reset_index()
)

selected_outcomes = national_outcome_oof.loc[
    national_outcome_oof["Model"].eq("NestedSelected")
]
scorecard_rows = []
for year in historical_years:
    row = {
        "Test Election": int(year),
        "Training Elections": ", ".join(
            map(str, [y for y in historical_years if int(y) != int(year)])
        ),
    }
    for outcome, short in [
        ("Popular Vote Margin (pp)", "Popular Margin"),
        ("Democratic House Seats", "D House Seats"),
        ("Democratic Senate Seats (national prior)", "D Senate Prior"),
    ]:
        value = selected_outcomes.loc[
            (selected_outcomes["Year"].eq(int(year)))
            & (selected_outcomes["Outcome"].eq(outcome))
        ].iloc[0]
        row[f"{short} Predicted"] = float(value["Predicted"])
        row[f"{short} Actual"] = float(value["Actual"])
        row[f"{short} Abs Error"] = float(value["Absolute Error"])
    for group, short in [
        ("Popular Vote", "Vote Strategy"),
        ("House Buckets", "House Strategy"),
        ("Senate Buckets", "Senate Strategy"),
    ]:
        match = national_nested_fold_diagnostics.loc[
            (national_nested_fold_diagnostics["Outer Test Year"].eq(int(year)))
            & (national_nested_fold_diagnostics["Group"].eq(group)),
            "Selected Strategy",
        ]
        row[short] = match.iloc[0]
    scorecard_rows.append(row)

national_time_machine_scorecard = pd.DataFrame(scorecard_rows)

# The first exam happens inside the four-cycle training sample. The second exam
# is the untouched outer election. This table makes the two levels explicit and
# compares the refined winner with fixed baselines on the same outer exam.
outer_group_mae = (
    loocv_df.groupby(["Year", "Group", "Model"])["Absolute Error"]
    .mean()
    .unstack("Model")
    .reset_index()
)
national_exam_progression = national_nested_fold_diagnostics.merge(
    outer_group_mae,
    left_on=["Outer Test Year", "Group"],
    right_on=["Year", "Group"],
    how="left",
    validate="one_to_one",
)
national_exam_progression["First Exam Winner MAE"] = national_exam_progression.apply(
    lambda row: row[
        {
            "HistoricalMean": "Inner HistoricalMean MAE",
            "Ridge": "Inner Ridge MAE",
            "TreeEnsemble": "Inner TreeEnsemble MAE",
            "ExpectedVoteAnchor": "Inner ExpectedVoteAnchor MAE",
            "ExpectedVoteErrorCorrected": (
                "Inner ExpectedVoteErrorCorrected MAE"
            ),
        }[row["Selected Strategy"]]
    ],
    axis=1,
)
national_exam_progression["Second Exam Selected MAE"] = (
    national_exam_progression["Outer MAE"]
)
national_exam_progression["Generalization Gap"] = (
    national_exam_progression["Second Exam Selected MAE"]
    - national_exam_progression["First Exam Winner MAE"]
)
national_exam_progression["Improvement vs HistoricalMean"] = (
    national_exam_progression["HistoricalMean"]
    - national_exam_progression["Second Exam Selected MAE"]
)
national_exam_progression["Improvement vs TreeEnsemble"] = (
    national_exam_progression["TreeEnsemble"]
    - national_exam_progression["Second Exam Selected MAE"]
)
national_exam_progression["Improved vs HistoricalMean"] = (
    national_exam_progression["Improvement vs HistoricalMean"] > 0
)
national_exam_progression["Improved vs TreeEnsemble"] = (
    national_exam_progression["Improvement vs TreeEnsemble"] > 0
)
national_exam_progression = national_exam_progression[[
    "Outer Test Year", "Group", "Selected Strategy",
    "First Exam Winner MAE", "Second Exam Selected MAE",
    "Generalization Gap", "HistoricalMean", "TreeEnsemble", "Ridge",
    "ExpectedVoteAnchor", "ExpectedVoteErrorCorrected",
    "Selection Metric", "Popular Vote Correction Weight",
    "Improvement vs HistoricalMean", "Improvement vs TreeEnsemble",
    "Improved vs HistoricalMean", "Improved vs TreeEnsemble",
]]

outcome_mae_wide = (
    national_outcome_summary.pivot(
        index="Outcome", columns="Model", values="MAE"
    )
    .reset_index()
)
national_exam_outcome_comparison = outcome_mae_wide[[
    "Outcome", "HistoricalMean", "TreeEnsemble", "Ridge", "NestedSelected"
]].copy()
national_exam_outcome_comparison["Improvement vs HistoricalMean"] = (
    national_exam_outcome_comparison["HistoricalMean"]
    - national_exam_outcome_comparison["NestedSelected"]
)
national_exam_outcome_comparison["Improvement vs TreeEnsemble"] = (
    national_exam_outcome_comparison["TreeEnsemble"]
    - national_exam_outcome_comparison["NestedSelected"]
)
national_exam_outcome_comparison["Improvement vs HistoricalMean (%)"] = (
    national_exam_outcome_comparison["Improvement vs HistoricalMean"]
    / national_exam_outcome_comparison["HistoricalMean"]
    * 100.0
)
national_exam_outcome_comparison["Improvement vs TreeEnsemble (%)"] = (
    national_exam_outcome_comparison["Improvement vs TreeEnsemble"]
    / national_exam_outcome_comparison["TreeEnsemble"]
    * 100.0
)
national_exam_outcome_comparison["Second Exam Verdict"] = np.where(
    national_exam_outcome_comparison["Improvement vs TreeEnsemble"] > 0,
    "Refinement improved outer-test MAE",
    "Refinement did not improve outer-test MAE",
)


# ------------------------------------------------------------
# 3C. SEGUNDO EXAMEN: PIPELINE COMPLETO SIN REENTRENAMIENTO
# ------------------------------------------------------------
#
# The first sealed result is the raw selected prediction for each learned
# target. The second result sends exactly that prediction through the same
# non-negativity, Hamilton allocation and deterministic identities used in
# production. It is an end-to-end audit, not a second fit and not feedback.

raw_pipeline_exam = national_nested_oof[[
    "Year", "Variable", "Group", "Actual", "Predicted"
]].rename(columns={
    "Actual": "Raw Actual",
    "Predicted": "Raw Selected Prediction",
})
constrained_pipeline_exam = national_time_machine_selected.loc[
    national_time_machine_selected["Variable"].isin(ml_target_cols),
    ["Year", "Variable", "Actual", "Predicted"],
].rename(columns={
    "Actual": "Constrained Actual",
    "Predicted": "Constrained Pipeline Prediction",
})
national_pipeline_target_exam = raw_pipeline_exam.merge(
    constrained_pipeline_exam,
    on=["Year", "Variable"],
    how="inner",
    validate="one_to_one",
)
national_pipeline_target_exam["Raw Absolute Error"] = np.abs(
    national_pipeline_target_exam["Raw Selected Prediction"]
    - national_pipeline_target_exam["Raw Actual"]
)
national_pipeline_target_exam["Pipeline Absolute Error"] = np.abs(
    national_pipeline_target_exam["Constrained Pipeline Prediction"]
    - national_pipeline_target_exam["Constrained Actual"]
)
national_pipeline_target_exam["Pipeline Error Change"] = (
    national_pipeline_target_exam["Pipeline Absolute Error"]
    - national_pipeline_target_exam["Raw Absolute Error"]
)

national_pipeline_stage_summary = (
    national_pipeline_target_exam.groupby("Group")
    .agg(
        Raw_Selected_MAE=("Raw Absolute Error", "mean"),
        Constrained_Pipeline_MAE=("Pipeline Absolute Error", "mean"),
        Mean_Error_Change=("Pipeline Error Change", "mean"),
        Learned_Target_Comparisons=("Variable", "size"),
        Election_Exams=("Year", "nunique"),
    )
    .reset_index()
)

derived_pipeline_exam = national_time_machine_selected.loc[
    national_time_machine_selected["Variable"].isin(derived_cols)
].copy()
national_pipeline_derived_summary = (
    derived_pipeline_exam.groupby("Group")
    .agg(
        Derived_Identity_MAE=("Absolute Error", "mean"),
        Derived_Comparisons=("Variable", "size"),
        Election_Exams=("Year", "nunique"),
    )
    .reset_index()
)

national_validation_stage_contract = pd.DataFrame([
    {
        "Stage": "Selection exam",
        "Input": "Four historical cycles inside each outer fold",
        "Output": "Chosen family, hyperparameters and raw target predictions",
        "Information Allowed": "Training cycles only",
        "Information Prohibited": "Held-out result and 2026 result",
        "Can Retrain Upstream": False,
    },
    {
        "Stage": "Pipeline exam",
        "Input": "Sealed outer raw predictions",
        "Output": "Constrained 42 targets, derived identities and headline outcomes",
        "Information Allowed": "Pre-election institutional totals for that cycle",
        "Information Prohibited": "Observed election result as a correction",
        "Can Retrain Upstream": False,
    },
    {
        "Stage": "Monte Carlo",
        "Input": "Frozen point forecasts and validated uncertainty",
        "Output": "Control probabilities and forecast distributions",
        "Information Allowed": "Upstream point estimates and OOF residuals",
        "Information Prohibited": "Writing simulated values back into the 42 targets",
        "Can Retrain Upstream": False,
    },
])


# Five 2026 predictions, each omitting a different historical election.
fold_2026_raw = pd.DataFrame(fold_2026_raw_records)
fold_2026_rows = []
fold_2026_headline_rows = []
for omitted_year, fold in fold_2026_raw.groupby("Omitted Historical Cycle"):
    raw_predictions = fold.set_index("Variable")["Raw Prediction"].to_dict()
    projected = project_all_42_targets(raw_predictions, pred_df.iloc[0])
    training_cycles = fold["Training Cycles"].iloc[0]

    for column in p_cols:
        fold_2026_rows.append({
            "Omitted Historical Cycle": int(omitted_year),
            "Training Cycles": training_cycles,
            "Variable": column,
            "Group": target_groups[column],
            "Target Type": "Learned" if column in ml_target_cols else "Derived identity",
            "Prediction": float(projected[column]),
        })

    strategy_by_group = (
        fold.drop_duplicates("Group").set_index("Group")["Selected Strategy"].to_dict()
    )
    fold_2026_headline_rows.append({
        "Omitted Historical Cycle": int(omitted_year),
        "Training Cycles": training_cycles,
        "Popular Vote Strategy": strategy_by_group["Popular Vote"],
        "House Strategy": strategy_by_group["House Buckets"],
        "Senate Strategy": strategy_by_group["Senate Buckets"],
        "D Popular Vote (%)": projected["DPP"] * 100.0,
        "R Popular Vote (%)": projected["RPP"] * 100.0,
        "D-R Popular Margin (pp)": (projected["DPP"] - projected["RPP"]) * 100.0,
        "D House Seats": projected["D House After"],
        "R House Seats": projected["R House After"],
        "D Senate Seats (national prior)": projected["D Senate After"],
        "R Senate Seats (national prior)": projected["R Senate After"],
    })

national_2026_fold_targets = pd.DataFrame(fold_2026_rows)
national_2026_fold_forecasts = (
    pd.DataFrame(fold_2026_headline_rows)
    .sort_values("Omitted Historical Cycle")
    .reset_index(drop=True)
)

if len(national_time_machine_selected) != len(historical_years) * len(p_cols):
    raise AssertionError("Time Machine no produjo 42 comparaciones por elección.")
if national_time_machine_selected.duplicated(["Year", "Variable"]).any():
    raise AssertionError("Time Machine contiene targets históricos duplicados.")
if len(national_2026_fold_targets) != len(historical_years) * len(p_cols):
    raise AssertionError("El jackknife 2026 no produjo 42 targets por modelo parcial.")

# ------------------------------------------------------------
# 4. SELECCIÓN FINAL USANDO TODO EL HISTÓRICO DISPONIBLE
# ------------------------------------------------------------

final_selection, national_tuning_summary = tune_on_available_cycles(
    train_df, stage_label="Final training selection"
)
final_fitted_cache, final_prediction_cache = predictions_for_selected_configs(
    final_selection, train_df, pred_df
)
final_mean_prediction = historical_mean_predictions(train_df, pred_df)[0]

ensemble_weights = {}
national_selected_strategy = {}
trained_models = {"ExtraTrees": {}, "RandomForest": {}, "Ridge": {}}
final_model_predictions = {
    "ExtraTrees": np.full(len(ml_target_cols), np.nan),
    "RandomForest": np.full(len(ml_target_cols), np.nan),
    "Ridge": np.full(len(ml_target_cols), np.nan),
    "HistoricalMean": final_mean_prediction.copy(),
    "TreeEnsemble": np.full(len(ml_target_cols), np.nan),
    "ExpectedVoteAnchor": np.full(len(ml_target_cols), np.nan),
    "ExpectedVoteErrorCorrected": np.full(len(ml_target_cols), np.nan),
}
final_selected_prediction = np.full(len(ml_target_cols), np.nan)
final_selection_rows = []

for group in modeling_groups:
    columns = [c for c in ml_target_cols if target_groups[c] == group]
    positions = [ml_target_cols.index(c) for c in columns]
    selection = final_selection[group]
    if group == "Popular Vote":
        final_popular_anchor = predict_popular_vote_expectation(
            train_df, pred_df, correction_weight=0.0
        )
        final_popular_corrected = predict_popular_vote_expectation(
            train_df,
            pred_df,
            correction_weight=selection["popular_vote_correction_weight"],
        )
        for strategy_name, matrix in {
            "ExpectedVoteAnchor": final_popular_anchor,
            "ExpectedVoteErrorCorrected": final_popular_corrected,
        }.items():
            final_model_predictions[strategy_name][positions] = (
                matrix.iloc[0][columns].to_numpy(float)
            )
    ensemble_weights[group] = selection["tree_weights"]
    national_selected_strategy[group] = selection["selected_strategy"]

    group_family_predictions = {}
    for family in ["ExtraTrees", "RandomForest", "Ridge"]:
        selected = selection["selected_configs"][family]
        key = (
            family,
            selected["number"],
            parameter_label(selected["parameters"]),
        )
        group_family_predictions[family] = final_prediction_cache[key][0]
        trained_models[family][group] = final_fitted_cache[key]
        final_model_predictions[family][positions] = (
            group_family_predictions[family][positions]
        )

    tree_vector = (
        selection["tree_weights"]["ExtraTrees"]
        * group_family_predictions["ExtraTrees"]
        + selection["tree_weights"]["RandomForest"]
        * group_family_predictions["RandomForest"]
    )
    final_model_predictions["TreeEnsemble"][positions] = tree_vector[positions]
    strategy_vectors = {
        "HistoricalMean": final_mean_prediction,
        "Ridge": group_family_predictions["Ridge"],
        "TreeEnsemble": tree_vector,
    }
    if group == "Popular Vote":
        strategy_vectors.update({
            "ExpectedVoteAnchor": final_model_predictions[
                "ExpectedVoteAnchor"
            ],
            "ExpectedVoteErrorCorrected": final_model_predictions[
                "ExpectedVoteErrorCorrected"
            ],
        })
    selected_strategy = selection["selected_strategy"]
    final_selected_prediction[positions] = strategy_vectors[selected_strategy][positions]

    configs = selection["selected_configs"]
    final_selection_rows.append({
        "Group": group,
        "Selected Strategy": selected_strategy,
        "HistoricalMean Inner OOF MAE": selection["strategy_mae"]["HistoricalMean"],
        "Ridge Inner OOF MAE": selection["strategy_mae"]["Ridge"],
        "TreeEnsemble Inner OOF MAE": selection["strategy_mae"]["TreeEnsemble"],
        "ExtraTrees Weight": selection["tree_weights"]["ExtraTrees"],
        "RandomForest Weight": selection["tree_weights"]["RandomForest"],
        "ExtraTrees Parameters": parameter_label(
            configs["ExtraTrees"]["parameters"]
        ),
        "RandomForest Parameters": parameter_label(
            configs["RandomForest"]["parameters"]
        ),
        "Ridge Parameters": parameter_label(configs["Ridge"]["parameters"]),
        "Complexity Tolerance": NATIONAL_COMPLEXITY_TOLERANCE,
        "Selection Metric": selection.get(
            "selection_metric", "Mean target MAE in native units"
        ),
        "ExpectedVoteAnchor Inner OOF MAE": selection["strategy_mae"].get(
            "ExpectedVoteAnchor", np.nan
        ),
        "ExpectedVoteErrorCorrected Inner OOF MAE": selection[
            "strategy_mae"
        ].get("ExpectedVoteErrorCorrected", np.nan),
        "Popular Vote Correction Weight": selection.get(
            "popular_vote_correction_weight", np.nan
        ),
    })

national_final_selection = pd.DataFrame(final_selection_rows)

final_2026_raw_predictions = {
    column: float(final_selected_prediction[position])
    for position, column in enumerate(ml_target_cols)
}
final_2026_full_projection_42 = project_all_42_targets(
    final_2026_raw_predictions, pred_df.iloc[0]
)


# Production bridge and explicit dependency tests.
popular_selection_final = final_selection["Popular Vote"]
popular_anchor_2026 = predict_popular_vote_expectation(
    train_df, pred_df, correction_weight=0.0
)
popular_corrected_2026 = predict_popular_vote_expectation(
    train_df,
    pred_df,
    correction_weight=popular_selection_final[
        "popular_vote_correction_weight"
    ],
)
popular_actual_other_history = (
    1.0 - train_df["DPP"].to_numpy(float) - train_df["RPP"].to_numpy(float)
)
popular_vote_production_bridge = pd.DataFrame([{
    "Selected Strategy": popular_selection_final["selected_strategy"],
    "EDPP Input (%)": float(pred_df.iloc[0]["EDPP"]) * 100.0,
    "ERPP Input (%)": float(pred_df.iloc[0]["ERPP"]) * 100.0,
    "Expected Two-Party Margin PP": float(
        popular_vote_two_party_margin_pp(pred_df, "EDPP", "ERPP")[0]
    ),
    "Historical Mean Other (%)": float(
        np.mean(popular_actual_other_history) * 100.0
    ),
    "Selected Historical Error-Correction Weight": float(
        popular_selection_final["popular_vote_correction_weight"]
    ),
    "Anchor D (%)": float(popular_anchor_2026.iloc[0]["DPP"]) * 100.0,
    "Anchor R (%)": float(popular_anchor_2026.iloc[0]["RPP"]) * 100.0,
    "Error-Corrected Candidate D (%)": float(
        popular_corrected_2026.iloc[0]["DPP"]
    ) * 100.0,
    "Error-Corrected Candidate R (%)": float(
        popular_corrected_2026.iloc[0]["RPP"]
    ) * 100.0,
    "Production D (%)": float(final_2026_full_projection_42["DPP"]) * 100.0,
    "Production R (%)": float(final_2026_full_projection_42["RPP"]) * 100.0,
    "Production Other (%)": (
        1.0
        - float(final_2026_full_projection_42["DPP"])
        - float(final_2026_full_projection_42["RPP"])
    ) * 100.0,
    "Production Raw D-R Margin PP": (
        float(final_2026_full_projection_42["DPP"])
        - float(final_2026_full_projection_42["RPP"])
    ) * 100.0,
    "Production Two-Party Margin PP": (
        100.0
        * (
            float(final_2026_full_projection_42["DPP"])
            - float(final_2026_full_projection_42["RPP"])
        )
        / (
            float(final_2026_full_projection_42["DPP"])
            + float(final_2026_full_projection_42["RPP"])
        )
    ),
}])

senate_like_columns = [
    column
    for column in feature_cols
    if any(
        token in str(column).lower()
        for token in ["senate", "dsen", "rsen", "ds before", "rs before"]
    )
]
counterfactual_2026 = pred_df.copy()
for column in senate_like_columns:
    if column in numeric_features:
        counterfactual_2026[column] = (
            counterfactual_2026[column].astype(float) + 10_000.0
        )
    else:
        counterfactual_2026[column] = "COUNTERFACTUAL"

baseline_popular_dependency_test = predict_popular_vote_expectation(
    train_df,
    pred_df,
    correction_weight=popular_selection_final[
        "popular_vote_correction_weight"
    ],
)
counterfactual_popular_dependency_test = predict_popular_vote_expectation(
    train_df,
    counterfactual_2026,
    correction_weight=popular_selection_final[
        "popular_vote_correction_weight"
    ],
)
forbidden_popular_dependencies = sorted(
    set(POPULAR_VOTE_INPUT_COLUMNS).intersection(senate_like_columns)
)
popular_vote_dependency_audit = pd.DataFrame([
    {
        "Check": "Popular-vote production inputs exclude Senate variables",
        "Passed": len(forbidden_popular_dependencies) == 0,
        "Detail": (
            "Inputs: EDPP, ERPP; historical DPP/RPP only define targets and "
            "Other/error estimates"
        ),
    },
    {
        "Check": "Extreme Senate-input counterfactual leaves popular vote unchanged",
        "Passed": bool(np.allclose(
            baseline_popular_dependency_test[popular_vote_cols],
            counterfactual_popular_dependency_test[popular_vote_cols],
            atol=0.0,
            rtol=0.0,
        )),
        "Detail": f"{len(senate_like_columns)} Senate-related national columns perturbed",
    },
    {
        "Check": "P targets are absent from the national feature matrix",
        "Passed": set(p_cols).isdisjoint(feature_cols),
        "Detail": f"{len(p_cols)} targets; {len(feature_cols)} predictors",
    },
])
if not popular_vote_dependency_audit["Passed"].all():
    raise AssertionError(
        "Falló la auditoría de dependencias del voto popular:\n"
        + popular_vote_dependency_audit.loc[
            ~popular_vote_dependency_audit["Passed"]
        ].to_string(index=False)
    )

national_module_contract = pd.DataFrame([
    {
        "Module": "National 42-target engine",
        "Consumes": "71 pre-election features",
        "Produces": "34 learned targets + 8 deterministic identities",
        "May Consume Upstream": "Model worksheet",
        "May Not Mutate": "Source inputs",
    },
    {
        "Module": "Popular-vote selector",
        "Consumes": "EDPP, ERPP; historical DPP/RPP for sealed validation",
        "Produces": "DPP, RPP and residual Other",
        "May Consume Upstream": "National pre-election expectation",
        "May Not Mutate": "House targets; Senate targets; state Senate data",
    },
    {
        "Module": "State Senate model",
        "Consumes": "Frozen national signals + Senate Parser/Historic",
        "Produces": "State margins, probabilities and final Senate seats",
        "May Consume Upstream": "National vote and Senate-bucket signals",
        "May Not Mutate": "Any of the 42 national targets",
    },
])
national_projection_snapshot_before_senate = {
    key: float(value)
    for key, value in final_2026_full_projection_42.items()
}

fold_summary_specs = [
    ("D-R Popular Margin (pp)", "D-R Popular Margin (pp)",
     (final_2026_full_projection_42["DPP"] - final_2026_full_projection_42["RPP"]) * 100.0),
    ("D House Seats", "D House Seats", final_2026_full_projection_42["D House After"]),
    ("D Senate Seats (national prior)", "D Senate Seats (national prior)",
     final_2026_full_projection_42["D Senate After"]),
]
fold_summary_rows = []
for metric, column, full_history_value in fold_summary_specs:
    values = national_2026_fold_forecasts[column].to_numpy(float)
    fold_mean = float(np.mean(values))
    jackknife_se = float(np.sqrt(
        (len(values) - 1) / len(values) * np.sum((values - fold_mean) ** 2)
    ))
    fold_summary_rows.append({
        "Metric": metric,
        "Full-History Production Forecast": float(full_history_value),
        "Five-Fold Mean (Sensitivity Only)": fold_mean,
        "Mean Difference vs Production": fold_mean - float(full_history_value),
        "Fold Standard Deviation": float(np.std(values, ddof=1)),
        "Jackknife Sensitivity SE": jackknife_se,
        "Minimum": float(np.min(values)),
        "Maximum": float(np.max(values)),
        "Range": float(np.ptp(values)),
        "Production Rule": "Refit on all five historical cycles",
    })
national_2026_fold_summary = pd.DataFrame(fold_summary_rows)


# ------------------------------------------------------------
# 4B. 2026 JACKKNIFE STABILITY FOR ALL 42 TARGETS
# ------------------------------------------------------------
#
# Each row summarizes five independently refitted four-cycle forecasts.
# Their mean and median are descriptive jackknife diagnostics. They are never
# appended to the training table and never replace the five-cycle production fit.

target_stability_rows = []
for variable, fold in national_2026_fold_targets.groupby("Variable"):
    group = target_groups[variable]
    scale = 100.0 if group == "Popular Vote" else 1.0
    unit = "percentage points" if group == "Popular Vote" else "seats/count"
    values = fold["Prediction"].to_numpy(float) * scale
    production = float(final_2026_full_projection_42[variable]) * scale
    historical_errors = national_time_machine_selected.loc[
        national_time_machine_selected["Variable"].eq(variable),
        "Absolute Error",
    ].to_numpy(float) * scale
    historical_mae = float(np.mean(historical_errors))
    maximum_shift = float(np.max(np.abs(values - production)))
    sensitivity_to_error_ratio = (
        maximum_shift / historical_mae
        if historical_mae > 1e-12 else np.nan
    )
    target_stability_rows.append({
        "Variable": variable,
        "Group": group,
        "Target Type": (
            "Learned" if variable in ml_target_cols else "Derived identity"
        ),
        "Unit": unit,
        "Production Forecast": production,
        "Five-Fold Mean (Diagnostic Only)": float(np.mean(values)),
        "Five-Fold Median (Diagnostic Only)": float(np.median(values)),
        "Mean Minus Production": float(np.mean(values) - production),
        "Fold Standard Deviation": float(np.std(values, ddof=1)),
        "Minimum": float(np.min(values)),
        "Maximum": float(np.max(values)),
        "Range": float(np.ptp(values)),
        "Maximum Absolute Deletion Shift": maximum_shift,
        "Historical Outer MAE": historical_mae,
        "Deletion Shift / Historical MAE": sensitivity_to_error_ratio,
        "Production Inside Fold Range": bool(
            np.min(values) - 1e-12 <= production <= np.max(values) + 1e-12
        ),
        "Production Rule": "Refit once on all five historical cycles",
        "Fold Mean Role": "Sensitivity diagnostic; never training data",
    })

national_2026_target_stability = (
    pd.DataFrame(target_stability_rows)
    .sort_values(
        ["Maximum Absolute Deletion Shift", "Variable"],
        ascending=[False, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

national_architecture_contract = pd.DataFrame([
    {
        "Order": 1,
        "Module": "National nested selector",
        "Consumes": "71 pre-election inputs",
        "Produces": "34 learned targets",
        "May Feed": "Constraint/identity engine",
        "May Not Feed": "Its own training rows",
        "Production Status": "Upstream",
    },
    {
        "Order": 2,
        "Module": "Constraint and identity engine",
        "Consumes": "34 frozen raw targets + institutional totals",
        "Produces": "All 42 national targets",
        "May Feed": "National dashboard and downstream Senate context",
        "May Not Feed": "National model selection",
        "Production Status": "Upstream",
    },
    {
        "Order": 3,
        "Module": "Five-model 2026 jackknife",
        "Consumes": "Five four-cycle refits",
        "Produces": "Mean, median, range and deletion sensitivity",
        "May Feed": "Diagnostics only",
        "May Not Feed": "Training, target values or production point forecast",
        "Production Status": "Diagnostic only",
    },
    {
        "Order": 4,
        "Module": "Full-history 2026 fit",
        "Consumes": "2006, 2010, 2014, 2018 and 2022",
        "Produces": "Single national production forecast",
        "May Feed": "Monte Carlo and state-Senate context",
        "May Not Feed": "Historical backtests",
        "Production Status": "Official point forecast",
    },
    {
        "Order": 5,
        "Module": "State Senate forecast",
        "Consumes": "Frozen national context + state polls/history",
        "Produces": "Race margins, probabilities and canonical Senate total",
        "May Feed": "Senate dashboard and simulation",
        "May Not Feed": "Any national target",
        "Production Status": "Downstream",
    },
    {
        "Order": 6,
        "Module": "Monte Carlo",
        "Consumes": "Frozen point estimates and validated uncertainty",
        "Produces": "Probability distributions",
        "May Feed": "Probability displays",
        "May Not Feed": "Any fitted point estimate",
        "Production Status": "Downstream uncertainty",
    },
])

if set(national_2026_target_stability["Fold Mean Role"]) != {
    "Sensitivity diagnostic; never training data"
}:
    raise AssertionError("El promedio de folds recibió un rol de producción.")
if len(national_2026_target_stability) != len(p_cols):
    raise AssertionError("La estabilidad 2026 no contiene los 42 targets.")

# Alias de compatibilidad: los modelos finales ahora están organizados por familia y grupo.
models = trained_models

# ------------------------------------------------------------
# 5. PRONÓSTICO 2026 Y AUDITORÍA
# ------------------------------------------------------------

nested_group_mae = (
    national_nested_oof.groupby("Group")["Absolute Error"].mean().to_dict()
)
nested_variable_mae = (
    national_nested_oof.groupby("Variable")["Absolute Error"].mean().to_dict()
)
historical_mean_variable_mae = (
    loocv_df.loc[loocv_df["Model"] == "HistoricalMean"]
    .groupby("Variable")["Absolute Error"].mean().to_dict()
)
prediction_rows = []

for position, column in enumerate(ml_target_cols):
    group = target_groups[column]
    ensemble_pred = float(final_selected_prediction[position])
    if group == "Popular Vote":
        final_ml_pred = float(np.clip(ensemble_pred, 0, 1))
    else:
        final_ml_pred = max(ensemble_pred, 0.0)

    prediction_rows.append({
        "Variable": column,
        "Group": group,
        "ExtraTrees": float(final_model_predictions["ExtraTrees"][position]),
        "RandomForest": float(final_model_predictions["RandomForest"][position]),
        "Ridge": float(final_model_predictions["Ridge"][position]),
        "HistoricalMean": float(
            final_model_predictions["HistoricalMean"][position]
        ),
        "TreeEnsemble": float(
            final_model_predictions["TreeEnsemble"][position]
        ),
        "ExpectedVoteAnchor": float(
            final_model_predictions["ExpectedVoteAnchor"][position]
        ) if group == "Popular Vote" else np.nan,
        "ExpectedVoteErrorCorrected": float(
            final_model_predictions[
                "ExpectedVoteErrorCorrected"
            ][position]
        ) if group == "Popular Vote" else np.nan,
        "Selected Strategy": national_selected_strategy[group],
        "ML Prediction": final_ml_pred,
        "Historical MAE": float(nested_variable_mae[column]),
        "Group Historical MAE": float(nested_group_mae[group]),
        "HistoricalMean MAE": float(historical_mean_variable_mae[column]),
        "Status": "ML Predicted",
    })

for column in derived_cols:
    prediction_rows.append({
        "Variable": column,
        "Group": "Derived Later",
        "ExtraTrees": np.nan,
        "RandomForest": np.nan,
        "Ridge": np.nan,
        "HistoricalMean": np.nan,
        "TreeEnsemble": np.nan,
        "ExpectedVoteAnchor": np.nan,
        "ExpectedVoteErrorCorrected": np.nan,
        "Selected Strategy": "Calculated in Constraint Engine",
        "ML Prediction": np.nan,
        "Historical MAE": np.nan,
        "Group Historical MAE": np.nan,
        "HistoricalMean MAE": np.nan,
        "Status": "Calculated in Constraint Engine",
    })

block2_predictions = pd.DataFrame(prediction_rows)
block2_predictions["order"] = block2_predictions["Variable"].apply(p_cols.index)
block2_predictions = (
    block2_predictions.sort_values("order")
    .drop(columns="order")
    .reset_index(drop=True)
)

if national_nested_oof.duplicated(["Year", "Variable"]).any():
    raise AssertionError("La tabla nacional nested OOF tiene duplicados.")
if not np.isfinite(
    national_nested_oof[["Actual", "Predicted", "Signed Error", "Absolute Error"]]
).all().all():
    raise FloatingPointError("La validación nacional nested contiene NaN o infinitos.")
if block2_predictions.loc[
    block2_predictions["Status"] == "ML Predicted", "ML Prediction"
].isna().any():
    raise AssertionError("Hay pronósticos nacionales 2026 faltantes.")

print("\n============================================================")
print("BLOQUE 2 v12 COMPLETADO")
print("============================================================")
print("Validación exterior: nested leave-one-election-year-out")
print("Ciclos históricos:", historical_years)
print("Estrategias finales por grupo:")
display(national_final_selection)
print("\nDesempeño exterior por modelo y grupo:")
display(loocv_summary)
print("\nTIME MACHINE — cinco elecciones históricas predichas fuera de muestra:")
display(national_time_machine_scorecard)
print("\nPRIMER EXAMEN INTERNO → SEGUNDO EXAMEN EXTERIOR:")
display(national_exam_progression)
print("\n¿MEJORÓ EL REFINAMIENTO EN RESULTADOS ELECTORALES?")
display(national_exam_outcome_comparison)
print("\nMétricas OOF alineadas con resultados electorales:")
display(national_outcome_summary)
print("\n2026 — cinco pronósticos leave-cycle-out (sensibilidad, no producción):")
display(national_2026_fold_forecasts)
print("\n2026 — comparación del promedio parcial con el ajuste completo:")
display(national_2026_fold_summary)
print("\nPredicciones Bloque 2:")
display(block2_predictions)

# ------------------------------------------------------------
# 6. EXPORTAR RESULTADOS
# ------------------------------------------------------------

block2_predictions.to_excel(OUTPUT_DIR / "Block2_ML_Predictions_v12.xlsx", index=False)
loocv_df.to_excel(OUTPUT_DIR / "Block2_Nested_LOEO_Details_v12.xlsx", index=False)
loocv_summary.to_excel(OUTPUT_DIR / "Block2_Nested_LOEO_Summary_v12.xlsx", index=False)
national_nested_oof.to_excel(OUTPUT_DIR / "Block2_National_Nested_OOF_v12.xlsx", index=False)
national_nested_fold_diagnostics.to_excel(
    OUTPUT_DIR / "Block2_National_Nested_Folds_v12.xlsx", index=False
)
national_final_selection.to_excel(
    OUTPUT_DIR / "Block2_National_Final_Selection_v12.xlsx", index=False
)
national_tuning_summary.to_excel(
    OUTPUT_DIR / "Block2_National_Final_Tuning_v12.xlsx", index=False
)
national_outcome_oof.to_excel(
    OUTPUT_DIR / "Block2_National_Outcome_OOF_v12.xlsx", index=False
)
national_outcome_summary.to_excel(
    OUTPUT_DIR / "Block2_National_Outcome_Summary_v12.xlsx", index=False
)
national_time_machine_selected.to_excel(
    OUTPUT_DIR / "Block2_TimeMachine_42_Targets_v12.xlsx", index=False
)
historical_input_reconciliation.to_excel(
    OUTPUT_DIR / "Block2_Historical_Input_Reconciliation_v12.xlsx", index=False
)
national_time_machine_scorecard.to_excel(
    OUTPUT_DIR / "Block2_TimeMachine_Scorecard_v12.xlsx", index=False
)
national_time_machine_target_summary.to_excel(
    OUTPUT_DIR / "Block2_TimeMachine_Target_Summary_v12.xlsx", index=False
)
national_exam_progression.to_excel(
    OUTPUT_DIR / "Block2_First_Second_Exam_Progression_v12.xlsx", index=False
)
national_exam_outcome_comparison.to_excel(
    OUTPUT_DIR / "Block2_Second_Exam_Improvement_v12.xlsx", index=False
)
national_2026_fold_targets.to_excel(
    OUTPUT_DIR / "Block2_2026_Fold_Targets_v12.xlsx", index=False
)
national_2026_fold_forecasts.to_excel(
    OUTPUT_DIR / "Block2_2026_Fold_Forecasts_v12.xlsx", index=False
)
national_2026_fold_summary.to_excel(
    OUTPUT_DIR / "Block2_2026_Fold_Summary_v12.xlsx", index=False
)
national_2026_target_stability.to_excel(
    OUTPUT_DIR / "Block2_2026_Target_Stability_v12.xlsx", index=False
)
national_pipeline_target_exam.to_excel(
    OUTPUT_DIR / "Block2_Pipeline_Target_Exam_v12.xlsx", index=False
)
national_pipeline_stage_summary.to_excel(
    OUTPUT_DIR / "Block2_Pipeline_Stage_Summary_v12.xlsx", index=False
)
national_pipeline_derived_summary.to_excel(
    OUTPUT_DIR / "Block2_Pipeline_Derived_Summary_v12.xlsx", index=False
)
national_validation_stage_contract.to_excel(
    OUTPUT_DIR / "Block2_Validation_Stage_Contract_v12.xlsx", index=False
)
national_architecture_contract.to_excel(
    OUTPUT_DIR / "Block2_Architecture_Contract_v12.xlsx", index=False
)
national_popular_vote_validation.to_excel(
    OUTPUT_DIR / "Block2_Popular_Vote_Validation_v12.xlsx", index=False
)
national_popular_vote_method_summary.to_excel(
    OUTPUT_DIR / "Block2_Popular_Vote_Methods_v12.xlsx", index=False
)
popular_vote_production_bridge.to_excel(
    OUTPUT_DIR / "Block2_Popular_Vote_Bridge_v12.xlsx", index=False
)
popular_vote_dependency_audit.to_excel(
    OUTPUT_DIR / "Block2_Popular_Vote_Dependency_Audit_v12.xlsx", index=False
)
national_module_contract.to_excel(
    OUTPUT_DIR / "Block2_Module_Contract_v12.xlsx", index=False
)

print("\nArchivos de auditoría v12 guardados.")


## BLOQUE 3 — CONSTRAINT ENGINE

In [ ]:
# ============================================================
# BLOQUE 3 v3 — CONSTRAINT ENGINE
# Cuotas condicionadas a 2026 + Hamilton / largest remainder
# ============================================================

import numpy as np
import pandas as pd


def get_2026_value(column):
    if column not in pred_df.columns:
        raise ValueError(f"No encontré la columna: {column}")
    return float(pred_df[column].iloc[0])


DH_BEFORE = int(round(get_2026_value("DH Before")))
RH_BEFORE = int(round(get_2026_value("RH Before")))
DS_BEFORE = int(round(get_2026_value("DS before")))
RS_BEFORE = int(round(get_2026_value("RS Before")))
DSS_UP = int(round(get_2026_value("DSS UP")))
RSS_UP = int(round(get_2026_value("RSS UP")))
HOUSE_TOTAL = 435
SENATE_TOTAL = 100

if DH_BEFORE + RH_BEFORE != HOUSE_TOTAL:
    raise ValueError("DH Before + RH Before debe sumar 435.")
if DS_BEFORE + RS_BEFORE != SENATE_TOTAL:
    raise ValueError("DS before + RS Before debe sumar 100.")

DHOU_RES_COLS = [c for c in p_cols if c.startswith("DHou")]
RHOU_RES_COLS = [c for c in p_cols if c.startswith("RHou")]
DSEN_RES_COLS = [c for c in p_cols if c.startswith("DSen")]
RSEN_RES_COLS = [c for c in p_cols if c.startswith("RSen")]


def get_ml_prediction(pred_table, variable):
    row = pred_table[pred_table["Variable"] == variable]
    if row.empty or pd.isna(row["ML Prediction"].iloc[0]):
        return 0.0
    return max(float(row["ML Prediction"].iloc[0]), 0.0)


def normalized_hamilton(pred_table, columns, required_total):
    """
    1) Conserva la distribución relativa aprendida por ML.
    2) La condiciona al número real de escaños del grupo en 2026.
    3) Hamilton solo resuelve los decimales; cada bucket recibe como máximo
       una unidad adicional sobre su piso.
    """
    values = np.array(
        [get_ml_prediction(pred_table, column) for column in columns],
        dtype=float
    )
    required_total = int(required_total)
    raw_total = float(values.sum())
    if not np.isfinite(raw_total) or raw_total <= 0:
        raise ValueError(f"Predicción no positiva para: {columns}")

    quotas = values / raw_total * required_total
    allocation = np.floor(quotas).astype(int)
    seats_left = required_total - int(allocation.sum())
    if seats_left < 0 or seats_left > len(columns):
        raise RuntimeError("Hamilton recibió cuotas incoherentes.")
    if seats_left:
        remainders = quotas - allocation
        order = np.argsort(-remainders, kind="stable")
        allocation[order[:seats_left]] += 1

    return {
        "allocation": dict(zip(columns, allocation)),
        "quotas": dict(zip(columns, quotas)),
        "raw_total": raw_total,
        "scale_factor": required_total / raw_total
    }


group_specs = {
    "D House": (DHOU_RES_COLS, DH_BEFORE),
    "R House": (RHOU_RES_COLS, RH_BEFORE),
    "D Senate": (DSEN_RES_COLS, DSS_UP),
    "R Senate": (RSEN_RES_COLS, RSS_UP),
}

allocations = {
    name: normalized_hamilton(block2_predictions, columns, total)
    for name, (columns, total) in group_specs.items()
}

constrained = {
    column: get_ml_prediction(block2_predictions, column)
    for column in popular_vote_cols
}
normalized_quotas = {column: constrained[column] for column in popular_vote_cols}
scale_factors = {column: 1.0 for column in popular_vote_cols}

for group_name, result in allocations.items():
    constrained.update(result["allocation"])
    normalized_quotas.update(result["quotas"])
    for column in result["allocation"]:
        scale_factors[column] = result["scale_factor"]


def loss_columns(columns, incumbent_party):
    marker = "ResLR" if incumbent_party == "D" else "ResLD"
    return [column for column in columns if marker in column]


D_HOUSE_LOSS_EXPECTED_PRIOR = sum(
    normalized_quotas[c] for c in loss_columns(DHOU_RES_COLS, "D")
)
R_HOUSE_LOSS_EXPECTED_PRIOR = sum(
    normalized_quotas[c] for c in loss_columns(RHOU_RES_COLS, "R")
)
D_SENATE_LOSS_EXPECTED_PRIOR = sum(
    normalized_quotas[c] for c in loss_columns(DSEN_RES_COLS, "D")
)
R_SENATE_LOSS_EXPECTED_PRIOR = sum(
    normalized_quotas[c] for c in loss_columns(RSEN_RES_COLS, "R")
)

D_HOUSE_LOST = sum(constrained[c] for c in loss_columns(DHOU_RES_COLS, "D"))
R_HOUSE_LOST = sum(constrained[c] for c in loss_columns(RHOU_RES_COLS, "R"))
D_SENATE_LOST = sum(constrained[c] for c in loss_columns(DSEN_RES_COLS, "D"))
R_SENATE_LOST = sum(constrained[c] for c in loss_columns(RSEN_RES_COLS, "R"))

constrained.update({
    "D House Seats LOST": int(D_HOUSE_LOST),
    "R House Seats Lost": int(R_HOUSE_LOST),
    "D Senate Seats LOST": int(D_SENATE_LOST),
    "R Senate Seats LOST": int(R_SENATE_LOST),
    "D House After": int(DH_BEFORE - D_HOUSE_LOST + R_HOUSE_LOST),
    "R House After": int(RH_BEFORE + D_HOUSE_LOST - R_HOUSE_LOST),
    # Este es el prior nacional entero. El pronóstico estatal del Bloque 5B
    # producirá la proyección central canónica del Senado.
    "D Senate After": int(DS_BEFORE - D_SENATE_LOST + R_SENATE_LOST),
    "R Senate After": int(RS_BEFORE + D_SENATE_LOST - R_SENATE_LOST),
})

block3_predictions = block2_predictions.copy()
block3_predictions["Normalized Quota"] = block3_predictions["Variable"].map(normalized_quotas)
block3_predictions["Scale Factor"] = block3_predictions["Variable"].map(scale_factors)
block3_predictions["Constrained Prediction"] = block3_predictions["Variable"].map(constrained)
block3_predictions["Final Type"] = np.where(
    block3_predictions["Group"] == "Popular Vote", "Continuous", "Integer"
)

# Constraint Adjustment mide únicamente el redondeo Hamilton.
# El reescalamiento legítimo queda separado y auditable.
block3_predictions["Rescaling Adjustment"] = (
    block3_predictions["Normalized Quota"] - block3_predictions["ML Prediction"]
)
block3_predictions["Constraint Adjustment"] = (
    block3_predictions["Constrained Prediction"] - block3_predictions["Normalized Quota"]
)
block3_predictions.loc[
    block3_predictions["Group"] == "Derived Later",
    ["Rescaling Adjustment", "Constraint Adjustment"]
] = np.nan

for name, (columns, required_total) in group_specs.items():
    observed = sum(constrained[c] for c in columns)
    if observed != required_total:
        raise AssertionError(f"{name}: {observed} != {required_total}")

if constrained["D House After"] + constrained["R House After"] != HOUSE_TOTAL:
    raise AssertionError("La Cámara final no suma 435.")
if constrained["D Senate After"] + constrained["R Senate After"] != SENATE_TOTAL:
    raise AssertionError("El Senado final no suma 100.")

print("=" * 72)
print("BLOQUE 3 v3 — CONSTRAINT ENGINE COMPLETADO")
print("=" * 72)
for name, result in allocations.items():
    print(
        f"{name}: raw={result['raw_total']:.3f}, "
        f"scale={result['scale_factor']:.4f}, "
        f"final={sum(result['allocation'].values())}"
    )
print("Senate expected losses prior:",
      round(D_SENATE_LOSS_EXPECTED_PRIOR, 3),
      round(R_SENATE_LOSS_EXPECTED_PRIOR, 3))
print("Senate integer prior:", constrained["D Senate After"], constrained["R Senate After"])
display(block3_predictions)

block3_predictions.to_excel(OUTPUT_DIR / "Block3_Constrained_Predictions.xlsx", index=False)


## BLOQUE 4 — VALIDATION & DIAGNOSTIC SENSITIVITY


In [ ]:
# ============================================================
# BLOQUE 4 — DIAGNÓSTICOS DE VALIDACIÓN Y SENSIBILIDAD (v12)
# ============================================================

import numpy as np
import pandas as pd

block4 = block3_predictions.copy()
ml_mask = block4["Status"].eq("ML Predicted")

# Diferencia entre el pronóstico de producción y la alternativa de árboles
# heredada de la arquitectura original. Esto sí mide sensibilidad a la familia
# de modelos que se elige; ExtraTrees vs RandomForest ya no era pertinente
# cuando la estrategia productiva seleccionada era Ridge.
block4["Alternative Strategy"] = "TreeEnsemble"
block4["Alternative Prediction"] = block4["TreeEnsemble"]
block4["Model Difference"] = (
    block4["ML Prediction"] - block4["Alternative Prediction"]
).abs()

# Impacto total del posprocesamiento: normalización de cuotas + Hamilton.
block4["Constraint Impact"] = (
    block4["Constrained Prediction"] - block4["ML Prediction"]
).abs()

denominator = block4["Historical MAE"].replace(0, np.nan)
block4["Relative Model Difference"] = block4["Model Difference"] / denominator
block4["Relative Constraint Impact"] = block4["Constraint Impact"] / denominator
block4["OOF Error Ratio vs Historical Mean"] = (
    block4["Historical MAE"]
    / block4["HistoricalMean MAE"].replace(0, np.nan)
)

# Índice heurístico de sensibilidad; no es una probabilidad, una medida de
# precisión ni un objetivo de optimización. Se limita el efecto de un único
# target para impedir que una escala extrema domine todo el resumen.
stability_risk = (
    0.65 * block4["Relative Model Difference"].clip(upper=3).fillna(0)
    + 0.35 * block4["Relative Constraint Impact"].clip(upper=3).fillna(0)
)
block4["Model Stability Score"] = (100 * np.exp(-stability_risk)).clip(0, 100)


def stability_label(score):
    if pd.isna(score):
        return ""
    if score >= 90:
        return "Low sensitivity"
    if score >= 75:
        return "Moderate-low sensitivity"
    if score >= 60:
        return "Moderate sensitivity"
    if score >= 40:
        return "High sensitivity"
    return "Very high sensitivity"


block4["Model Stability"] = block4["Model Stability Score"].apply(stability_label)
block4.loc[~ml_mask, [
    "Alternative Prediction", "Model Difference", "Constraint Impact",
    "Relative Model Difference", "Relative Constraint Impact",
    "OOF Error Ratio vs Historical Mean", "Model Stability Score",
]] = np.nan
block4.loc[~ml_mask, "Model Stability"] = ""

ml_rows = block4.loc[ml_mask].copy()
overall_stability = float(ml_rows["Model Stability Score"].mean())
ensemble_agreement = float(ml_rows["Model Difference"].mean())
constraint_efficiency = float(ml_rows["Constraint Impact"].mean())
max_constraint = float(ml_rows["Constraint Impact"].max())

group_summary = (
    ml_rows.groupby("Group")
    .agg(
        Stability=("Model Stability Score", "mean"),
        AlternativeDifference=("Model Difference", "mean"),
        ConstraintImpact=("Constraint Impact", "mean"),
        HistoricalMAE=("Historical MAE", "mean"),
        OOFErrorRatio=("OOF Error Ratio vs Historical Mean", "mean"),
    )
    .reset_index()
    .round(3)
)

highest_stability = ml_rows.sort_values(
    "Model Stability Score", ascending=False
).head(10)
largest_adjustments = ml_rows.sort_values(
    "Constraint Impact", ascending=False
).head(10)
largest_disagreement = ml_rows.sort_values(
    "Model Difference", ascending=False
).head(10)
largest_disagreements = largest_disagreement.copy()

print("=" * 72)
print("BLOQUE 4 — DIAGNÓSTICOS v12")
print("=" * 72)
print(f"Variables evaluadas: {len(ml_rows)}")
print(f"Índice diagnóstico de estabilidad: {overall_stability:.1f}/100")
print("Nota: índice heurístico de sensibilidad; no es probabilidad ni precisión.")
print(f"Diferencia media frente a TreeEnsemble: {ensemble_agreement:.3f}")
print(f"Impacto medio total de restricciones: {constraint_efficiency:.3f}")
print(f"Impacto máximo total de restricciones: {max_constraint:.3f}")
print("\nResumen por grupo:")
display(group_summary)
print("\nResultados centrales:")


def diagnostic_value(variable):
    row = block4[block4["Variable"] == variable]
    if row.empty:
        raise ValueError(f"No encontré {variable} en Block 4.")
    return float(row["Constrained Prediction"].iloc[0])


print(
    f"House: D {int(diagnostic_value('D House After'))} - "
    f"R {int(diagnostic_value('R House After'))}"
)
print(
    f"Senate national prior: D {int(diagnostic_value('D Senate After'))} - "
    f"R {int(diagnostic_value('R Senate After'))}"
)
print(f"D Popular Vote: {diagnostic_value('DPP'):.3f}")
print(f"R Popular Vote: {diagnostic_value('RPP'):.3f}")
print(f"Other: {1-diagnostic_value('DPP')-diagnostic_value('RPP'):.3f}")

block4.to_excel(OUTPUT_DIR / "Block4_Diagnostics_v12.xlsx", index=False)
group_summary.to_excel(OUTPUT_DIR / "Block4_Diagnostic_Stability_Summary_v12.xlsx", index=False)
highest_stability.to_excel(OUTPUT_DIR / "Block4_Lowest_Sensitivity_v12.xlsx", index=False)
largest_adjustments.to_excel(OUTPUT_DIR / "Block4_Largest_Adjustments_v12.xlsx", index=False)
largest_disagreement.to_excel(OUTPUT_DIR / "Block4_Largest_Alternative_Differences_v12.xlsx", index=False)


## BLOQUE 5 ELECTION PROJECTION ENGINE

In [ ]:
# ============================================================
# BLOQUE 5 — ELECTION PROJECTION ENGINE
# Versión limpia desde cero
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. BASES DE TRABAJO
# ------------------------------------------------------------

projection = block3_predictions.copy()

# Usar block4 si existe; si no, usar projection y crear métricas mínimas
if "block4" in globals():
    diagnostics = block4.copy()
else:
    diagnostics = projection.copy()

# Asegurar columnas necesarias
if "Model Stability Score" not in diagnostics.columns:
    diagnostics["Model Difference"] = (
        diagnostics["ExtraTrees"] - diagnostics["RandomForest"]
    ).abs()

    diagnostics["Constraint Impact"] = (
        diagnostics["Constraint Adjustment"]
        .abs()
        .fillna(0)
    )

    diagnostics["Relative Model Difference"] = (
        diagnostics["Model Difference"] /
        diagnostics["Historical MAE"]
    )

    diagnostics["Historical Error"] = (
        diagnostics["Historical MAE"] /
        diagnostics["Historical MAE"].max()
    )

    risk = (
          0.50 * diagnostics["Relative Model Difference"].fillna(0)
        + 0.20 * diagnostics["Constraint Impact"].fillna(0)
        + 0.30 * diagnostics["Historical Error"].fillna(0)
    )

    diagnostics["Model Stability Score"] = (
        100 * np.exp(-risk)
    ).clip(0, 100)

if "Model Stability" not in diagnostics.columns:
    def stability_label(score):
        if pd.isna(score):
            return ""
        if score >= 90:
            return "Excellent"
        elif score >= 80:
            return "Very High"
        elif score >= 70:
            return "High"
        elif score >= 60:
            return "Moderate"
        else:
            return "Needs Review"

    diagnostics["Model Stability"] = diagnostics["Model Stability Score"].apply(stability_label)

# ------------------------------------------------------------
# 2. FUNCIÓN PARA EXTRAER VALORES
# ------------------------------------------------------------

def get_projection_value(variable):
    row = projection[projection["Variable"] == variable]

    if len(row) == 0:
        raise ValueError(f"No encontré la variable: {variable}")

    return row["Constrained Prediction"].iloc[0]

# ------------------------------------------------------------
# 3. VARIABLES PRINCIPALES
# ------------------------------------------------------------

DPP = float(get_projection_value("DPP"))
RPP = float(get_projection_value("RPP"))
OTHER = max(0, 1 - DPP - RPP)

D_HOUSE = int(get_projection_value("D House After"))
R_HOUSE = int(get_projection_value("R House After"))

D_SENATE = int(get_projection_value("D Senate After"))
R_SENATE = int(get_projection_value("R Senate After"))

D_HOUSE_LOST = int(get_projection_value("D House Seats LOST"))
R_HOUSE_LOST = int(get_projection_value("R House Seats Lost"))

D_SENATE_LOST = int(get_projection_value("D Senate Seats LOST"))
R_SENATE_LOST = int(get_projection_value("R Senate Seats LOST"))

HOUSE_NET_D = R_HOUSE_LOST - D_HOUSE_LOST
SENATE_NET_D = R_SENATE_LOST - D_SENATE_LOST

HOUSE_CONTROL = "Democratic" if D_HOUSE > R_HOUSE else "Republican"
SENATE_CONTROL = "Democratic" if D_SENATE > R_SENATE else "Republican"

HOUSE_MARGIN = abs(D_HOUSE - R_HOUSE)
SENATE_MARGIN = abs(D_SENATE - R_SENATE)

# ------------------------------------------------------------
# 4. POPULAR VOTE
# ------------------------------------------------------------

popular_vote_table = pd.DataFrame({
    "Party": ["Democratic", "Republican", "Other"],
    "Share": [DPP, RPP, OTHER],
    "Share (%)": [DPP * 100, RPP * 100, OTHER * 100]
})

popular_vote_table["Share (%)"] = popular_vote_table["Share (%)"].round(2)

# ------------------------------------------------------------
# 5. CONTROL SUMMARY
# ------------------------------------------------------------

control_summary = pd.DataFrame({
    "Chamber": ["House", "Senate"],
    "Democratic Seats": [D_HOUSE, D_SENATE],
    "Republican Seats": [R_HOUSE, R_SENATE],
    "Projected Control": [HOUSE_CONTROL, SENATE_CONTROL],
    "Margin": [HOUSE_MARGIN, SENATE_MARGIN],
    "Democratic Net Change": [HOUSE_NET_D, SENATE_NET_D],
    "Republican Net Change": [-HOUSE_NET_D, -SENATE_NET_D]
})

# ------------------------------------------------------------
# 6. BUCKET DISTRIBUTIONS
# ------------------------------------------------------------

house_distribution = projection[
    projection["Variable"].str.contains("Hou", na=False)
    & projection["Variable"].str.contains("Tilt|Lean|Lik|Saf", regex=True, na=False)
][[
    "Variable",
    "ML Prediction",
    "Constrained Prediction",
    "Constraint Adjustment"
]].copy()

senate_distribution = projection[
    projection["Variable"].str.contains("Sen", na=False)
    & projection["Variable"].str.contains("Tilt|Lean|Lik|Saf", regex=True, na=False)
][[
    "Variable",
    "ML Prediction",
    "Constrained Prediction",
    "Constraint Adjustment"
]].copy()

house_distribution.rename(columns={"Constrained Prediction": "Projected Seats"}, inplace=True)
senate_distribution.rename(columns={"Constrained Prediction": "Projected Seats"}, inplace=True)

# ------------------------------------------------------------
# 7. DIAGNOSTIC STABILITY SUMMARY
# ------------------------------------------------------------

ml_diagnostics = diagnostics[
    diagnostics["Status"] == "ML Predicted"
].copy()

stability_summary = (
    ml_diagnostics
    .groupby("Group")
    .agg(
        Stability=("Model Stability Score", "mean"),
        AlternativeDifference=("Model Difference", "mean"),
        ConstraintImpact=("Constraint Impact", "mean"),
        HistoricalMAE=("Historical MAE", "mean")
    )
    .round(3)
    .reset_index()
)

overall_stability = round(ml_diagnostics["Model Stability Score"].mean(), 1)

def component_assessment(group_name):
    row = stability_summary[stability_summary["Group"] == group_name]
    if len(row) == 0:
        return np.nan
    return row["Stability"].iloc[0]

projection_reliability = pd.DataFrame({
    "Component": ["Popular Vote", "House", "Senate", "Derived Values"],
    "Stability Score": [
        component_assessment("Popular Vote"),
        component_assessment("House Buckets"),
        component_assessment("Senate Buckets"),
        100.0
    ],
    "Assessment": [
        stability_label(component_assessment("Popular Vote")),
        stability_label(component_assessment("House Buckets")),
        stability_label(component_assessment("Senate Buckets")),
        "Deterministic identity"
    ],
    "Comment": [
        "Sensitivity to the alternative TreeEnsemble; no seat constraint required.",
        "Sensitivity includes normalization to the 435-seat identity.",
        "National prior sensitivity includes the contested-seat identity.",
        "Calculated directly from constrained seat identities."
    ]
})

# ------------------------------------------------------------
# 8. EXECUTIVE SNAPSHOT
# ------------------------------------------------------------

election_snapshot = pd.DataFrame({
    "Metric": [
        "Democratic Popular Vote",
        "Republican Popular Vote",
        "Other Popular Vote",
        "House Projection",
        "House Control",
        "House Margin",
        "House Democratic Net Change",
        "Senate Projection",
        "Senate Control",
        "Senate Margin",
        "Senate Democratic Net Change",
        "Overall Diagnostic Stability"
    ],
    "Projection": [
        f"{DPP*100:.2f}%",
        f"{RPP*100:.2f}%",
        f"{OTHER*100:.2f}%",
        f"D {D_HOUSE} - R {R_HOUSE}",
        HOUSE_CONTROL,
        HOUSE_MARGIN,
        HOUSE_NET_D,
        f"D {D_SENATE} - R {R_SENATE}",
        SENATE_CONTROL,
        SENATE_MARGIN,
        SENATE_NET_D,
        f"{overall_stability}/100"
    ]
})

# Objetos ejecutivos consumidos por el Bloque 7.
executive_summary = election_snapshot.copy()
final_projection = control_summary.copy()
final_snapshot = election_snapshot.copy()
model_quality = stability_summary.rename(columns={
    "Stability": "Diagnostic Stability (0-100)",
    "AlternativeDifference": "Production vs Tree Difference",
    "ConstraintImpact": "Constraint Impact",
    "HistoricalMAE": "Nested OOF MAE",
}).copy()

# ------------------------------------------------------------
# 9. RANKINGS DE AUDITORÍA
# ------------------------------------------------------------

largest_adjustments = (
    diagnostics[
        diagnostics["Status"] == "ML Predicted"
    ]
    .sort_values("Constraint Impact", ascending=False)
    .head(15)
)

largest_disagreements = (
    diagnostics[
        diagnostics["Status"] == "ML Predicted"
    ]
    .sort_values("Model Difference", ascending=False)
    .head(15)
)

most_stable = (
    diagnostics[
        diagnostics["Status"] == "ML Predicted"
    ]
    .sort_values("Model Stability Score", ascending=False)
    .head(15)
)

least_stable = (
    diagnostics[
        diagnostics["Status"] == "ML Predicted"
    ]
    .sort_values("Model Stability Score", ascending=True)
    .head(15)
)

# ------------------------------------------------------------
# 10. IMPRESIÓN DEL REPORTE
# ------------------------------------------------------------

print("="*70)
print("BLOQUE 5 — ELECTION PROJECTION ENGINE")
print("="*70)

print("\nPOPULAR VOTE")
display(popular_vote_table)

print("\nCONTROL SUMMARY")
display(control_summary)

print("\nELECTION SNAPSHOT")
display(election_snapshot)

print("\nHOUSE DISTRIBUTION")
display(house_distribution)

print("\nSENATE DISTRIBUTION")
display(senate_distribution)

print("\nPROJECTION RELIABILITY")
display(projection_reliability)

print("\nDIAGNOSTIC STABILITY SUMMARY")
display(stability_summary)

print("\nLARGEST CONSTRAINT ADJUSTMENTS")
display(
    largest_adjustments[
        ["Variable", "Group", "ML Prediction", "Constrained Prediction", "Constraint Impact"]
    ]
)

print("\nLARGEST PRODUCTION-vs-TREE DIFFERENCES")
display(
    largest_disagreements[
        ["Variable", "Group", "Selected Strategy", "ML Prediction", "Alternative Prediction", "Model Difference"]
    ]
)

print("\nLOWEST-SENSITIVITY VARIABLES")
display(
    most_stable[
        ["Variable", "Group", "Model Stability Score", "Model Stability"]
    ]
)

print("\nHIGHEST-SENSITIVITY VARIABLES")
display(
    least_stable[
        ["Variable", "Group", "Model Stability Score", "Model Stability"]
    ]
)

# ------------------------------------------------------------
# 11. EXPORTAR ARCHIVOS
# ------------------------------------------------------------

popular_vote_table.to_excel(OUTPUT_DIR / "Block5_Popular_Vote.xlsx", index=False)
control_summary.to_excel(OUTPUT_DIR / "Block5_Control_Summary.xlsx", index=False)
election_snapshot.to_excel(OUTPUT_DIR / "Block5_Election_Snapshot.xlsx", index=False)
house_distribution.to_excel(OUTPUT_DIR / "Block5_House_Distribution.xlsx", index=False)
senate_distribution.to_excel(OUTPUT_DIR / "Block5_Senate_Distribution.xlsx", index=False)
projection_reliability.to_excel(OUTPUT_DIR / "Block5_Projection_Reliability.xlsx", index=False)
stability_summary.to_excel(OUTPUT_DIR / "Block5_Model_Stability_Summary.xlsx", index=False)
largest_adjustments.to_excel(OUTPUT_DIR / "Block5_Largest_Adjustments.xlsx", index=False)
largest_disagreements.to_excel(OUTPUT_DIR / "Block5_Largest_Disagreements.xlsx", index=False)
most_stable.to_excel(OUTPUT_DIR / "Block5_Most_Stable.xlsx", index=False)
least_stable.to_excel(OUTPUT_DIR / "Block5_Least_Stable.xlsx", index=False)

# Archivo consolidado con varias hojas
with pd.ExcelWriter(OUTPUT_DIR / "Block5_Election_Projection_Report.xlsx") as writer:
    election_snapshot.to_excel(writer, sheet_name="Snapshot", index=False)
    popular_vote_table.to_excel(writer, sheet_name="Popular Vote", index=False)
    control_summary.to_excel(writer, sheet_name="Control Summary", index=False)
    house_distribution.to_excel(writer, sheet_name="House Distribution", index=False)
    senate_distribution.to_excel(writer, sheet_name="Senate Distribution", index=False)
    # Los resultados de 5B se consolidan después, en el Bloque 7.
    projection_reliability.to_excel(writer, sheet_name="Reliability", index=False)
    stability_summary.to_excel(writer, sheet_name="Stability Summary", index=False)
    largest_adjustments.to_excel(writer, sheet_name="Largest Adjustments", index=False)
    largest_disagreements.to_excel(writer, sheet_name="Model Disagreement", index=False)
    most_stable.to_excel(writer, sheet_name="Most Stable", index=False)
    least_stable.to_excel(writer, sheet_name="Least Stable", index=False)

print("\nArchivos guardados:")
print("- Block5_Popular_Vote.xlsx")
print("- Block5_Control_Summary.xlsx")
print("- Block5_Election_Snapshot.xlsx")
print("- Block5_House_Distribution.xlsx")
print("- Block5_Senate_Distribution.xlsx")
print("- Block5_Projection_Reliability.xlsx")
print("- Block5_Model_Stability_Summary.xlsx")
print("- Block5_Election_Projection_Report.xlsx")

print("\nBLOQUE 5 COMPLETADO.")

## BLOQUE 5B — STATE SENATE AFTER THE NATIONAL MODEL v12

El módulo estatal recibe una copia congelada del contexto nacional. Hace competir encuesta, voto nacional, contexto senatorial agregado y partial pooling estatal bajo validación anidada. El total final se reconstruye exclusivamente desde los ganadores estatales; el prior nacional es una señal opcional y nunca una cuota de escaños.


In [ ]:
# ============================================================
# BLOQUE 5B — NESTED POLLING-ERROR SENATE FORECAST (v12)
# Poll anchor + cross-fitted polling-error correction + hierarchical Monte Carlo
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("=" * 76)
print("BLOQUE 5B v12 — NESTED POLLING-ERROR SENATE FORECAST")
print("=" * 76)

# ------------------------------------------------------------
# 1. VERIFICACIONES Y RUTA
# ------------------------------------------------------------

required_objects_5b = [
    "block3_predictions", "block2_predictions", "pred_df", "train_df",
    "national_nested_oof", "national_nested_fold_diagnostics",
    "DSEN_RES_COLS", "RSEN_RES_COLS",
    "D_SENATE_LOSS_EXPECTED_PRIOR", "R_SENATE_LOSS_EXPECTED_PRIOR"
]
missing_5b = [obj for obj in required_objects_5b if obj not in globals()]
if missing_5b:
    raise ValueError("Faltan objetos necesarios para Bloque 5B:\n" + "\n".join(missing_5b))

if "MODEL_FILE" in globals():
    senate_excel_path = MODEL_FILE
elif "excel_path" in globals():
    senate_excel_path = excel_path
elif "file_path" in globals():
    senate_excel_path = file_path
elif "xlsx_path" in globals():
    senate_excel_path = xlsx_path
else:
    senate_excel_path = "Model.xlsx"

projection_5b = block3_predictions.copy()


def get_projection_5b(variable):
    row = projection_5b[projection_5b["Variable"] == variable]
    if row.empty:
        raise ValueError(f"No encontré la variable en block3_predictions: {variable}")
    return row["Constrained Prediction"].iloc[0]


def get_pred_value_5b(variable):
    if variable not in pred_df.columns:
        raise ValueError(f"No encontré la columna en pred_df: {variable}")
    return pred_df[variable].iloc[0]


def clean_percent_to_pp(value):
    """Convierte 0.49, 49 o '49%' a 49 puntos porcentuales."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        value = value.strip().replace("%", "").replace(",", "")
        if value == "":
            return np.nan
    value = float(value)
    return value * 100 if abs(value) <= 1 else value


def weighted_mean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        return np.nan
    return float(np.average(values[valid], weights=weights[valid]))


def outcome_label(incumbent, winner):
    incumbent = str(incumbent).strip().upper()
    if winner == "D":
        return "Democratic Hold" if incumbent == "D" else "Democratic Flip"
    return "Republican Hold" if incumbent == "R" else "Republican Flip"


def rating_bucket_from_text(rating):
    rating = str(rating)
    if "TosPoll" in rating or "Tilt" in rating:
        return "Toss-Up"
    if "Lean" in rating:
        return "Lean"
    if "Lik" in rating:
        return "Likely"
    if "Saf" in rating:
        return "Safe"
    return "Unclassified"


def party_margin_rating(margin_pp):
    """Rating del modelo según los umbrales definidos en la base."""
    margin_pp = float(margin_pp)
    magnitude = abs(margin_pp)
    if magnitude <= 1:
        return "Toss-Up"
    party = "D" if margin_pp > 0 else "R"
    if magnitude <= 5:
        bucket = "Lean"
    elif magnitude <= 10:
        bucket = "Likely"
    else:
        bucket = "Safe"
    return f"{party} {bucket}"


def consensus_rating(row):
    bucket = rating_bucket_from_text(row["RATING"])
    if bucket in {"Toss-Up", "Unclassified"}:
        return bucket
    party = "D" if float(row["MARGIN"]) > 0 else "R"
    return f"{party} {bucket}"


def ensemble_oof_target(year, variable):
    """Señal nacional canónica: pronóstico exterior ya seleccionado en Bloque 2."""
    rows = national_nested_oof[
        (national_nested_oof["Year"].astype(int) == int(year))
        & (national_nested_oof["Variable"] == variable)
    ]
    if len(rows) != 1:
        raise ValueError(
            f"Esperaba una predicción nacional nested para {year} / {variable}; "
            f"encontré {len(rows)}."
        )
    return float(rows["Predicted"].iloc[0])


def final_national_target(variable):
    row = block2_predictions[block2_predictions["Variable"] == variable]
    if row.empty:
        raise ValueError(f"No existe la predicción nacional 2026 para {variable}.")
    return float(row["ML Prediction"].iloc[0])


def expected_losses_from_buckets(predictions, columns, total_up, loss_marker):
    values = np.array([max(float(predictions[c]), 0.0) for c in columns], dtype=float)
    if not np.isfinite(values).all() or values.sum() <= 0:
        raise ValueError("Los buckets senatoriales nacionales no forman una distribución válida.")
    quotas = values / values.sum() * float(total_up)
    return float(sum(q for c, q in zip(columns, quotas) if loss_marker in c))


# ------------------------------------------------------------
# 2. COMPOSICIÓN INSTITUCIONAL DE PARTIDA
# ------------------------------------------------------------

DS_BEFORE_5B = int(round(float(get_pred_value_5b("DS before"))))
RS_BEFORE_5B = int(round(float(get_pred_value_5b("RS Before"))))
DSS_UP_5B = int(round(float(get_pred_value_5b("DSS UP"))))
RSS_UP_5B = int(round(float(get_pred_value_5b("RSS UP"))))
D_SENATE_AFTER_5B = int(round(float(get_projection_5b("D Senate After"))))
R_SENATE_AFTER_5B = int(round(float(get_projection_5b("R Senate After"))))

# ------------------------------------------------------------
# 3. CARGAR SENATE PARSER 2026
# ------------------------------------------------------------

raw_senate = pd.read_excel(senate_excel_path, sheet_name="Senate Parser", header=None)
parser_headers = ["STATE", "INCUMBENT", "DEM MEAN", "REP MEAN", "MARGIN", "RATING"]
parser_header_row = None
parser_positions = {}

for idx, row in raw_senate.iterrows():
    normalized = list(row.astype(str).str.strip().str.upper())
    if all(header in normalized for header in parser_headers):
        parser_header_row = idx
        parser_positions = {header: normalized.index(header) for header in parser_headers}
        break

if parser_header_row is None:
    raise ValueError("No pude detectar los encabezados requeridos en Senate Parser.")

senate_parser = raw_senate.iloc[
    parser_header_row + 1:, list(parser_positions.values())
].copy()
senate_parser.columns = parser_headers

valid_states = {
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado",
    "Connecticut", "Delaware", "Florida", "Georgia", "Hawaii", "Idaho",
    "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky", "Louisiana",
    "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota",
    "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada",
    "New Hampshire", "New Jersey", "New Mexico", "New York",
    "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon",
    "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota",
    "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington",
    "West Virginia", "Wisconsin", "Wyoming"
}

senate_parser["STATE"] = senate_parser["STATE"].astype(str).str.strip()
senate_parser = senate_parser[senate_parser["STATE"].isin(valid_states)].copy()
if senate_parser.empty:
    raise ValueError("Senate Parser no contiene filas estatales válidas.")
if senate_parser["STATE"].duplicated().any():
    duplicates = senate_parser.loc[senate_parser["STATE"].duplicated(False), "STATE"].tolist()
    raise ValueError(f"Senate Parser tiene estados duplicados: {duplicates}")

senate_parser["INCUMBENT"] = senate_parser["INCUMBENT"].astype(str).str.strip().str.upper()
for column in ["DEM MEAN", "REP MEAN", "MARGIN"]:
    senate_parser[column] = senate_parser[column].apply(clean_percent_to_pp)
senate_parser["RATING"] = senate_parser["RATING"].astype(str).str.strip()

if senate_parser[["DEM MEAN", "REP MEAN"]].isna().any().any():
    raise ValueError("Hay encuestas 2026 faltantes en DEM MEAN o REP MEAN.")
if not senate_parser["INCUMBENT"].isin(["D", "R"]).all():
    raise ValueError("INCUMBENT debe contener únicamente D o R.")

senate_parser["MARGIN"] = senate_parser["MARGIN"].fillna(
    senate_parser["DEM MEAN"] - senate_parser["REP MEAN"]
)
current_total_2p = senate_parser["DEM MEAN"] + senate_parser["REP MEAN"]
senate_parser["D Poll 2P"] = 100 * senate_parser["DEM MEAN"] / current_total_2p
senate_parser["R Poll 2P"] = 100 * senate_parser["REP MEAN"] / current_total_2p
senate_parser["Poll Margin 2P"] = senate_parser["D Poll 2P"] - senate_parser["R Poll 2P"]
# Shared feature name keeps the historical and 2026 design matrices identical.
senate_parser["Expected Margin 2P"] = senate_parser["Poll Margin 2P"]
senate_parser["Unallocated Poll Share"] = np.clip(100 - current_total_2p, 0, 100)
senate_parser["Rating Bucket"] = senate_parser["RATING"].apply(rating_bucket_from_text)
senate_parser["Consensus Rating"] = senate_parser.apply(consensus_rating, axis=1)

# ------------------------------------------------------------
# 4. CARGAR Y PREPARAR SENATE HISTORIC
# ------------------------------------------------------------

historic_headers = [
    "STATE", "YEAR", "INCUMBENT", "PARTY AFTER",
    "D EXPECTED", "R EXPECTED", "D RESULT", "R RESULT"
]
senate_historic = pd.read_excel(senate_excel_path, sheet_name="Senate Historic")
senate_historic.columns = senate_historic.columns.astype(str).str.strip().str.upper()
missing_historic_columns = [c for c in historic_headers if c not in senate_historic.columns]
if missing_historic_columns:
    raise ValueError("Faltan columnas en Senate Historic: " + ", ".join(missing_historic_columns))

senate_historic = senate_historic[historic_headers].copy()
senate_historic["STATE"] = senate_historic["STATE"].ffill().astype(str).str.strip()
senate_historic["YEAR"] = pd.to_numeric(senate_historic["YEAR"], errors="coerce")
senate_historic = senate_historic[senate_historic["YEAR"].notna()].copy()
senate_historic["YEAR"] = senate_historic["YEAR"].astype(int)
senate_historic["INCUMBENT"] = senate_historic["INCUMBENT"].astype(str).str.strip().str.upper()
senate_historic["PARTY AFTER"] = senate_historic["PARTY AFTER"].astype(str).str.strip().str.upper()
for column in ["D EXPECTED", "R EXPECTED", "D RESULT", "R RESULT"]:
    senate_historic[column] = senate_historic[column].apply(clean_percent_to_pp)

if senate_historic.duplicated(["STATE", "YEAR"]).any():
    duplicates = senate_historic.loc[
        senate_historic.duplicated(["STATE", "YEAR"], False), ["STATE", "YEAR"]
    ].to_dict("records")
    raise ValueError(f"Senate Historic tiene state-years duplicados: {duplicates}")
if senate_historic[["D EXPECTED", "R EXPECTED", "D RESULT", "R RESULT"]].isna().any().any():
    raise ValueError("Senate Historic contiene porcentajes faltantes o no numéricos.")
if not senate_historic["INCUMBENT"].isin(["D", "R"]).all():
    raise ValueError("INCUMBENT histórico debe contener únicamente D o R.")
if not senate_historic["PARTY AFTER"].isin(["D", "R"]).all():
    raise ValueError("PARTY AFTER debe contener únicamente D o R.")

states_without_history = sorted(set(senate_parser["STATE"]) - set(senate_historic["STATE"]))
if states_without_history:
    raise ValueError("Estados 2026 sin historia en Senate Historic: " + ", ".join(states_without_history))

expected_total = senate_historic["D EXPECTED"] + senate_historic["R EXPECTED"]
result_total = senate_historic["D RESULT"] + senate_historic["R RESULT"]
if (expected_total <= 0).any() or (result_total <= 0).any():
    raise ValueError("Senate Historic contiene totales bipartidistas no positivos.")

senate_historic["D Expected 2P"] = 100 * senate_historic["D EXPECTED"] / expected_total
senate_historic["R Expected 2P"] = 100 * senate_historic["R EXPECTED"] / expected_total
senate_historic["D Result 2P"] = 100 * senate_historic["D RESULT"] / result_total
senate_historic["R Result 2P"] = 100 * senate_historic["R RESULT"] / result_total
senate_historic["Expected Margin 2P"] = (
    senate_historic["D Expected 2P"] - senate_historic["R Expected 2P"]
)
senate_historic["Result Margin 2P"] = (
    senate_historic["D Result 2P"] - senate_historic["R Result 2P"]
)
senate_historic["Raw Margin Error PP"] = (
    senate_historic["Result Margin 2P"] - senate_historic["Expected Margin 2P"]
)
senate_historic["Unallocated Poll Share"] = np.clip(100 - expected_total, 0, 100)
senate_historic["Abs Poll Margin 2P"] = senate_historic["Expected Margin 2P"].abs()
senate_historic["Expected Winner"] = np.where(
    senate_historic["Expected Margin 2P"] >= 0, "D", "R"
)
senate_historic["Actual Winner"] = np.where(
    senate_historic["Result Margin 2P"] >= 0, "D", "R"
)
senate_historic["Polling Miss"] = senate_historic["Expected Winner"] != senate_historic["Actual Winner"]
senate_historic["Flip"] = senate_historic["INCUMBENT"] != senate_historic["PARTY AFTER"]
if not (senate_historic["PARTY AFTER"] == senate_historic["Actual Winner"]).all():
    bad = senate_historic.loc[
        senate_historic["PARTY AFTER"] != senate_historic["Actual Winner"],
        ["STATE", "YEAR", "PARTY AFTER", "Actual Winner"]
    ]
    raise ValueError("PARTY AFTER no coincide con el resultado:\n" + bad.to_string(index=False))

# ------------------------------------------------------------
# 5. SEÑALES NACIONALES OUT-OF-FOLD DERIVADAS DE LAS 71 VARIABLES
# ------------------------------------------------------------

national_signal_rows = []
historic_years = sorted(senate_historic["YEAR"].unique())

for year in historic_years:
    dpp_pred = ensemble_oof_target(year, "DPP")
    rpp_pred = ensemble_oof_target(year, "RPP")
    two_party_total = dpp_pred + rpp_pred
    national_vote_margin = 100 * (dpp_pred - rpp_pred) / two_party_total

    year_row = train_df[train_df["Midterm Year"].astype(int) == int(year)]
    if year_row.empty:
        raise ValueError(f"El año {year} no existe en la hoja Model histórica.")
    dss_up = float(year_row["DSS UP"].iloc[0])
    rss_up = float(year_row["RSS UP"].iloc[0])
    d_bucket_preds = {c: ensemble_oof_target(year, c) for c in DSEN_RES_COLS}
    r_bucket_preds = {c: ensemble_oof_target(year, c) for c in RSEN_RES_COLS}
    d_losses = expected_losses_from_buckets(d_bucket_preds, DSEN_RES_COLS, dss_up, "ResLR")
    r_losses = expected_losses_from_buckets(r_bucket_preds, RSEN_RES_COLS, rss_up, "ResLD")

    national_signal_rows.append({
        "YEAR": int(year),
        "National Vote Margin 2P": national_vote_margin,
        "D-Held Loss Pressure": d_losses,
        "R-Held Loss Pressure": r_losses,
        "National Senate Net Gain Signal": r_losses - d_losses,
        "National Senate Flip Pressure": r_losses + d_losses,
    })

national_signals = pd.DataFrame(national_signal_rows)
senate_training = senate_historic.merge(national_signals, on="YEAR", how="left")

from types import MappingProxyType

dpp_2026 = final_national_target("DPP")
rpp_2026 = final_national_target("RPP")
_national_signal_payload_2026 = {
    "DPP": float(dpp_2026),
    "RPP": float(rpp_2026),
    "National Vote Margin 2P": float(
        100 * (dpp_2026 - rpp_2026) / (dpp_2026 + rpp_2026)
    ),
    "D-Held Loss Pressure": float(D_SENATE_LOSS_EXPECTED_PRIOR),
    "R-Held Loss Pressure": float(R_SENATE_LOSS_EXPECTED_PRIOR),
}
_national_signal_payload_2026["National Senate Net Gain Signal"] = (
    _national_signal_payload_2026["R-Held Loss Pressure"]
    - _national_signal_payload_2026["D-Held Loss Pressure"]
)
_national_signal_payload_2026["National Senate Flip Pressure"] = (
    _national_signal_payload_2026["R-Held Loss Pressure"]
    + _national_signal_payload_2026["D-Held Loss Pressure"]
)
national_signal_contract_2026 = MappingProxyType(
    _national_signal_payload_2026
)
national_vote_margin_2026 = national_signal_contract_2026[
    "National Vote Margin 2P"
]
d_loss_pressure_2026 = national_signal_contract_2026[
    "D-Held Loss Pressure"
]
r_loss_pressure_2026 = national_signal_contract_2026[
    "R-Held Loss Pressure"
]
national_senate_net_gain_2026 = national_signal_contract_2026[
    "National Senate Net Gain Signal"
]
national_senate_flip_pressure_2026 = national_signal_contract_2026[
    "National Senate Flip Pressure"
]

for frame in [senate_training, senate_parser]:
    frame["Incumbent D Indicator"] = (frame["INCUMBENT"] == "D").astype(float)

senate_training["Incumbent Vulnerability Signal"] = np.where(
    senate_training["INCUMBENT"] == "D",
    -senate_training["D-Held Loss Pressure"],
    senate_training["R-Held Loss Pressure"],
)

senate_parser["National Vote Margin 2P"] = national_vote_margin_2026
senate_parser["D-Held Loss Pressure"] = d_loss_pressure_2026
senate_parser["R-Held Loss Pressure"] = r_loss_pressure_2026
senate_parser["National Senate Net Gain Signal"] = national_senate_net_gain_2026
senate_parser["National Senate Flip Pressure"] = national_senate_flip_pressure_2026
senate_parser["Incumbent Vulnerability Signal"] = np.where(
    senate_parser["INCUMBENT"] == "D",
    -d_loss_pressure_2026,
    r_loss_pressure_2026,
)
senate_parser["Abs Poll Margin 2P"] = senate_parser["Poll Margin 2P"].abs()

# ------------------------------------------------------------
# 6. MODELO AGRUPADO DEL ERROR DE ENCUESTA
# ------------------------------------------------------------

import platform
import warnings

import scipy
import sklearn
from scipy.linalg import LinAlgWarning
from scipy.optimize import minimize_scalar
from sklearn.base import clone
from sklearn.metrics import brier_score_loss, log_loss

state_categorical_features = []
fundamentals_numeric_features = [
    "Expected Margin 2P",
    "Abs Poll Margin 2P",
    "Unallocated Poll Share",
    "National Vote Margin 2P",
    "D-Held Loss Pressure",
    "R-Held Loss Pressure",
    "National Senate Net Gain Signal",
    "National Senate Flip Pressure",
    "Incumbent Vulnerability Signal",
]

# The pooled specification deliberately omits state fixed effects. With only
# 34 state-years, nested validation preferred generalizable error patterns over
# memorizing a state's average historical miss.


# D+R bajo indica contiendas con terceros, write-ins o sistemas electorales menos
# equivalentes a una carrera D-R ordinaria. Se conservan, pero con influencia menor.
expected_coverage = np.clip(
    (senate_training["D EXPECTED"] + senate_training["R EXPECTED"]) / 100, 0, 1
)
result_coverage = np.clip(
    (senate_training["D RESULT"] + senate_training["R RESULT"]) / 100, 0, 1
)
senate_training["Contest Comparability Weight"] = np.clip(
    np.sqrt(expected_coverage * result_coverage), 0.35, 1.0
)
senate_training["Recency Weight"] = 0.84 ** (
    (senate_training["YEAR"].max() - senate_training["YEAR"]) / 4.0
)
senate_training["Training Weight"] = (
    senate_training["Contest Comparability Weight"] * senate_training["Recency Weight"]
)


def make_fundamentals_model(alpha):
    return Pipeline([
        ("scale", StandardScaler()),
        ("model", Ridge(alpha=float(alpha), solver="svd")),
    ])



def assert_finite_5b(name, values):
    values = np.asarray(values, dtype=float)
    if not np.isfinite(values).all():
        bad = int((~np.isfinite(values)).sum())
        raise FloatingPointError(f"{name} contiene {bad} valores NaN o infinitos.")


numerical_warning_log_5b = []


def safe_fundamentals_fit_predict(alpha, train_idx, test_idx, label):
    model = make_fundamentals_model(alpha)
    X_train_local = senate_training.loc[
        train_idx, state_categorical_features + fundamentals_numeric_features
    ]
    X_test_local = senate_training.loc[
        test_idx, state_categorical_features + fundamentals_numeric_features
    ]
    # Learn the comparable historical miss, not the final margin itself.
    y_train_local = senate_training.loc[
        train_idx, "Raw Margin Error PP"
    ].to_numpy(float)
    weights_local = local_training_weights_5b(train_idx)
    assert_finite_5b(f"{label} y", y_train_local)
    assert_finite_5b(f"{label} weights", weights_local)
    assert_finite_5b(
        f"{label} numeric inputs",
        X_train_local[fundamentals_numeric_features].to_numpy(float),
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        warnings.filterwarnings("error", category=RuntimeWarning)
        warnings.filterwarnings("error", category=LinAlgWarning)
        fitted = clone(model)
        fitted.fit(
            X_train_local,
            y_train_local,
            model__sample_weight=weights_local,
        )
        predicted_error = np.asarray(
            fitted.predict(X_test_local), dtype=float
        )
        poll_anchor = senate_training.loc[
            test_idx, "Expected Margin 2P"
        ].to_numpy(float)
        prediction = poll_anchor + predicted_error
    for item in caught:
        numerical_warning_log_5b.append({
            "Stage": label,
            "Category": item.category.__name__,
            "Message": str(item.message),
        })
    assert_finite_5b(f"{label} prediction", prediction)
    assert_finite_5b(
        f"{label} coefficients",
        np.asarray(fitted.named_steps["model"].coef_, dtype=float),
    )
    return fitted, prediction


X_fundamentals = senate_training[
    state_categorical_features + fundamentals_numeric_features
].copy()
y_margin_state = senate_training["Result Margin 2P"].to_numpy(float)
poll_margin_state = senate_training["Expected Margin 2P"].to_numpy(float)
groups_state = senate_training["YEAR"].to_numpy(int)
all_indices_5b = senate_training.index.to_numpy(int)

assert_finite_5b(
    "Fundamentals numeric matrix",
    X_fundamentals[fundamentals_numeric_features].to_numpy(float),
)
assert_finite_5b("Historical result margins", y_margin_state)
assert_finite_5b("Historical poll margins", poll_margin_state)

ridge_candidates = [0.25, 1.0, 4.0, 12.0, 36.0, 100.0]


def local_training_weights_5b(indices):
    """Recalcula recencia dentro de cada training fold."""
    indices = np.asarray(indices, dtype=int)
    local_years = senate_training.loc[indices, "YEAR"].to_numpy(int)
    local_max_year = int(local_years.max())
    local_recency = 0.84 ** ((local_max_year - local_years) / 4.0)
    local_comparability = senate_training.loc[
        indices, "Contest Comparability Weight"
    ].to_numpy(float)
    weights = local_comparability * local_recency
    assert_finite_5b("Local training weights", weights)
    return weights


def fundamentals_oof_for_indices(indices, alpha):
    """Corrected-margin OOF interno; no usa el ciclo pronosticado."""
    indices = np.asarray(indices, dtype=int)
    years_here = sorted(np.unique(groups_state[indices]))
    predictions = np.full(len(indices), np.nan)
    for test_year in years_here:
        local_test = indices[groups_state[indices] == int(test_year)]
        local_train = indices[groups_state[indices] != int(test_year)]
        _, fold_prediction = safe_fundamentals_fit_predict(
            alpha,
            local_train,
            local_test,
            f"Polling-error alpha={alpha} inner-year={test_year}",
        )
        positions = np.flatnonzero(np.isin(indices, local_test))
        predictions[positions] = fold_prediction
    assert_finite_5b(f"Fundamentals OOF alpha={alpha}", predictions)
    return predictions


def select_fundamentals_alpha(indices):
    indices = np.asarray(indices, dtype=int)
    actual = y_margin_state[indices]
    evaluation_weights = local_training_weights_5b(indices)
    rows = []
    prediction_cache = {}
    for alpha in ridge_candidates:
        predictions = fundamentals_oof_for_indices(indices, alpha)
        prediction_cache[float(alpha)] = predictions
        rows.append({
            "Alpha": float(alpha),
            "Weighted MAE PP": float(np.average(
                np.abs(actual - predictions), weights=evaluation_weights
            )),
            "Unweighted MAE PP": float(np.mean(np.abs(actual - predictions))),
        })
    table = pd.DataFrame(rows).sort_values(
        ["Weighted MAE PP", "Unweighted MAE PP", "Alpha"],
        kind="mergesort",
    ).reset_index(drop=True)
    selected_alpha = float(table.iloc[0]["Alpha"])
    return selected_alpha, table, prediction_cache[selected_alpha]


correction_weight_grid_5b = [
    0.0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.75, 1.0
]


def guarded_precision_weights_5b(actual, polls, fundamentals, weights):
    """Selects how much of the cross-fitted error correction to apply."""
    actual = np.asarray(actual, dtype=float)
    polls = np.asarray(polls, dtype=float)
    fundamentals = np.asarray(fundamentals, dtype=float)
    weights = np.asarray(weights, dtype=float)

    poll_errors = actual - polls
    candidate_errors = actual - fundamentals
    poll_variance = float(np.average(poll_errors ** 2, weights=weights))
    fundamentals_variance = float(np.average(
        candidate_errors ** 2, weights=weights
    ))
    if poll_variance <= 0 or fundamentals_variance <= 0:
        raise ValueError("Las varianzas del baseline/corrección no son válidas.")

    candidate_rows = []
    for correction_weight in correction_weight_grid_5b:
        poll_weight = 1.0 - correction_weight
        corrected_margin = (
            poll_weight * polls + correction_weight * fundamentals
        )
        candidate_rows.append({
            "poll_weight": float(poll_weight),
            "fundamentals_weight": float(correction_weight),
            "posterior": corrected_margin,
            "mae": float(np.average(
                np.abs(actual - corrected_margin), weights=weights
            )),
        })

    candidate_rows.sort(
        key=lambda row: (
            row["mae"],
            row["fundamentals_weight"],
        )
    )
    selected = candidate_rows[0]
    poll_mae = float(np.average(np.abs(actual - polls), weights=weights))
    if selected["fundamentals_weight"] > 0 and selected["mae"] < poll_mae - 1e-9:
        status = "Polling-error correction retained — inner LOEO improved MAE"
    else:
        selected = {
            "poll_weight": 1.0,
            "fundamentals_weight": 0.0,
            "posterior": polls.copy(),
            "mae": poll_mae,
        }
        status = "Poll baseline retained — error correction did not improve inner MAE"

    return {
        "poll_weight": selected["poll_weight"],
        "fundamentals_weight": selected["fundamentals_weight"],
        "precision_poll_weight": selected["poll_weight"],
        "precision_fundamentals_weight": selected["fundamentals_weight"],
        "poll_variance": poll_variance,
        "fundamentals_variance": fundamentals_variance,
        "poll_mae": poll_mae,
        "precision_posterior_mae": selected["mae"],
        "guarded_posterior_mae": selected["mae"],
        "status": status,
        "posterior": selected["posterior"],
    }



def choose_probability_scale_5b(margins, sigmas, actual_winners):
    margins = np.asarray(margins, dtype=float)
    sigmas = np.asarray(sigmas, dtype=float)
    actual_winners = np.asarray(actual_winners, dtype=int)

    def objective(scale):
        probability = np.clip(
            norm.cdf(margins / np.clip(sigmas * scale, 1e-6, None)),
            1e-6,
            1 - 1e-6,
        )
        return float(log_loss(
            actual_winners, probability, labels=[0, 1]
        ))

    result = minimize_scalar(
        objective, bounds=(0.60, 2.50), method="bounded"
    )
    return float(result.x) if result.success and np.isfinite(result.x) else 1.0


def estimate_uncertainty_5b(
    indices,
    posterior_margins,
    poll_margins,
    fundamentals_margins,
    poll_weight,
    fundamentals_weight,
):
    """Aprende shocks, dispersión estatal y calibración sin datos exteriores."""
    indices = np.asarray(indices, dtype=int)
    posterior_margins = np.asarray(posterior_margins, dtype=float)
    poll_margins = np.asarray(poll_margins, dtype=float)
    fundamentals_margins = np.asarray(fundamentals_margins, dtype=float)
    actual = y_margin_state[indices]
    years = groups_state[indices]
    states = senate_training.loc[indices, "STATE"].to_numpy(str)

    residuals = actual - posterior_margins
    cycle_means = pd.Series(residuals).groupby(years).mean()
    common_sigma = float(cycle_means.std(ddof=1))
    if not np.isfinite(common_sigma) or common_sigma < 1.0:
        common_sigma = 1.0

    cycle_map = cycle_means.to_dict()
    idiosyncratic = np.array([
        residual - cycle_map[int(year)]
        for residual, year in zip(residuals, years)
    ], dtype=float)
    global_idio_variance = float(np.var(idiosyncratic, ddof=1))
    if not np.isfinite(global_idio_variance) or global_idio_variance <= 0:
        global_idio_variance = 9.0

    state_sigma_map = {}
    prior_strength = 3.0
    for state in np.unique(states):
        local = idiosyncratic[states == state]
        local_variance = (
            float(np.var(local, ddof=1))
            if len(local) > 1 else global_idio_variance
        )
        numerator = (
            max(len(local) - 1, 0) * local_variance
            + prior_strength * global_idio_variance
        )
        denominator = max(len(local) - 1, 0) + prior_strength
        state_sigma_map[state] = float(
            np.sqrt(max(numerator / denominator, 1.0))
        )

    fallback_state_sigma = float(
        np.sqrt(max(global_idio_variance, 1.0))
    )
    state_sigmas = np.array([
        state_sigma_map.get(state, fallback_state_sigma)
        for state in states
    ], dtype=float)
    disagreement = (
        np.sqrt(poll_weight * fundamentals_weight)
        * np.abs(poll_margins - fundamentals_margins)
    )
    raw_sigmas = np.sqrt(
        common_sigma ** 2 + state_sigmas ** 2 + disagreement ** 2
    )
    actual_winners = (actual > 0).astype(int)
    logloss_scale = choose_probability_scale_5b(
        posterior_margins, raw_sigmas, actual_winners
    )
    standardized_error = (
        np.abs(actual - posterior_margins)
        / np.clip(raw_sigmas, 1e-6, None)
    )
    coverage_scale = float(
        np.quantile(
            standardized_error, 0.90, method="higher"
        ) / norm.ppf(0.95)
    )
    probability_scale = max(logloss_scale, coverage_scale, 0.60)

    poll_raw_sigmas = np.sqrt(common_sigma ** 2 + state_sigmas ** 2)
    poll_logloss_scale = choose_probability_scale_5b(
        poll_margins, poll_raw_sigmas, actual_winners
    )
    return {
        "residuals": residuals,
        "cycle_means": cycle_means,
        "idiosyncratic": idiosyncratic,
        "common_sigma": common_sigma,
        "global_idio_variance": global_idio_variance,
        "state_sigma_map": state_sigma_map,
        "fallback_state_sigma": fallback_state_sigma,
        "raw_sigmas": raw_sigmas,
        "probability_scale": float(probability_scale),
        "logloss_scale": float(logloss_scale),
        "coverage_scale": float(coverage_scale),
        "poll_logloss_scale": float(poll_logloss_scale),
    }


# ------------------------------------------------------------
# 7. VALIDACIÓN EXTERIOR: ALPHA, TAMAÑO DE CORRECCIÓN Y CALIBRACIÓN
# ------------------------------------------------------------

fundamentals_oof_5b = np.full(len(senate_training), np.nan)
posterior_oof_margins_5b = np.full(len(senate_training), np.nan)
posterior_oof_disagreement_5b = np.full(len(senate_training), np.nan)
raw_oof_sigmas_5b = np.full(len(senate_training), np.nan)
calibrated_oof_sigmas_5b = np.full(len(senate_training), np.nan)
oof_probabilities_5b = np.full(len(senate_training), np.nan)
poll_probabilities_oof_5b = np.full(len(senate_training), np.nan)

outer_fold_records_5b = []
for outer_year in sorted(np.unique(groups_state)):
    outer_test_idx = all_indices_5b[groups_state == int(outer_year)]
    outer_train_idx = all_indices_5b[groups_state != int(outer_year)]

    selected_alpha, inner_alpha_table, inner_fundamentals = (
        select_fundamentals_alpha(outer_train_idx)
    )
    training_weights = local_training_weights_5b(outer_train_idx)
    weight_fit = guarded_precision_weights_5b(
        y_margin_state[outer_train_idx],
        poll_margin_state[outer_train_idx],
        inner_fundamentals,
        training_weights,
    )
    inner_posterior = weight_fit["posterior"]
    uncertainty_fit = estimate_uncertainty_5b(
        outer_train_idx,
        inner_posterior,
        poll_margin_state[outer_train_idx],
        inner_fundamentals,
        weight_fit["poll_weight"],
        weight_fit["fundamentals_weight"],
    )

    _, outer_fundamentals = safe_fundamentals_fit_predict(
        selected_alpha,
        outer_train_idx,
        outer_test_idx,
        f"Polling-error outer-year={outer_year}",
    )
    fundamentals_oof_5b[outer_test_idx] = outer_fundamentals
    outer_posterior = (
        weight_fit["poll_weight"] * poll_margin_state[outer_test_idx]
        + weight_fit["fundamentals_weight"] * outer_fundamentals
    )
    posterior_oof_margins_5b[outer_test_idx] = outer_posterior

    outer_states = senate_training.loc[
        outer_test_idx, "STATE"
    ].to_numpy(str)
    outer_state_sigmas = np.array([
        uncertainty_fit["state_sigma_map"].get(
            state, uncertainty_fit["fallback_state_sigma"]
        )
        for state in outer_states
    ], dtype=float)
    outer_disagreement = (
        np.sqrt(
            weight_fit["poll_weight"]
            * weight_fit["fundamentals_weight"]
        )
        * np.abs(
            poll_margin_state[outer_test_idx] - outer_fundamentals
        )
    )
    posterior_oof_disagreement_5b[outer_test_idx] = outer_disagreement
    outer_raw_sigma = np.sqrt(
        uncertainty_fit["common_sigma"] ** 2
        + outer_state_sigmas ** 2
        + outer_disagreement ** 2
    )
    outer_calibrated_sigma = (
        outer_raw_sigma * uncertainty_fit["probability_scale"]
    )
    raw_oof_sigmas_5b[outer_test_idx] = outer_raw_sigma
    calibrated_oof_sigmas_5b[outer_test_idx] = outer_calibrated_sigma
    oof_probabilities_5b[outer_test_idx] = np.clip(
        norm.cdf(outer_posterior / outer_calibrated_sigma),
        0.001,
        0.999,
    )

    outer_poll_raw_sigma = np.sqrt(
        uncertainty_fit["common_sigma"] ** 2
        + outer_state_sigmas ** 2
    )
    poll_probabilities_oof_5b[outer_test_idx] = np.clip(
        norm.cdf(
            poll_margin_state[outer_test_idx]
            / (
                outer_poll_raw_sigma
                * uncertainty_fit["poll_logloss_scale"]
            )
        ),
        0.001,
        0.999,
    )

    outer_fold_records_5b.append({
        "Outer Test Year": int(outer_year),
        "Selected Polling-Error Ridge Alpha": selected_alpha,
        "Inner Full-Correction Weighted MAE PP": float(
            inner_alpha_table.iloc[0]["Weighted MAE PP"]
        ),
        "Inner Poll MAE PP": weight_fit["poll_mae"],
        "Inner Selected-Correction MAE PP": (
            weight_fit["precision_posterior_mae"]
        ),
        "Applied Poll-Only Weight": weight_fit["poll_weight"],
        "Applied Error-Correction Candidate Weight": weight_fit["fundamentals_weight"],
        "Baseline Safeguard Status": weight_fit["status"],
        "Outer Poll MAE PP": float(np.mean(np.abs(
            y_margin_state[outer_test_idx]
            - poll_margin_state[outer_test_idx]
        ))),
        "Outer Full-Correction Candidate MAE PP": float(np.mean(np.abs(
            y_margin_state[outer_test_idx]
            - fundamentals_oof_5b[outer_test_idx]
        ))),
        "Outer Validated Error-Corrected MAE PP": float(np.mean(np.abs(
            y_margin_state[outer_test_idx]
            - posterior_oof_margins_5b[outer_test_idx]
        ))),
        "Outer Probability Scale": uncertainty_fit["probability_scale"],
        "Outer Poll Probability Scale": (
            uncertainty_fit["poll_logloss_scale"]
        ),
    })

for name, values in {
    "Nested fundamentals OOF": fundamentals_oof_5b,
    "Nested posterior OOF": posterior_oof_margins_5b,
    "Nested posterior sigmas": calibrated_oof_sigmas_5b,
    "Nested posterior probabilities": oof_probabilities_5b,
    "Nested poll probabilities": poll_probabilities_oof_5b,
}.items():
    assert_finite_5b(name, values)

senate_nested_fold_diagnostics = pd.DataFrame(outer_fold_records_5b)

# ------------------------------------------------------------
# 8. AJUSTE DE PRODUCCIÓN 2026 CON TODO EL HISTÓRICO
# ------------------------------------------------------------

RIDGE_ALPHA_5B, ridge_cv, final_fundamentals_inner_oof_5b = (
    select_fundamentals_alpha(all_indices_5b)
)
final_weight_fit_5b = guarded_precision_weights_5b(
    y_margin_state,
    poll_margin_state,
    final_fundamentals_inner_oof_5b,
    local_training_weights_5b(all_indices_5b),
)
POLL_WEIGHT_5B = final_weight_fit_5b["poll_weight"]
FUNDAMENTALS_WEIGHT_5B = final_weight_fit_5b["fundamentals_weight"]
poll_error_variance_5b = final_weight_fit_5b["poll_variance"]
fundamentals_error_variance_5b = (
    final_weight_fit_5b["fundamentals_variance"]
)
BASELINE_SAFEGUARD_STATUS_5B = final_weight_fit_5b["status"]

final_training_posterior_5b = final_weight_fit_5b["posterior"]
production_uncertainty_5b = estimate_uncertainty_5b(
    all_indices_5b,
    final_training_posterior_5b,
    poll_margin_state,
    final_fundamentals_inner_oof_5b,
    POLL_WEIGHT_5B,
    FUNDAMENTALS_WEIGHT_5B,
)

final_fundamentals_model, _ = safe_fundamentals_fit_predict(
    RIDGE_ALPHA_5B,
    all_indices_5b,
    all_indices_5b[:1],
    "Polling-error final fit check",
)
current_fundamentals_features = senate_parser[
    state_categorical_features + fundamentals_numeric_features
].copy()
with warnings.catch_warnings():
    warnings.simplefilter("error", RuntimeWarning)
    raw_predicted_polling_error_2026 = np.asarray(
        final_fundamentals_model.predict(
            current_fundamentals_features
        ),
        dtype=float,
    )
    fundamentals_margin_2026 = (
        senate_parser["Poll Margin 2P"].to_numpy(float)
        + raw_predicted_polling_error_2026
    )
assert_finite_5b("2026 fundamentals margins", fundamentals_margin_2026)

senate_parser["Raw Predicted Polling Error PP"] = (
    raw_predicted_polling_error_2026
)
senate_parser["Error-Corrected Candidate Margin 2P"] = (
    fundamentals_margin_2026
)
# Compatibility alias retained for downstream workbook readers.
senate_parser["Fundamentals Margin 2P"] = fundamentals_margin_2026
senate_parser["Poll Weight"] = POLL_WEIGHT_5B
senate_parser["Fundamentals Weight"] = FUNDAMENTALS_WEIGHT_5B
senate_parser["Poll-Only Candidate Weight"] = POLL_WEIGHT_5B
senate_parser["Error-Corrected Candidate Weight"] = FUNDAMENTALS_WEIGHT_5B
senate_parser["Model Projected Margin 2P"] = (
    POLL_WEIGHT_5B * senate_parser["Poll Margin 2P"]
    + FUNDAMENTALS_WEIGHT_5B * senate_parser["Fundamentals Margin 2P"]
)
senate_parser["Model Polling Error Correction PP"] = (
    senate_parser["Model Projected Margin 2P"]
    - senate_parser["Poll Margin 2P"]
)
senate_parser["Model Disagreement PP"] = (
    np.sqrt(POLL_WEIGHT_5B * FUNDAMENTALS_WEIGHT_5B)
    * np.abs(
        senate_parser["Poll Margin 2P"]
        - senate_parser["Fundamentals Margin 2P"]
    )
)
senate_parser["Baseline Safeguard Status"] = (
    BASELINE_SAFEGUARD_STATUS_5B
)

# Los objetos OOF que siguen son genuinamente exteriores: cada fila fue
# construida con alpha, pesos y calibración aprendidos sin su ciclo.
posterior_oof_residuals_5b = (
    y_margin_state - posterior_oof_margins_5b
)
cycle_residual_means_5b = pd.Series(
    posterior_oof_residuals_5b
).groupby(groups_state).mean()
cycle_residual_map_5b = cycle_residual_means_5b.to_dict()
idiosyncratic_residuals_5b = np.array([
    residual - cycle_residual_map_5b[int(year)]
    for residual, year in zip(
        posterior_oof_residuals_5b, groups_state
    )
], dtype=float)

raw_common_sigma_5b = production_uncertainty_5b["common_sigma"]
state_sigma_map_5b = production_uncertainty_5b["state_sigma_map"]
fallback_state_sigma_5b = (
    production_uncertainty_5b["fallback_state_sigma"]
)
PROBABILITY_SCALE_5B = production_uncertainty_5b["probability_scale"]
logloss_scale_5b = production_uncertainty_5b["logloss_scale"]
coverage_scale_90_5b = production_uncertainty_5b["coverage_scale"]

current_state_base_sigma_5b = np.array([
    state_sigma_map_5b.get(state, fallback_state_sigma_5b)
    for state in senate_parser["STATE"]
], dtype=float)
senate_common_sigma_pp = raw_common_sigma_5b * PROBABILITY_SCALE_5B
senate_parser["State Idiosyncratic Sigma PP"] = (
    current_state_base_sigma_5b
)
senate_parser["State Simulation Sigma PP"] = (
    np.sqrt(
        current_state_base_sigma_5b ** 2
        + senate_parser["Model Disagreement PP"] ** 2
    ) * PROBABILITY_SCALE_5B
)
senate_parser["Forecast Sigma PP"] = np.sqrt(
    senate_common_sigma_pp ** 2
    + senate_parser["State Simulation Sigma PP"] ** 2
)
senate_parser["Probability Calibration Scale"] = PROBABILITY_SCALE_5B

senate_parser["Adjusted Margin 2P"] = (
    senate_parser["Model Projected Margin 2P"]
)
senate_parser["D Win Probability Decimal"] = np.clip(
    norm.cdf(
        senate_parser["Adjusted Margin 2P"]
        / senate_parser["Forecast Sigma PP"]
    ),
    0.001,
    0.999,
)
senate_parser["R Win Probability Decimal"] = (
    1 - senate_parser["D Win Probability Decimal"]
)
senate_parser["Projected Winner"] = np.where(
    senate_parser["Adjusted Margin 2P"] >= 0, "D", "R"
)
senate_parser["Model Assigned Outcome"] = senate_parser.apply(
    lambda row: outcome_label(
        row["INCUMBENT"], row["Projected Winner"]
    ),
    axis=1,
)
senate_parser["Projected D 2P"] = np.clip(
    50 + senate_parser["Adjusted Margin 2P"] / 2, 0, 100
)
senate_parser["Projected R 2P"] = (
    100 - senate_parser["Projected D 2P"]
)
senate_parser["Forecast Rating"] = (
    senate_parser["Adjusted Margin 2P"].apply(party_margin_rating)
)
senate_parser["Polling Leader"] = np.where(
    senate_parser["MARGIN"] >= 0, "D", "R"
)
senate_parser["Adjusted Leader"] = senate_parser["Projected Winner"]
senate_parser["Polling Outcome"] = senate_parser.apply(
    lambda row: outcome_label(
        row["INCUMBENT"], row["Polling Leader"]
    ),
    axis=1,
)
senate_parser["Vulnerability Score"] = np.where(
    senate_parser["INCUMBENT"] == "D",
    100 * senate_parser["R Win Probability Decimal"],
    100 * senate_parser["D Win Probability Decimal"],
)

actual_d_wins_5b = (y_margin_state > 0).astype(int)
posterior_brier_5b = float(
    brier_score_loss(actual_d_wins_5b, oof_probabilities_5b)
)
poll_brier_5b = float(
    brier_score_loss(
        actual_d_wins_5b, poll_probabilities_oof_5b
    )
)
posterior_log_loss_5b = float(log_loss(
    actual_d_wins_5b, oof_probabilities_5b, labels=[0, 1]
))
poll_log_loss_5b = float(log_loss(
    actual_d_wins_5b, poll_probabilities_oof_5b, labels=[0, 1]
))
coverage_80_5b = float(np.mean(
    np.abs(y_margin_state - posterior_oof_margins_5b)
    <= norm.ppf(0.90) * calibrated_oof_sigmas_5b
))
coverage_90_5b = float(np.mean(
    np.abs(y_margin_state - posterior_oof_margins_5b)
    <= norm.ppf(0.95) * calibrated_oof_sigmas_5b
))

# ------------------------------------------------------------
# 9. AUDITORÍA HISTÓRICA, VALIDACIÓN Y ATRIBUCIÓN POR CARRERA
# ------------------------------------------------------------

senate_training["Fundamentals OOF Margin 2P"] = fundamentals_oof_5b
senate_training["Posterior OOF Margin 2P"] = posterior_oof_margins_5b
senate_training["Posterior OOF Residual PP"] = posterior_oof_residuals_5b
senate_training["Posterior OOF D Win Probability"] = oof_probabilities_5b
senate_training["Posterior OOF Forecast Sigma PP"] = calibrated_oof_sigmas_5b
senate_training["OOF Cycle Residual PP"] = senate_training["YEAR"].map(
    cycle_residual_means_5b
)
senate_training["OOF Idiosyncratic Residual PP"] = idiosyncratic_residuals_5b

cv_records_5b = []
for row_index, row in senate_training.iterrows():
    common = {
        "Row Index": int(row_index),
        "STATE": row["STATE"],
        "YEAR": int(row["YEAR"]),
        "Actual Margin 2P": float(row["Result Margin 2P"]),
        "Poll Margin 2P": float(row["Expected Margin 2P"]),
    }
    for model_name, prediction in [
        ("PollBaseline", row["Expected Margin 2P"]),
        ("PollingErrorCandidate", row["Fundamentals OOF Margin 2P"]),
        ("ValidatedErrorCorrection", row["Posterior OOF Margin 2P"]),
    ]:
        record = {
            **common,
            "Model": model_name,
            "Predicted Margin 2P": float(prediction),
            "Absolute Error PP": abs(float(row["Result Margin 2P"] - prediction)),
        }
        if model_name == "ValidatedErrorCorrection":
            record["D Win Probability"] = float(row["Posterior OOF D Win Probability"])
            record["Forecast Sigma PP"] = float(row["Posterior OOF Forecast Sigma PP"])
        cv_records_5b.append(record)

senate_state_model_cv = pd.DataFrame(cv_records_5b)
state_model_diagnostics = (
    senate_state_model_cv.groupby("Model")["Absolute Error PP"]
    .mean().rename("Nested LOEO MAE PP").reset_index()
)
weight_map_5b = {
    "PollBaseline": POLL_WEIGHT_5B,
    "PollingErrorCandidate": FUNDAMENTALS_WEIGHT_5B,
    "ValidatedErrorCorrection": 1.0,
}
state_model_diagnostics["Final Ensemble Weight"] = state_model_diagnostics["Model"].map(
    weight_map_5b
)
state_model_diagnostics["Selected Polling-Error Ridge Alpha"] = np.where(
    state_model_diagnostics["Model"] == "PollingErrorCandidate", RIDGE_ALPHA_5B, np.nan
)
state_model_diagnostics["Target"] = np.where(
    state_model_diagnostics["Model"] == "PollBaseline",
    "Observed poll margin",
    np.where(
        state_model_diagnostics["Model"] == "PollingErrorCandidate",
        "Poll anchor plus fully predicted polling error",
        "Validated poll-anchor plus error correction",
    ),
)

cycle_validation_rows_5b = []
for year in sorted(np.unique(groups_state)):
    mask = groups_state == year
    cycle_validation_rows_5b.append({
        "YEAR": int(year),
        "Poll MAE PP": float(np.mean(np.abs(
            y_margin_state[mask] - poll_margin_state[mask]
        ))),
        "Full Error-Correction Candidate MAE PP": float(np.mean(np.abs(
            y_margin_state[mask] - fundamentals_oof_5b[mask]
        ))),
        "Validated Error-Corrected MAE PP": float(np.mean(np.abs(
            y_margin_state[mask] - posterior_oof_margins_5b[mask]
        ))),
    })
senate_cycle_validation = pd.DataFrame(cycle_validation_rows_5b)
senate_cycle_validation["Validated Correction Better Than Poll"] = (
    senate_cycle_validation["Validated Error-Corrected MAE PP"]
    < senate_cycle_validation["Poll MAE PP"] - 1e-9
)

global_historic_bias_5b = weighted_mean(
    senate_historic["Raw Margin Error PP"],
    senate_training["Training Weight"],
)
historic_summary_rows = []
for state, history in senate_training.groupby("STATE"):
    weights = history["Training Weight"].to_numpy(float)
    errors = history["Raw Margin Error PP"].to_numpy(float)
    local_bias = weighted_mean(errors, weights)
    effective_n = float(weights.sum())
    shrinkage = effective_n / (effective_n + 3.0)
    shrunk_bias = global_historic_bias_5b + shrinkage * (
        local_bias - global_historic_bias_5b
    )
    historic_summary_rows.append({
        "STATE": state,
        "Historic Elections": len(history),
        "Effective Comparable Elections": effective_n,
        "Shrunk Historic Bias PP": shrunk_bias,
        "Historic MAE PP": weighted_mean(np.abs(errors), weights),
        "Historic Flip Rate": 100 * weighted_mean(history["Flip"].astype(float), weights),
        "Historic Polling Miss Rate": 100 * weighted_mean(
            history["Polling Miss"].astype(float), weights
        ),
        "Mean Comparability Weight": float(history["Contest Comparability Weight"].mean()),
        "First Historic Year": int(history["YEAR"].min()),
        "Last Historic Year": int(history["YEAR"].max()),
    })
senate_historic_summary = pd.DataFrame(historic_summary_rows)
senate_parser = senate_parser.merge(senate_historic_summary, on="STATE", how="left")

parsed_d_incumbents = int((senate_parser["INCUMBENT"] == "D").sum())
SENATE_FIXED_D_SEATS_5B = DS_BEFORE_5B - parsed_d_incumbents
modeled_d_wins = int((senate_parser["Projected Winner"] == "D").sum())
D_SENATE_CENTRAL_5B = SENATE_FIXED_D_SEATS_5B + modeled_d_wins
R_SENATE_CENTRAL_5B = 100 - D_SENATE_CENTRAL_5B
D_SENATE_LOST_CENTRAL_5B = int((
    (senate_parser["INCUMBENT"] == "D")
    & (senate_parser["Projected Winner"] == "R")
).sum())
R_SENATE_LOST_CENTRAL_5B = int((
    (senate_parser["INCUMBENT"] == "R")
    & (senate_parser["Projected Winner"] == "D")
).sum())

probability_winner = np.where(senate_parser["D Win Probability Decimal"] >= 0.50, "D", "R")
margin_winner = np.where(senate_parser["Adjusted Margin 2P"] >= 0, "D", "R")
if not np.array_equal(probability_winner, senate_parser["Projected Winner"]):
    raise AssertionError("Probabilidad y ganador senatorial no coinciden.")
if not np.array_equal(margin_winner, senate_parser["Projected Winner"]):
    raise AssertionError("Margen y ganador senatorial no coinciden.")
if D_SENATE_CENTRAL_5B != DS_BEFORE_5B - D_SENATE_LOST_CENTRAL_5B + R_SENATE_LOST_CENTRAL_5B:
    raise AssertionError("Los flips no reconstruyen la composición central del Senado.")
if not np.isclose(POLL_WEIGHT_5B + FUNDAMENTALS_WEIGHT_5B, 1.0):
    raise AssertionError("Los pesos del posterior no suman 1.")
if FUNDAMENTALS_WEIGHT_5B == 0:
    if not np.allclose(
        senate_parser["Model Projected Margin 2P"],
        senate_parser["Poll Margin 2P"],
    ):
        raise AssertionError("El guardrail seleccionó polls, pero el margen fue alterado.")
else:
    if np.allclose(
        senate_parser["Model Projected Margin 2P"],
        senate_parser["Poll Margin 2P"],
    ):
        raise AssertionError("Fundamentals recibió peso positivo sin modificar el margen.")

senate_parser["D Win Probability"] = 100 * senate_parser["D Win Probability Decimal"]
senate_parser["R Win Probability"] = 100 * senate_parser["R Win Probability Decimal"]


# ------------------------------------------------------------
# 9A. SPECIFICATION CHALLENGE AND FIVE-MODEL 2026 SENATE STABILITY
# ------------------------------------------------------------
#
# Separate state-by-state regressions would have only three or four elections
# per state. PartialPoolFull provides the statistically safer version of that
# idea: state intercepts are regularized inside a pooled model. Every candidate
# faces nested leave-one-election-year-out selection.

senate_specifications_5b = {
    "PooledPollStructure": {
        "numeric": [
            "Expected Margin 2P", "Abs Poll Margin 2P",
            "Unallocated Poll Share", "Incumbent D Indicator",
        ],
        "categorical": [],
        "complexity": 0,
        "role": "Poll structure and incumbency only",
    },
    "PooledPopularContext": {
        "numeric": [
            "Expected Margin 2P", "Abs Poll Margin 2P",
            "Unallocated Poll Share", "Incumbent D Indicator",
            "National Vote Margin 2P",
        ],
        "categorical": [],
        "complexity": 1,
        "role": "Adds frozen national popular-vote context",
    },
    "PooledNationalReduced": {
        "numeric": [
            "Expected Margin 2P", "Abs Poll Margin 2P",
            "Unallocated Poll Share", "Incumbent D Indicator",
            "National Vote Margin 2P",
            "National Senate Net Gain Signal",
            "National Senate Flip Pressure",
            "Incumbent Vulnerability Signal",
        ],
        "categorical": [],
        "complexity": 2,
        "role": "Adds non-quota national Senate context",
    },
    "PooledFullV11": {
        "numeric": list(fundamentals_numeric_features),
        "categorical": [],
        "complexity": 3,
        "role": "Exact pooled production specification inherited from v12",
    },
    "PartialPoolFull": {
        "numeric": list(fundamentals_numeric_features),
        "categorical": ["STATE"],
        "complexity": 4,
        "role": "Regularized state effects; safer alternative to 11 tiny models",
    },
}


def make_senate_specification_model_5b(specification_name, alpha):
    specification = senate_specifications_5b[specification_name]
    numeric = specification["numeric"]
    categorical = specification["categorical"]
    if categorical:
        preprocessor = ColumnTransformer([
            (
                "state",
                OneHotEncoder(handle_unknown="ignore"),
                categorical,
            ),
            ("numeric", StandardScaler(), numeric),
        ])
        return Pipeline([
            ("prep", preprocessor),
            ("model", Ridge(alpha=float(alpha), solver="lsqr")),
        ])
    return Pipeline([
        ("scale", StandardScaler()),
        ("model", Ridge(alpha=float(alpha), solver="svd")),
    ])


def fit_predict_senate_specification_5b(
    specification_name,
    alpha,
    train_indices,
    prediction_frame,
):
    specification = senate_specifications_5b[specification_name]
    feature_columns = (
        specification["categorical"] + specification["numeric"]
    )
    train_indices = np.asarray(train_indices, dtype=int)
    model = make_senate_specification_model_5b(
        specification_name, alpha
    )
    model.fit(
        senate_training.loc[train_indices, feature_columns],
        senate_training.loc[
            train_indices, "Raw Margin Error PP"
        ].to_numpy(float),
        model__sample_weight=local_training_weights_5b(train_indices),
    )
    predicted_error = np.asarray(
        model.predict(prediction_frame[feature_columns]),
        dtype=float,
    )
    poll_anchor = prediction_frame["Expected Margin 2P"].to_numpy(float)
    corrected_candidate = poll_anchor + predicted_error
    assert_finite_5b(
        f"{specification_name} corrected candidate",
        corrected_candidate,
    )
    return model, corrected_candidate


def senate_specification_inner_oof_5b(
    indices,
    specification_name,
    alpha,
):
    indices = np.asarray(indices, dtype=int)
    predictions = np.full(len(indices), np.nan)
    for test_year in sorted(np.unique(groups_state[indices])):
        test_indices = indices[groups_state[indices] == int(test_year)]
        train_indices = indices[groups_state[indices] != int(test_year)]
        _, fold_prediction = fit_predict_senate_specification_5b(
            specification_name,
            alpha,
            train_indices,
            senate_training.loc[test_indices],
        )
        positions = np.flatnonzero(np.isin(indices, test_indices))
        predictions[positions] = fold_prediction
    assert_finite_5b(
        f"{specification_name} alpha={alpha} inner OOF",
        predictions,
    )
    return predictions


def select_senate_specification_5b(indices):
    indices = np.asarray(indices, dtype=int)
    evaluation_weights = local_training_weights_5b(indices)
    candidate_rows = []
    prediction_cache = {}
    for specification_name, specification in senate_specifications_5b.items():
        for alpha in ridge_candidates:
            predictions = senate_specification_inner_oof_5b(
                indices,
                specification_name,
                alpha,
            )
            mae = float(np.average(
                np.abs(y_margin_state[indices] - predictions),
                weights=evaluation_weights,
            ))
            candidate_rows.append({
                "Specification": specification_name,
                "Alpha": float(alpha),
                "Inner Full-Correction Weighted MAE PP": mae,
                "Complexity": int(specification["complexity"]),
            })
            prediction_cache[(specification_name, float(alpha))] = predictions

    candidate_table = pd.DataFrame(candidate_rows).sort_values(
        [
            "Inner Full-Correction Weighted MAE PP",
            "Complexity",
            "Alpha",
            "Specification",
        ],
        kind="mergesort",
    ).reset_index(drop=True)
    best_mae = float(
        candidate_table["Inner Full-Correction Weighted MAE PP"].min()
    )
    tolerance = max(
        best_mae * NATIONAL_COMPLEXITY_TOLERANCE,
        1e-9,
    )
    eligible = candidate_table[
        candidate_table["Inner Full-Correction Weighted MAE PP"]
        <= best_mae + tolerance
    ].sort_values(
        ["Complexity", "Inner Full-Correction Weighted MAE PP", "Alpha"],
        kind="mergesort",
    )
    winner = eligible.iloc[0]
    selected_specification = str(winner["Specification"])
    selected_alpha = float(winner["Alpha"])
    full_correction_oof = prediction_cache[
        (selected_specification, selected_alpha)
    ]
    weight_fit = guarded_precision_weights_5b(
        y_margin_state[indices],
        poll_margin_state[indices],
        full_correction_oof,
        evaluation_weights,
    )
    return {
        "specification": selected_specification,
        "alpha": selected_alpha,
        "correction_weight": float(weight_fit["fundamentals_weight"]),
        "poll_weight": float(weight_fit["poll_weight"]),
        "candidate_table": candidate_table,
        "inner_selected_mae": float(weight_fit["precision_posterior_mae"]),
        "inner_poll_mae": float(weight_fit["poll_mae"]),
    }


specification_outer_records_5b = []
specification_outer_predictions_5b = {}
fixed_specification_scores_5b = []

for specification_name in senate_specifications_5b:
    fixed_predictions = np.full(len(senate_training), np.nan)
    fixed_fold_rows = []
    for outer_year in sorted(np.unique(groups_state)):
        outer_test_idx = all_indices_5b[groups_state == int(outer_year)]
        outer_train_idx = all_indices_5b[groups_state != int(outer_year)]
        evaluation_weights = local_training_weights_5b(outer_train_idx)
        alpha_rows = []
        alpha_cache = {}
        for alpha in ridge_candidates:
            inner_predictions = senate_specification_inner_oof_5b(
                outer_train_idx,
                specification_name,
                alpha,
            )
            score = float(np.average(
                np.abs(
                    y_margin_state[outer_train_idx] - inner_predictions
                ),
                weights=evaluation_weights,
            ))
            alpha_rows.append((score, float(alpha)))
            alpha_cache[float(alpha)] = inner_predictions
        alpha_rows.sort(key=lambda item: (item[0], item[1]))
        selected_alpha = alpha_rows[0][1]
        selected_inner = alpha_cache[selected_alpha]
        weight_fit = guarded_precision_weights_5b(
            y_margin_state[outer_train_idx],
            poll_margin_state[outer_train_idx],
            selected_inner,
            evaluation_weights,
        )
        _, candidate = fit_predict_senate_specification_5b(
            specification_name,
            selected_alpha,
            outer_train_idx,
            senate_training.loc[outer_test_idx],
        )
        prediction = (
            weight_fit["poll_weight"] * poll_margin_state[outer_test_idx]
            + weight_fit["fundamentals_weight"] * candidate
        )
        fixed_predictions[outer_test_idx] = prediction
        fixed_fold_rows.append({
            "Outer Test Year": int(outer_year),
            "Specification": specification_name,
            "Selected Alpha": selected_alpha,
            "Applied Correction Weight": float(
                weight_fit["fundamentals_weight"]
            ),
            "Outer MAE PP": float(np.mean(np.abs(
                y_margin_state[outer_test_idx] - prediction
            ))),
        })
    specification_outer_predictions_5b[specification_name] = fixed_predictions
    fixed_specification_scores_5b.append({
        "Specification": specification_name,
        "Architecture": senate_specifications_5b[
            specification_name
        ]["role"],
        "Nested LOEO MAE PP": float(np.mean(np.abs(
            y_margin_state - fixed_predictions
        ))),
        "Cycles Better Than Poll": int(sum(
            row["Outer MAE PP"]
            < float(senate_cycle_validation.loc[
                senate_cycle_validation["YEAR"].eq(
                    row["Outer Test Year"]
                ),
                "Poll MAE PP",
            ].iloc[0]) - 1e-9
            for row in fixed_fold_rows
        )),
        "Uses State Effects": bool(
            senate_specifications_5b[
                specification_name
            ]["categorical"]
        ),
    })

dynamic_specification_oof_5b = np.full(len(senate_training), np.nan)
dynamic_selection_by_year_5b = {}
for outer_year in sorted(np.unique(groups_state)):
    outer_test_idx = all_indices_5b[groups_state == int(outer_year)]
    outer_train_idx = all_indices_5b[groups_state != int(outer_year)]
    selection = select_senate_specification_5b(outer_train_idx)
    dynamic_selection_by_year_5b[int(outer_year)] = selection
    _, candidate = fit_predict_senate_specification_5b(
        selection["specification"],
        selection["alpha"],
        outer_train_idx,
        senate_training.loc[outer_test_idx],
    )
    prediction = (
        selection["poll_weight"] * poll_margin_state[outer_test_idx]
        + selection["correction_weight"] * candidate
    )
    dynamic_specification_oof_5b[outer_test_idx] = prediction
    specification_outer_records_5b.append({
        "Outer Test Year": int(outer_year),
        "Selected Specification": selection["specification"],
        "Selected Alpha": selection["alpha"],
        "Applied Poll Weight": selection["poll_weight"],
        "Applied Correction Weight": selection["correction_weight"],
        "Inner Poll MAE PP": selection["inner_poll_mae"],
        "Inner Selected MAE PP": selection["inner_selected_mae"],
        "Outer Poll MAE PP": float(np.mean(np.abs(
            y_margin_state[outer_test_idx]
            - poll_margin_state[outer_test_idx]
        ))),
        "Outer Selected MAE PP": float(np.mean(np.abs(
            y_margin_state[outer_test_idx] - prediction
        ))),
    })

senate_specification_nested_folds = pd.DataFrame(
    specification_outer_records_5b
)
senate_specification_summary = pd.DataFrame(
    fixed_specification_scores_5b
)
senate_specification_summary = pd.concat([
    senate_specification_summary,
    pd.DataFrame([{
        "Specification": "NestedSpecificationSelector",
        "Architecture": (
            "Specification, alpha and correction size selected inside "
            "each outer fold"
        ),
        "Nested LOEO MAE PP": float(np.mean(np.abs(
            y_margin_state - dynamic_specification_oof_5b
        ))),
        "Cycles Better Than Poll": int(sum(
            senate_specification_nested_folds[
                "Outer Selected MAE PP"
            ] < senate_specification_nested_folds[
                "Outer Poll MAE PP"
            ] - 1e-9
        )),
        "Uses State Effects": "Selected only if validated",
    }]),
], ignore_index=True).sort_values(
    ["Nested LOEO MAE PP", "Specification"],
    kind="mergesort",
).reset_index(drop=True)

production_specification_selection_5b = (
    select_senate_specification_5b(all_indices_5b)
)
if production_specification_selection_5b["specification"] != "PooledFullV11":
    raise AssertionError(
        "La especificación senatorial de producción ya no gana el examen "
        "anidado. Se requiere revisión deliberada antes de publicar."
    )
if not np.isclose(
    production_specification_selection_5b["alpha"],
    RIDGE_ALPHA_5B,
):
    raise AssertionError(
        "El alpha del specification challenge no coincide con producción."
    )
if not np.isclose(
    production_specification_selection_5b["correction_weight"],
    FUNDAMENTALS_WEIGHT_5B,
):
    raise AssertionError(
        "El peso del specification challenge no coincide con producción."
    )


def build_fold_specific_senate_frame_5b(omitted_year):
    fold_targets = national_2026_fold_targets.loc[
        national_2026_fold_targets[
            "Omitted Historical Cycle"
        ].eq(int(omitted_year))
    ].set_index("Variable")["Prediction"].to_dict()
    frame = senate_parser.copy()
    dpp = float(fold_targets["DPP"])
    rpp = float(fold_targets["RPP"])
    d_losses = expected_losses_from_buckets(
        fold_targets,
        DSEN_RES_COLS,
        DSS_UP_5B,
        "ResLR",
    )
    r_losses = expected_losses_from_buckets(
        fold_targets,
        RSEN_RES_COLS,
        RSS_UP_5B,
        "ResLD",
    )
    frame["National Vote Margin 2P"] = (
        100.0 * (dpp - rpp) / (dpp + rpp)
    )
    frame["D-Held Loss Pressure"] = d_losses
    frame["R-Held Loss Pressure"] = r_losses
    frame["National Senate Net Gain Signal"] = r_losses - d_losses
    frame["National Senate Flip Pressure"] = r_losses + d_losses
    frame["Incumbent Vulnerability Signal"] = np.where(
        frame["INCUMBENT"].eq("D"),
        -d_losses,
        r_losses,
    )
    frame["Incumbent D Indicator"] = (
        frame["INCUMBENT"].eq("D").astype(float)
    )
    return frame


senate_2026_fold_race_rows = []
senate_2026_fold_summary_rows = []
for omitted_year in sorted(np.unique(groups_state)):
    outer_train_idx = all_indices_5b[
        groups_state != int(omitted_year)
    ]
    selection = dynamic_selection_by_year_5b[int(omitted_year)]
    fold_frame = build_fold_specific_senate_frame_5b(omitted_year)
    _, candidate = fit_predict_senate_specification_5b(
        selection["specification"],
        selection["alpha"],
        outer_train_idx,
        fold_frame,
    )
    poll_anchor = fold_frame["Poll Margin 2P"].to_numpy(float)
    projected_margin = (
        selection["poll_weight"] * poll_anchor
        + selection["correction_weight"] * candidate
    )
    projected_winner = np.where(projected_margin >= 0, "D", "R")
    d_total = int(
        SENATE_FIXED_D_SEATS_5B + np.sum(projected_winner == "D")
    )
    senate_2026_fold_summary_rows.append({
        "Omitted Historical Cycle": int(omitted_year),
        "Training Cycles": ", ".join(map(
            str,
            sorted(
                year for year in np.unique(groups_state)
                if int(year) != int(omitted_year)
            ),
        )),
        "Selected Specification": selection["specification"],
        "Selected Alpha": selection["alpha"],
        "Applied Poll Weight": selection["poll_weight"],
        "Applied Correction Weight": selection["correction_weight"],
        "D Senate Seats": d_total,
        "R Senate Seats": 100 - d_total,
    })
    for state, incumbent, poll_margin, margin, winner in zip(
        fold_frame["STATE"],
        fold_frame["INCUMBENT"],
        poll_anchor,
        projected_margin,
        projected_winner,
    ):
        senate_2026_fold_race_rows.append({
            "Omitted Historical Cycle": int(omitted_year),
            "STATE": state,
            "INCUMBENT": incumbent,
            "Poll Margin 2P": float(poll_margin),
            "Projected Margin 2P": float(margin),
            "Projected Winner": winner,
            "Selected Specification": selection["specification"],
        })

senate_2026_fold_races = pd.DataFrame(senate_2026_fold_race_rows)
senate_2026_fold_summary = pd.DataFrame(
    senate_2026_fold_summary_rows
).sort_values("Omitted Historical Cycle").reset_index(drop=True)

senate_2026_race_stability_rows = []
production_by_state = senate_parser.set_index("STATE")
for state, fold in senate_2026_fold_races.groupby("STATE"):
    values = fold["Projected Margin 2P"].to_numpy(float)
    d_wins = int((fold["Projected Winner"] == "D").sum())
    production_margin = float(
        production_by_state.at[state, "Adjusted Margin 2P"]
    )
    production_winner = str(
        production_by_state.at[state, "Projected Winner"]
    )
    senate_2026_race_stability_rows.append({
        "STATE": state,
        "Production Margin 2P": production_margin,
        "Production Winner": production_winner,
        "Five-Fold Mean Margin 2P": float(np.mean(values)),
        "Five-Fold Median Margin 2P": float(np.median(values)),
        "Fold Standard Deviation PP": float(np.std(values, ddof=1)),
        "Minimum Margin 2P": float(np.min(values)),
        "Maximum Margin 2P": float(np.max(values)),
        "Range PP": float(np.ptp(values)),
        "D Winners Across Five Folds": d_wins,
        "R Winners Across Five Folds": 5 - d_wins,
        "Winner Agreement (%)": 20.0 * max(d_wins, 5 - d_wins),
        "Production Inside Fold Range": bool(
            np.min(values) - 1e-12
            <= production_margin
            <= np.max(values) + 1e-12
        ),
        "Role": "Sensitivity diagnostic; never a Senate seat quota",
    })

senate_2026_race_stability = pd.DataFrame(
    senate_2026_race_stability_rows
).sort_values(
    ["Winner Agreement (%)", "Range PP", "STATE"],
    ascending=[True, False, True],
    kind="mergesort",
).reset_index(drop=True)

senate_specification_contract = pd.DataFrame([
    {
        "Candidate": name,
        "Architecture": spec["role"],
        "Numeric Features": ", ".join(spec["numeric"]),
        "Categorical Features": (
            ", ".join(spec["categorical"]) or "None"
        ),
        "Can Force National Seat Total": False,
        "Can Mutate National Targets": False,
    }
    for name, spec in senate_specifications_5b.items()
])

round_columns = [
    "DEM MEAN", "REP MEAN", "MARGIN", "D Poll 2P", "R Poll 2P", "Poll Margin 2P",
    "National Vote Margin 2P", "D-Held Loss Pressure", "R-Held Loss Pressure",
    "National Senate Net Gain Signal", "National Senate Flip Pressure",
    "Incumbent Vulnerability Signal", "Raw Predicted Polling Error PP",
    "Error-Corrected Candidate Margin 2P", "Fundamentals Margin 2P",
    "Poll-Only Candidate Weight", "Error-Corrected Candidate Weight",
    "Model Polling Error Correction PP",
    "Model Projected Margin 2P", "Adjusted Margin 2P", "Model Disagreement PP",
    "State Idiosyncratic Sigma PP", "State Simulation Sigma PP", "Forecast Sigma PP",
    "Probability Calibration Scale", "Projected D 2P", "Projected R 2P",
    "D Win Probability", "R Win Probability", "Vulnerability Score",
    "Effective Comparable Elections", "Shrunk Historic Bias PP", "Historic MAE PP",
    "Historic Flip Rate", "Historic Polling Miss Rate", "Mean Comparability Weight",
]
senate_parser[round_columns] = senate_parser[round_columns].round(2)

race_columns = [
    "STATE", "INCUMBENT", "DEM MEAN", "REP MEAN", "MARGIN", "RATING",
    "Consensus Rating", "Rating Bucket", "D Poll 2P", "R Poll 2P", "Poll Margin 2P",
    "Raw Predicted Polling Error PP",
    "Error-Corrected Candidate Margin 2P",
    "Fundamentals Margin 2P", "Poll-Only Candidate Weight",
    "Error-Corrected Candidate Weight",
    "Projected D 2P", "Projected R 2P", "Model Polling Error Correction PP",
    "Model Projected Margin 2P", "Adjusted Margin 2P", "D Win Probability",
    "R Win Probability", "Forecast Rating", "Projected Winner", "Model Assigned Outcome",
    "Polling Leader", "Polling Outcome", "Vulnerability Score", "Baseline Safeguard Status",
    "National Vote Margin 2P", "D-Held Loss Pressure", "R-Held Loss Pressure",
    "National Senate Net Gain Signal", "National Senate Flip Pressure",
    "Incumbent Vulnerability Signal", "Model Disagreement PP",
    "State Idiosyncratic Sigma PP", "State Simulation Sigma PP", "Forecast Sigma PP",
    "Probability Calibration Scale", "Historic Elections", "Effective Comparable Elections",
    "Shrunk Historic Bias PP", "Historic MAE PP", "Historic Flip Rate",
    "Historic Polling Miss Rate", "Mean Comparability Weight",
]
senate_race_attribution = senate_parser[race_columns].copy().sort_values(
    "D Win Probability", ascending=False
).reset_index(drop=True)
senate_model_flips = senate_race_attribution[
    senate_race_attribution["Model Assigned Outcome"].isin([
        "Democratic Flip", "Republican Flip"
    ])
].copy()
senate_competitive_races = senate_race_attribution[
    (senate_race_attribution[["D Win Probability", "R Win Probability"]].max(axis=1) < 70)
    | senate_race_attribution["Forecast Rating"].str.contains("Toss-Up|Lean", regex=True)
].copy()

poll_mae_5b = float(np.mean(np.abs(y_margin_state - poll_margin_state)))
fundamentals_mae_5b = float(np.mean(np.abs(
    y_margin_state - fundamentals_oof_5b
)))
posterior_mae_5b = float(np.mean(np.abs(
    y_margin_state - posterior_oof_margins_5b
)))

senate_validation_summary = pd.DataFrame({
    "Metric": [
        "Historic observations", "Election cycles", "Historic states",
        "Poll baseline nested LOEO MAE PP", "Full polling-error candidate nested LOEO MAE PP",
        "Validated polling-error forecast fully nested LOEO MAE PP", "Validated forecast MAE change vs poll PP",
        "Cycles validated correction improved vs poll", "Poll-only candidate weight",
        "Error-corrected candidate weight", "Poll error variance", "Full correction candidate error variance",
        "Posterior OOF Brier score", "Poll-anchor Brier score", "Posterior OOF log loss",
        "Poll-anchor log loss", "80% interval coverage", "90% interval coverage",
        "Probability calibration scale", "Log-loss calibration scale",
        "90% coverage scale floor", "Numerical warnings", "NumPy version",
        "SciPy version", "scikit-learn version", "Python version",
    ],
    "Value": [
        len(senate_training), senate_training["YEAR"].nunique(),
        senate_training["STATE"].nunique(), poll_mae_5b, fundamentals_mae_5b,
        posterior_mae_5b, posterior_mae_5b - poll_mae_5b,
        int(senate_cycle_validation["Validated Correction Better Than Poll"].sum()),
        POLL_WEIGHT_5B, FUNDAMENTALS_WEIGHT_5B,
        poll_error_variance_5b, fundamentals_error_variance_5b,
        posterior_brier_5b, poll_brier_5b, posterior_log_loss_5b, poll_log_loss_5b,
        coverage_80_5b, coverage_90_5b, PROBABILITY_SCALE_5B, logloss_scale_5b,
        coverage_scale_90_5b, len(numerical_warning_log_5b), np.__version__,
        scipy.__version__, sklearn.__version__, platform.python_version(),
    ],
})

senate_validation_summary = pd.concat([
    senate_validation_summary,
    pd.DataFrame({
        "Metric": [
            "Validation architecture",
            "Production baseline safeguard status",
            "Median outer-fold probability scale",
            "Outer folds retaining polling-error correction",
        ],
        "Value": [
            "Fully nested by election cycle: error-model alpha, correction size and calibration",
            BASELINE_SAFEGUARD_STATUS_5B,
            float(senate_nested_fold_diagnostics["Outer Probability Scale"].median()),
            int((senate_nested_fold_diagnostics["Applied Error-Correction Candidate Weight"] > 0).sum()),
        ],
    }),
], ignore_index=True)

senate_attribution_summary = pd.DataFrame({
    "Metric": [
        "Democratic Senate Before", "Republican Senate Before",
        "National bucket prior D after", "National bucket prior R after",
        "Canonical Democratic Senate After", "Canonical Republican Senate After",
        "Canonical Democratic-held seats lost", "Canonical Republican-held seats lost",
        "Modeled Senate races", "Historic race observations", "Historic states",
        "Competitive races", "Assigned flips", "Fixed Democratic seats outside modeled races",
        "2026 national vote margin signal PP", "2026 D-held loss pressure",
        "2026 R-held loss pressure", "2026 national Senate net-gain signal",
        "2026 national Senate flip pressure", "Poll-only candidate weight",
        "Error-corrected candidate weight", "Poll nested LOEO MAE PP",
        "Fundamentals nested LOEO MAE PP", "Posterior nested LOEO MAE PP",
        "Common national shock sigma PP", "Probability calibration scale",
        "Selected polling-error Ridge alpha", "Numerical warnings",
        "National reconciliation shift",
    ],
    "Value": [
        DS_BEFORE_5B, RS_BEFORE_5B, D_SENATE_AFTER_5B, R_SENATE_AFTER_5B,
        D_SENATE_CENTRAL_5B, R_SENATE_CENTRAL_5B,
        D_SENATE_LOST_CENTRAL_5B, R_SENATE_LOST_CENTRAL_5B,
        len(senate_race_attribution), len(senate_training),
        senate_training["STATE"].nunique(), len(senate_competitive_races),
        len(senate_model_flips), SENATE_FIXED_D_SEATS_5B,
        round(national_vote_margin_2026, 4), round(d_loss_pressure_2026, 4),
        round(r_loss_pressure_2026, 4), round(national_senate_net_gain_2026, 4),
        round(national_senate_flip_pressure_2026, 4), round(POLL_WEIGHT_5B, 4),
        round(FUNDAMENTALS_WEIGHT_5B, 4), round(poll_mae_5b, 4),
        round(fundamentals_mae_5b, 4), round(posterior_mae_5b, 4),
        round(senate_common_sigma_pp, 4), round(PROBABILITY_SCALE_5B, 4),
        RIDGE_ALPHA_5B, len(numerical_warning_log_5b),
        "Removed — no forced seat quota",
    ],
})

senate_attribution_summary = pd.concat([
    senate_attribution_summary,
    pd.DataFrame({
        "Metric": ["Baseline safeguard status", "Validation architecture"],
        "Value": [
            BASELINE_SAFEGUARD_STATUS_5B,
            "Fully nested by election cycle",
        ],
    }),
], ignore_index=True)

senate_calibrated_probabilities = senate_parser.set_index("STATE")[
    "D Win Probability Decimal"
].copy()
senate_projected_margins = senate_parser.set_index("STATE")["Adjusted Margin 2P"].copy()
senate_idiosyncratic_sigmas = senate_parser.set_index("STATE")[
    "State Simulation Sigma PP"
].copy()

# Actualizar los objetos ejecutivos creados en Bloque 5 con la proyección estatal.
D_SENATE = D_SENATE_CENTRAL_5B
R_SENATE = R_SENATE_CENTRAL_5B
SENATE_CONTROL = "Democratic" if D_SENATE >= 51 else "Republican"
SENATE_MARGIN = abs(D_SENATE - R_SENATE)
SENATE_NET_D = D_SENATE - DS_BEFORE_5B

senate_mask = control_summary["Chamber"] == "Senate"
control_summary.loc[senate_mask, "Democratic Seats"] = D_SENATE
control_summary.loc[senate_mask, "Republican Seats"] = R_SENATE
control_summary.loc[senate_mask, "Projected Control"] = SENATE_CONTROL
control_summary.loc[senate_mask, "Margin"] = SENATE_MARGIN
control_summary.loc[senate_mask, "Democratic Net Change"] = SENATE_NET_D
control_summary.loc[senate_mask, "Republican Net Change"] = -SENATE_NET_D

snapshot_updates = {
    "Senate Projection": f"D {D_SENATE} - R {R_SENATE}",
    "Senate Control": SENATE_CONTROL,
    "Senate Margin": SENATE_MARGIN,
    "Senate Democratic Net Change": SENATE_NET_D,
}
for metric, value in snapshot_updates.items():
    election_snapshot.loc[
        election_snapshot["Metric"] == metric, "Projection"
    ] = value
executive_summary = election_snapshot.copy()
final_projection = control_summary.copy()
final_snapshot = election_snapshot.copy()

# ------------------------------------------------------------
# 10. EXPORTAR Y MOSTRAR
# ------------------------------------------------------------

senate_historic_clean = senate_training.sort_values(["STATE", "YEAR"]).reset_index(drop=True)
senate_historic_clean.to_excel(OUTPUT_DIR / "Block5B_Senate_Historic_Clean.xlsx", index=False)
senate_historic_summary.to_excel(OUTPUT_DIR / "Block5B_Senate_Historic_Summary.xlsx", index=False)
senate_state_model_cv.to_excel(OUTPUT_DIR / "Block5B_Senate_State_Model_CV.xlsx", index=False)
senate_nested_fold_diagnostics.to_excel(
    OUTPUT_DIR / "Block5B_Senate_Nested_Fold_Diagnostics.xlsx", index=False
)
senate_cycle_validation.to_excel(OUTPUT_DIR / "Block5B_Senate_Cycle_Validation.xlsx", index=False)
state_model_diagnostics.to_excel(OUTPUT_DIR / "Block5B_Senate_State_Model_Diagnostics.xlsx", index=False)
senate_validation_summary.to_excel(OUTPUT_DIR / "Block5B_Senate_Validation_Summary.xlsx", index=False)
senate_specification_summary.to_excel(
    OUTPUT_DIR / "Block5B_Senate_Specification_Summary.xlsx", index=False
)
senate_specification_nested_folds.to_excel(
    OUTPUT_DIR / "Block5B_Senate_Specification_Folds.xlsx", index=False
)
senate_specification_contract.to_excel(
    OUTPUT_DIR / "Block5B_Senate_Specification_Contract.xlsx", index=False
)
senate_2026_fold_summary.to_excel(
    OUTPUT_DIR / "Block5B_Senate_2026_Fold_Summary.xlsx", index=False
)
senate_2026_fold_races.to_excel(
    OUTPUT_DIR / "Block5B_Senate_2026_Fold_Races.xlsx", index=False
)
senate_2026_race_stability.to_excel(
    OUTPUT_DIR / "Block5B_Senate_2026_Race_Stability.xlsx", index=False
)
senate_race_attribution.to_excel(OUTPUT_DIR / "Block5B_Senate_Race_Attribution.xlsx", index=False)
senate_model_flips.to_excel(OUTPUT_DIR / "Block5B_Senate_Model_Flips.xlsx", index=False)
senate_competitive_races.to_excel(OUTPUT_DIR / "Block5B_Senate_Competitive_Races.xlsx", index=False)
senate_attribution_summary.to_excel(OUTPUT_DIR / "Block5B_Senate_Attribution_Summary.xlsx", index=False)

print("\nPOLL ANCHOR, ERROR CORRECTION AND VALIDATED FORECAST — OUTER TESTS")
display(state_model_diagnostics.round(4))
print("\nVALIDATION SUMMARY")
display(senate_validation_summary)
print("\n2026 SENATE RACE FORECASTS")
display(senate_race_attribution)
print("\nSENATE ATTRIBUTION SUMMARY")
display(senate_attribution_summary)
print("\nBLOQUE 5B v12 COMPLETADO SIN RECONCILIACIÓN NACIONAL FORZADA")

# One-way dependency audit: the state module may consume the frozen national
# signal contract, but it cannot rewrite any national target.
national_projection_after_senate = {
    key: float(value)
    for key, value in final_2026_full_projection_42.items()
}
national_state_isolation_audit = pd.concat(
    [
        popular_vote_dependency_audit.copy(),
        pd.DataFrame([
            {
                "Check": "State Senate module did not mutate national 42-target projection",
                "Passed": (
                    national_projection_snapshot_before_senate
                    == national_projection_after_senate
                ),
                "Detail": "Exact dictionary equality before vs after Block 5B",
            },
            {
                "Check": "State Senate module consumed a read-only national signal contract",
                "Passed": isinstance(
                    national_signal_contract_2026, MappingProxyType
                ),
                "Detail": ", ".join(sorted(national_signal_contract_2026.keys())),
            },
            {
                "Check": "National popular vote passed downstream without Senate override",
                "Passed": bool(
                    np.isclose(
                        national_signal_contract_2026["DPP"],
                        final_2026_full_projection_42["DPP"],
                    )
                    and np.isclose(
                        national_signal_contract_2026["RPP"],
                        final_2026_full_projection_42["RPP"],
                    )
                ),
                "Detail": (
                    f"DPP={national_signal_contract_2026['DPP']:.8f}; "
                    f"RPP={national_signal_contract_2026['RPP']:.8f}"
                ),
            },
        ]),
    ],
    ignore_index=True,
)
if not national_state_isolation_audit["Passed"].all():
    raise AssertionError(
        "Falló la auditoría de aislamiento nacional/Senado:\n"
        + national_state_isolation_audit.loc[
            ~national_state_isolation_audit["Passed"]
        ].to_string(index=False)
    )



## BLOQUE 5C — PROBABILISTIC HOUSE DISTRICT LAYER v17

Capa externa posterior a la señal nacional congelada: 435 probabilidades, ratings, simulación correlacionada y auditoría por distrito.

In [ ]:
# ============================================================
# BLOQUE 5C — PROBABILISTIC HOUSE DISTRICT LAYER (v17)
# 435 district probabilities after the frozen national model
# ============================================================

import re
import warnings
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit, logit
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss

print("=" * 78)
print("BLOQUE 5C v17 — PROBABILISTIC HOUSE DISTRICT LAYER")
print("=" * 78)

HOUSE_YEARS = [2006, 2010, 2014, 2018, 2022, 2026]
HOUSE_HISTORICAL_YEARS = HOUSE_YEARS[:-1]
HOUSE_RANDOM_STATE = 2026
HOUSE_SIMULATIONS = 50_000


def _read_house_sheet(year):
    frame = pd.read_excel(MODEL_FILE, sheet_name=f"House {year}", header=2)
    frame = frame.loc[:, ~frame.columns.astype(str).str.startswith("Unnamed")].copy()
    frame["Election Year"] = pd.to_numeric(frame["Election Year"], errors="coerce")
    geoid_numeric = pd.to_numeric(frame["GEOID"], errors="coerce")
    if geoid_numeric.isna().any():
        raise ValueError(f"House {year}: hay GEOID no numéricos.")
    frame["GEOID4"] = geoid_numeric.astype("Int64").astype(str).str.zfill(4)
    if len(frame) != 435:
        raise AssertionError(f"House {year}: se esperaban 435 distritos y hay {len(frame)}.")
    if frame["District ID"].duplicated().any() or frame["GEOID4"].duplicated().any():
        raise AssertionError(f"House {year}: los identificadores distritales no son únicos.")
    return frame


house_by_year = {year: _read_house_sheet(year) for year in HOUSE_YEARS}
house_historical = pd.concat(
    [house_by_year[year] for year in HOUSE_HISTORICAL_YEARS], ignore_index=True
)
house_2026 = house_by_year[2026].copy()


def _last_result_margin_pp(value):
    """Parse '54.6% D' as a D-minus-R two-party margin in percentage points."""
    if pd.isna(value):
        return np.nan
    match = re.search(r"([0-9]+(?:\.[0-9]+)?)\s*%?\s*([DR])", str(value), re.I)
    if not match:
        return np.nan
    share = float(match.group(1))
    sign = 1.0 if match.group(2).upper() == "D" else -1.0
    return sign * (2.0 * share - 100.0)


def _as_pp(value):
    value = float(value)
    return 100.0 * value if abs(value) <= 1.0 else value


# Historical national context is out-of-sample. The 2026 signal is the frozen
# national production forecast. This layer never writes information upstream.
national_popular_oof = national_outcome_oof.loc[
    national_outcome_oof["Model"].eq("NestedSelected")
    & national_outcome_oof["Outcome"].eq("Popular Vote Margin (pp)"),
    ["Year", "Predicted"],
].copy()
if set(national_popular_oof["Year"].astype(int)) != set(HOUSE_HISTORICAL_YEARS):
    raise AssertionError("El contexto nacional House no contiene cinco márgenes OOF.")
national_house_margin_pp = {
    int(row["Year"]): _as_pp(row["Predicted"])
    for _, row in national_popular_oof.iterrows()
}
national_house_margin_pp[2026] = _as_pp(
    get_projection_5b("DPP") - get_projection_5b("RPP")
)


# Core feature contract: only stable, pre-election and temporally comparable
# signals may estimate probabilities. The remaining 148+ fields are retained in
# HouseRaceDetail for source attribution, candidates, audit and visualization.
HOUSE_CORE_FEATURES = [
    "All Source Signed Median",
    "PVI D-Signed",
    "Last Result D-R Margin PP",
    "National D-R Popular Margin PP",
]
HOUSE_C_VALUES = [0.03, 0.10, 0.30, 1.00, 3.00]
HOUSE_FORBIDDEN_OUTCOME_COLUMNS = {
    "Winner Name", "Winner Party", "Winner Vote Share", "Runner-up Name",
    "Runner-up Party", "Runner-up Vote Share", "Democratic Vote Share",
    "Republican Vote Share", "Other Vote Share", "Two-Party D Share",
    "D Minus R Margin", "Absolute Margin", "Result Valid", "Result Status",
    "Incumbent Won", "Party Change vs Baseline", "Actual Result Strength",
    "Actual Signed Score", "Actual Model Bucket", "Forecast Party Correct",
    "Rating Error Signed Scale", "Upset Core3", "Cook Correct", "Inside Correct",
    "Sabato Correct",
}


def _prepare_house_features(frame):
    prepared = frame.copy()
    prepared["Last Result D-R Margin PP"] = prepared["Last Result Raw"].map(
        _last_result_margin_pp
    )
    prepared["National D-R Popular Margin PP"] = prepared["Election Year"].map(
        national_house_margin_pp
    )
    for column in HOUSE_CORE_FEATURES:
        prepared[column] = pd.to_numeric(prepared[column], errors="coerce")
        prepared.loc[~np.isfinite(prepared[column]), column] = np.nan
    return prepared


house_historical_model = _prepare_house_features(house_historical)
house_2026_model = _prepare_house_features(house_2026)
house_historical_model["D Minus R Margin"] = pd.to_numeric(
    house_historical_model["D Minus R Margin"], errors="coerce"
)
house_training = house_historical_model.dropna(subset=["D Minus R Margin"]).reset_index(drop=True)
house_training["D Won"] = (house_training["D Minus R Margin"] >= 0).astype(int)

X_house = house_training[HOUSE_CORE_FEATURES].copy()
y_house = house_training["D Won"].to_numpy(dtype=int)
g_house = house_training["Election Year"].to_numpy(dtype=int)
X_house_2026 = house_2026_model[HOUSE_CORE_FEATURES].copy()


class _StableHouseLogit:
    """Finite float64 logistic regression with bounded L2 optimization.

    This local estimator preserves the v17 model contract (median imputation,
    training-fold standardization and C-controlled L2 regularization) without
    relying on the sklearn decision_function matmul that overflows in some
    Python 3.9 / NumPy / sklearn combinations.
    """

    def __init__(self, c_value):
        self.c_value = float(c_value)

    @staticmethod
    def _as_float_matrix(values):
        matrix = np.asarray(values, dtype=np.float64)
        if matrix.ndim != 2:
            raise ValueError("House X debe ser una matriz bidimensional.")
        return matrix

    def _transform(self, values):
        matrix = self._as_float_matrix(values)
        matrix = np.where(np.isfinite(matrix), matrix, self.medians_)
        standardized = (matrix - self.means_) / self.scales_
        standardized = np.clip(standardized, -12.0, 12.0)
        if not np.isfinite(standardized).all():
            raise FloatingPointError("House produjo predictores estandarizados no finitos.")
        return standardized

    def fit(self, values, target):
        matrix = self._as_float_matrix(values)
        target = np.asarray(target, dtype=np.float64).reshape(-1)
        if len(matrix) != len(target):
            raise ValueError("House X e y tienen longitudes distintas.")
        if not np.isin(target, [0.0, 1.0]).all() or np.unique(target).size != 2:
            raise ValueError("Cada ajuste House requiere observaciones D y R.")

        medians = []
        for column in matrix.T:
            finite = column[np.isfinite(column)]
            medians.append(float(np.median(finite)) if finite.size else 0.0)
        self.medians_ = np.asarray(medians, dtype=np.float64)
        matrix = np.where(np.isfinite(matrix), matrix, self.medians_)
        self.means_ = np.mean(matrix, axis=0, dtype=np.float64)
        self.scales_ = np.std(matrix, axis=0, dtype=np.float64)
        self.scales_ = np.where(
            np.isfinite(self.scales_) & (self.scales_ > 1e-8), self.scales_, 1.0
        )
        standardized = np.clip((matrix - self.means_) / self.scales_, -12.0, 12.0)
        design = np.column_stack([
            np.ones(len(standardized), dtype=np.float64), standardized
        ])
        sample_size = float(len(target))
        l2_strength = 1.0 / max(self.c_value * sample_size, 1e-12)
        prior = np.clip(float(np.mean(target)), 1e-6, 1.0 - 1e-6)
        initial = np.zeros(design.shape[1], dtype=np.float64)
        initial[0] = float(logit(prior))

        def objective(coefficients):
            # Explicit elementwise reductions intentionally avoid every use of
            # np.matmul / @. Some Python 3.9 macOS builds emit false floating-
            # point warnings inside BLAS even for finite, standardized values.
            linear = np.sum(
                design * coefficients[np.newaxis, :], axis=1, dtype=np.float64
            )
            residual = expit(linear) - target
            loss = np.mean(np.logaddexp(0.0, linear) - target * linear)
            penalty = np.sum(coefficients[1:] ** 2, dtype=np.float64)
            loss += 0.5 * l2_strength * float(penalty)
            gradient = np.sum(
                design * residual[:, np.newaxis], axis=0, dtype=np.float64
            ) / sample_size
            gradient[1:] += l2_strength * coefficients[1:]
            return float(loss), np.asarray(gradient, dtype=np.float64)

        bounds = [(-16.0, 16.0)] + [(-10.0, 10.0)] * (design.shape[1] - 1)
        result = minimize(
            objective,
            initial,
            method="L-BFGS-B",
            jac=True,
            bounds=bounds,
            options={"maxiter": 2_000, "ftol": 1e-12, "gtol": 1e-8},
        )
        self.coef_ = np.asarray(result.x, dtype=np.float64)
        self.optimization_success_ = bool(result.success)
        self.optimization_message_ = str(result.message)
        if not np.isfinite(self.coef_).all():
            raise FloatingPointError(
                "El optimizador House produjo coeficientes no finitos: "
                + self.optimization_message_
            )
        return self

    def predict_proba(self, values):
        standardized = self._transform(values)
        design = np.column_stack([
            np.ones(len(standardized), dtype=np.float64), standardized
        ])
        linear = np.sum(
            design * self.coef_[np.newaxis, :], axis=1, dtype=np.float64
        )
        linear = np.clip(linear, -35.0, 35.0)
        democratic = expit(linear)
        probabilities = np.column_stack([1.0 - democratic, democratic])
        if not np.isfinite(probabilities).all():
            raise FloatingPointError("House produjo probabilidades no finitas.")
        return probabilities


def _make_house_model(c_value):
    return _StableHouseLogit(c_value)


def _weighted_brier(actual, predicted, rating_score):
    # Competitive pre-election ratings receive greater weight so safe seats do
    # not hide poor probability calibration in the races that determine control.
    competitive_weight = np.where(np.abs(rating_score) <= 2.0, 3.0, 1.0)
    return float(np.average((actual - predicted) ** 2, weights=competitive_weight))


# Fully nested leave-one-election-cycle-out selection and evaluation.
house_oof_rows = []
house_fold_rows = []
house_numeric_warning_log = []
with warnings.catch_warnings(record=True) as caught_house_warnings:
    warnings.simplefilter("always", RuntimeWarning)
    for outer_year in HOUSE_HISTORICAL_YEARS:
        outer_train = g_house != outer_year
        outer_test = g_house == outer_year
        inner_years = sorted(set(g_house[outer_train]))
        inner_scores = {}
        for c_value in HOUSE_C_VALUES:
            fold_scores = []
            for inner_year in inner_years:
                inner_train = outer_train & (g_house != inner_year)
                inner_test = outer_train & (g_house == inner_year)
                candidate = _make_house_model(c_value)
                candidate.fit(X_house.loc[inner_train], y_house[inner_train])
                probability = candidate.predict_proba(X_house.loc[inner_test])[:, 1]
                rating_score = pd.to_numeric(
                    house_training.loc[inner_test, "All Source Signed Median"],
                    errors="coerce",
                ).fillna(0.0).to_numpy()
                fold_scores.append(_weighted_brier(
                    y_house[inner_test], probability, rating_score
                ))
            inner_scores[c_value] = float(np.mean(fold_scores))

        best_inner = min(inner_scores.values())
        # Complexity guardrail: choose the strongest regularization still
        # statistically indistinguishable (0.001 Brier) from the inner minimum.
        eligible = [c for c in HOUSE_C_VALUES if inner_scores[c] <= best_inner + 0.001]
        selected_c = min(eligible)
        outer_model = _make_house_model(selected_c)
        outer_model.fit(X_house.loc[outer_train], y_house[outer_train])
        outer_probability = outer_model.predict_proba(X_house.loc[outer_test])[:, 1]
        outer_actual = y_house[outer_test]
        outer_source = house_training.loc[outer_test].reset_index(drop=True)
        for row_number, probability in enumerate(outer_probability):
            source = outer_source.iloc[row_number]
            actual = int(outer_actual[row_number])
            actual_margin_pp = 100.0 * float(source["D Minus R Margin"])
            house_oof_rows.append({
                "Election Year": int(outer_year),
                "District ID": source["District ID"],
                "Actual D Win": actual,
                "Predicted D Win Probability": float(probability),
                "Actual D-R Margin PP": actual_margin_pp,
                "All Source Signed Median": source["All Source Signed Median"],
                "PVI D-Signed": source["PVI D-Signed"],
                "Selected C": float(selected_c),
                "Brier Contribution": float((actual - probability) ** 2),
            })
        rating_outer = pd.to_numeric(
            outer_source["All Source Signed Median"], errors="coerce"
        ).fillna(0.0).to_numpy()
        competitive_outer = np.abs(rating_outer) <= 2.0
        house_fold_rows.append({
            "Outer Election Year": int(outer_year),
            "Selected C": float(selected_c),
            "Outer Brier": brier_score_loss(outer_actual, outer_probability),
            "Outer Weighted Brier": _weighted_brier(
                outer_actual, outer_probability, rating_outer
            ),
            "Outer Log Loss": log_loss(
                outer_actual, np.clip(outer_probability, 1e-6, 1 - 1e-6)
            ),
            "Outer Winner Accuracy": accuracy_score(
                outer_actual, outer_probability >= 0.5
            ),
            "Competitive Brier": brier_score_loss(
                outer_actual[competitive_outer], outer_probability[competitive_outer]
            ),
            "Competitive Accuracy": accuracy_score(
                outer_actual[competitive_outer], outer_probability[competitive_outer] >= 0.5
            ),
            **{f"Inner Weighted Brier C={c:g}": score for c, score in inner_scores.items()},
        })
    house_numeric_warning_log = [
        str(item.message) for item in caught_house_warnings
        if issubclass(item.category, RuntimeWarning)
    ]

if house_numeric_warning_log:
    print(
        "Aviso numérico de plataforma capturado; las invariantes finitas "
        "posteriores determinarán si la ejecución es válida."
    )

house_validation_oof = pd.DataFrame(house_oof_rows)
house_validation_folds = pd.DataFrame(house_fold_rows)
oof_probability = house_validation_oof["Predicted D Win Probability"].to_numpy()
oof_actual = house_validation_oof["Actual D Win"].to_numpy(dtype=int)
oof_rating = pd.to_numeric(
    house_validation_oof["All Source Signed Median"], errors="coerce"
).fillna(0.0).to_numpy()
oof_competitive = np.abs(oof_rating) <= 2.0


# Production regularization is selected only from inner validation results.
production_scores = {
    c: float(house_validation_folds[f"Inner Weighted Brier C={c:g}"].mean())
    for c in HOUSE_C_VALUES
}
production_minimum = min(production_scores.values())
production_eligible = [
    c for c in HOUSE_C_VALUES if production_scores[c] <= production_minimum + 0.001
]
production_c = min(production_eligible)
house_production_model = _make_house_model(production_c)
house_production_model.fit(X_house, y_house)
house_d_probability = house_production_model.predict_proba(X_house_2026)[:, 1]
house_d_probability = np.clip(house_d_probability, 1e-5, 1 - 1e-5)
if not np.isfinite(house_d_probability).all():
    raise FloatingPointError("House 2026 contiene probabilidades no finitas.")


# Convert probability to a readable projected margin through a zero-centered
# OOF calibration. The positive scale preserves winner/probability signs.
oof_logit = logit(np.clip(oof_probability, 1e-5, 1 - 1e-5))
oof_margin_pp = house_validation_oof["Actual D-R Margin PP"].to_numpy(float)
margin_scale_fit = minimize_scalar(
    lambda scale: np.mean(np.abs(oof_margin_pp - scale * oof_logit)),
    bounds=(0.25, 20.0), method="bounded",
)
HOUSE_MARGIN_SCALE_PP = float(margin_scale_fit.x)
house_projected_margin_pp = np.clip(
    HOUSE_MARGIN_SCALE_PP * logit(house_d_probability), -100.0, 100.0
)


def _house_rating_from_probability(probability):
    probability = float(probability)
    advantage = abs(probability - 0.5)
    if advantage < 0.05:
        return "Toss-Up"
    party = "D" if probability > 0.5 else "R"
    if advantage < 0.15:
        return f"Tilt {party}"
    if advantage < 0.30:
        return f"Lean {party}"
    if advantage < 0.45:
        return f"Likely {party}"
    return f"Safe {party}"


house_central_winner = np.where(house_d_probability >= 0.5, "D", "R")
HOUSE_RACE_LEADER_D_SEATS = int((house_central_winner == "D").sum())
HOUSE_RACE_LEADER_R_SEATS = 435 - HOUSE_RACE_LEADER_D_SEATS
HOUSE_EXPECTED_D_SEATS = float(house_d_probability.sum())
HOUSE_EXPECTED_R_SEATS = 435.0 - HOUSE_EXPECTED_D_SEATS


# Estimate the election-wide uncertainty from sealed cycle seat residuals.
# Then calibrate a shared logit shock so the 2026 simulation reproduces that
# historical aggregate uncertainty without changing district point estimates.
oof_cycle_seats = house_validation_oof.groupby("Election Year").agg(
    Actual_D=("Actual D Win", "sum"),
    Expected_D=("Predicted D Win Probability", "sum"),
).reset_index()
oof_cycle_seats["Seat Residual"] = oof_cycle_seats["Actual_D"] - oof_cycle_seats["Expected_D"]
HOUSE_TARGET_SEAT_SD = max(
    float(oof_cycle_seats["Seat Residual"].std(ddof=1)),
    float(np.sqrt(np.sum(house_d_probability * (1.0 - house_d_probability)))),
)

calibration_rng = np.random.default_rng(HOUSE_RANDOM_STATE + 31)
calibration_common = calibration_rng.standard_normal(10_000)
calibration_uniform = calibration_rng.random((10_000, 435))
house_current_logit = logit(house_d_probability)


def _seat_sd_for_common_sigma(sigma):
    shifted = expit(house_current_logit[None, :] + sigma * calibration_common[:, None])
    seats = (calibration_uniform < shifted).sum(axis=1)
    return float(seats.std(ddof=1))


low_sigma, high_sigma = 0.0, 2.5
for _ in range(24):
    middle_sigma = (low_sigma + high_sigma) / 2.0
    if _seat_sd_for_common_sigma(middle_sigma) < HOUSE_TARGET_SEAT_SD:
        low_sigma = middle_sigma
    else:
        high_sigma = middle_sigma
HOUSE_COMMON_LOGIT_SIGMA = (low_sigma + high_sigma) / 2.0

house_rng = np.random.default_rng(HOUSE_RANDOM_STATE)
house_simulated_d = np.empty(HOUSE_SIMULATIONS, dtype=np.int16)
batch_size = 500
for start in range(0, HOUSE_SIMULATIONS, batch_size):
    stop = min(start + batch_size, HOUSE_SIMULATIONS)
    size = stop - start
    common_shock = house_rng.normal(0.0, HOUSE_COMMON_LOGIT_SIGMA, size=(size, 1))
    shifted_probability = expit(house_current_logit[None, :] + common_shock)
    house_simulated_d[start:stop] = (
        house_rng.random((size, 435)) < shifted_probability
    ).sum(axis=1)

house_simulations = pd.DataFrame({
    "Simulation": np.arange(1, HOUSE_SIMULATIONS + 1),
    "D House Seats": house_simulated_d,
    "R House Seats": 435 - house_simulated_d,
})
HOUSE_DISTRICT_D_CENTRAL = int(np.median(house_simulated_d))
HOUSE_DISTRICT_R_CENTRAL = 435 - HOUSE_DISTRICT_D_CENTRAL
HOUSE_NATIONAL_D_SEAT_PRIOR = int(round(float(get_projection_5b("D House After"))))
HOUSE_NATIONAL_R_SEAT_PRIOR = 435 - HOUSE_NATIONAL_D_SEAT_PRIOR
HOUSE_PRIOR_DIFFERENCE_D = HOUSE_DISTRICT_D_CENTRAL - HOUSE_NATIONAL_D_SEAT_PRIOR
HOUSE_D_CONTROL_PROBABILITY = float(100.0 * np.mean(house_simulated_d >= 218))
HOUSE_R_CONTROL_PROBABILITY = 100.0 - HOUSE_D_CONTROL_PROBABILITY


# Explain the log-odds contribution of every modeled signal for every district.
# Coefficient 0 is the intercept; the four remaining coefficients map exactly
# to HOUSE_CORE_FEATURES after fold-specific imputation and standardization.
scaled_2026 = house_production_model._transform(X_house_2026)
logit_contributions = scaled_2026 * house_production_model.coef_[1:]

house_race_detail = house_2026.copy()
house_race_detail["Projected D-R Margin"] = house_projected_margin_pp / 100.0
house_race_detail["Projected Margin PP"] = house_projected_margin_pp
house_race_detail["D Win Probability"] = 100.0 * house_d_probability
house_race_detail["R Win Probability"] = 100.0 * (1.0 - house_d_probability)
house_race_detail["Expected D Seat Contribution"] = house_d_probability
house_race_detail["Projected Winner"] = house_central_winner
house_race_detail["Forecast Rating"] = [
    _house_rating_from_probability(value) for value in house_d_probability
]
house_race_detail["Competitive"] = ~house_race_detail["Forecast Rating"].str.startswith("Safe")
house_race_detail["Rating Signal Log-Odds"] = logit_contributions[:, 0]
house_race_detail["PVI Signal Log-Odds"] = logit_contributions[:, 1]
house_race_detail["Prior Result Signal Log-Odds"] = logit_contributions[:, 2]
house_race_detail["National Context Signal Log-Odds"] = logit_contributions[:, 3]
baseline_party = house_race_detail["Baseline Party Model"].astype(str).str.upper().str[0]
house_race_detail["Projected Flip"] = np.where(
    baseline_party.isin(["D", "R"]) & (baseline_party != house_race_detail["Projected Winner"]),
    baseline_party + "→" + house_race_detail["Projected Winner"], "Hold",
)
consensus_numeric = pd.to_numeric(
    house_race_detail["All Source Signed Median"], errors="coerce"
)
house_race_detail["Model vs Consensus Disagreement"] = (
    consensus_numeric.notna()
    & (np.sign(house_projected_margin_pp) != np.sign(consensus_numeric))
    & (consensus_numeric != 0)
)

house_competitive_races = (
    house_race_detail.loc[house_race_detail["Competitive"]]
    .assign(Distance_From_50=lambda frame: abs(frame["D Win Probability"] - 50.0))
    .sort_values(["Distance_From_50", "District ID"])
    .drop(columns="Distance_From_50")
    .reset_index(drop=True)
)

house_simulation_summary = pd.DataFrame({
    "Metric": [
        "Expected Democratic seats", "Expected Republican seats",
        "Race-leader Democratic seats", "Race-leader Republican seats",
        "Simulation median Democratic seats", "Simulation median Republican seats",
        "Frozen national comparator D seats", "Frozen national comparator R seats",
        "Median minus comparator D seats", "D seats 10th percentile",
        "D seats 90th percentile", "Democratic control probability",
        "Republican control probability", "Target aggregate seat SD",
        "Simulated aggregate seat SD", "Common logit shock sigma",
    ],
    "Value": [
        HOUSE_EXPECTED_D_SEATS, HOUSE_EXPECTED_R_SEATS,
        HOUSE_RACE_LEADER_D_SEATS, HOUSE_RACE_LEADER_R_SEATS,
        HOUSE_DISTRICT_D_CENTRAL, HOUSE_DISTRICT_R_CENTRAL,
        HOUSE_NATIONAL_D_SEAT_PRIOR, HOUSE_NATIONAL_R_SEAT_PRIOR,
        HOUSE_PRIOR_DIFFERENCE_D, float(np.quantile(house_simulated_d, 0.10)),
        float(np.quantile(house_simulated_d, 0.90)), HOUSE_D_CONTROL_PROBABILITY,
        HOUSE_R_CONTROL_PROBABILITY, HOUSE_TARGET_SEAT_SD,
        float(house_simulated_d.std(ddof=1)), HOUSE_COMMON_LOGIT_SIGMA,
    ],
})

house_validation_summary = pd.DataFrame({
    "Metric": [
        "Historical observations", "Sealed election cycles", "Nested OOF Brier",
        "Nested OOF weighted Brier", "Nested OOF log loss",
        "Nested OOF winner accuracy", "Competitive OOF Brier",
        "Competitive OOF winner accuracy", "OOF margin calibration MAE PP",
        "Production regularization C", "Numerical runtime warnings",
    ],
    "Value": [
        len(house_validation_oof), len(HOUSE_HISTORICAL_YEARS),
        brier_score_loss(oof_actual, oof_probability),
        _weighted_brier(oof_actual, oof_probability, oof_rating),
        log_loss(oof_actual, np.clip(oof_probability, 1e-6, 1 - 1e-6)),
        accuracy_score(oof_actual, oof_probability >= 0.5),
        brier_score_loss(oof_actual[oof_competitive], oof_probability[oof_competitive]),
        accuracy_score(oof_actual[oof_competitive], oof_probability[oof_competitive] >= 0.5),
        float(np.mean(np.abs(oof_margin_pp - HOUSE_MARGIN_SCALE_PP * oof_logit))),
        production_c, len(house_numeric_warning_log),
    ],
})

house_feature_contract = pd.DataFrame({
    "Field": [
        "Layer position", "Training observations", "Historical cycles",
        "Probability estimator", "Core modeled signals", "Display/audit fields",
        "Explicitly excluded temporal leakage", "Historical national signal",
        "2026 national signal", "Probability validation", "Margin display",
        "Seat aggregation", "National House comparator", "Uncertainty",
    ],
    "Value": [
        "Downstream of frozen national model; no feedback to national or Senate",
        len(house_training), ", ".join(map(str, HOUSE_HISTORICAL_YEARS)),
        f"Regularized logistic probability model; nested-selected C={production_c:g}",
        "; ".join(HOUSE_CORE_FEATURES),
        f"All {len(house_2026.columns)} source columns retained in HouseRaceDetail",
        "Incumbent Status and every result/correctness field",
        "Nested-selected OOF popular-vote margin for each held-out cycle",
        "Frozen production popular-vote margin from the national layer",
        "Nested leave-one-election-cycle-out Brier, log loss and competitive diagnostics",
        f"Zero-centered OOF logit scale ({HOUSE_MARGIN_SCALE_PP:.3f} pp per logit unit)",
        "Expected seats=sum probabilities; race leaders=p>=50%; central=simulation median",
        f"D {HOUSE_NATIONAL_D_SEAT_PRIOR} / R {HOUSE_NATIONAL_R_SEAT_PRIOR}; display only",
        "Correlated district simulations calibrated to OOF cycle-level seat residuals",
    ],
})

house_leakage_audit = pd.DataFrame({
    "Check": [
        "No outcome field in model features", "Incumbent Status excluded",
        "Candidate identities excluded", "Historical national context is OOF",
        "Exactly five historical cycles", "Exactly 435 current districts",
        "Unique district IDs", "Unique GEOID", "Finite probabilities",
        "Probability complements", "Winner follows probability",
        "Expected seats equal probability sum", "No forced national reconciliation",
        "Every simulation sums 435", "Central total is simulation median",
        "Finite House outputs after numerical audit",
    ],
    "Passed": [
        not bool(HOUSE_FORBIDDEN_OUTCOME_COLUMNS.intersection(HOUSE_CORE_FEATURES)),
        "Incumbent Status" not in HOUSE_CORE_FEATURES,
        not any(name in HOUSE_CORE_FEATURES for name in [
            "Incumbent Name", "Democratic Candidate", "Republican Candidate"
        ]),
        set(national_popular_oof["Year"].astype(int)) == set(HOUSE_HISTORICAL_YEARS),
        set(house_training["Election Year"].astype(int)) == set(HOUSE_HISTORICAL_YEARS),
        len(house_race_detail) == 435,
        house_race_detail["District ID"].nunique() == 435,
        house_race_detail["GEOID4"].nunique() == 435,
        bool(np.isfinite(house_d_probability).all()),
        np.allclose(
            house_race_detail["D Win Probability"] + house_race_detail["R Win Probability"],
            100.0, atol=1e-8,
        ),
        bool(((house_race_detail["D Win Probability"] >= 50.0)
              == (house_race_detail["Projected Winner"] == "D")).all()),
        np.isclose(HOUSE_EXPECTED_D_SEATS, house_d_probability.sum()),
        "National Reconciliation Shift" not in house_race_detail.columns,
        bool((house_simulations["D House Seats"] + house_simulations["R House Seats"] == 435).all()),
        HOUSE_DISTRICT_D_CENTRAL == int(np.median(house_simulated_d)),
        bool(np.isfinite(house_d_probability).all()),
    ],
    "Detail": [
        "; ".join(HOUSE_CORE_FEATURES), "Outcome-dependent historical text is display only",
        "Names remain visible in Race Desk", ", ".join(map(str, HOUSE_HISTORICAL_YEARS)),
        str(len(HOUSE_HISTORICAL_YEARS)), str(len(house_race_detail)),
        str(house_race_detail["District ID"].nunique()),
        str(house_race_detail["GEOID4"].nunique()), "all 435 probabilities",
        "D% + R% = 100", "all 435 race leaders", f"D={HOUSE_EXPECTED_D_SEATS:.3f}",
        f"central D={HOUSE_DISTRICT_D_CENTRAL}; comparator D={HOUSE_NATIONAL_D_SEAT_PRIOR}",
        f"{len(house_simulations):,} simulations", f"D={HOUSE_DISTRICT_D_CENTRAL}",
        f"finite probabilities; captured platform warnings={len(house_numeric_warning_log)}",
    ],
})
if not house_leakage_audit["Passed"].all():
    raise AssertionError(
        "Falló la auditoría House:\n"
        + house_leakage_audit.loc[~house_leakage_audit["Passed"]].to_string(index=False)
    )

# Put model outputs first; retain every source column after them.
house_output_columns = [
    "Election Year", "Congress", "District ID", "District Label", "State", "State Abbr",
    "GEOID4", "District Number", "At-Large", "Projected Winner", "Forecast Rating",
    "Projected Margin PP", "D Win Probability", "R Win Probability",
    "Expected D Seat Contribution", "Competitive", "Projected Flip",
    "Model vs Consensus Disagreement", "All Source Consensus Rating",
    "Core3 Consensus Rating", "All Source Signed Median", "PVI Raw", "PVI D-Signed",
    "Last Result Raw", "Incumbent Name", "Incumbent Party Raw", "Baseline Party Model",
    "Incumbent Running", "Open Seat", "Democratic Candidate", "Republican Candidate",
    "Rating Signal Log-Odds", "PVI Signal Log-Odds", "Prior Result Signal Log-Odds",
    "National Context Signal Log-Odds",
]
house_race_detail = house_race_detail[
    house_output_columns
    + [column for column in house_race_detail.columns if column not in house_output_columns]
].copy()
house_competitive_races = house_race_detail.loc[
    house_race_detail["Competitive"]
].assign(Distance_From_50=lambda frame: abs(frame["D Win Probability"] - 50.0)).sort_values(
    ["Distance_From_50", "District ID"]
).drop(columns="Distance_From_50").reset_index(drop=True)

print(f"House historical observations: {len(house_training):,}")
print(f"Production probability model: regularized logistic (C={production_c:g})")
print(f"Nested OOF Brier: {brier_score_loss(oof_actual, oof_probability):.4f}")
print(f"Competitive OOF Brier: {brier_score_loss(oof_actual[oof_competitive], oof_probability[oof_competitive]):.4f}")
print(f"Expected seats: D {HOUSE_EXPECTED_D_SEATS:.1f} / R {HOUSE_EXPECTED_R_SEATS:.1f}")
print(f"Race leaders: D {HOUSE_RACE_LEADER_D_SEATS} / R {HOUSE_RACE_LEADER_R_SEATS}")
print(f"Simulation median: D {HOUSE_DISTRICT_D_CENTRAL} / R {HOUSE_DISTRICT_R_CENTRAL}")
print(f"House control probability: D {HOUSE_D_CONTROL_PROBABILITY:.1f}% / R {HOUSE_R_CONTROL_PROBABILITY:.1f}%")
print(f"Frozen national comparator: D {HOUSE_NATIONAL_D_SEAT_PRIOR} / R {HOUSE_NATIONAL_R_SEAT_PRIOR}")
print(f"Competitive districts: {len(house_competitive_races)}")
print(f"House simulations: {len(house_simulations):,}")
print(f"House numerical RuntimeWarnings: {len(house_numeric_warning_log)}")


## BLOQUE 6 — UNCERTAINTY ENGINE v5

Los errores nacionales proceden de la tabla cross-fitted canónica; Monte Carlo no vuelve a estimar pesos con los ciclos evaluados.


In [ ]:
# ============================================================
# BLOQUE 6 — UNCERTAINTY ENGINE v5
# LOOCV nacional + Monte Carlo estatal para el Senado
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import norm

rng = np.random.default_rng(42)

# ------------------------------------------------------------
# 1. VERIFICACIONES Y FUNCIONES
# ------------------------------------------------------------

projection = block3_predictions.copy()
required_objects_6 = [
    "national_nested_oof", "control_summary", "senate_race_attribution",
    "senate_calibrated_probabilities", "senate_projected_margins",
    "senate_idiosyncratic_sigmas", "senate_common_sigma_pp",
    "SENATE_FIXED_D_SEATS_5B", "D_SENATE_CENTRAL_5B", "R_SENATE_CENTRAL_5B",
    "HOUSE_DISTRICT_D_CENTRAL", "HOUSE_DISTRICT_R_CENTRAL", "house_simulations"
]
missing_6 = [obj for obj in required_objects_6 if obj not in globals()]
if missing_6:
    raise ValueError("Faltan objetos para Bloque 6:\n" + "\n".join(missing_6))


def get_final(variable):
    row = projection[projection["Variable"] == variable]
    if row.empty:
        raise ValueError(f"No encontré la variable: {variable}")
    return row["Constrained Prediction"].iloc[0]


def summarize_distribution(values, name):
    values = np.asarray(values)
    return {
        "Metric": name, "Mean": np.mean(values), "SD": np.std(values, ddof=1),
        "Median": np.median(values), "P01": np.percentile(values, 1),
        "P05": np.percentile(values, 5), "P10": np.percentile(values, 10),
        "P25": np.percentile(values, 25), "P75": np.percentile(values, 75),
        "P90": np.percentile(values, 90), "P95": np.percentile(values, 95),
        "P99": np.percentile(values, 99), "Min": np.min(values), "Max": np.max(values)
    }


def format_range(low, high):
    return f"{low:.2f} – {high:.2f}"


def format_seat_range(low, high):
    return f"{int(round(low))} – {int(round(high))}"


# ------------------------------------------------------------
# 2. VALORES CENTRALES
# ------------------------------------------------------------

DPP_CENTRAL = float(get_final("DPP"))
RPP_CENTRAL = float(get_final("RPP"))
OTHER_CENTRAL = max(0, 1 - DPP_CENTRAL - RPP_CENTRAL)
D_HOUSE_CENTRAL = int(HOUSE_DISTRICT_D_CENTRAL)
R_HOUSE_CENTRAL = int(HOUSE_DISTRICT_R_CENTRAL)
D_SENATE_CENTRAL = int(D_SENATE_CENTRAL_5B)
R_SENATE_CENTRAL = int(R_SENATE_CENTRAL_5B)
HOUSE_TOTAL = 435
SENATE_TOTAL = 100
HOUSE_DEM_MARGIN_CENTRAL = D_HOUSE_CENTRAL - R_HOUSE_CENTRAL
SENATE_DEM_MARGIN_CENTRAL = D_SENATE_CENTRAL - R_SENATE_CENTRAL

# ------------------------------------------------------------
# 3. ERROR HISTÓRICO NACIONAL CON ENSEMBLE LOOCV
# ------------------------------------------------------------

# Artefacto canónico del Bloque 2: cada error procede de un ciclo exterior.
# Hiperparámetros, pesos y estrategia fueron aprendidos sin ese ciclo.
historical_errors = national_nested_oof[
    [
        "Year", "Variable", "Group", "Actual", "Predicted",
        "Signed Error", "Absolute Error", "Selected Strategy",
    ]
].copy()
if historical_errors.duplicated(["Year", "Variable"]).any():
    raise AssertionError("Los errores nacionales nested contienen duplicados.")

error_summary = historical_errors.groupby(["Group", "Variable"]).agg(
    MAE=("Absolute Error", "mean"),
    RMSE=("Signed Error", lambda x: np.sqrt(np.mean(np.square(x)))),
    Bias=("Signed Error", "mean"), SD=("Signed Error", "std"),
    MaxAbsError=("Absolute Error", "max")
).reset_index()
group_error_summary = historical_errors.groupby("Group").agg(
    MAE=("Absolute Error", "mean"),
    RMSE=("Signed Error", lambda x: np.sqrt(np.mean(np.square(x)))),
    Bias=("Signed Error", "mean"), SD=("Signed Error", "std"),
    MaxAbsError=("Absolute Error", "max"), N=("Signed Error", "count")
).reset_index()

popular_errors_by_year = historical_errors[
    historical_errors["Variable"].isin(["DPP", "RPP"])
].pivot(index="Year", columns="Variable", values="Signed Error").dropna()
if popular_errors_by_year.empty:
    raise ValueError("No hay errores LOOCV emparejados para DPP/RPP.")

# Reconstruir el error histórico del resultado agregado de Cámara, no usar
# residuos de buckets individuales como si fueran errores de escaños finales.
house_error_records = []
for year in sorted(historical_errors["Year"].unique()):
    historic_row = train_df[train_df["Midterm Year"] == year].iloc[0]
    predicted_by_variable = historical_errors[
        historical_errors["Year"] == year
    ].set_index("Variable")["Predicted"]

    d_values = np.array([max(float(predicted_by_variable[c]), 0) for c in DHOU_RES_COLS])
    r_values = np.array([max(float(predicted_by_variable[c]), 0) for c in RHOU_RES_COLS])
    d_quotas = d_values / d_values.sum() * float(historic_row["DH Before"])
    r_quotas = r_values / r_values.sum() * float(historic_row["RH Before"])
    d_losses = sum(
        quota for column, quota in zip(DHOU_RES_COLS, d_quotas) if "ResLR" in column
    )
    r_losses = sum(
        quota for column, quota in zip(RHOU_RES_COLS, r_quotas) if "ResLD" in column
    )
    predicted_d_after = float(historic_row["DH Before"]) - d_losses + r_losses
    actual_d_after = float(historic_row["D House After"])
    house_error_records.append({
        "Year": int(year),
        "Predicted D House After": predicted_d_after,
        "Actual D House After": actual_d_after,
        "Signed Error": predicted_d_after - actual_d_after
    })
house_loocv_errors = pd.DataFrame(house_error_records)
house_error_pool = house_loocv_errors["Signed Error"].to_numpy(float)
# ------------------------------------------------------------
# 4. MONTE CARLO
# ------------------------------------------------------------

N_SIM = 50000

def normal_draw(pool, size):
    mean = float(np.mean(pool))
    sd = float(np.std(pool, ddof=1)) if len(pool) > 1 else 0.0
    if not np.isfinite(sd) or sd == 0:
        sd = max(abs(mean), 1e-6)
    return rng.normal(mean, sd, size=size)


pv_error_matrix = popular_errors_by_year[["DPP", "RPP"]].to_numpy(float)
sampled_years = rng.integers(0, len(pv_error_matrix), size=N_SIM)
sampled_pv_errors = pv_error_matrix[sampled_years]

# Signed Error = predicted - actual; por eso se resta al simular el observado.
sim_dpp = np.clip(DPP_CENTRAL - sampled_pv_errors[:, 0], 0, 1)
sim_rpp = np.clip(RPP_CENTRAL - sampled_pv_errors[:, 1], 0, 1)
major_total = sim_dpp + sim_rpp
overflow = major_total > 0.995
sim_dpp[overflow] = sim_dpp[overflow] / major_total[overflow] * 0.995
sim_rpp[overflow] = sim_rpp[overflow] / major_total[overflow] * 0.995
sim_other = 1 - sim_dpp - sim_rpp

# Cámara: la distribución procede directamente de los 435 distritos.
# No se aplica una cuota nacional ni se vuelve a simular un total agregado.
if len(house_simulations) != N_SIM:
    raise AssertionError(
        f"House produjo {len(house_simulations):,} simulaciones y Bloque 6 espera {N_SIM:,}."
    )
sim_d_house = house_simulations["D House Seats"].to_numpy(dtype=int, copy=True)
sim_r_house = house_simulations["R House Seats"].to_numpy(dtype=int, copy=True)
if not np.all(sim_d_house + sim_r_house == HOUSE_TOTAL):
    raise AssertionError("La simulación distrital House contiene totales distintos de 435.")

# Senado: simulación jerárquica directamente en puntos de margen.
# El shock común se estima con errores OOF por ciclo; el error estatal se estima
# con los residuos idiosincráticos parcialmente pooled. No existe cuota de flips.
race_states = senate_race_attribution["STATE"].tolist()
race_margins = senate_projected_margins.reindex(race_states).to_numpy(float)
race_idio_sigmas = senate_idiosyncratic_sigmas.reindex(race_states).to_numpy(float)
if not np.isfinite(race_margins).all() or not np.isfinite(race_idio_sigmas).all():
    raise ValueError("Los márgenes o sigmas estatales del Senado contienen faltantes.")

common_shock_pp = rng.normal(
    loc=0.0, scale=senate_common_sigma_pp, size=(N_SIM, 1)
)
state_shocks_pp = rng.normal(
    loc=0.0, scale=race_idio_sigmas[None, :], size=(N_SIM, len(race_states))
)
simulated_race_margins = race_margins[None, :] + common_shock_pp + state_shocks_pp
race_dem_wins = simulated_race_margins > 0
sim_d_senate = SENATE_FIXED_D_SEATS_5B + race_dem_wins.sum(axis=1)
sim_r_senate = SENATE_TOTAL - sim_d_senate
race_probabilities = senate_calibrated_probabilities.reindex(race_states).to_numpy(float)

sim_house_margin = sim_d_house - sim_r_house
sim_senate_margin = sim_d_senate - sim_r_senate
sim_house_control = np.where(sim_d_house >= 218, "Democratic", "Republican")
# En 2026 la vicepresidencia es republicana: 50-50 implica control republicano.
sim_senate_control = np.where(sim_d_senate >= 51, "Democratic", "Republican")

simulations = pd.DataFrame({
    "Simulation": np.arange(1, N_SIM + 1),
    "DPP": sim_dpp, "RPP": sim_rpp, "Other": sim_other,
    "D House Seats": sim_d_house, "R House Seats": sim_r_house,
    "House Democratic Margin": sim_house_margin, "House Control": sim_house_control,
    "D Senate Seats": sim_d_senate, "R Senate Seats": sim_r_senate,
    "Senate Democratic Margin": sim_senate_margin, "Senate Control": sim_senate_control
})

senate_race_simulation_summary = pd.DataFrame({
    "STATE": race_states,
    "Model D Win Probability (%)": np.round(100 * race_probabilities, 2),
    "Simulated D Win Rate (%)": np.round(100 * race_dem_wins.mean(axis=0), 2),
    "Simulated R Win Rate (%)": np.round(100 * (1 - race_dem_wins.mean(axis=0)), 2)
})

# ------------------------------------------------------------
# 5. RESÚMENES E INTERVALOS
# ------------------------------------------------------------

house_control_probs = simulations["House Control"].value_counts(normalize=True).mul(100).round(2).reset_index()
house_control_probs.columns = ["Outcome", "Probability (%)"]
senate_control_probs = simulations["Senate Control"].value_counts(normalize=True).mul(100).round(2).reset_index()
senate_control_probs.columns = ["Outcome", "Probability (%)"]

distribution_summary = pd.DataFrame([
    summarize_distribution(simulations["D House Seats"], "Democratic House Seats"),
    summarize_distribution(simulations["R House Seats"], "Republican House Seats"),
    summarize_distribution(simulations["House Democratic Margin"], "House Democratic Margin"),
    summarize_distribution(simulations["D Senate Seats"], "Democratic Senate Seats"),
    summarize_distribution(simulations["R Senate Seats"], "Republican Senate Seats"),
    summarize_distribution(simulations["Senate Democratic Margin"], "Senate Democratic Margin"),
    summarize_distribution(simulations["DPP"] * 100, "Democratic Popular Vote (%)"),
    summarize_distribution(simulations["RPP"] * 100, "Republican Popular Vote (%)"),
    summarize_distribution(simulations["Other"] * 100, "Other Popular Vote (%)")
]).round(2)

interval_specs = [
    ("Democratic Popular Vote", DPP_CENTRAL * 100, simulations["DPP"] * 100, False),
    ("Republican Popular Vote", RPP_CENTRAL * 100, simulations["RPP"] * 100, False),
    ("Other Popular Vote", OTHER_CENTRAL * 100, simulations["Other"] * 100, False),
    ("Democratic House", D_HOUSE_CENTRAL, simulations["D House Seats"], True),
    ("Republican House", R_HOUSE_CENTRAL, simulations["R House Seats"], True),
    ("House Democratic Margin", HOUSE_DEM_MARGIN_CENTRAL, simulations["House Democratic Margin"], True),
    ("Democratic Senate", D_SENATE_CENTRAL, simulations["D Senate Seats"], True),
    ("Republican Senate", R_SENATE_CENTRAL, simulations["R Senate Seats"], True),
    ("Senate Democratic Margin", SENATE_DEM_MARGIN_CENTRAL, simulations["Senate Democratic Margin"], True)
]

interval_rows = []
for outcome, central, values, integer in interval_specs:
    row = {
        "Outcome": outcome, "Central": central,
        "Lower 90": np.percentile(values, 5), "Upper 90": np.percentile(values, 95),
        "Lower 95": np.percentile(values, 2.5), "Upper 95": np.percentile(values, 97.5)
    }
    if integer:
        row = {key: int(round(value)) if key != "Outcome" else value for key, value in row.items()}
    else:
        row = {key: round(value, 2) if key != "Outcome" else value for key, value in row.items()}
    interval_rows.append(row)
prediction_intervals = pd.DataFrame(interval_rows)

range_summary = pd.DataFrame({
    "Outcome": [
        "Democratic House", "Republican House", "Democratic Senate", "Republican Senate",
        "Democratic PV", "Republican PV", "Other Vote"
    ],
    "Most Likely": [
        D_HOUSE_CENTRAL, R_HOUSE_CENTRAL, D_SENATE_CENTRAL, R_SENATE_CENTRAL,
        round(DPP_CENTRAL * 100, 2), round(RPP_CENTRAL * 100, 2), round(OTHER_CENTRAL * 100, 2)
    ],
    "90% Range": [
        format_seat_range(*np.percentile(simulations["D House Seats"], [5, 95])),
        format_seat_range(*np.percentile(simulations["R House Seats"], [5, 95])),
        format_seat_range(*np.percentile(simulations["D Senate Seats"], [5, 95])),
        format_seat_range(*np.percentile(simulations["R Senate Seats"], [5, 95])),
        format_range(*np.percentile(simulations["DPP"] * 100, [5, 95])),
        format_range(*np.percentile(simulations["RPP"] * 100, [5, 95])),
        format_range(*np.percentile(simulations["Other"] * 100, [5, 95]))
    ],
    "95% Range": [
        format_seat_range(*np.percentile(simulations["D House Seats"], [2.5, 97.5])),
        format_seat_range(*np.percentile(simulations["R House Seats"], [2.5, 97.5])),
        format_seat_range(*np.percentile(simulations["D Senate Seats"], [2.5, 97.5])),
        format_seat_range(*np.percentile(simulations["R Senate Seats"], [2.5, 97.5])),
        format_range(*np.percentile(simulations["DPP"] * 100, [2.5, 97.5])),
        format_range(*np.percentile(simulations["RPP"] * 100, [2.5, 97.5])),
        format_range(*np.percentile(simulations["Other"] * 100, [2.5, 97.5]))
    ]
})

print("Simulaciones generadas:", len(simulations))
print("Media simulada de escaños D en Senado:", round(simulations["D Senate Seats"].mean(), 3))
display(senate_race_simulation_summary)
display(prediction_intervals)
display(range_summary)
display(distribution_summary)
print("=" * 70)
print("BLOQUE 6 COMPLETADO")
print("=" * 70)


## BLOQUE 6C — UNCERTAINTY REPORT CONSOLIDATION Probability Snapshot + Risk Summary + Exports

In [ ]:
# ============================================================
# BLOQUE 6C — UNCERTAINTY REPORT CONSOLIDATION
# Probability Snapshot + Risk Summary + Exports
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. VERIFICACIONES
# ------------------------------------------------------------

required_objects = [
    "simulations",
    "prediction_intervals",
    "range_summary",
    "distribution_summary",
    "group_error_summary",
    "error_summary"
]

for obj in required_objects:
    if obj not in globals():
        raise ValueError(f"No encontré {obj}. Corre primero Bloque 6A y 6B.")

# ------------------------------------------------------------
# 2. PROBABILIDADES DE CONTROL
# ------------------------------------------------------------

house_dem_prob = (simulations["D House Seats"] >= 218).mean() * 100
house_rep_prob = (simulations["R House Seats"] >= 218).mean() * 100

senate_dem_prob = (simulations["D Senate Seats"] >= 51).mean() * 100
senate_rep_prob = (simulations["R Senate Seats"] >= 50).mean() * 100
senate_tie_prob = (simulations["D Senate Seats"] == 50).mean() * 100

probability_snapshot = pd.DataFrame({
    "Outcome": [
        "Democratic House Control",
        "Republican House Control",
        "Democratic Senate Control",
        "Republican Senate Control",
        "Senate 50-50"
    ],
    "Probability (%)": [
        house_dem_prob,
        house_rep_prob,
        senate_dem_prob,
        senate_rep_prob,
        senate_tie_prob
    ]
})

probability_snapshot["Probability (%)"] = (
    probability_snapshot["Probability (%)"]
    .round(2)
)

# ------------------------------------------------------------
# 3. MÁRGENES Y RIESGO ELECTORAL
# ------------------------------------------------------------

house_close_5 = (simulations["House Democratic Margin"].abs() <= 5).mean() * 100
house_close_10 = (simulations["House Democratic Margin"].abs() <= 10).mean() * 100

senate_close_1 = (simulations["Senate Democratic Margin"].abs() <= 2).mean() * 100
senate_close_2 = (simulations["Senate Democratic Margin"].abs() <= 4).mean() * 100

dem_house_220_plus = (simulations["D House Seats"] >= 220).mean() * 100
rep_house_220_plus = (simulations["R House Seats"] >= 220).mean() * 100

dem_senate_52_plus = (simulations["D Senate Seats"] >= 52).mean() * 100
rep_senate_50_plus = (simulations["R Senate Seats"] >= 50).mean() * 100

upset_risk = pd.DataFrame({
    "Metric": [
        "House margin within 5 seats",
        "House margin within 10 seats",
        "Senate margin within 1 seat equivalent",
        "Senate margin within 2 seat equivalent",
        "Democratic House >= 220",
        "Republican House >= 220",
        "Democratic Senate >= 52",
        "Republican Senate >= 50"
    ],
    "Probability (%)": [
        house_close_5,
        house_close_10,
        senate_close_1,
        senate_close_2,
        dem_house_220_plus,
        rep_house_220_plus,
        dem_senate_52_plus,
        rep_senate_50_plus
    ]
})

upset_risk["Probability (%)"] = upset_risk["Probability (%)"].round(2)

# ------------------------------------------------------------
# 4. MARGIN SUMMARY
# ------------------------------------------------------------

margin_summary = pd.DataFrame([
    {
        "Chamber": "House",
        "Central Democratic Margin": HOUSE_DEM_MARGIN_CENTRAL,
        "Mean Simulated Margin": simulations["House Democratic Margin"].mean(),
        "Median Simulated Margin": simulations["House Democratic Margin"].median(),
        "P05": np.percentile(simulations["House Democratic Margin"], 5),
        "P95": np.percentile(simulations["House Democratic Margin"], 95),
        "Probability Democratic Control (%)": house_dem_prob,
        "Probability Republican Control (%)": house_rep_prob
    },
    {
        "Chamber": "Senate",
        "Central Democratic Margin": SENATE_DEM_MARGIN_CENTRAL,
        "Mean Simulated Margin": simulations["Senate Democratic Margin"].mean(),
        "Median Simulated Margin": simulations["Senate Democratic Margin"].median(),
        "P05": np.percentile(simulations["Senate Democratic Margin"], 5),
        "P95": np.percentile(simulations["Senate Democratic Margin"], 95),
        "Probability Democratic Control (%)": senate_dem_prob,
        "Probability Republican Control (%)": senate_rep_prob
    }
])

margin_summary = margin_summary.round(2)

# ------------------------------------------------------------
# 5. VARIABILITY METRICS
# ------------------------------------------------------------

def variability_metric(series, label):
    mean = series.mean()
    sd = series.std(ddof=1)
    cv = sd / abs(mean) if mean != 0 else np.nan

    return {
        "Metric": label,
        "Mean": mean,
        "SD": sd,
        "CV": cv
    }

variability_metrics = pd.DataFrame([
    variability_metric(simulations["DPP"] * 100, "Democratic Popular Vote"),
    variability_metric(simulations["RPP"] * 100, "Republican Popular Vote"),
    variability_metric(simulations["Other"] * 100, "Other Popular Vote"),
    variability_metric(simulations["D House Seats"], "Democratic House Seats"),
    variability_metric(simulations["R House Seats"], "Republican House Seats"),
    variability_metric(simulations["D Senate Seats"], "Democratic Senate Seats"),
    variability_metric(simulations["R Senate Seats"], "Republican Senate Seats")
]).round(3)

# ------------------------------------------------------------
# 6. ELECTORAL RISK SUMMARY
# ------------------------------------------------------------

def risk_label(prob_control, margin_p05, margin_p95):
    crosses_zero = margin_p05 <= 0 <= margin_p95

    if crosses_zero and 45 <= prob_control <= 55:
        return "Very Competitive"

    if crosses_zero:
        return "Competitive"

    if prob_control >= 80:
        return "Favored"

    if prob_control >= 65:
        return "Lean Favored"

    return "Uncertain"

house_risk_label = risk_label(
    house_dem_prob,
    np.percentile(simulations["House Democratic Margin"], 5),
    np.percentile(simulations["House Democratic Margin"], 95)
)

senate_risk_label = risk_label(
    senate_dem_prob,
    np.percentile(simulations["Senate Democratic Margin"], 5),
    np.percentile(simulations["Senate Democratic Margin"], 95)
)

electoral_risk = pd.DataFrame({
    "Component": [
        "House",
        "Senate",
        "Popular Vote"
    ],
    "Risk / Status": [
        house_risk_label,
        senate_risk_label,
        "Range-Based Estimate"
    ],
    "Interpretation": [
        "Based on the national House Monte Carlo distribution and nested historical error.",
        "Based on state-level Senate simulations calibrated with historical polling error.",
        "Based only on simulated popular vote uncertainty."
    ]
})

# ------------------------------------------------------------
# 7. UNCERTAINTY SNAPSHOT FINAL
# ------------------------------------------------------------

def get_range(outcome, column):
    row = range_summary[range_summary["Outcome"] == outcome]
    if len(row) == 0:
        return ""
    return row[column].iloc[0]

uncertainty_snapshot_final = pd.DataFrame({
    "Metric": [
        "Central Popular Vote",
        "Democratic Popular Vote 90% Range",
        "Republican Popular Vote 90% Range",
        "Other Popular Vote 90% Range",
        "Central House Projection",
        "Democratic House 90% Range",
        "Republican House 90% Range",
        "House Democratic Margin 90% Range",
        "House Democratic Control Probability",
        "House Republican Control Probability",
        "Central Senate Projection",
        "Democratic Senate 90% Range",
        "Republican Senate 90% Range",
        "Senate Democratic Margin 90% Range",
        "Senate Democratic Control Probability",
        "Senate Republican Control Probability",
        "Senate 50-50 Probability",
        "Monte Carlo Simulations"
    ],
    "Value": [
        f"D {DPP_CENTRAL*100:.2f}% - R {RPP_CENTRAL*100:.2f}% - O {OTHER_CENTRAL*100:.2f}%",
        get_range("Democratic PV", "90% Range"),
        get_range("Republican PV", "90% Range"),
        get_range("Other Vote", "90% Range"),
        f"D {D_HOUSE_CENTRAL} - R {R_HOUSE_CENTRAL}",
        get_range("Democratic House", "90% Range"),
        get_range("Republican House", "90% Range"),
        f"{int(np.percentile(simulations['House Democratic Margin'], 5))} – {int(np.percentile(simulations['House Democratic Margin'], 95))}",
        f"{house_dem_prob:.2f}%",
        f"{house_rep_prob:.2f}%",
        f"D {D_SENATE_CENTRAL} - R {R_SENATE_CENTRAL}",
        get_range("Democratic Senate", "90% Range"),
        get_range("Republican Senate", "90% Range"),
        f"{int(np.percentile(simulations['Senate Democratic Margin'], 5))} – {int(np.percentile(simulations['Senate Democratic Margin'], 95))}",
        f"{senate_dem_prob:.2f}%",
        f"{senate_rep_prob:.2f}%",
        f"{senate_tie_prob:.2f}%",
        len(simulations)
    ]
})

# ------------------------------------------------------------
# 8. REPORTE EN PANTALLA
# ------------------------------------------------------------

print("="*70)
print("BLOQUE 6C — UNCERTAINTY REPORT CONSOLIDATION")
print("="*70)

print("\nPROBABILITY SNAPSHOT")
display(probability_snapshot)

print("\nUPSET / CLOSE-RACE RISK")
display(upset_risk)

print("\nMARGIN SUMMARY")
display(margin_summary)

print("\nVARIABILITY METRICS")
display(variability_metrics)

print("\nELECTORAL RISK SUMMARY")
display(electoral_risk)

print("\nFINAL UNCERTAINTY SNAPSHOT")
display(uncertainty_snapshot_final)

# ------------------------------------------------------------
# 9. EXPORTACIONES
# ------------------------------------------------------------

probability_snapshot.to_excel(
    OUTPUT_DIR / "Block6C_Probability_Snapshot.xlsx",
    index=False
)

upset_risk.to_excel(
    OUTPUT_DIR / "Block6C_Close_Race_Risk.xlsx",
    index=False
)

margin_summary.to_excel(
    OUTPUT_DIR / "Block6C_Margin_Summary.xlsx",
    index=False
)

variability_metrics.to_excel(
    OUTPUT_DIR / "Block6C_Variability_Metrics.xlsx",
    index=False
)

electoral_risk.to_excel(
    OUTPUT_DIR / "Block6C_Electoral_Risk.xlsx",
    index=False
)

uncertainty_snapshot_final.to_excel(
    OUTPUT_DIR / "Block6C_Final_Uncertainty_Snapshot.xlsx",
    index=False
)

with pd.ExcelWriter(OUTPUT_DIR / "Block6_Final_Uncertainty_Report.xlsx") as writer:
    uncertainty_snapshot_final.to_excel(writer, sheet_name="Final Snapshot", index=False)
    probability_snapshot.to_excel(writer, sheet_name="Probability Snapshot", index=False)
    prediction_intervals.to_excel(writer, sheet_name="Prediction Intervals", index=False)
    range_summary.to_excel(writer, sheet_name="Range Summary", index=False)
    distribution_summary.to_excel(writer, sheet_name="Distribution Summary", index=False)
    margin_summary.to_excel(writer, sheet_name="Margin Summary", index=False)
    upset_risk.to_excel(writer, sheet_name="Close Race Risk", index=False)
    variability_metrics.to_excel(writer, sheet_name="Variability", index=False)
    electoral_risk.to_excel(writer, sheet_name="Electoral Risk", index=False)
    group_error_summary.to_excel(writer, sheet_name="Historical Group Error", index=False)
    error_summary.to_excel(writer, sheet_name="Variable Error", index=False)
    simulations.head(5000).to_excel(writer, sheet_name="Monte Carlo Sample", index=False)

print("\nArchivos guardados:")
print("- Block6C_Probability_Snapshot.xlsx")
print("- Block6C_Close_Race_Risk.xlsx")
print("- Block6C_Margin_Summary.xlsx")
print("- Block6C_Variability_Metrics.xlsx")
print("- Block6C_Electoral_Risk.xlsx")
print("- Block6C_Final_Uncertainty_Snapshot.xlsx")
print("- Block6_Final_Uncertainty_Report.xlsx")

print("\nBLOQUE 6 COMPLETADO.")

## BLOQUE 7 — FINAL REPORT GENERATOR

In [ ]:
# ============================================================
# BLOQUE 7 — FINAL REPORT & WORKBOOK EXPORTER
# Versión limpia, robusta y preparada para Bloque 8 HTML
# ============================================================

import pandas as pd
import numpy as np
from datetime import datetime, timezone

print("="*70)
print("BLOQUE 7 — FINAL REPORT & WORKBOOK EXPORTER")
print("="*70)

# ------------------------------------------------------------
# 1. VERIFICACIONES
# ------------------------------------------------------------

required_objects = [
    # Core outputs
    "block3_predictions",
    "group_summary",
    "block2_predictions",
    "national_nested_oof",
    "national_nested_fold_diagnostics",
    "national_final_selection",
    "national_tuning_summary",
    "national_outer_tuning",
    "loocv_df",
    "loocv_summary",
    "national_outcome_oof",
    "national_outcome_summary",
    "national_time_machine_selected",
    "national_time_machine_scorecard",
    "national_time_machine_target_summary",
    "national_exam_progression",
    "national_exam_outcome_comparison",
    "national_2026_fold_targets",
    "national_2026_fold_forecasts",
    "national_2026_fold_summary",
    "national_2026_target_stability",
    "national_pipeline_target_exam",
    "national_pipeline_stage_summary",
    "national_pipeline_derived_summary",
    "national_validation_stage_contract",
    "national_architecture_contract",
    "final_2026_full_projection_42",
    "unit_normalization_audit",
    "historical_input_reconciliation",
    "national_popular_vote_validation",
    "national_popular_vote_method_summary",
    "popular_vote_production_bridge",
    "popular_vote_dependency_audit",
    "national_module_contract",
    "national_state_isolation_audit",
    "senate_specification_summary",
    "senate_specification_nested_folds",
    "senate_specification_contract",
    "senate_2026_fold_summary",
    "senate_2026_fold_races",
    "senate_2026_race_stability",

    # Block 5
    "executive_summary",
    "final_projection",
    "final_snapshot",
    "model_quality",
    "election_snapshot",
    "popular_vote_table",
    "control_summary",
    "house_distribution",
    "senate_distribution",

    # Block 5B
    "senate_attribution_summary",
    "senate_model_flips",
    "senate_competitive_races",
    "senate_race_attribution",
    "senate_historic_clean",
    "senate_historic_summary",
    "senate_race_simulation_summary",
    "senate_state_model_cv",
    "state_model_diagnostics",
    "senate_validation_summary",
    "senate_nested_fold_diagnostics",
    "senate_cycle_validation",

    # Block 6 / 6C
    "group_error_summary",
    "prediction_intervals",
    "range_summary",
    "distribution_summary",
    "probability_snapshot",
    "upset_risk",
    "margin_summary",
    "variability_metrics",
    "electoral_risk",
    "uncertainty_snapshot_final",
    "simulations",
    # Block 5C — district House layer
    "house_race_detail", "house_competitive_races",
    "house_validation_oof", "house_validation_folds",
    "house_validation_summary", "house_simulations",
    "house_simulation_summary", "house_feature_contract",
    "house_leakage_audit"
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise ValueError(
        "Faltan objetos necesarios. Ejecuta los bloques anteriores:\n"
        + "\n".join(missing_objects)
    )

# ------------------------------------------------------------
# 2. FUNCIONES AUXILIARES
# ------------------------------------------------------------

projection = block3_predictions.copy()

if "Group" not in group_summary.columns:
    group_summary = group_summary.reset_index()

def get_projection_value(variable):
    row = projection[projection["Variable"] == variable]

    if len(row) == 0:
        return np.nan

    return row["Constrained Prediction"].iloc[0]

def get_stability(group_name):
    row = group_summary[group_summary["Group"] == group_name]

    if len(row) == 0:
        return np.nan

    return float(row["Stability"].iloc[0])

def safe_percent(x):
    try:
        return round(float(x) * 100, 2)
    except Exception:
        return np.nan

def safe_value_from_snapshot(metric_name):
    if "Metric" not in uncertainty_snapshot_final.columns:
        return np.nan

    row = uncertainty_snapshot_final[
        uncertainty_snapshot_final["Metric"] == metric_name
    ]

    if len(row) == 0:
        return np.nan

    return row["Value"].iloc[0]

# ------------------------------------------------------------
# 3. DASHBOARD DATA
# ------------------------------------------------------------

DPP = float(get_projection_value("DPP"))
RPP = float(get_projection_value("RPP"))
OTHER = max(0, 1 - DPP - RPP)

DHOUSE = int(HOUSE_DISTRICT_D_CENTRAL)
RHOUSE = int(HOUSE_DISTRICT_R_CENTRAL)

DSEN = int(D_SENATE_CENTRAL_5B)
RSEN = int(R_SENATE_CENTRAL_5B)

HOUSE_MARGIN = DHOUSE - RHOUSE
SENATE_MARGIN = DSEN - RSEN

HOUSE_CONTROL = "Democratic" if DHOUSE > RHOUSE else "Republican"

# Preserve the national House output as an explicit comparator, then make the
# downstream district simulation median the canonical House headline.
house_national_comparator = final_projection.loc[
    final_projection["Chamber"].eq("House")
].copy()
_old_house = house_national_comparator.iloc[0]
_baseline_d_house = int(_old_house["Democratic Seats"] - _old_house["Democratic Net Change"])
_baseline_r_house = int(_old_house["Republican Seats"] - _old_house["Republican Net Change"])
for _frame in [final_projection, control_summary]:
    _mask = _frame["Chamber"].eq("House")
    _frame.loc[_mask, "Democratic Seats"] = DHOUSE
    _frame.loc[_mask, "Republican Seats"] = RHOUSE
    _frame.loc[_mask, "Projected Control"] = HOUSE_CONTROL
    _frame.loc[_mask, "Margin"] = abs(DHOUSE - RHOUSE)
    _frame.loc[_mask, "Democratic Net Change"] = DHOUSE - _baseline_d_house
    _frame.loc[_mask, "Republican Net Change"] = RHOUSE - _baseline_r_house
for _frame in [executive_summary, final_snapshot]:
    _frame.loc[_frame["Metric"].eq("House Projection"), "Projection"] = f"D {DHOUSE} - R {RHOUSE}"
    _frame.loc[_frame["Metric"].eq("House Control"), "Projection"] = HOUSE_CONTROL
    _frame.loc[_frame["Metric"].eq("House Margin"), "Projection"] = abs(DHOUSE - RHOUSE)
    _frame.loc[_frame["Metric"].eq("House Democratic Net Change"), "Projection"] = DHOUSE - _baseline_d_house
SENATE_CONTROL = "Democratic" if DSEN > RSEN else "Republican"

overall_stability = float(
    ml_diagnostics["Model Stability Score"].mean()
)
popular_stability = get_stability("Popular Vote")
house_stability = get_stability("House Buckets")
senate_stability = get_stability("Senate Buckets")

# Probabilidades principales
def get_probability(metric):
    if "Outcome" in probability_snapshot.columns:
        row = probability_snapshot[probability_snapshot["Outcome"] == metric]
    elif "Metric" in probability_snapshot.columns:
        row = probability_snapshot[probability_snapshot["Metric"] == metric]
    else:
        return np.nan

    if len(row) == 0:
        return np.nan

    if "Probability (%)" in row.columns:
        return row["Probability (%)"].iloc[0]
    if "Probability" in row.columns:
        return row["Probability"].iloc[0]

    return np.nan

dashboard_data = pd.DataFrame({
    "Metric": [
        "Democratic Popular Vote",
        "Republican Popular Vote",
        "Other Popular Vote",

        "Democratic House Seats",
        "Republican House Seats",
        "House Control",
        "House Democratic Margin",

        "Democratic Senate Seats",
        "Republican Senate Seats",
        "Senate Control",
        "Senate Democratic Margin",

        "Overall Diagnostic Stability",
        "Popular Vote Diagnostic Stability",
        "House Diagnostic Stability",
        "Senate Diagnostic Stability",

        "House Democratic Control Probability",
        "House Republican Control Probability",
        "Senate Democratic Control Probability",
        "Senate Republican Control Probability",
        "Senate 50-50 Probability",

        "Monte Carlo Simulations",

        "Senate Assigned Flips",
        "Senate Competitive Races"
    ],
    "Value": [
        f"{DPP*100:.2f}%",
        f"{RPP*100:.2f}%",
        f"{OTHER*100:.2f}%",

        DHOUSE,
        RHOUSE,
        HOUSE_CONTROL,
        HOUSE_MARGIN,

        DSEN,
        RSEN,
        SENATE_CONTROL,
        SENATE_MARGIN,

        f"{overall_stability:.1f}/100",
        f"{popular_stability:.1f}/100",
        f"{house_stability:.1f}/100",
        f"{senate_stability:.1f}/100",

        get_probability("Democratic House Control"),
        get_probability("Republican House Control"),
        get_probability("Democratic Senate Control"),
        get_probability("Republican Senate Control"),
        get_probability("Senate 50-50"),

        len(simulations),

        len(senate_model_flips),
        len(senate_competitive_races)
    ]
})

# ------------------------------------------------------------
# 3B. CONTRATO DE COHERENCIA Y TRAZABILIDAD
# ------------------------------------------------------------

detail_d_wins = int((senate_race_attribution["Projected Winner"] == "D").sum())
detail_d_total = int(SENATE_FIXED_D_SEATS_5B + detail_d_wins)
detail_flip_rows = int(senate_race_attribution["Model Assigned Outcome"].isin(
    ["Democratic Flip", "Republican Flip"]
).sum())

consistency_checks = [
    ("Input unit contract passed", bool(unit_normalization_audit["Passed"].all()),
     f"{int(unit_normalization_audit['Passed'].sum())}/{len(unit_normalization_audit)} checks"),
    ("Historical identity exceptions are resolved deterministically", bool(
        historical_input_reconciliation["Resolution Status"].eq("Resolved").all()
    ), (
        f"{int(historical_input_reconciliation['Source Exception'].sum())} "
        "documented exceptions; resolved by final totals or non-two-party composition"
    )),
    ("Diagnostic stability is bounded", bool(
        block4.loc[block4["Status"] == "ML Predicted", "Model Stability Score"]
        .between(0, 100).all()
    ), "heuristic sensitivity index; not a probability"),
    ("Time Machine covers five historical tests",
     national_time_machine_selected["Year"].nunique() == 5,
     f"{national_time_machine_selected['Year'].nunique()} outer test cycles"),
    ("Time Machine has 42 targets per historical test",
     len(national_time_machine_selected) == 5 * len(p_cols),
     f"{len(national_time_machine_selected)} comparisons"),
    ("2026 jackknife has five leave-cycle-out forecasts",
     len(national_2026_fold_forecasts) == 5,
     f"{len(national_2026_fold_forecasts)} partial forecasts"),
    ("Nested two-exam design covers every outer cycle and target group",
     len(national_exam_progression) == 5 * len(modeling_groups),
     f"{len(national_exam_progression)} cycle-group second exams"),
    ("2026 production forecast uses all historical cycles",
     set(train_df["Midterm Year"].astype(int)) == {2006, 2010, 2014, 2018, 2022},
     "partial-fold mean is diagnostic only"),
    ("Popular-vote dependency audit passed", bool(
        popular_vote_dependency_audit["Passed"].all()
    ), f"{int(popular_vote_dependency_audit['Passed'].sum())}/"
       f"{len(popular_vote_dependency_audit)} checks"),
    ("National/State-Senate isolation audit passed", bool(
        national_state_isolation_audit["Passed"].all()
    ), f"{int(national_state_isolation_audit['Passed'].sum())}/"
       f"{len(national_state_isolation_audit)} checks"),
    ("Popular vote production sums to 100", bool(np.isclose(
        DPP + RPP + OTHER, 1.0, atol=1e-10
    )), f"{(DPP + RPP + OTHER) * 100:.10f}%"),
    ("All 42 targets have five 2026 sensitivity forecasts", bool(
        len(national_2026_target_stability) == 42
    ), f"{len(national_2026_target_stability)}/42 targets"),
    ("Fold means are diagnostic-only", bool(
        national_2026_target_stability["Fold Mean Role"].eq(
            "Sensitivity diagnostic; never training data"
        ).all()
    ), "No fold mean enters production training"),
    ("Pipeline exam never retrains upstream", bool(
        (~national_validation_stage_contract[
            "Can Retrain Upstream"
        ]).all()
    ), "Selection, constraints and Monte Carlo remain one-way"),
    ("Senate specification challenge confirms production package", bool(
        production_specification_selection_5b["specification"]
        == "PooledFullV11"
    ), production_specification_selection_5b["specification"]),
    ("State effects were tested without forcing them", bool(
        "PartialPoolFull" in set(
            senate_specification_summary["Specification"]
        )
    ), "Partial pooling evaluated under nested LOEO"),
    ("Five state-Senate 2026 sensitivity models completed", bool(
        senate_2026_fold_summary["Omitted Historical Cycle"].nunique()
        == 5
    ), f"{senate_2026_fold_summary['Omitted Historical Cycle'].nunique()}/5 folds"),
    ("House totals 435", DHOUSE + RHOUSE == 435, f"{DHOUSE}+{RHOUSE}"),
    ("Senate totals 100", DSEN + RSEN == 100, f"{DSEN}+{RSEN}"),
    ("Every House simulation totals 435", bool((
        simulations["D House Seats"] + simulations["R House Seats"] == 435
    ).all()), "all simulations"),
    ("Every Senate simulation totals 100", bool((
        simulations["D Senate Seats"] + simulations["R Senate Seats"] == 100
    ).all()), "all simulations"),
    ("House Monte Carlo is not discretely degenerate",
     simulations["D House Seats"].nunique() >= 10,
     f"{simulations['D House Seats'].nunique()} distinct Democratic seat totals"),
    ("Senate detail reconstructs dashboard", detail_d_total == DSEN,
     f"detail D={detail_d_total}, dashboard D={DSEN}"),
    ("Flip table equals race detail", len(senate_model_flips) == detail_flip_rows,
     f"table={len(senate_model_flips)}, detail={detail_flip_rows}"),
    ("D/R probability complements", np.allclose(
        senate_race_attribution["D Win Probability"]
        + senate_race_attribution["R Win Probability"], 100, atol=0.02
    ), "race probabilities"),
    ("Margin/probability/winner coherent", bool((
        ((senate_race_attribution["Adjusted Margin 2P"] >= 0)
         == (senate_race_attribution["Projected Winner"] == "D"))
    ).all()), "all Senate races"),
    ("Parser margin matches DEM MEAN minus REP MEAN", bool(np.allclose(
        senate_race_attribution["MARGIN"],
        senate_race_attribution["DEM MEAN"] - senate_race_attribution["REP MEAN"],
        atol=0.03
    )), "all Senate races"),
    ("Projected margin is the state-model output", bool(np.allclose(
        senate_race_attribution["Adjusted Margin 2P"],
        senate_race_attribution["Model Projected Margin 2P"],
        atol=0.02
    )), "poll anchor plus nested polling-error correction; no seat-quota reconciliation"),
    ("Projected two-party vote reconstructs margin", bool(np.allclose(
        senate_race_attribution["Projected D 2P"]
        - senate_race_attribution["Projected R 2P"],
        senate_race_attribution["Adjusted Margin 2P"],
        atol=0.04
    )), "D 2P minus R 2P"),
    ("Model ratings follow database margin thresholds", bool((
        senate_race_attribution.apply(
            lambda row: party_margin_rating(row["Adjusted Margin 2P"]), axis=1
        ) == senate_race_attribution["Forecast Rating"]
    ).all()), "Toss-Up <=1; Lean <=5; Likely <=10; Safe >10"),
    ("Poll-only and error-corrected candidate weights form a valid convex combination", bool(
        0 <= POLL_WEIGHT_5B <= 1
        and 0 <= FUNDAMENTALS_WEIGHT_5B <= 1
        and np.isclose(POLL_WEIGHT_5B + FUNDAMENTALS_WEIGHT_5B, 1.0)
    ), f"poll_only_candidate={POLL_WEIGHT_5B:.4f}, error_corrected_candidate={FUNDAMENTALS_WEIGHT_5B:.4f}"),
    ("Baseline safeguard is coherent with the applied margin", bool(
        (
            FUNDAMENTALS_WEIGHT_5B == 0
            and np.allclose(
                senate_race_attribution["Adjusted Margin 2P"],
                senate_race_attribution["Poll Margin 2P"]
            )
        )
        or (
            FUNDAMENTALS_WEIGHT_5B > 0
            and not np.allclose(
                senate_race_attribution["Adjusted Margin 2P"],
                senate_race_attribution["Poll Margin 2P"]
            )
        )
    ), BASELINE_SAFEGUARD_STATUS_5B),
    ("Senate forecast values are finite", bool(np.isfinite(
        senate_race_attribution[[
            "Adjusted Margin 2P", "D Win Probability", "R Win Probability",
            "Forecast Sigma PP"
        ]].to_numpy(float)
    ).all()), "margins, probabilities and sigmas"),
    ("No numerical warnings in polling-error model", len(numerical_warning_log_5b) == 0,
     f"warnings={len(numerical_warning_log_5b)}"),
]
consistency_checks.extend([
    ("House district audit passed", bool(house_leakage_audit["Passed"].all()),
     f"{int(house_leakage_audit['Passed'].sum())}/{len(house_leakage_audit)} checks"),
    ("House layer contains 435 unique districts",
     len(house_race_detail) == 435 and house_race_detail["District ID"].nunique() == 435,
     f"rows={len(house_race_detail)}, unique={house_race_detail['District ID'].nunique()}"),
    ("House winners follow district margin signs", bool((
        (house_race_detail["Projected D-R Margin"] >= 0)
        == (house_race_detail["Projected Winner"] == "D")
    ).all()), "all 435 districts"),
    ("House central total is the simulation median",
     int(np.median(house_simulations["D House Seats"])) == DHOUSE,
     f"median D={int(np.median(house_simulations['D House Seats']))}; dashboard D={DHOUSE}"),
    ("House expected seats equal summed probabilities",
     np.isclose(HOUSE_EXPECTED_D_SEATS,
                house_race_detail["D Win Probability"].sum() / 100.0),
     f"expected D={HOUSE_EXPECTED_D_SEATS:.3f}"),
    ("House national prior is comparator only",
     "National Reconciliation Shift" not in house_race_detail.columns,
     f"district D={DHOUSE}; frozen national prior D={HOUSE_NATIONAL_D_SEAT_PRIOR}"),
    ("House model emitted no RuntimeWarning", len(house_numeric_warning_log) == 0,
     f"warnings={len(house_numeric_warning_log)}"),
])
consistency_audit = pd.DataFrame(consistency_checks, columns=["Check", "Passed", "Detail"])

if not consistency_audit["Passed"].all():
    raise AssertionError(
        "EL MODELO NO SUPERÓ LA AUDITORÍA FINAL:\n"
        + consistency_audit.loc[~consistency_audit["Passed"]].to_string(index=False)
    )

run_metadata = pd.DataFrame({
    "Field": [
        "Run ID", "Source File", "Source SHA-256", "Source Modified UTC",
        "Run Started UTC", "Report Generated UTC", "Historical Years",
        "Feature Count", "P Target Count", "Monte Carlo Simulations"
    ],
    "Value": [
        RUN_ID, str(MODEL_FILE.resolve()), MODEL_SHA256, MODEL_MODIFIED_UTC,
        RUN_STARTED_UTC, datetime.now(timezone.utc).isoformat(),
        ", ".join(map(str, train_df["Midterm Year"].tolist())),
        len(feature_cols), len(p_cols), len(simulations)
    ]
})

run_metadata = pd.concat([
    run_metadata,
    pd.DataFrame({
        "Field": [
            "Model Version",
            "National Validation",
            "Popular Vote Validation",
            "Module Isolation",
            "Senate Validation",
            "National Complexity Tolerance",
            "Senate Baseline Safeguard",
            "Input Unit Contract",
            "Diagnostic Stability Definition",
            "Validation Limitation",
        ],
        "Value": [
            "v17",
            "Five outer historical tests with inner LOEO selection; 42 targets per test",
            "Margin-first nested competition: expectation anchor, guarded error correction, Ridge, trees and historical mean",
            "One-way frozen national signal contract; counterfactual and exact-mutation tests",
            "Fully nested alpha, weights and calibration by election cycle",
            NATIONAL_COMPLEXITY_TOLERANCE,
            BASELINE_SAFEGUARD_STATUS_5B,
            "Economic batteries normalized to proportions; net favorability recomputed",
            "Sensitivity to TreeEnsemble and constraint engine; heuristic, not accuracy",
            "Five completed midterm cycles; 210 target comparisons are repeated measures, not 210 independent elections; source-derived identities reconciled in InputReconciliation",
        ],
    }),
], ignore_index=True)

# ------------------------------------------------------------
# 4. BLOQUE 4 OPCIONALES
# ------------------------------------------------------------

if "largest_disagreements" not in globals():
    if "diagnostics" in globals() and "Model Difference" in diagnostics.columns:
        largest_disagreements = (
            diagnostics[diagnostics["Status"] == "ML Predicted"]
            .sort_values("Model Difference", ascending=False)
            .head(15)
            .copy()
        )
    else:
        largest_disagreements = pd.DataFrame()

if "largest_adjustments" not in globals():
    if "diagnostics" in globals() and "Constraint Impact" in diagnostics.columns:
        largest_adjustments = (
            diagnostics[diagnostics["Status"] == "ML Predicted"]
            .sort_values("Constraint Impact", ascending=False)
            .head(15)
            .copy()
        )
    else:
        largest_adjustments = pd.DataFrame()

if "most_stable" not in globals():
    if "diagnostics" in globals() and "Model Stability Score" in diagnostics.columns:
        most_stable = (
            diagnostics[diagnostics["Status"] == "ML Predicted"]
            .sort_values("Model Stability Score", ascending=False)
            .head(15)
            .copy()
        )
    else:
        most_stable = pd.DataFrame()

if "least_stable" not in globals():
    if "diagnostics" in globals() and "Model Stability Score" in diagnostics.columns:
        least_stable = (
            diagnostics[diagnostics["Status"] == "ML Predicted"]
            .sort_values("Model Stability Score", ascending=True)
            .head(15)
            .copy()
        )
    else:
        least_stable = pd.DataFrame()

# ------------------------------------------------------------
# 5. REPORTE EN PANTALLA
# ------------------------------------------------------------

print("\nEXECUTIVE SUMMARY")
display(executive_summary)

print("\nFINAL PROJECTION")
display(final_projection)

print("\nMODEL QUALITY")
display(model_quality)

print("\nFINAL SNAPSHOT")
display(final_snapshot)

print("\nSENATE MODEL FLIPS")
display(senate_model_flips)

print("\nDASHBOARD DATA")
display(dashboard_data)

# ------------------------------------------------------------
# 6. EXPORTACIÓN MAESTRA
# ------------------------------------------------------------

output_file = FINAL_REPORT_PATH

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    # ========================================================
    # DASHBOARD / EXECUTIVE
    # ========================================================

    dashboard_data.to_excel(writer, sheet_name="Dashboard_Data", index=False)
    run_metadata.to_excel(writer, sheet_name="RunMetadata", index=False)
    consistency_audit.to_excel(writer, sheet_name="ConsistencyAudit", index=False)
    unit_normalization_audit.to_excel(writer, sheet_name="InputUnitAudit", index=False)
    historical_input_reconciliation.to_excel(
        writer, sheet_name="InputReconciliation", index=False
    )
    executive_summary.to_excel(writer, sheet_name="ExecutiveSummary", index=False)
    final_projection.to_excel(writer, sheet_name="FinalProjection", index=False)
    house_national_comparator.to_excel(writer, sheet_name="HouseNationalComparator", index=False)
    final_snapshot.to_excel(writer, sheet_name="FinalSnapshot", index=False)
    model_quality.to_excel(writer, sheet_name="ModelQuality", index=False)

    # ========================================================
    # CENTRAL PROJECTION
    # ========================================================

    election_snapshot.to_excel(writer, sheet_name="ElectionSnapshot", index=False)
    popular_vote_table.to_excel(writer, sheet_name="PopularVote", index=False)
    control_summary.to_excel(writer, sheet_name="ControlSummary", index=False)
    house_distribution.to_excel(writer, sheet_name="HouseDistribution", index=False)
    senate_distribution.to_excel(writer, sheet_name="SenateDistribution", index=False)

    # ========================================================
    # SENATE ATTRIBUTION — BLOCK 5B
    # ========================================================

    senate_attribution_summary.to_excel(
        writer,
        sheet_name="SenateAttribution",
        index=False
    )

    senate_model_flips.to_excel(
        writer,
        sheet_name="SenateModelFlips",
        index=False
    )

    senate_competitive_races.to_excel(
        writer,
        sheet_name="SenateCompetitive",
        index=False
    )

    senate_race_attribution.to_excel(
        writer,
        sheet_name="SenateRaceDetail",
        index=False
    )

    senate_historic_clean.to_excel(
        writer,
        sheet_name="SenateHistoricClean",
        index=False
    )

    senate_historic_summary.to_excel(
        writer,
        sheet_name="SenateHistoricSummary",
        index=False
    )

    senate_race_simulation_summary.to_excel(
        writer,
        sheet_name="SenateRaceSimulation",
        index=False
    )

    senate_state_model_cv.to_excel(
        writer, sheet_name="SenateStateModelCV", index=False
    )

    state_model_diagnostics.to_excel(
        writer, sheet_name="SenateStateModel", index=False
    )

    senate_validation_summary.to_excel(
        writer, sheet_name="SenateValidation", index=False
    )

    senate_nested_fold_diagnostics.to_excel(
        writer, sheet_name="SenateNestedFolds", index=False
    )

    senate_cycle_validation.to_excel(
        writer, sheet_name="SenateCycleValidation", index=False
    )
    senate_specification_summary.to_excel(
        writer, sheet_name="SenateSpecSummary", index=False
    )
    senate_specification_nested_folds.to_excel(
        writer, sheet_name="SenateSpecFolds", index=False
    )
    senate_specification_contract.to_excel(
        writer, sheet_name="SenateSpecContract", index=False
    )
    senate_2026_fold_summary.to_excel(
        writer, sheet_name="Senate2026Folds", index=False
    )
    senate_2026_fold_races.to_excel(
        writer, sheet_name="Senate2026FoldRaces", index=False
    )
    senate_2026_race_stability.to_excel(
        writer, sheet_name="SenateRaceStability", index=False
    )

    # ========================================================
    # NATIONAL MODEL VALIDATION — BLOCK 2 v12
    # ========================================================

    block2_predictions.to_excel(
        writer, sheet_name="NationalForecast", index=False
    )
    national_nested_oof.to_excel(
        writer, sheet_name="NationalNestedOOF", index=False
    )
    national_nested_fold_diagnostics.to_excel(
        writer, sheet_name="NationalNestedFolds", index=False
    )
    national_final_selection.to_excel(
        writer, sheet_name="NationalFinalSelection", index=False
    )
    national_tuning_summary.to_excel(
        writer, sheet_name="NationalFinalTuning", index=False
    )
    national_outer_tuning.to_excel(
        writer, sheet_name="NationalOuterTuning", index=False
    )
    loocv_df.to_excel(
        writer, sheet_name="NationalModelOOF", index=False
    )
    loocv_summary.to_excel(
        writer, sheet_name="NationalModelSummary", index=False
    )
    national_outcome_oof.to_excel(
        writer, sheet_name="NationalOutcomeOOF", index=False
    )
    national_outcome_summary.to_excel(
        writer, sheet_name="NationalOutcomeSummary", index=False
    )
    national_time_machine_selected.to_excel(
        writer, sheet_name="TimeMachine42Targets", index=False
    )
    national_time_machine_scorecard.to_excel(
        writer, sheet_name="TimeMachineScorecard", index=False
    )
    national_time_machine_target_summary.to_excel(
        writer, sheet_name="TimeMachineTargetSummary", index=False
    )
    national_exam_progression.to_excel(
        writer, sheet_name="NestedExamProgression", index=False
    )
    national_exam_outcome_comparison.to_excel(
        writer, sheet_name="SecondExamImprovement", index=False
    )
    national_2026_fold_targets.to_excel(
        writer, sheet_name="TimeMachine2026Targets", index=False
    )
    national_2026_fold_forecasts.to_excel(
        writer, sheet_name="TimeMachine2026Folds", index=False
    )
    national_2026_fold_summary.to_excel(
        writer, sheet_name="TimeMachine2026Summary", index=False
    )
    national_2026_target_stability.to_excel(
        writer, sheet_name="TargetStability2026", index=False
    )
    national_pipeline_target_exam.to_excel(
        writer, sheet_name="PipelineTargetExam", index=False
    )
    national_pipeline_stage_summary.to_excel(
        writer, sheet_name="PipelineStageSummary", index=False
    )
    national_pipeline_derived_summary.to_excel(
        writer, sheet_name="PipelineDerived", index=False
    )
    national_validation_stage_contract.to_excel(
        writer, sheet_name="ValidationStages", index=False
    )
    national_architecture_contract.to_excel(
        writer, sheet_name="ArchitectureContract", index=False
    )
    national_popular_vote_validation.to_excel(
        writer, sheet_name="PopularVoteValidation", index=False
    )
    national_popular_vote_method_summary.to_excel(
        writer, sheet_name="PopularVoteMethods", index=False
    )
    popular_vote_production_bridge.to_excel(
        writer, sheet_name="PopularVoteBridge", index=False
    )
    national_state_isolation_audit.to_excel(
        writer, sheet_name="ModuleIsolation", index=False
    )
    national_module_contract.to_excel(
        writer, sheet_name="ModuleContract", index=False
    )


    # ========================================================
    # HOUSE DISTRICT ATTRIBUTION — BLOCK 5C
    # ========================================================
    house_race_detail.to_excel(writer, sheet_name="HouseRaceDetail", index=False)
    house_competitive_races.to_excel(writer, sheet_name="HouseCompetitive", index=False)
    house_validation_oof.to_excel(writer, sheet_name="HouseValidationOOF", index=False)
    house_validation_folds.to_excel(writer, sheet_name="HouseValidationFolds", index=False)
    house_validation_summary.to_excel(writer, sheet_name="HouseValidationSummary", index=False)
    house_simulations.to_excel(writer, sheet_name="HouseSimulations", index=False)
    house_simulation_summary.to_excel(writer, sheet_name="HouseSimulationSummary", index=False)
    house_feature_contract.to_excel(writer, sheet_name="HouseFeatureContract", index=False)
    house_leakage_audit.to_excel(writer, sheet_name="HouseLeakageAudit", index=False)

    # ========================================================
    # CONSTRAINTS / MODEL DIAGNOSTICS
    # ========================================================

    block3_predictions.to_excel(
        writer,
        sheet_name="Block3_Constraints",
        index=False
    )

    group_summary.to_excel(
        writer,
        sheet_name="Block4_ModelQuality",
        index=False
    )

    if len(largest_adjustments) > 0:
        largest_adjustments.to_excel(
            writer,
            sheet_name="Block4_Adjustments",
            index=False
        )

    if len(largest_disagreements) > 0:
        largest_disagreements.to_excel(
            writer,
            sheet_name="Block4_Disagreement",
            index=False
        )

    if len(most_stable) > 0:
        most_stable.to_excel(
            writer,
            sheet_name="Block4_MostStable",
            index=False
        )

    if len(least_stable) > 0:
        least_stable.to_excel(
            writer,
            sheet_name="Block4_LeastStable",
            index=False
        )

    # ========================================================
    # UNCERTAINTY / MONTE CARLO
    # ========================================================

    group_error_summary.to_excel(
        writer,
        sheet_name="HistoricalErrors",
        index=False
    )

    prediction_intervals.to_excel(
        writer,
        sheet_name="PredictionIntervals",
        index=False
    )

    range_summary.to_excel(
        writer,
        sheet_name="PredictionRanges",
        index=False
    )

    distribution_summary.to_excel(
        writer,
        sheet_name="MonteCarloSummary",
        index=False
    )

    probability_snapshot.to_excel(
        writer,
        sheet_name="ControlProbability",
        index=False
    )

    upset_risk.to_excel(
        writer,
        sheet_name="CloseRaceRisk",
        index=False
    )

    margin_summary.to_excel(
        writer,
        sheet_name="MarginSummary",
        index=False
    )

    variability_metrics.to_excel(
        writer,
        sheet_name="Variability",
        index=False
    )

    electoral_risk.to_excel(
        writer,
        sheet_name="ElectoralRisk",
        index=False
    )

    uncertainty_snapshot_final.to_excel(
        writer,
        sheet_name="FinalUncertainty",
        index=False
    )

    simulations.head(5000).to_excel(
        writer,
        sheet_name="MonteCarloSample",
        index=False
    )

    # Reader-facing workbook formatting. The data remain unchanged.
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.utils import get_column_letter

    narrative_headers = {
        "Check", "Detail", "Architecture", "Input", "Output",
        "Information Allowed", "Information Prohibited", "Consumes",
        "Produces", "May Feed", "May Not Feed", "Production Rule",
        "Fold Mean Role", "Role", "Numeric Features",
        "Categorical Features", "Resolution Status", "Source Exception",
    }
    for worksheet in writer.book.worksheets:
        worksheet.freeze_panes = "A2"
        worksheet.sheet_view.showGridLines = False
        worksheet.auto_filter.ref = worksheet.dimensions
        worksheet.row_dimensions[1].height = 30
        for cell in worksheet[1]:
            cell.fill = PatternFill("solid", fgColor="16324F")
            cell.font = Font(color="FFFFFF", bold=True)
            cell.alignment = Alignment(
                horizontal="center", vertical="center", wrap_text=True
            )
        for column_index, column_cells in enumerate(
            worksheet.iter_cols(1, worksheet.max_column), start=1
        ):
            header = str(column_cells[0].value or "")
            sampled_cells = column_cells[: min(len(column_cells), 250)]
            maximum_length = max(
                len(str(cell.value)) if cell.value is not None else 0
                for cell in sampled_cells
            )
            if header in narrative_headers:
                width = min(max(maximum_length + 2, 24), 70)
            else:
                width = min(max(maximum_length + 2, 11), 28)
            worksheet.column_dimensions[
                get_column_letter(column_index)
            ].width = width
        worksheet.auto_filter.ref = worksheet.dimensions

print("\nArchivo generado:")
print(output_file)

print("\n" + "="*70)
print("MODELO COMPLETADO")
print("="*70)

print(f"""
Resumen Final

Popular Vote
-------------
Democratic : {DPP*100:.2f}%
Republican : {RPP*100:.2f}%
Other       : {OTHER*100:.2f}%

House
-------------
Democrats   : {DHOUSE}
Republicans : {RHOUSE}
Control     : {HOUSE_CONTROL}
Margin      : {HOUSE_MARGIN}

Senate
-------------
Democrats   : {DSEN}
Republicans : {RSEN}
Control     : {SENATE_CONTROL}
Margin      : {SENATE_MARGIN}

Senate Assigned Flips
-------------
{len(senate_model_flips)}

Overall Diagnostic Stability
-------------
{overall_stability:.1f}/100

Monte Carlo Simulations
-------------
{len(simulations):,}

Workbook Exported
-------------
{output_file}
""")

In [ ]:
# ============================================================
# BLOQUE 8 — HTML FORECAST DASHBOARD
# U.S. Midterm Elections 2026
# Dashboard visual profesional inspirado en forecast pages
# ============================================================

import pandas as pd
import base64
import gzip
import html as html_lib
import json
import hashlib
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.offline.offline import get_plotlyjs
from datetime import datetime
from pathlib import Path

print("="*70)
print("BLOQUE 8 — HTML FORECAST DASHBOARD")
print("="*70)

# ------------------------------------------------------------
# 1. ARCHIVOS
# ------------------------------------------------------------

final_report_path = FINAL_REPORT_PATH
output_html = HTML_OUTPUT_PATH

if not final_report_path.exists():
    raise FileNotFoundError(
        "No encontré Election_Model_Final_Report.xlsx. Ejecutá primero el Bloque 7."
    )

model_excel_path = Path(MODEL_FILE) if "MODEL_FILE" in globals() else Path("Model.xlsx")
if not model_excel_path.exists():
    raise FileNotFoundError(f"No encontré la fuente canónica: {model_excel_path}")

# ------------------------------------------------------------
# 2. CARGAR REPORTE FINAL
# ------------------------------------------------------------

dashboard_data = pd.read_excel(final_report_path, sheet_name="Dashboard_Data")
final_projection = pd.read_excel(final_report_path, sheet_name="FinalProjection")
model_quality = pd.read_excel(final_report_path, sheet_name="ModelQuality")
senate_flips = pd.read_excel(final_report_path, sheet_name="SenateModelFlips")
senate_detail = pd.read_excel(final_report_path, sheet_name="SenateRaceDetail")
senate_historic_summary = pd.read_excel(final_report_path, sheet_name="SenateHistoricSummary")
senate_competitive = pd.read_excel(final_report_path, sheet_name="SenateCompetitive")
probability_snapshot = pd.read_excel(final_report_path, sheet_name="ControlProbability")
electoral_risk = pd.read_excel(final_report_path, sheet_name="ElectoralRisk")
final_uncertainty = pd.read_excel(final_report_path, sheet_name="FinalUncertainty")
monte_carlo = pd.read_excel(final_report_path, sheet_name="MonteCarloSample")
run_metadata = pd.read_excel(final_report_path, sheet_name="RunMetadata")
consistency_audit = pd.read_excel(final_report_path, sheet_name="ConsistencyAudit")
time_machine_scorecard = pd.read_excel(
    final_report_path, sheet_name="TimeMachineScorecard"
)
time_machine_outcomes = pd.read_excel(
    final_report_path, sheet_name="NationalOutcomeOOF"
)
time_machine_2026_folds = pd.read_excel(
    final_report_path, sheet_name="TimeMachine2026Folds"
)
time_machine_2026_summary = pd.read_excel(
    final_report_path, sheet_name="TimeMachine2026Summary"
)
nested_exam_progression = pd.read_excel(
    final_report_path, sheet_name="NestedExamProgression"
)
second_exam_improvement = pd.read_excel(
    final_report_path, sheet_name="SecondExamImprovement"
)
popular_vote_methods = pd.read_excel(
    final_report_path, sheet_name="PopularVoteMethods"
)
popular_vote_bridge = pd.read_excel(
    final_report_path, sheet_name="PopularVoteBridge"
)
module_isolation = pd.read_excel(
    final_report_path, sheet_name="ModuleIsolation"
)
senate_cycle_validation = pd.read_excel(
    final_report_path, sheet_name="SenateCycleValidation"
)
senate_validation = pd.read_excel(
    final_report_path, sheet_name="SenateValidation"
)
house_detail = pd.read_excel(final_report_path, sheet_name="HouseRaceDetail")
house_competitive = pd.read_excel(final_report_path, sheet_name="HouseCompetitive")
house_validation_folds = pd.read_excel(final_report_path, sheet_name="HouseValidationFolds")
house_validation_summary = pd.read_excel(final_report_path, sheet_name="HouseValidationSummary")
house_simulation_summary = pd.read_excel(final_report_path, sheet_name="HouseSimulationSummary")
house_feature_contract = pd.read_excel(final_report_path, sheet_name="HouseFeatureContract")
house_leakage_audit = pd.read_excel(final_report_path, sheet_name="HouseLeakageAudit")

if not consistency_audit["Passed"].astype(bool).all():
    raise ValueError("El reporte final contiene auditorías fallidas; no se generará el HTML.")
metadata = run_metadata.set_index("Field")["Value"]
current_model_hash = hashlib.sha256(model_excel_path.read_bytes()).hexdigest()
report_model_hash = str(metadata.get("Source SHA-256", ""))
if current_model_hash != report_model_hash:
    raise ValueError(
        "Election_Model_Final_Report.xlsx fue generado con otro Model.xlsx. "
        "Ejecuta nuevamente los Bloques 1–7 antes del HTML."
    )

# ------------------------------------------------------------
# 3. CARGAR FILA 2026 DESDE HOJA MODEL
# ------------------------------------------------------------

context_metrics = {}

try:
    raw_model = pd.read_excel(model_excel_path, sheet_name="Model", header=None)

    header_row = None
    for idx, row in raw_model.iterrows():
        values = [str(v).strip() for v in row.tolist()]
        if "Midterm Year" in values:
            header_row = idx
            break

    if header_row is not None:
        model_df = raw_model.iloc[header_row + 1:].copy()
        model_df.columns = raw_model.iloc[header_row].astype(str).str.strip()
        model_df = model_df[model_df["Midterm Year"].notna()].copy()
        model_df["Midterm Year"] = pd.to_numeric(model_df["Midterm Year"], errors="coerce")
        row_2026 = model_df[model_df["Midterm Year"] == 2026].iloc[0]

        context_cols = {
            "Presidential Approval": "Presidential Approval",
            "Presidential Disapproval": "Presidential Disapproval",
            "Right Track": "Right Track",
            "Wrong Track": "Wrong Track",
            "Democratic Net Favorability": "DEM NET FAVORABILITY",
            "Republican Net Favorability": "REP NET FAVORABILITY",
            "Unemployment": "Uneyployment",
            "Inflation": "Inflation",
            "Economy Excellent": "ECC Excellent",
            "Economy Good": "ECC Good",
            "Economy Only Fair": "ECC Only Fair",
            "Economy Poor": "ECC Poor",
            "Good Time to Find a Job": "Job Good Time",
            "Bad Time to Find a Job": "Job Bad Time"
        }

        for label, col in context_cols.items():
            if col in model_df.columns:
                context_metrics[label] = row_2026[col]

except Exception as e:
    print("Aviso: no pude cargar contexto nacional desde la hoja Model.")
    print(e)

# ------------------------------------------------------------
# 4. FUNCIONES
# ------------------------------------------------------------

def get_dash(metric):
    row = dashboard_data[dashboard_data["Metric"] == metric]
    if len(row) == 0:
        return np.nan
    return row["Value"].iloc[0]

def clean_pct(x):
    if isinstance(x, str):
        return float(x.replace("%", "").replace(",", "").strip())
    return float(x)

def safe_num(x):
    try:
        return float(str(x).replace("%", "").replace(",", "").strip())
    except:
        return np.nan

def format_party_margin(value):
    numeric = safe_num(value)
    if not np.isfinite(numeric):
        return "—"
    if numeric > 0:
        return f"D+{numeric:.2f}"
    if numeric < 0:
        return f"R+{abs(numeric):.2f}"
    return "EVEN"

def fmt_context(x):
    if pd.isna(x):
        return "—"
    if isinstance(x, str):
        return x
    try:
        if abs(float(x)) <= 1:
            return f"{float(x)*100:.2f}%"
        return f"{float(x):.2f}%"
    except:
        return str(x)

def _format_table_cell(value, column, row=None):
    """Añade color semántico y jerarquía sin cambiar ningún valor calculado."""
    import html as html_lib

    if pd.isna(value):
        return "<span class='cell-muted'>—</span>"

    text = html_lib.escape(str(value))
    normalized = str(value).strip().upper()
    column_lower = str(column).lower()

    # Voto y probabilidades: ambos partidos conservan su color; solo el mayor
    # valor de cada pareja recibe negrita y fondo suave.
    paired_metrics = {
        "Projected D": ("Projected R", "dem"),
        "Projected R": ("Projected D", "rep"),
        "D Win %": ("R Win %", "dem"),
        "R Win %": ("D Win %", "rep"),
    }
    if column in paired_metrics:
        counterpart, party = paired_metrics[column]
        try:
            numeric = float(value)
            other = float(row[counterpart]) if row is not None else np.nan
            emphasis = "is-leading" if np.isfinite(other) and numeric > other else "is-trailing"
            return (
                f"<span class='party-number party-number-{party} {emphasis}'>"
                f"{numeric:.2f}%</span>"
            )
        except (TypeError, ValueError, KeyError):
            pass

    if normalized == "D" or normalized.startswith("DEMOCRATIC") or normalized.startswith("D "):
        return f"<span class='party-pill party-dem'>{text}</span>"
    if normalized == "R" or normalized.startswith("REPUBLICAN") or normalized.startswith("R "):
        return f"<span class='party-pill party-rep'>{text}</span>"
    if "toss-up" in normalized.lower():
        return f"<span class='rating-pill rating-tossup'>{text}</span>"
    if "safe" in normalized.lower():
        return f"<span class='rating-pill rating-safe'>{text}</span>"
    if "likely" in normalized.lower():
        return f"<span class='rating-pill rating-likely'>{text}</span>"
    if "lean" in normalized.lower():
        return f"<span class='rating-pill rating-lean'>{text}</span>"
    directional_columns = (
        "margin" in column_lower
        or "historic adj" in column_lower
        or "historic bias" in column_lower
        or "national adj" in column_lower
    )
    if directional_columns:
        try:
            numeric = float(value)
            if numeric > 0:
                return f"<span class='margin-pill margin-dem'>D+{numeric:.2f}</span>"
            if numeric < 0:
                return f"<span class='margin-pill margin-rep'>R+{abs(numeric):.2f}</span>"
            return "<span class='margin-pill margin-even'>EVEN</span>"
        except (TypeError, ValueError):
            pass
    if column in {"DEM MEAN", "REP MEAN"}:
        try:
            return f"{float(value):.2f}%"
        except (TypeError, ValueError):
            pass
    return text


def table_html(df, max_rows=12, label="Data table"):
    if df is None or len(df) == 0:
        return "<p class='muted'>No data available.</p>"

    clean_df = df.head(max_rows).copy().replace({np.nan: "—"})
    formatted_df = clean_df.astype(object).copy()
    for row_index, row in clean_df.iterrows():
        for column in clean_df.columns:
            formatted_df.at[row_index, column] = _format_table_cell(
                row[column], column, row
            )
    clean_df = formatted_df

    table = clean_df.to_html(
        index=False,
        classes="data-table",
        border=0,
        escape=False
    )
    return f"""
    <div class="table-shell" role="region" aria-label="{label}" tabindex="0">
        {table}
    </div>
    <div class="scroll-hint">Scroll horizontally to inspect every variable →</div>
    """

# ------------------------------------------------------------
# 5. VARIABLES PRINCIPALES
# ------------------------------------------------------------

DPP = clean_pct(get_dash("Democratic Popular Vote"))
RPP = clean_pct(get_dash("Republican Popular Vote"))
OTHER = clean_pct(get_dash("Other Popular Vote"))

D_HOUSE = int(get_dash("Democratic House Seats"))
R_HOUSE = int(get_dash("Republican House Seats"))
HOUSE_CONTROL = str(get_dash("House Control"))
HOUSE_MARGIN = int(get_dash("House Democratic Margin"))

D_SENATE = int(get_dash("Democratic Senate Seats"))
R_SENATE = int(get_dash("Republican Senate Seats"))
SENATE_CONTROL = str(get_dash("Senate Control"))
SENATE_MARGIN = int(get_dash("Senate Democratic Margin"))

OVERALL_STABILITY = str(get_dash("Overall Diagnostic Stability"))

HOUSE_D_PROB = safe_num(get_dash("House Democratic Control Probability"))
HOUSE_R_PROB = safe_num(get_dash("House Republican Control Probability"))
SENATE_D_PROB = safe_num(get_dash("Senate Democratic Control Probability"))
SENATE_R_PROB = safe_num(get_dash("Senate Republican Control Probability"))
SENATE_5050 = safe_num(get_dash("Senate 50-50 Probability"))

SIMS = int(get_dash("Monte Carlo Simulations"))
ASSIGNED_FLIPS = int(get_dash("Senate Assigned Flips"))
COMPETITIVE_RACES = int(get_dash("Senate Competitive Races"))

last_updated = datetime.now().strftime("%B %d, %Y · %I:%M %p")

# ------------------------------------------------------------
# 6. PALETA
# ------------------------------------------------------------

DEM = "#0B5CAB"
DEM_DARK = "#073B75"
DEM_LIGHT = "#8AB6FF"

REP = "#C1121F"
REP_DARK = "#7F0000"
REP_LIGHT = "#FF7F8A"

PURPLE = "#8B5CF6"
GREEN = "#10B981"
ORANGE = "#F97316"
GOLD = "#F59E0B"
TEAL = "#14B8A6"
GRAY = "#D9D9D9"
TEXT = "#111827"

# Canonical Núcleo 42 brand asset. The SVG is embedded so the generated
# dashboard remains a single offline HTML file.
logo_candidates = [
    BASE_DIR / "Midterms_2026_Logo.svg", BASE_DIR / "Midterms_2026_Logo.png",
    BASE_DIR / "assets" / "branding" / "Midterms_2026_Logo.svg",
]
logo_path = next((path for path in logo_candidates if path.exists()), None)
if logo_path is None:
    brand_mark_html = '<div class="brand-mark" aria-label="Model logo placeholder"></div>'
else:
    logo_mime = "image/svg+xml" if logo_path.suffix.lower() == ".svg" else "image/png"
    logo_payload = base64.b64encode(logo_path.read_bytes()).decode("ascii")
    brand_mark_html = (
        f'<img class="brand-logo" src="data:{logo_mime};base64,{logo_payload}" '
        'alt="Midterms 2026 model logo">'
    )

def probability_card(title, party, value, color):
    return f"""
    <div class="prob-card">
        <div class="prob-title">{title}</div>
        <div class="prob-circle" style="--p:{value}; --c:{color};">
            <div class="prob-inner">
                <div class="prob-value">{value:.1f}%</div>
                <div class="prob-party">{party}</div>
            </div>
        </div>
    </div>
    """

probability_cards_html = f"""
<div class="prob-grid">
    {probability_card("House Control", "Democratic", HOUSE_D_PROB, DEM)}
    {probability_card("House Control", "Republican", HOUSE_R_PROB, REP)}
    {probability_card("Senate Control", "Democratic", SENATE_D_PROB, DEM)}
    {probability_card("Senate Control", "Republican", SENATE_R_PROB, REP)}
</div>
"""


def context_color(label):
    label = label.lower()

    if "approval" in label and "disapproval" not in label:
        return GREEN
    if "disapproval" in label:
        return ORANGE
    if "right track" in label:
        return GREEN
    if "wrong track" in label:
        return ORANGE
    if "democratic" in label:
        return DEM
    if "republican" in label:
        return REP
    if "unemployment" in label:
        return GOLD
    if "inflation" in label:
        return ORANGE
    if "excellent" in label or "good time" in label:
        return GREEN
    if "poor" in label or "bad time" in label:
        return ORANGE
    if "only fair" in label:
        return GOLD
    if "economy good" in label:
        return TEAL

    return PURPLE

# ------------------------------------------------------------
# 7. CLASES VISUALES DINÁMICAS
# ------------------------------------------------------------

house_card_class = "card card-rep" if HOUSE_CONTROL == "Republican" else "card card-dem"
senate_card_class = "card card-dem" if SENATE_CONTROL == "Democratic" else "card card-rep"

house_control_class = "rep" if HOUSE_CONTROL == "Republican" else "dem"
senate_control_class = "dem" if SENATE_CONTROL == "Democratic" else "rep"

house_d_color = DEM
house_r_color = REP
senate_d_color = DEM
senate_r_color = REP

# ------------------------------------------------------------
# 8. MAPAS HOUSE Y SENADO
# ------------------------------------------------------------

# Official TIGER/Line 2026 CD120 geometry supplied as an Esri File
# Geodatabase is converted once to an offline SVG path package.  The display
# uses the standard composite Albers USA parameters (the same conventional
# treatment used by D3 geoAlbersUsa), so no state is independently stretched.
HOUSE_CD120_SOURCE_FILE = "tlgdb_2026_us_legislative.gdb.zip"
HOUSE_CD120_SOURCE_SHA256 = "6708d7052ac4c07c32c241f341685572fdf5c38d57a0c3144fab4751e702cfbc"
HOUSE_CD120_SOURCE_LAYER = "Congressional_Districts"
HOUSE_CD120_SOURCE_CRS = "EPSG:4269"
HOUSE_CARTOGRAPHIC_MASK_FILE = "cb_2025_us_state_500k.zip"
HOUSE_CARTOGRAPHIC_MASK_SHA256 = "9cbfe171dad1555e11770c981d8f4db9e687a65c86f5bdae684eeb487e2e9b80"
HOUSE_CARTOGRAPHIC_MASK_LAYER = "cb_2025_us_state_500k"
HOUSE_CARTOGRAPHIC_MASK_CRS = "EPSG:4269"
HOUSE_CD120_PATHS_ASSET = BASE_DIR / "assets" / "house_cd120_albers_paths.json.gz"
HOUSE_CD120_PATHS_SHA256 = "cd0c01239022760c8ad5b0d85d8868eaee97ee7c8e1c4f6c24e2ade113cd576e"
HOUSE_CD120_PATHS_B64 = "..."

if HOUSE_CD120_PATHS_ASSET.exists():
    house_map_geometry_bytes = HOUSE_CD120_PATHS_ASSET.read_bytes()
    if hashlib.sha256(house_map_geometry_bytes).hexdigest() != HOUSE_CD120_PATHS_SHA256:
        raise AssertionError("El activo externo de geometría House no coincide con su checksum.")
    HOUSE_CD120_GEOMETRY_MODE = "external-auditable-asset"
else:
    house_map_geometry_bytes = base64.b64decode(HOUSE_CD120_PATHS_B64)
    HOUSE_CD120_GEOMETRY_MODE = "embedded-portable-fallback"

house_map_geometry = json.loads(gzip.decompress(house_map_geometry_bytes))
if len(house_map_geometry.get("districts", [])) != 435:
    raise AssertionError("La geometría oficial CD120 no contiene 435 distritos con voto.")
if len(house_map_geometry.get("states", [])) != 50:
    raise AssertionError("La geometría oficial CD120 no contiene los 50 contornos estatales.")

house_detail = house_detail.reset_index(drop=True)
house_detail["GEOID4"] = (
    house_detail["GEOID4"].astype(str).str.replace(".0", "", regex=False).str.zfill(4)
)
for column in ["Projected Margin PP", "D Win Probability", "R Win Probability"]:
    house_detail[column] = pd.to_numeric(house_detail[column], errors="coerce")
if house_detail["GEOID4"].nunique() != 435:
    raise AssertionError("HouseRaceDetail no contiene 435 GEOID únicos.")
house_index_by_geoid = {
    geoid: index for index, geoid in enumerate(house_detail["GEOID4"])
}


def _probability_svg_color(probability):
    probability = float(probability)
    if probability >= 95: return "#073B75"
    if probability >= 80: return "#1769AA"
    if probability >= 65: return "#5B9BD5"
    if probability >= 55: return "#A8D5EC"
    if probability > 45: return "#F4D35E"
    if probability > 35: return "#F2B5B9"
    if probability > 20: return "#DF6670"
    if probability > 5: return "#C1121F"
    return "#7F0000"


house_svg_paths = []
for district_geometry in house_map_geometry["districts"]:
    geoid = str(district_geometry["GEOID"]).zfill(4)
    if geoid not in house_index_by_geoid:
        raise AssertionError(f"Geometría CD120 sin fila de pronóstico: {geoid}")
    row_index = house_index_by_geoid[geoid]
    row = house_detail.iloc[row_index]
    probability = safe_num(row.get("D Win Probability", np.nan))
    margin = safe_num(row.get("Projected Margin PP", np.nan))
    rating = str(row.get("Forecast Rating", "Toss-Up"))
    consensus = str(row.get("All Source Consensus Rating", "Not rated"))
    winner = str(row.get("Projected Winner", ""))
    flip = str(row.get("Projected Flip", "Hold"))
    district = str(row.get("District ID", geoid))
    source_count = row.get("All Source Count", "")
    title = (
        f"{district} · {rating} · D {probability:.1f}% · "
        f"{format_party_margin(margin)}"
    )
    house_svg_paths.append(
        f'<path class="house-district" tabindex="0" d="{district_geometry["d"]}" '
        f'fill="{_probability_svg_color(probability)}" '
        f'data-index="{row_index}" data-geoid="{geoid}" '
        f'data-district="{html_lib.escape(district, quote=True)}" '
        f'data-prob="{probability:.6f}" data-rprob="{100.0 - probability:.6f}" data-margin="{margin:.6f}" '
        f'data-rating="{html_lib.escape(rating, quote=True)}" '
        f'data-consensus="{html_lib.escape(consensus, quote=True)}" '
        f'data-winner="{html_lib.escape(winner, quote=True)}" '
        f'data-flip="{html_lib.escape(flip, quote=True)}" '
        f'data-incumbent="{html_lib.escape(str(row.get("Incumbent Name", "Open seat")), quote=True)}" '
        f'data-pvi="{html_lib.escape(str(row.get("PVI Raw", "")), quote=True)}" '
        f'data-sources="{html_lib.escape(str(source_count), quote=True)}" '
        f'aria-label="{html_lib.escape(title, quote=True)}"><title>'
        f'{html_lib.escape(title)}</title></path>'
    )

house_state_outline_paths = [
    f'<path class="house-state-outline" d="{state_geometry["d"]}"></path>'
    for state_geometry in house_map_geometry["states"]
]

alaska_inset = house_map_geometry["insets"]["alaska"]
hawaii_inset = house_map_geometry["insets"]["hawaii"]
house_inset_frame_svg = "".join(
    f'<rect class="house-inset-frame" x="{extent[0]:.2f}" y="{extent[1]:.2f}" '
    f'width="{extent[2] - extent[0]:.2f}" height="{extent[3] - extent[1]:.2f}" rx="8"></rect>'
    for extent in [alaska_inset, hawaii_inset]
)
house_inset_label_svg = "".join(
    f'<g class="house-inset-label"><rect x="{extent[0] + 6:.2f}" y="{extent[1] + 6:.2f}" '
    f'width="{badge_width:.2f}" height="14" rx="5"></rect>'
    f'<text x="{extent[0] + 11:.2f}" y="{extent[1] + 16:.2f}">{label}</text></g>'
    for label, extent, badge_width in [
        ("ALASKA", alaska_inset, 49.0),
        ("HAWAII", hawaii_inset, 47.0),
    ]
)

if len(house_svg_paths) != 435:
    raise AssertionError(f"El mapa SVG House contiene {len(house_svg_paths)} distritos.")


# Equal-area district cartogram. One square is one voting district, so tiny
# urban seats, Alaska and Hawaii remain as visible and interactive as every
# other seat. States are ordered west-to-east in five geographic bands.
HOUSE_CARTOGRAM_REGIONS = [
    ("Pacific", ["AK", "HI", "WA", "OR", "CA"]),
    ("Mountain", ["ID", "MT", "WY", "NV", "UT", "CO", "AZ", "NM"]),
    ("Midwest", ["ND", "SD", "NE", "KS", "MN", "IA", "MO", "WI", "IL", "MI", "IN", "OH"]),
    ("South", ["TX", "OK", "AR", "LA", "MS", "AL", "TN", "KY", "WV", "VA", "NC", "SC", "GA", "FL"]),
    ("Northeast", ["ME", "NH", "VT", "MA", "RI", "CT", "NY", "NJ", "PA", "DE", "MD"]),
]


def _cartogram_district_tile(row_index, row):
    district = str(row.get("District ID", ""))
    suffix = district.split("-", 1)[-1]
    label = "AL" if suffix in {"AL", "00"} else str(int(suffix))
    probability = safe_num(row.get("D Win Probability", np.nan))
    margin = safe_num(row.get("Projected Margin PP", np.nan))
    rating = str(row.get("Forecast Rating", "Toss-Up"))
    consensus = str(row.get("All Source Consensus Rating", "Not rated"))
    winner = str(row.get("Projected Winner", ""))
    flip = str(row.get("Projected Flip", "Hold"))
    source_count = row.get("All Source Count", "")
    title = f"{district} · {rating} · D {probability:.1f}% · {format_party_margin(margin)}"
    return (
        f'<button type="button" class="house-cart-tile" '
        f'style="background:{_probability_svg_color(probability)}" '
        f'data-index="{row_index}" data-district="{html_lib.escape(district, quote=True)}" '
        f'data-prob="{probability:.6f}" data-rprob="{100.0 - probability:.6f}" '
        f'data-margin="{margin:.6f}" data-rating="{html_lib.escape(rating, quote=True)}" '
        f'data-consensus="{html_lib.escape(consensus, quote=True)}" '
        f'data-winner="{html_lib.escape(winner, quote=True)}" '
        f'data-flip="{html_lib.escape(flip, quote=True)}" '
        f'data-incumbent="{html_lib.escape(str(row.get("Incumbent Name", "Open seat")), quote=True)}" '
        f'data-pvi="{html_lib.escape(str(row.get("PVI Raw", "")), quote=True)}" '
        f'data-sources="{html_lib.escape(str(source_count), quote=True)}" '
        f'aria-label="{html_lib.escape(title, quote=True)}"><span>{label}</span></button>'
    )


cartogram_region_html = []
cartogram_district_count = 0
cartogram_state_count = 0
for region_name, state_order in HOUSE_CARTOGRAM_REGIONS:
    state_blocks = []
    region_seat_count = 0
    for state_abbr in state_order:
        positions = [
            position for position in range(len(house_detail))
            if str(house_detail.iloc[position].get("District ID", "")).startswith(state_abbr + "-")
        ]
        if not positions:
            raise AssertionError(f"El cartograma House no encontró distritos para {state_abbr}.")
        seat_count = len(positions)
        column_count = min(10, max(1, int(np.ceil(np.sqrt(seat_count * 1.65)))))
        district_tiles = "".join(
            _cartogram_district_tile(position, house_detail.iloc[position])
            for position in positions
        )
        state_blocks.append(
            f'<section class="cart-state" aria-label="{state_abbr}: {seat_count} House districts">'
            f'<div class="cart-state-head"><b>{state_abbr}</b><span>{seat_count}</span></div>'
            f'<div class="cart-state-grid" style="grid-template-columns:repeat({column_count},17px)">'
            f'{district_tiles}</div></section>'
        )
        region_seat_count += seat_count
        cartogram_district_count += seat_count
        cartogram_state_count += 1
    cartogram_region_html.append(
        f'<div class="cart-region">'
        f'<div class="cart-region-title"><b>{region_name}</b><span>{region_seat_count} seats</span></div>'
        f'{"".join(state_blocks)}</div>'
    )

if cartogram_district_count != 435 or cartogram_state_count != 50:
    raise AssertionError(
        f"Cartograma House incompleto: {cartogram_district_count} distritos, "
        f"{cartogram_state_count} estados."
    )
house_cartogram_html = "".join(cartogram_region_html)

house_map_html = f'''
<div class="map-toolbar">
  <div>
    <div class="map-eyebrow">435 official CD120 district forecasts</div>
    <div class="map-toolbar-title" id="houseMapTitle">House geographic forecast map</div>
    <div class="map-toolbar-subtitle" id="houseMapDescription">Official CD120 districts · Census cartographic shorelines · standard composite Albers USA.</div>
  </div>
  <div class="map-toolbar-controls">
    <label>Layout
      <select id="houseMapLayout" aria-label="House map layout">
        <option value="geographic" selected>Geographic forecast</option>
        <option value="cartogram">District cartogram</option>
      </select>
    </label>
    <label>View
      <select id="houseMapMetric" aria-label="House map view">
        <option value="probability">Win probability</option>
        <option value="rating">Model ratings</option>
        <option value="consensus">Source consensus</option>
        <option value="margin">Projected margin</option>
        <option value="flips">Holds & flips</option>
      </select>
    </label>
  </div>
</div>
<div class="map-dynamic-legend" id="houseMapLegend">
  <span><i style="background:#7F0000"></i>R &gt;95%</span>
  <span><i style="background:#DF6670"></i>R advantage</span>
  <span><i style="background:#F4D35E"></i>Toss-up</span>
  <span><i style="background:#5B9BD5"></i>D advantage</span>
  <span><i style="background:#073B75"></i>D &gt;95%</span>
</div>
<div class="house-map-visual" id="houseMapVisual">
  <div class="house-cartogram-shell" id="houseCartogram" hidden
       role="group" aria-label="Equal-area House district cartogram grouped by geographic region and state">
    <div class="house-cartogram">{house_cartogram_html}</div>
    <div class="cartogram-note"><b>One square = one voting district.</b> Every seat has equal visual weight; states are grouped west to east.</div>
  </div>
  <div class="house-svg-shell" id="houseGeographicMap">
    <svg id="houseDistrictMap" class="house-svg" viewBox="0 0 960 510" preserveAspectRatio="xMidYMid meet"
         role="img" aria-label="Interactive geographic forecast for all 435 official CD120 voting districts">
      <defs>
        <pattern id="houseFlipD" width="8" height="8" patternUnits="userSpaceOnUse" patternTransform="rotate(45)">
          <rect width="8" height="8" fill="#1769AA"></rect><rect width="3" height="8" fill="#D9ECF7"></rect>
        </pattern>
        <pattern id="houseFlipR" width="8" height="8" patternUnits="userSpaceOnUse" patternTransform="rotate(45)">
          <rect width="8" height="8" fill="#C1121F"></rect><rect width="3" height="8" fill="#F8D5D8"></rect>
        </pattern>
      </defs>
      {house_inset_frame_svg}
      {''.join(house_svg_paths)}
      {''.join(house_state_outline_paths)}
      {house_inset_label_svg}
    </svg>
    <div class="geographic-note"><b>Official geographic forecast.</b> TIGER/Line 2026 CD120 districts clipped to the independent Census 2025 cartographic state shoreline · EPSG:4269 · standard composite Albers USA. Legally assigned water is excluded only from the display; all 435 official GEOID records remain interactive.</div>
  </div>
  <div class="map-tooltip" id="houseMapTooltip" role="status"></div>
</div>
'''

def _canonical_senate_rating(value):
    text = str(value).strip().replace("Tossup", "Toss-Up").replace("Toss Up", "Toss-Up")
    text = text.split(" ·", 1)[0].strip()
    party_first = re.fullmatch(r"([DR])\s+(Safe|Likely|Lean|Tilt)", text, re.I)
    if party_first:
        return f"{party_first.group(2).title()} {party_first.group(1).upper()}"
    strength_first = re.fullmatch(r"(Safe|Likely|Lean|Tilt)\s+([DR])", text, re.I)
    if strength_first:
        return f"{strength_first.group(1).title()} {strength_first.group(2).upper()}"
    if text.lower() in {"toss-up", "toss up"}:
        return "Toss-Up"
    return text if text and text.lower() != "nan" else "Not rated"


state_abbrev = {
    "Alabama":"AL","Alaska":"AK","Arizona":"AZ","Arkansas":"AR","California":"CA",
    "Colorado":"CO","Connecticut":"CT","Delaware":"DE","Florida":"FL","Georgia":"GA",
    "Hawaii":"HI","Idaho":"ID","Illinois":"IL","Indiana":"IN","Iowa":"IA",
    "Kansas":"KS","Kentucky":"KY","Louisiana":"LA","Maine":"ME","Maryland":"MD",
    "Massachusetts":"MA","Michigan":"MI","Minnesota":"MN","Mississippi":"MS",
    "Missouri":"MO","Montana":"MT","Nebraska":"NE","Nevada":"NV",
    "New Hampshire":"NH","New Jersey":"NJ","New Mexico":"NM","New York":"NY",
    "North Carolina":"NC","North Dakota":"ND","Ohio":"OH","Oklahoma":"OK",
    "Oregon":"OR","Pennsylvania":"PA","Rhode Island":"RI","South Carolina":"SC",
    "South Dakota":"SD","Tennessee":"TN","Texas":"TX","Utah":"UT","Vermont":"VT",
    "Virginia":"VA","Washington":"WA","West Virginia":"WV","Wisconsin":"WI","Wyoming":"WY",
}
safe_d_states = {"CO","DE","IL","MA","MN","NJ","NM","OR","RI","VA"}
safe_r_states = {"AL","AR","FL","ID","KS","KY","LA","MS","OK","SC","SD","TN","WV","WY"}
monitored_states = {"AK","GA","IA","ME","MI","MT","NE","NH","NC","OH","TX"}
# Michigan and every other monitored contest retain the state-model output.
senate_lookup = senate_detail.copy()
senate_lookup["ABBR"] = senate_lookup["STATE"].map(state_abbrev).fillna(senate_lookup["STATE"])
senate_lookup = senate_lookup.set_index("ABBR", drop=False)
senate_rows = []
for state_name, abbr in state_abbrev.items():
    if abbr in monitored_states and abbr in senate_lookup.index:
        row = senate_lookup.loc[abbr]
        margin = safe_num(row.get("Adjusted Margin 2P", np.nan))
        outcome = str(row.get("Model Assigned Outcome", "Model monitored"))
        rating = _canonical_senate_rating(row.get("Forecast Rating", "Model monitored"))
        dprob = safe_num(row.get("D Win Probability", np.nan))
        projected_d_share = safe_num(row.get("Projected D 2P", np.nan))
        projected_r_share = safe_num(row.get("Projected R 2P", np.nan))
        map_score = float(np.clip(margin / 8.0, -1.65, 1.65))
        tier = "Monitored"
    elif abbr in safe_d_states:
        outcome, rating, dprob, map_score, tier = (
            "Democratic hold", "Safe D · non-competitive", 99.0, 2.0, "Safe"
        )
        margin = 20.0
        projected_d_share, projected_r_share = 60.0, 40.0
    elif abbr in safe_r_states:
        outcome, rating, dprob, map_score, tier = (
            "Republican hold", "Safe R · non-competitive", 1.0, -2.0, "Safe"
        )
        margin = -20.0
        projected_d_share, projected_r_share = 40.0, 60.0
    else:
        outcome, rating, dprob, map_score, tier = (
            "No 2026 Senate election", "No election", np.nan, np.nan, "None"
        )
        margin = np.nan
        projected_d_share, projected_r_share = np.nan, np.nan
    if tier != "None" and (
        not np.isfinite(projected_d_share) or not np.isfinite(projected_r_share)
    ):
        projected_d_share = 50.0 + margin / 2.0
        projected_r_share = 50.0 - margin / 2.0
    senate_rows.append({
        "STATE": state_name, "ABBR": abbr, "Outcome": outcome, "Rating": rating,
        "D Win Probability": dprob, "Map Score": map_score, "Tier": tier,
        "Forecast Margin": margin, "Projected D 2P": projected_d_share,
        "Projected R 2P": projected_r_share,
        "Rating Key": _canonical_senate_rating(rating),
    })
senate_map = pd.DataFrame(senate_rows)
senate_map["Hover"] = senate_map.apply(
    lambda row: (
        f"<b>{row['STATE']}</b><br>{row['Rating']}<br>{row['Outcome']}"
        + (f"<br>D win probability: {row['D Win Probability']:.1f}%"
           if np.isfinite(row['D Win Probability']) else "")
    ), axis=1,
)

# A self-contained state cartogram replaces Plotly's USA base layer, whose
# default topology is fetched from a CDN. Every tile and tooltip works offline.
senate_tile_positions = {
    "AK":(1,1), "ME":(1,11),
    "WA":(2,1), "ID":(2,2), "MT":(2,3), "ND":(2,4), "MN":(2,5),
    "WI":(2,6), "MI":(2,8), "VT":(2,10), "NH":(2,11),
    "OR":(3,1), "NV":(3,2), "WY":(3,3), "SD":(3,4), "IA":(3,5),
    "IL":(3,6), "IN":(3,7), "OH":(3,8), "PA":(3,9), "NY":(3,10), "MA":(3,11),
    "CA":(4,1), "UT":(4,2), "CO":(4,3), "NE":(4,4), "MO":(4,5),
    "KY":(4,6), "WV":(4,7), "VA":(4,8), "MD":(4,9), "NJ":(4,10),
    "CT":(4,11), "RI":(4,12),
    "AZ":(5,2), "NM":(5,3), "KS":(5,4), "AR":(5,5), "TN":(5,6),
    "NC":(5,8), "SC":(5,9), "DE":(5,10),
    "OK":(6,4), "LA":(6,5), "MS":(6,6), "AL":(6,7), "GA":(6,8),
    "HI":(7,1), "TX":(7,4), "FL":(7,9),
}
flip_abbr = set()
if "Model Assigned Outcome" in senate_lookup.columns:
    flip_abbr = set(senate_lookup.loc[
        senate_lookup["Model Assigned Outcome"].isin(["Democratic Flip", "Republican Flip"]),
        "ABBR",
    ])
senate_tiles = []
for _, row in senate_map.iterrows():
    abbr = row["ABBR"]
    grid_row, grid_column = senate_tile_positions[abbr]
    score = safe_num(row.get("Map Score", np.nan))
    probability = safe_num(row.get("D Win Probability", np.nan))
    margin = safe_num(row.get("Forecast Margin", np.nan))
    projected_d_share = safe_num(row.get("Projected D 2P", np.nan))
    projected_r_share = safe_num(row.get("Projected R 2P", np.nan))
    rating = str(row.get("Rating", "No election"))
    outcome = str(row.get("Outcome", ""))
    tier = str(row.get("Tier", "None"))
    if tier == "None":
        tile_class = "state-none"
    elif np.isfinite(probability) and 45.0 < probability < 55.0:
        tile_class = "state-watch-toss"
    elif np.isfinite(probability) and probability >= 55.0:
        tile_class = "state-watch-dem" if tier == "Monitored" else "state-safe-dem"
    else:
        tile_class = "state-watch-rep" if tier == "Monitored" else "state-safe-rep"
    monitored_class = " state-monitored" if tier == "Monitored" else ""
    flip = "D" if abbr in flip_abbr and "Democratic" in outcome else "R" if abbr in flip_abbr else ""
    star = "<span class='state-star'>★</span>" if flip else ""
    probability_value = f"{probability:.6f}" if np.isfinite(probability) else ""
    margin_value = f"{margin:.6f}" if np.isfinite(margin) else ""
    projected_d_share_value = f"{projected_d_share:.6f}" if np.isfinite(projected_d_share) else ""
    projected_r_share_value = f"{projected_r_share:.6f}" if np.isfinite(projected_r_share) else ""
    senate_tiles.append(
        f'<button type="button" class="state-tile {tile_class}{monitored_class}" '
        f'style="grid-row:{grid_row};grid-column:{grid_column};" '
        f'data-state="{html_lib.escape(str(row["STATE"]), quote=True)}" '
        f'data-abbr="{abbr}" data-tier="{tier}" data-prob="{probability_value}" '
        f'data-margin="{margin_value}" data-dshare="{projected_d_share_value}" '
        f'data-rshare="{projected_r_share_value}" data-rating="{html_lib.escape(rating, quote=True)}" '
        f'data-rating-key="{html_lib.escape(str(row.get("Rating Key", rating)), quote=True)}" '
        f'data-outcome="{html_lib.escape(outcome, quote=True)}" data-flip="{flip}" '
        f'aria-label="{html_lib.escape(str(row["STATE"] + " · " + rating + " · " + outcome), quote=True)}">'
        f'{abbr}{star}<span class="state-status">{"WATCH" if tier == "Monitored" else ("SAFE" if tier == "Safe" else "—")}</span></button>'
    )
senate_map_html = f'''
<div class="map-toolbar senate-toolbar">
  <div><div class="map-eyebrow">All 35 elections</div><div class="map-toolbar-title">Senate forecast map</div></div>
  <label>View
    <select id="senateMapMetric" aria-label="Senate map view">
      <option value="forecast">Forecast</option>
      <option value="probability">Win probability</option>
      <option value="ratings">Ratings</option>
      <option value="margin">Projected margin</option>
      <option value="flips">Holds & flips</option>
    </select>
  </label>
</div>
<div class="map-dynamic-legend" id="senateMapLegend">
  <span><i style="background:#7F0000"></i>Republican</span>
  <span><i style="background:#F4D35E"></i>Toss-up</span>
  <span><i style="background:#073B75"></i>Democratic</span>
  <span><i class="outline-key"></i>Model monitored</span>
</div>
<div class="senate-map-shell">
  <div class="state-map-grid">{''.join(senate_tiles)}</div>
  <div class="map-tooltip" id="senateMapTooltip" role="status"></div>
</div>
<div class="map-selection" id="senateMapSelection">Hover or click a state to inspect the forecast.</div>
'''


def _normalized_rating(value):
    return _canonical_senate_rating(value)


RATING_ORDER = [
    "Safe D", "Likely D", "Lean D", "Tilt D", "Toss-Up",
    "Tilt R", "Lean R", "Likely R", "Safe R",
]
RATING_COLORS = {
    "Safe D":"#073B75", "Likely D":"#1769AA", "Lean D":"#5B9BD5",
    "Tilt D":"#A8D5EC", "Toss-Up":"#F4D35E", "Tilt R":"#F2B5B9",
    "Lean R":"#DF6670", "Likely R":"#C1121F", "Safe R":"#7F0000",
}


def _ratings_bar(values, label, total, majority_line=None):
    counts = pd.Series([_normalized_rating(value) for value in values]).value_counts()
    segments = []
    legend = []
    for rating_name in RATING_ORDER:
        count = int(counts.get(rating_name, 0))
        if count <= 0:
            continue
        segments.append(
            f'<div class="rating-segment" style="flex:{count};background:{RATING_COLORS[rating_name]}" '
            f'title="{rating_name}: {count}"><span>{count if count >= 4 else ""}</span></div>'
        )
        legend.append(
            f'<span><i style="background:{RATING_COLORS[rating_name]}"></i>{rating_name} <b>{count}</b></span>'
        )
    marker = (
        f'<div class="majority-marker" style="left:{majority_line/total*100:.4f}%"></div>'
        if majority_line is not None else ""
    )
    return (
        f'<div class="rating-row"><div class="rating-row-label">{label}</div>'
        f'<div class="rating-track">{"".join(segments)}{marker}</div>'
        f'<div class="rating-legend">{"".join(legend)}</div></div>'
    )


house_model_ratings_html = _ratings_bar(
    house_detail["Forecast Rating"], "Model forecast", 435, majority_line=218
)
house_source_ratings_html = _ratings_bar(
    house_detail["All Source Consensus Rating"], "Source consensus", 435, majority_line=218
)
senate_rating_values = senate_map.loc[senate_map["Tier"].ne("None"), "Rating"]
senate_ratings_html = _ratings_bar(senate_rating_values, "2026 Senate races", 35)


def _house_summary_value(metric, default=np.nan):
    row = house_simulation_summary.loc[house_simulation_summary["Metric"].eq(metric), "Value"]
    return safe_num(row.iloc[0]) if len(row) else default


HOUSE_EXPECTED_D_DISPLAY = _house_summary_value("Expected Democratic seats", D_HOUSE)
HOUSE_EXPECTED_R_DISPLAY = _house_summary_value("Expected Republican seats", R_HOUSE)
HOUSE_LEADER_D_DISPLAY = int(round(_house_summary_value("Race-leader Democratic seats", D_HOUSE)))
HOUSE_LEADER_R_DISPLAY = 435 - HOUSE_LEADER_D_DISPLAY
HOUSE_HEADLINE_PARTY = "Democrats" if HOUSE_D_PROB >= HOUSE_R_PROB else "Republicans"
HOUSE_HEADLINE_PROB = max(HOUSE_D_PROB, HOUSE_R_PROB)
HOUSE_HEADLINE_CLASS = "dem" if HOUSE_D_PROB >= HOUSE_R_PROB else "rep"

house_forecast_panel_html = f'''
<div class="forecast-poster">
  <div class="forecast-poster-copy">
    <div class="map-eyebrow">Probabilistic House forecast</div>
    <h3><span class="{HOUSE_HEADLINE_CLASS}">{HOUSE_HEADLINE_PARTY}</span> have a
      <span class="{HOUSE_HEADLINE_CLASS}">{HOUSE_HEADLINE_PROB:.1f}% chance</span><br>of controlling the House.</h3>
    <p>Simulation median <b>D {D_HOUSE} · R {R_HOUSE}</b> · Expected seats
       <b>D {HOUSE_EXPECTED_D_DISPLAY:.1f} · R {HOUSE_EXPECTED_R_DISPLAY:.1f}</b></p>
  </div>
  <div class="ratings-board">{house_model_ratings_html}{house_source_ratings_html}</div>
</div>
'''
SENATE_HEADLINE_PARTY = "Democrats" if SENATE_D_PROB >= SENATE_R_PROB else "Republicans"
SENATE_HEADLINE_PROB = max(SENATE_D_PROB, SENATE_R_PROB)
SENATE_HEADLINE_CLASS = "dem" if SENATE_D_PROB >= SENATE_R_PROB else "rep"
senate_forecast_panel_html = f'''
<div class="forecast-poster senate-poster">
  <div class="forecast-poster-copy">
    <div class="map-eyebrow">Probabilistic Senate forecast</div>
    <h3><span class="{SENATE_HEADLINE_CLASS}">{SENATE_HEADLINE_PARTY}</span> have a
      <span class="{SENATE_HEADLINE_CLASS}">{SENATE_HEADLINE_PROB:.1f}% chance</span><br>of controlling the Senate.</h3>
    <p>Simulation median <b>D {D_SENATE} · R {R_SENATE}</b> · all 35 scheduled elections shown below.</p>
  </div>
  <div class="ratings-board">{senate_ratings_html}</div>
</div>
'''


# ------------------------------------------------------------
# 9. GRÁFICO POPULAR VOTE
# ------------------------------------------------------------

fig_popular = go.Figure()

fig_popular.add_trace(go.Bar(
    x=["Democratic", "Republican", "Other"],
    y=[DPP, RPP, OTHER],
    marker_color=[DEM, REP, PURPLE],
    text=[f"{DPP:.2f}%", f"{RPP:.2f}%", f"{OTHER:.2f}%"],
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>%{y:.2f}%<extra></extra>"
))

fig_popular.update_layout(
    title="Projected National Popular Vote",
    yaxis_title="Vote Share (%)",
    height=360,
    showlegend=False,
    margin=dict(l=45, r=20, t=60, b=40),
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(color=TEXT)
)

popular_html = pio.to_html(fig_popular, include_plotlyjs=False, full_html=False)

# ------------------------------------------------------------
# 10. CONTROL PROBABILITY — DONUT CARDS
# ------------------------------------------------------------

def donut_chart(value, color, title, label):
    fig = go.Figure()

    fig.add_trace(go.Pie(
        values=[value, max(0, 100 - value)],
        labels=[label, "Other"],
        hole=0.72,
        marker=dict(colors=[color, "#E5E7EB"]),
        textinfo="none",
        hovertemplate="%{label}: %{value:.2f}%<extra></extra>",
        sort=False
    ))

    fig.add_annotation(
        text=f"<b>{value:.1f}%</b>",
        x=0.5,
        y=0.55,
        font=dict(size=28, color=color),
        showarrow=False
    )

    fig.add_annotation(
        text=label,
        x=0.5,
        y=0.39,
        font=dict(size=12, color="#6B7280"),
        showarrow=False
    )

    fig.update_layout(
        title=title,
        height=260,
        margin=dict(l=10, r=10, t=50, b=10),
        paper_bgcolor="white",
        plot_bgcolor="white",
        showlegend=False
    )

    return pio.to_html(fig, include_plotlyjs=False, full_html=False)

house_dem_donut = donut_chart(
    HOUSE_D_PROB,
    DEM,
    "House · Democratic Control",
    "Democratic"
)

house_rep_donut = donut_chart(
    HOUSE_R_PROB,
    REP,
    "House · Republican Control",
    "Republican"
)

senate_dem_donut = donut_chart(
    SENATE_D_PROB,
    DEM,
    "Senate · Democratic Control",
    "Democratic"
)

senate_rep_donut = donut_chart(
    SENATE_R_PROB,
    REP,
    "Senate · Republican Control",
    "Republican"
)

# ------------------------------------------------------------
# 11. MONTE CARLO SEAT DISTRIBUTIONS
# ------------------------------------------------------------

required_mc_columns = [
    "D House Seats", "R House Seats",
    "D Senate Seats", "R Senate Seats"
]
missing_mc_columns = [c for c in required_mc_columns if c not in monte_carlo.columns]
if missing_mc_columns:
    raise ValueError(
        "MonteCarloSample no contiene las columnas canónicas de escaños: "
        + ", ".join(missing_mc_columns)
    )

for column in required_mc_columns:
    monte_carlo[column] = pd.to_numeric(monte_carlo[column], errors="raise").astype("int64")

if not (monte_carlo["D House Seats"] + monte_carlo["R House Seats"] == 435).all():
    raise ValueError("La muestra Monte Carlo de Cámara contiene filas que no suman 435.")
if not (monte_carlo["D Senate Seats"] + monte_carlo["R Senate Seats"] == 100).all():
    raise ValueError("La muestra Monte Carlo de Senado contiene filas que no suman 100.")


def seat_dot_distribution(
    data,
    d_column,
    r_column,
    chamber,
    total_seats,
    democratic_control_at,
    target_dots=190
):
    """
    Distribución tipo dot plot: cada punto agrupa simulaciones con el mismo
    reparto de escaños. El color indica qué partido controlaría la cámara.
    Los ejes muestran simultáneamente escaños D (abajo) y R (arriba).
    """
    d_values = data[d_column].astype(int)
    r_values = data[r_column].astype(int)
    sample_size = len(data)
    if sample_size == 0:
        raise ValueError(f"No hay simulaciones para {chamber}.")

    counts = d_values.value_counts().sort_index()
    sample_per_dot = max(1, int(np.ceil(sample_size / target_dots)))
    scale_to_full_run = SIMS / sample_size

    traces = {
        "Democratic control": {"x": [], "y": [], "custom": []},
        "Republican control": {"x": [], "y": [], "custom": []},
    }

    for d_seats, sample_count in counts.items():
        r_seats = total_seats - int(d_seats)
        dot_count = int(np.ceil(sample_count / sample_per_dot))
        remaining = int(sample_count)
        control = (
            "Democratic control"
            if int(d_seats) >= democratic_control_at
            else "Republican control"
        )
        for stack_position in range(1, dot_count + 1):
            represented_sample = min(sample_per_dot, remaining)
            represented_full = int(round(represented_sample * scale_to_full_run))
            traces[control]["x"].append(int(d_seats))
            traces[control]["y"].append(stack_position)
            traces[control]["custom"].append([
                int(r_seats), represented_full, int(sample_count)
            ])
            remaining -= represented_sample

    figure = go.Figure()
    for name, color in [
        ("Democratic control", DEM),
        ("Republican control", REP),
    ]:
        values = traces[name]
        figure.add_trace(go.Scatter(
            x=list(values["x"]),
            y=list(values["y"]),
            customdata=list(values["custom"]),
            mode="markers",
            name=name,
            marker=dict(
                color=color,
                size=8 if chamber == "House" else 12,
                line=dict(color="white", width=1),
                opacity=0.95
            ),
            hovertemplate=(
                f"<b>{chamber} outcome</b><br>"
                "Democratic seats: %{x}<br>"
                "Republican seats: %{customdata[0]}<br>"
                "≈ %{customdata[1]:,} simulations represented"
                "<extra></extra>"
            )
        ))

    d_min = int(d_values.min())
    d_max = int(d_values.max())
    padding = 1 if chamber == "Senate" else 2
    if chamber == "Senate":
        tick_step = 1
    else:
        tick_step = max(2, int(np.ceil((d_max - d_min) / 8)))
    tick_values = list(range(d_min, d_max + 1, tick_step))
    if tick_values[-1] != d_max:
        tick_values.append(d_max)

    figure.add_vline(
        x=democratic_control_at - 0.5,
        line_width=2,
        line_dash="dot",
        line_color="#9CA3AF"
    )

    figure.update_layout(
        title=dict(
            text=f"{chamber} seat distribution",
            x=0.01,
            xanchor="left",
            font=dict(size=21, color=TEXT)
        ),
        height=390,
        margin=dict(l=18, r=18, t=82, b=48),
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(color=TEXT, family="Inter, Arial, sans-serif"),
        hovermode="closest",
        legend=dict(
            orientation="h",
            x=0,
            y=1.08,
            xanchor="left",
            yanchor="bottom",
            font=dict(size=11)
        ),
        xaxis=dict(
            range=[d_min - padding, d_max + padding],
            tickmode="array",
            tickvals=tick_values,
            ticktext=[str(value) for value in tick_values],
            tickfont=dict(color=DEM, size=11),
            showgrid=False,
            zeroline=False,
            linecolor="#D1D5DB",
            fixedrange=True
        ),
        xaxis2=dict(
            overlaying="x",
            side="top",
            range=[d_min - padding, d_max + padding],
            tickmode="array",
            tickvals=tick_values,
            ticktext=[str(total_seats - value) for value in tick_values],
            tickfont=dict(color=REP, size=11),
            showgrid=False,
            zeroline=False,
            linecolor="#D1D5DB",
            fixedrange=True
        ),
        yaxis=dict(
            visible=False,
            rangemode="tozero",
            fixedrange=True
        )
    )
    return figure, sample_per_dot, scale_to_full_run


fig_mc_house, house_sample_per_dot, house_scale = seat_dot_distribution(
    monte_carlo,
    "D House Seats",
    "R House Seats",
    "House",
    435,
    218,
    target_dots=220
)
fig_mc_senate, senate_sample_per_dot, senate_scale = seat_dot_distribution(
    monte_carlo,
    "D Senate Seats",
    "R Senate Seats",
    "Senate",
    100,
    51,
    target_dots=130
)

mc_house_html = pio.to_html(
    fig_mc_house,
    include_plotlyjs=False,
    full_html=False,
    config={"displayModeBar": False, "responsive": True}
)
mc_senate_html = pio.to_html(
    fig_mc_senate,
    include_plotlyjs=False,
    full_html=False,
    config={"displayModeBar": False, "responsive": True}
)

house_dot_full_run = int(round(house_sample_per_dot * house_scale))
senate_dot_full_run = int(round(senate_sample_per_dot * senate_scale))
# ------------------------------------------------------------
# 12. CONTEXTO NACIONAL
# ------------------------------------------------------------

context_cards_html = ""

for label, val in context_metrics.items():
    color = context_color(label)
    context_cards_html += f"""
    <div class="mini-card" style="border-top: 5px solid {color};">
        <div class="mini-label">{label}</div>
        <div class="mini-value" style="color:{color};">{fmt_context(val)}</div>
    </div>
    """

if context_cards_html.strip() == "":
    context_cards_html = "<p class='muted'>No national context indicators available.</p>"

# ------------------------------------------------------------
# 13. TABLAS HTML
# ------------------------------------------------------------

race_forecast_columns = [
    "STATE", "INCUMBENT", "MARGIN", "Poll Margin 2P",
    "Model Polling Error Correction PP", "Adjusted Margin 2P",
    "Projected D 2P", "Projected R 2P",
    "D Win Probability", "R Win Probability", "Forecast Rating",
    "Model Assigned Outcome"
]

def senate_display_table(dataframe):
    available = [column for column in race_forecast_columns if column in dataframe.columns]
    display_df = dataframe[available].copy()
    return display_df.rename(columns={
        "MARGIN": "Raw Poll Margin",
        "Poll Margin 2P": "Poll Margin 2P",
        "Model Polling Error Correction PP": "Error Correction",
        "Adjusted Margin 2P": "Projected Margin",
        "Projected D 2P": "Projected D",
        "Projected R 2P": "Projected R",
        "D Win Probability": "D Win %",
        "R Win Probability": "R Win %",
        "Forecast Rating": "Forecast Rating",
        "Model Assigned Outcome": "Forecast"
    })


def senate_diagnostic_table(dataframe):
    """Conserva todos los campos, pero distingue inputs de outputs del modelo."""
    return dataframe.copy().rename(columns={
        "RATING": "Parser Rating Code (Input)",
        "Consensus Rating": "Consensus Rating (Input)",
        "Rating Bucket": "Consensus Bucket (Input)",
        "Forecast Rating": "Forecast Rating (Model)",
        "Projected Winner": "Projected Winner (Model)",
        "Model Assigned Outcome": "Forecast Outcome (Model)",
        "Polling Leader": "Polling Leader (Input)",
        "Polling Outcome": "Polling Outcome (Input)",
    })


race_forecast_table = senate_display_table(senate_detail)
flips_display_table = senate_display_table(senate_flips)
race_diagnostics_table = senate_diagnostic_table(senate_detail)

race_forecast_table_html = table_html(
    race_forecast_table, 20, "Senate race forecasts"
)
flips_table_html = table_html(
    flips_display_table, 8, "Projected Senate flips"
)
race_diagnostics_html = table_html(
    race_diagnostics_table, 20, "Full diagnostics for all Senate races"
)
diagnostic_field_count = len(race_diagnostics_table.columns)
risk_table_html = table_html(electoral_risk, 8, "Electoral risk")
uncertainty_table_html = table_html(final_uncertainty, 20, "Final uncertainty")
quality_table_html = table_html(model_quality, 10, "Model quality")


# ------------------------------------------------------------
# 13A. HOUSE RACE DESK — 435 ROWS + ALL-VARIABLE INSPECTOR
# ------------------------------------------------------------

def _cell(value, fallback="—"):
    if pd.isna(value) or str(value).strip() == "":
        return fallback
    return html_lib.escape(str(value))

house_rows = []


def _rating_slug(value):
    return re.sub(r"[^a-z0-9]+", "-", str(value).lower()).strip("-") or "not-rated"


for row_index, row in house_detail.reset_index(drop=True).iterrows():
    rating = str(row.get("Forecast Rating", ""))
    consensus = str(row.get("All Source Consensus Rating", "Not rated"))
    search_text = " ".join(str(row.get(column, "")) for column in [
        "District ID", "State", "Incumbent Name", "Democratic Candidate",
        "Republican Candidate", "Forecast Rating", "All Source Consensus Rating", "PVI Raw",
    ]).lower()
    margin = safe_num(row.get("Projected Margin PP", np.nan))
    dprob = safe_num(row.get("D Win Probability", np.nan))
    party_switch = bool(row.get("Democratic Candidate Party Switch", False)) or bool(
        row.get("Republican Candidate Party Switch", False)
    )
    rating_slug = _rating_slug(rating)
    consensus_slug = _rating_slug(consensus)
    rprob = safe_num(row.get("R Win Probability", 100.0 - dprob))
    probability_width = min(max(dprob, 0.0), 100.0)
    republican_probability_width = min(max(rprob, 0.0), 100.0)
    house_rows.append(
        f'<tr class="house-race-row rating-row-{rating_slug}" data-index="{row_index}" '
        f'data-search="{html_lib.escape(search_text, quote=True)}" '
        f'data-rating="{html_lib.escape(rating, quote=True)}">'
        f'<td class="district-cell"><b>{_cell(row.get("District ID"))}</b><span class="row-rail"></span></td>'
        f'<td><span class="rating-chip rating-{rating_slug}">{_cell(rating)}</span></td>'
        f'<td><span class="rating-chip rating-{consensus_slug}">{_cell(consensus, "Not rated")}</span></td>'
        f'<td>{format_party_margin(margin)}</td>'
        f'<td><div class="probability-cell probability-cell-dem"><b>D {dprob:.1f}%</b><span><i style="width:{probability_width:.2f}%"></i></span></div></td>'
        f'<td><div class="probability-cell probability-cell-rep"><b>R {rprob:.1f}%</b><span><i style="width:{republican_probability_width:.2f}%"></i></span></div></td>'
        f'<td>{_cell(row.get("Incumbent Name"), "Open seat")}</td>'
        f'<td>{_cell(row.get("Incumbent Party Raw"))}</td>'
        f'<td>{_cell(row.get("Democratic Candidate"), "TBD")}</td>'
        f'<td>{_cell(row.get("Republican Candidate"), "TBD")}</td>'
        f'<td>{_cell(row.get("PVI Raw"))}</td>'
        f'<td>{_cell(row.get("Projected Flip"), "Hold")}</td>'
        f'<td>{"Yes" if bool(row.get("Open Seat", False)) else "No"}</td>'
        f'<td>{"Yes" if party_switch else "No"}</td></tr>'
    )
house_rows_html = "".join(house_rows)

house_records_json = house_detail.to_json(orient="records", date_format="iso")
house_competitive_html = table_html(
    house_competitive[[column for column in [
        "District ID", "Forecast Rating", "Projected Margin PP", "D Win Probability",
        "Incumbent Name", "Democratic Candidate", "Republican Candidate", "PVI Raw",
        "All Source Consensus Rating",
    ] if column in house_competitive.columns]],
    max_rows=len(house_competitive), label="Competitive House districts",
)
house_validation_display = house_validation_folds.rename(columns={
    "Outer Election Year": "Held-out year", "Outer MAE": "MAE",
    "Outer Winner Accuracy": "Winner accuracy",
}).copy()
for column in house_validation_display.select_dtypes(include=[np.number]).columns:
    house_validation_display[column] = house_validation_display[column].round(4)
house_validation_html = table_html(
    house_validation_display, max_rows=10, label="House validation folds",
)
house_contract_html = table_html(
    house_feature_contract, max_rows=len(house_feature_contract), label="House feature contract",
)

# ------------------------------------------------------------
# 13B. TIME-MACHINE VALIDATION VISUALS
# ------------------------------------------------------------

from plotly.subplots import make_subplots

selected_tm = time_machine_outcomes.loc[
    time_machine_outcomes["Model"].eq("NestedSelected")
].copy()

time_machine_plot_specs = [
    ("Popular Vote Margin (pp)", "National popular margin (D-R, pp)", 1, 1),
    ("Democratic House Seats", "Democratic House seats", 1, 2),
    ("Republican House Seats", "Republican House seats", 1, 3),
    (
        "Democratic Senate Seats (national prior)",
        "Democratic Senate seats · national prior",
        2, 1,
    ),
    (
        "Republican Senate Seats (national prior)",
        "Republican Senate seats · national prior",
        2, 2,
    ),
]
time_machine_figure = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=[spec[1] for spec in time_machine_plot_specs] + [""],
    horizontal_spacing=0.075,
    vertical_spacing=0.18,
)
for plot_index, (outcome, title, plot_row, plot_col) in enumerate(
    time_machine_plot_specs
):
    outcome_data = selected_tm.loc[
        selected_tm["Outcome"].eq(outcome)
    ].sort_values("Year")
    time_machine_figure.add_trace(
        go.Scatter(
            x=outcome_data["Year"],
            y=outcome_data["Actual"],
            mode="lines+markers",
            name="Actual",
            legendgroup="actual",
            showlegend=plot_index == 0,
            line=dict(color="#111827", width=3),
            marker=dict(size=8),
        ),
        row=plot_row, col=plot_col,
    )
    time_machine_figure.add_trace(
        go.Scatter(
            x=outcome_data["Year"],
            y=outcome_data["Predicted"],
            mode="lines+markers",
            name="Out-of-sample forecast",
            legendgroup="forecast",
            showlegend=plot_index == 0,
            line=dict(color="#7C3AED", width=3, dash="dash"),
            marker=dict(size=8),
        ),
        row=plot_row, col=plot_col,
    )

time_machine_figure.update_layout(
    height=720,
    margin=dict(l=38, r=25, t=95, b=45),
    paper_bgcolor="white",
    plot_bgcolor="white",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.09, x=0),
)
time_machine_figure.update_xaxes(
    tickmode="array",
    tickvals=sorted(selected_tm["Year"].unique()),
    gridcolor="#EEF2F7",
)
time_machine_figure.update_yaxes(
    gridcolor="#EEF2F7", zerolinecolor="#CBD5E1"
)
time_machine_chart_html = pio.to_html(
    time_machine_figure,
    include_plotlyjs=False,
    full_html=False,
    config={"displayModeBar": False, "responsive": True},
)

senate_cycle_display = senate_cycle_validation.rename(columns={
    "Fundamentals MAE PP": "Full Error-Correction Candidate MAE PP",
    "Posterior MAE PP": "Validated Error-Corrected MAE PP",
    "Posterior Better Than Poll": "Validated Correction Better Than Poll",
})
senate_error_figure = go.Figure()
senate_error_figure.add_bar(
    x=senate_cycle_display["YEAR"],
    y=senate_cycle_display["Poll MAE PP"],
    name="Poll baseline MAE",
    marker_color="#94A3B8",
)
senate_error_figure.add_bar(
    x=senate_cycle_display["YEAR"],
    y=senate_cycle_display["Validated Error-Corrected MAE PP"],
    name="Validated error-corrected MAE",
    marker_color="#7C3AED",
)
senate_error_figure.update_layout(
    barmode="group",
    title=dict(
        text="Senate margin MAE by held-out election",
        x=0.01,
        xanchor="left",
    ),
    xaxis_title="Outer test election",
    yaxis_title="Mean absolute error (pp)",
    height=390,
    margin=dict(l=55, r=25, t=80, b=55),
    paper_bgcolor="white",
    plot_bgcolor="white",
    legend=dict(orientation="h", y=1.12, x=0),
)
senate_error_figure.update_xaxes(
    tickmode="array",
    tickvals=senate_cycle_display["YEAR"].tolist(),
    gridcolor="#EEF2F7",
)
senate_error_figure.update_yaxes(
    rangemode="tozero",
    gridcolor="#EEF2F7",
)
senate_error_chart_html = pio.to_html(
    senate_error_figure,
    include_plotlyjs=False,
    full_html=False,
    config={"displayModeBar": False, "responsive": True},
)
senate_error_table_html = table_html(
    senate_cycle_display.round(3),
    5,
    "Senate polling-error validation by outer election",
)
popular_vote_methods_html = table_html(
    popular_vote_methods.round(3), 10,
    "Popular-vote methods · sealed outer tests",
)
popular_vote_bridge_html = table_html(
    popular_vote_bridge.round(3), 5,
    "2026 popular-vote production bridge",
)
module_isolation_html = table_html(
    module_isolation, 10,
    "One-way module isolation checks",
)
time_machine_scorecard_html = table_html(
    time_machine_scorecard.round(3), 5, "Five historical out-of-sample backtests"
)
nested_exam_progression_html = table_html(
    nested_exam_progression.round(3), 15,
    "Inner model-selection exam and untouched outer exam"
)
second_exam_improvement_html = table_html(
    second_exam_improvement.round(3), 6,
    "Outer-exam improvement versus fixed baselines"
)
time_machine_2026_folds_html = table_html(
    time_machine_2026_folds.round(3), 5, "Five 2026 leave-cycle-out forecasts"
)
time_machine_2026_summary_html = table_html(
    time_machine_2026_summary.round(3), 5, "2026 jackknife sensitivity summary"
)

selected_summary = (
    selected_tm.groupby("Outcome")["Absolute Error"].mean().to_dict()
)
TM_MARGIN_MAE = float(selected_summary["Popular Vote Margin (pp)"])
TM_HOUSE_MAE = float(selected_summary["Democratic House Seats"])
TM_SENATE_MAE = float(
    selected_summary["Democratic Senate Seats (national prior)"]
)

# ------------------------------------------------------------
# 14. HTML FINAL — VISUAL A / PLOTLY EDITORIAL UPGRADE
# ------------------------------------------------------------
# Presentation-only redesign. No model calculation is changed.

house_total = max(int(D_HOUSE) + int(R_HOUSE), 1)
senate_total = max(int(D_SENATE) + int(R_SENATE), 1)

house_dem_width = 100 * int(D_HOUSE) / house_total
house_rep_width = 100 * int(R_HOUSE) / house_total
senate_dem_width = 100 * int(D_SENATE) / senate_total
senate_rep_width = 100 * int(R_SENATE) / senate_total

house_control_color = DEM if HOUSE_CONTROL == "Democratic" else REP
senate_control_color = DEM if SENATE_CONTROL == "Democratic" else REP

html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>2026 Midterm Forecast — Visual A</title>
<script>{get_plotlyjs()}</script>

<style>
:root{{
--dem:{DEM};
--dem-dark:{DEM_DARK};
--dem-light:{DEM_LIGHT};
--rep:{REP};
--rep-dark:{REP_DARK};
--rep-light:{REP_LIGHT};
--purple:{PURPLE};
--green:{GREEN};
--orange:{ORANGE};
--gold:{GOLD};
--teal:{TEAL};
--gray:{GRAY};
--text:{TEXT};
--muted:#64748B;
--border:#E2E8F0;
--surface:#FFFFFF;
--surface-soft:#F8FAFC;
--navy:#0F172A;
--shadow:0 18px 50px rgba(15,23,42,.08);
--shadow-soft:0 8px 28px rgba(15,23,42,.055);
}}

*{{box-sizing:border-box;}}
html{{scroll-behavior:smooth;}}

body{{
margin:0;
font-family:Inter,-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Arial,sans-serif;
background:
radial-gradient(circle at 10% 0%,rgba(11,92,171,.10),transparent 27%),
radial-gradient(circle at 90% 2%,rgba(193,18,31,.08),transparent 28%),
linear-gradient(180deg,#F8FAFC 0%,#F7F8FC 54%,#FFFFFF 100%);
color:var(--text);
}}

.topbar{{
position:sticky;
top:0;
z-index:999;
height:62px;
display:flex;
align-items:center;
justify-content:space-between;
padding:0 28px;
background:rgba(255,255,255,.88);
backdrop-filter:blur(18px);
-webkit-backdrop-filter:blur(18px);
border-bottom:1px solid rgba(226,232,240,.9);
}}

.brand{{
display:flex;
align-items:center;
gap:11px;
font-size:12px;
font-weight:950;
letter-spacing:.095em;
text-transform:uppercase;
color:#334155;
white-space:nowrap;
}}

.brand-mark{{
width:28px;
height:28px;
border-radius:9px;
background:linear-gradient(135deg,var(--dem),var(--purple) 50%,var(--rep));
box-shadow:0 5px 16px rgba(79,70,229,.22);
}}

.nav{{
display:flex;
align-items:center;
gap:7px;
flex-wrap:wrap;
justify-content:flex-end;
}}

.nav a{{
text-decoration:none;
color:#475569;
font-size:12px;
font-weight:850;
padding:8px 10px;
border-radius:10px;
transition:.18s ease;
}}

.nav a:hover{{
background:#F1F5F9;
color:#0F172A;
}}

.header{{
max-width:1180px;
margin:0 auto;
padding:68px 20px 28px;
text-align:center;
}}

.kicker-row{{
display:flex;
align-items:center;
justify-content:center;
gap:10px;
flex-wrap:wrap;
}}

.kicker{{
font-size:11px;
letter-spacing:.20em;
font-weight:950;
text-transform:uppercase;
color:#64748B;
}}

.live-chip{{
display:inline-flex;
align-items:center;
gap:7px;
padding:6px 10px;
border-radius:999px;
background:#FFFFFF;
border:1px solid #E2E8F0;
font-size:11px;
font-weight:900;
color:#334155;
box-shadow:0 3px 10px rgba(15,23,42,.04);
}}

.live-dot{{
width:8px;
height:8px;
border-radius:50%;
background:var(--green);
box-shadow:0 0 0 4px rgba(16,185,129,.12);
}}

.title{{
font-size:clamp(50px,7vw,82px);
line-height:.93;
letter-spacing:-.055em;
font-weight:1000;
margin:18px 0 12px;
background:linear-gradient(90deg,var(--dem) 0%,#6D5CE7 50%,var(--rep) 100%);
-webkit-background-clip:text;
-webkit-text-fill-color:transparent;
}}

.subtitle{{
font-size:15px;
font-weight:750;
line-height:1.55;
color:#64748B;
}}

.author{{
margin-top:12px;
font-size:14px;
font-weight:700;
color:#334155;
}}

.quick-strip{{
max-width:1180px;
margin:12px auto 4px;
padding:0 20px;
display:grid;
grid-template-columns:repeat(4,1fr);
gap:12px;
}}

.quick-item{{
background:rgba(255,255,255,.85);
border:1px solid #E2E8F0;
border-radius:17px;
padding:13px 15px;
box-shadow:0 7px 22px rgba(15,23,42,.04);
}}

.quick-label{{
font-size:10px;
font-weight:950;
letter-spacing:.11em;
text-transform:uppercase;
color:#94A3B8;
}}

.quick-value{{
font-size:18px;
font-weight:1000;
margin-top:4px;
color:#0F172A;
}}

.hero{{
max-width:1180px;
margin:auto;
display:grid;
grid-template-columns:repeat(4,1fr);
gap:16px;
padding:20px;
}}

.card{{
position:relative;
overflow:hidden;
border-radius:26px;
padding:22px;
background:rgba(255,255,255,.96);
border:1px solid #E2E8F0;
box-shadow:var(--shadow-soft);
transition:transform .18s ease,box-shadow .18s ease;
}}

.card:hover{{
transform:translateY(-2px);
box-shadow:0 16px 38px rgba(15,23,42,.08);
}}

.card:before{{
content:"";
position:absolute;
left:0;right:0;top:0;
height:6px;
background:#CBD5E1;
}}

.card-dem:before{{background:var(--dem);}}
.card-rep:before{{background:var(--rep);}}
.card-vote:before{{background:linear-gradient(90deg,var(--dem),var(--purple),var(--rep));}}
.card-purple:before{{background:var(--purple);}}

.card-label{{
font-size:11px;
font-weight:950;
text-transform:uppercase;
letter-spacing:.14em;
color:#64748B;
}}

.big{{
font-size:34px;
line-height:1.05;
font-weight:1000;
margin-top:13px;
letter-spacing:-.025em;
}}

.note{{
margin-top:10px;
font-size:13px;
font-weight:750;
line-height:1.4;
color:#64748B;
}}

.dem{{color:var(--dem);}}
.rep{{color:var(--rep);}}
.purple{{color:var(--purple);}}
.muted{{color:var(--muted);}}

.section{{
max-width:1180px;
margin:42px auto;
padding:0 20px;
scroll-margin-top:82px;
}}

.section-heading{{
display:flex;
justify-content:space-between;
align-items:flex-end;
gap:25px;
margin-bottom:15px;
}}

.section-kicker{{
font-size:10px;
font-weight:950;
letter-spacing:.15em;
text-transform:uppercase;
color:#8B5CF6;
margin-bottom:5px;
}}

.section-title{{
font-size:31px;
line-height:1.06;
letter-spacing:-.035em;
font-weight:1000;
margin:0;
color:#0F172A;
}}

.section-copy{{
max-width:520px;
font-size:13px;
line-height:1.55;
color:#64748B;
text-align:right;
}}

.panel{{
background:rgba(255,255,255,.96);
border:1px solid #E2E8F0;
border-radius:26px;
padding:22px;
box-shadow:var(--shadow);
margin-bottom:22px;
}}

.panel-tight{{padding:18px;}}

.two{{
display:grid;
grid-template-columns:1fr 1fr;
gap:20px;
}}

.four{{
display:grid;
grid-template-columns:repeat(4,1fr);
gap:16px;
}}

.power-grid{{
display:grid;
grid-template-columns:1fr 1fr;
gap:18px;
}}

.power-card{{
background:#FFFFFF;
border:1px solid #E2E8F0;
border-radius:24px;
padding:20px;
box-shadow:var(--shadow-soft);
}}

.power-top{{
display:flex;
align-items:center;
justify-content:space-between;
gap:14px;
margin-bottom:17px;
}}

.power-title{{
font-size:13px;
font-weight:950;
letter-spacing:.10em;
text-transform:uppercase;
color:#475569;
}}

.control-badge{{
display:inline-flex;
align-items:center;
gap:7px;
padding:7px 10px;
border-radius:999px;
font-size:11px;
font-weight:950;
background:#F8FAFC;
border:1px solid #E2E8F0;
}}

.control-dot{{
width:8px;height:8px;border-radius:50%;
}}

.balance-numbers{{
display:flex;
align-items:flex-end;
justify-content:space-between;
gap:16px;
margin-bottom:10px;
}}

.balance-number{{
font-size:31px;
font-weight:1000;
letter-spacing:-.04em;
}}

.balance-caption{{
font-size:10px;
font-weight:900;
letter-spacing:.09em;
text-transform:uppercase;
color:#94A3B8;
margin-top:3px;
}}

.balance-track{{
position:relative;
display:flex;
height:19px;
overflow:hidden;
border-radius:999px;
background:#E2E8F0;
box-shadow:inset 0 0 0 1px rgba(148,163,184,.18);
}}

.balance-dem{{height:100%;background:linear-gradient(90deg,var(--dem-dark),var(--dem));}}
.balance-rep{{height:100%;background:linear-gradient(90deg,var(--rep),var(--rep-dark));}}

.balance-mid{{
position:absolute;
left:50%;
top:-4px;
bottom:-4px;
width:2px;
background:#FFFFFF;
box-shadow:0 0 0 1px rgba(15,23,42,.16);
z-index:3;
}}

.balance-foot{{
display:flex;
justify-content:space-between;
gap:10px;
margin-top:9px;
font-size:11px;
font-weight:800;
color:#94A3B8;
}}

.legend{{
display:flex;
gap:15px;
flex-wrap:wrap;
margin-bottom:14px;
}}

.legend-item{{
display:flex;
align-items:center;
gap:7px;
font-size:12px;
font-weight:850;
color:#475569;
}}

.swatch{{
width:13px;
height:13px;
border-radius:4px;
}}

.mini-grid{{
display:grid;
grid-template-columns:repeat(4,1fr);
gap:14px;
}}

.mini-card{{
background:white;
border:1px solid #E2E8F0;
border-radius:18px;
padding:17px;
box-shadow:0 5px 18px rgba(15,23,42,.04);
}}

.mini-label{{
font-size:10px;
letter-spacing:.09em;
text-transform:uppercase;
font-weight:950;
color:#64748B;
}}

.mini-value{{
font-size:24px;
font-weight:1000;
margin-top:8px;
letter-spacing:-.02em;
}}

.table-shell{{
width:100%;
max-width:100%;
overflow-x:auto;
overflow-y:auto;
border:1px solid #E5E7EB;
border-radius:18px;
background:#FFFFFF;
scrollbar-color:#CBD5E1 #F8FAFC;
scrollbar-width:thin;
}}

.table-shell:focus{{
outline:3px solid rgba(139,92,246,.24);
outline-offset:3px;
}}

.data-table{{
width:max-content;
min-width:100%;
border-collapse:separate;
border-spacing:0;
font-size:12px;
white-space:nowrap;
}}

.data-table th{{
position:sticky;
top:0;
z-index:3;
background:#F1F5F9;
padding:12px 13px;
text-align:left;
font-size:10px;
line-height:1.25;
letter-spacing:.055em;
text-transform:uppercase;
color:#475569;
border-bottom:1px solid #CBD5E1;
}}

.data-table td{{
padding:11px 13px;
border-bottom:1px solid #EEF2F7;
color:#1E293B;
background:#FFFFFF;
}}

.data-table tbody tr:nth-child(even) td{{background:#FAFCFF;}}
.data-table tbody tr:hover td{{background:#F5F3FF;}}

.data-table th:first-child,
.data-table td:first-child{{
position:sticky;
left:0;
z-index:2;
font-weight:900;
box-shadow:8px 0 12px -12px rgba(15,23,42,.5);
}}

.data-table th:first-child{{z-index:4;}}

.scroll-hint{{
display:none;
margin-top:8px;
font-size:11px;
font-weight:800;
letter-spacing:.04em;
color:#64748B;
text-align:right;
}}

.party-pill,.rating-pill{{
display:inline-flex;
align-items:center;
justify-content:center;
padding:4px 8px;
border-radius:999px;
font-weight:900;
line-height:1;
}}

.party-dem{{color:#07519A;background:#E8F2FF;}}
.party-rep{{color:#A30F1A;background:#FFF0F1;}}

.party-number{{
display:inline-flex;
align-items:center;
justify-content:flex-end;
min-width:68px;
padding:4px 7px;
border-radius:999px;
font-variant-numeric:tabular-nums;
}}

.party-number-dem{{color:#07519A;}}
.party-number-rep{{color:#A30F1A;}}
.party-number.is-leading{{font-weight:950;}}
.party-number-dem.is-leading{{background:#E8F2FF;}}
.party-number-rep.is-leading{{background:#FFF0F1;}}
.party-number.is-trailing{{font-weight:650;opacity:.78;}}

.rating-tossup{{color:#6D28D9;background:#F2EAFE;}}
.rating-lean{{color:#9A3412;background:#FFF1E8;}}
.rating-likely{{color:#075985;background:#E7F7FF;}}
.rating-safe{{color:#166534;background:#EAF8EF;}}
.number-dem{{color:var(--dem);font-weight:900;}}
.number-rep{{color:var(--rep);font-weight:900;}}

.margin-pill{{
display:inline-flex;
align-items:center;
justify-content:center;
min-width:58px;
padding:5px 8px;
border-radius:999px;
font-weight:950;
letter-spacing:.015em;
}}

.margin-dem{{color:#07519A;background:#E8F2FF;}}
.margin-rep{{color:#A30F1A;background:#FFF0F1;}}
.margin-even{{color:#6D28D9;background:#F2EAFE;}}
.cell-muted{{color:#94A3B8;}}

.diagnostic-details{{
margin-top:16px;
border-top:1px solid #E5E7EB;
padding-top:14px;
}}

.diagnostic-details summary{{
cursor:pointer;
font-size:12px;
font-weight:900;
color:#6D28D9;
letter-spacing:.04em;
}}

.diagnostic-note{{
margin:10px 0 12px;
font-size:13px;
line-height:1.5;
color:#64748B;
}}

.distribution-copy{{
margin:0 0 15px;
font-size:13px;
line-height:1.58;
color:#64748B;
}}

.prob-grid{{
display:grid;
grid-template-columns:repeat(4,1fr);
gap:18px;
align-items:center;
}}

.prob-card{{
background:#FFFFFF;
border-radius:22px;
padding:20px;
text-align:center;
border:1px solid #E2E8F0;
box-shadow:0 7px 22px rgba(15,23,42,.045);
}}

.prob-title{{
font-size:11px;
font-weight:950;
text-transform:uppercase;
letter-spacing:.105em;
color:#64748B;
margin-bottom:14px;
}}

.prob-circle{{
width:156px;
height:156px;
margin:0 auto;
border-radius:50%;
background:conic-gradient(var(--c) calc(var(--p)*1%),#E7ECF2 0);
display:flex;
align-items:center;
justify-content:center;
box-shadow:inset 0 0 0 1px rgba(148,163,184,.12);
}}

.prob-inner{{
width:114px;
height:114px;
background:white;
border-radius:50%;
display:flex;
flex-direction:column;
align-items:center;
justify-content:center;
box-shadow:0 4px 12px rgba(15,23,42,.035);
}}

.prob-value{{
font-size:30px;
font-weight:1000;
line-height:1;
color:var(--c);
letter-spacing:-.03em;
}}

.prob-party{{
font-size:12px;
font-weight:850;
color:#64748B;
margin-top:7px;
}}

.plot-stage{{
overflow:hidden;
border-radius:20px;
}}

.footer{{
max-width:1180px;
margin:50px auto 0;
padding:34px 20px 55px;
text-align:center;
border-top:1px solid #E2E8F0;
font-size:12px;
line-height:1.7;
color:#64748B;
}}


.forecast-callout{{display:grid;grid-template-columns:repeat(3,1fr);gap:14px;margin:18px 0}}
.brand-logo{{width:30px;height:30px;object-fit:contain;display:block;filter:drop-shadow(0 5px 12px rgba(67,56,202,.16))}}
.forecast-callout .mini-card{{background:linear-gradient(145deg,#FFFFFF 0%,#F7F9FC 62%,#F0ECFA 100%);box-shadow:0 18px 45px rgba(15,23,42,.07)}}
.house-map-wrap{{background:radial-gradient(circle at 18% 10%,rgba(11,92,171,.09),transparent 36%),radial-gradient(circle at 82% 12%,rgba(193,18,31,.08),transparent 34%),#fff}}
.race-toolbar{{display:grid;grid-template-columns:minmax(240px,1fr) 220px auto;gap:12px;align-items:center;margin-bottom:14px}}
.race-toolbar input,.race-toolbar select{{border:1px solid #CBD5E1;border-radius:12px;padding:12px 14px;background:white;color:#111827;font:inherit;outline:none}}
.race-toolbar input:focus,.race-toolbar select:focus{{border-color:#8B5CF6;box-shadow:0 0 0 3px rgba(139,92,246,.12)}}
.house-browser{{max-height:610px;overflow:auto;border:1px solid #E2E8F0;border-radius:14px}}
.house-browser tbody tr{{cursor:pointer}}
.house-browser tbody tr.is-selected td{{background:#F0ECFA!important}}
.inspector{{margin-top:18px;padding:20px;border:1px solid #DDD6FE;border-radius:16px;background:linear-gradient(145deg,#FFFFFF,#F7F5FF)}}
.inspector-title{{font-size:21px;font-weight:780;margin-bottom:12px}}
.inspector-grid{{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:8px 18px}}
.inspector-item{{padding:9px 0;border-bottom:1px solid #EDE9FE;min-width:0}}
.inspector-key{{font-size:10px;letter-spacing:.06em;text-transform:uppercase;color:#64748B}}
.inspector-value{{font-size:13px;color:#111827;overflow-wrap:anywhere;margin-top:3px}}
.map-fallback{{display:none}}
.state-map-grid{{display:grid;grid-template-columns:repeat(12,minmax(42px,1fr));grid-template-rows:repeat(7,62px);gap:8px;max-width:980px;margin:18px auto 8px;padding:18px;border-radius:22px;background:radial-gradient(circle at 50% 10%,rgba(139,92,246,.08),transparent 42%),#F8FAFC}}
.state-tile{{position:relative;display:flex;align-items:center;justify-content:center;border-radius:12px;color:white;font-size:16px;font-weight:820;letter-spacing:.02em;box-shadow:0 7px 16px rgba(15,23,42,.10);transition:transform .16s ease,box-shadow .16s ease;cursor:help;border:1px solid rgba(255,255,255,.88)}}
.state-tile:hover{{transform:translateY(-3px) scale(1.04);box-shadow:0 13px 24px rgba(15,23,42,.18);z-index:3}}
.state-status{{position:absolute;bottom:4px;font-size:7px;letter-spacing:.08em;opacity:.86;max-width:92%;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}}
.state-star{{position:absolute;right:5px;top:3px;font-size:10px;color:white}}
.state-none{{background:#D8DEE5;color:#667085;box-shadow:none}}
.state-safe-dem{{background:linear-gradient(145deg,#A9C8DE,#7EAAC9)}}
.state-safe-rep{{background:linear-gradient(145deg,#DFAEB3,#C98189)}}
.state-watch-dem{{background:linear-gradient(145deg,#1F78C8,#073B75);border:2px solid white;outline:2px solid rgba(11,92,171,.28)}}
.state-watch-rep{{background:linear-gradient(145deg,#E44B58,#7F0000);border:2px solid white;outline:2px solid rgba(193,18,31,.25)}}
.state-watch-toss{{background:linear-gradient(145deg,#9D8BC3,#65558A);border:2px solid white;outline:2px solid rgba(139,92,246,.26)}}
@media(max-width:760px){{.race-toolbar{{grid-template-columns:1fr}}.inspector-grid{{grid-template-columns:1fr}}.forecast-callout{{grid-template-columns:1fr}}}}
@media(max-width:760px){{.state-map-grid{{grid-template-columns:repeat(12,minmax(28px,1fr));grid-template-rows:repeat(7,44px);gap:4px;padding:10px}}.state-tile{{font-size:11px;border-radius:8px}}.state-status{{display:none}}}}


.forecast-poster{{position:relative;overflow:hidden;border:1px solid #E2E8F0;border-radius:26px;padding:30px;background:radial-gradient(circle at 0 0,rgba(11,92,171,.12),transparent 36%),radial-gradient(circle at 100% 0,rgba(193,18,31,.11),transparent 34%),linear-gradient(145deg,#FFFFFF,#F7F8FC);box-shadow:0 22px 60px rgba(15,23,42,.08);margin:18px 0 22px}}
.forecast-poster-copy{{text-align:center;max-width:880px;margin:0 auto 26px}}
.forecast-poster h3{{font-size:clamp(29px,4vw,54px);line-height:1.02;letter-spacing:-.045em;margin:12px 0 13px;color:#111827}}
.forecast-poster p{{color:#5A6B87;font-size:14px}}
.ratings-board{{display:grid;gap:18px}}
.rating-row{{display:grid;grid-template-columns:130px 1fr;gap:9px 14px;align-items:center}}
.rating-row-label{{font-size:11px;font-weight:800;letter-spacing:.08em;text-transform:uppercase;color:#50617C}}
.rating-track{{position:relative;height:28px;display:flex;overflow:hidden;border-radius:9px;background:#EEF2F7;box-shadow:inset 0 0 0 1px rgba(15,23,42,.06)}}
.rating-segment{{min-width:2px;display:flex;align-items:center;justify-content:center;color:white;font-size:10px;font-weight:850;border-right:1px solid rgba(255,255,255,.45)}}
.majority-marker{{position:absolute;top:-5px;bottom:-5px;width:2px;background:#111827;box-shadow:0 0 0 1px white}}
.rating-legend{{grid-column:2;display:flex;flex-wrap:wrap;gap:6px 12px;color:#5A6B87;font-size:10px}}
.rating-legend span{{display:flex;align-items:center;gap:5px}}.rating-legend i{{width:9px;height:9px;border-radius:3px}}
.map-toolbar{{display:flex;align-items:center;justify-content:space-between;gap:18px;margin:0 0 12px}}
.map-eyebrow{{font-size:10px;font-weight:850;letter-spacing:.15em;text-transform:uppercase;color:#8B5CF6}}
.map-toolbar-title{{font-size:23px;font-weight:820;letter-spacing:-.025em;margin-top:3px}}
.map-toolbar label{{display:flex;align-items:center;gap:8px;font-size:11px;color:#64748B;font-weight:750;text-transform:uppercase;letter-spacing:.08em}}
.map-toolbar select{{border:1px solid #CBD5E1;border-radius:12px;padding:10px 34px 10px 12px;background:white;color:#111827;font:inherit;text-transform:none;letter-spacing:0;outline:none}}
.map-toolbar select:focus{{border-color:#8B5CF6;box-shadow:0 0 0 3px rgba(139,92,246,.12)}}
.map-dynamic-legend{{display:flex;flex-wrap:wrap;gap:8px 15px;align-items:center;margin:8px 0 12px;font-size:10px;font-weight:700;color:#50617C}}
.map-dynamic-legend span{{display:flex;align-items:center;gap:6px}}.map-dynamic-legend i{{width:12px;height:12px;border-radius:4px;display:inline-block}}
.outline-key{{background:white!important;border:2px solid #8B5CF6}}
.house-svg-shell,.senate-map-shell{{position:relative;border-radius:22px;background:linear-gradient(145deg,#F8FAFC,#F3F5FA);overflow:hidden}}
.house-map-visual{{position:relative}}
.house-cartogram-shell{{position:relative;border-radius:22px;background:linear-gradient(145deg,#F8FAFC,#F3F5FA);padding:20px 18px 16px;overflow:hidden}}
.house-cartogram{{display:grid;grid-template-columns:repeat(5,minmax(0,1fr));gap:18px;align-items:start}}
.cart-region{{min-width:0}}
.cart-region-title{{display:flex;align-items:baseline;justify-content:space-between;gap:8px;padding:0 0 7px;border-bottom:1px solid #CBD5E1;color:#334155;font-size:10px;letter-spacing:.08em;text-transform:uppercase}}
.cart-region-title span{{font-size:9px;color:#7C8AA2;letter-spacing:0;text-transform:none;white-space:nowrap}}
.cart-state{{margin-top:10px}}
.cart-state-head{{display:flex;align-items:center;justify-content:space-between;width:100%;margin-bottom:4px;color:#334155;font-size:9px;line-height:1}}
.cart-state-head b{{font-size:10px;letter-spacing:.08em}}
.cart-state-head span{{color:#8A99AF;font-variant-numeric:tabular-nums}}
.cart-state-grid{{display:grid;gap:3px;justify-content:start}}
.house-cart-tile{{position:relative;width:17px;height:17px;min-width:17px;padding:0;border:0;border-radius:4px;color:white;cursor:pointer;display:flex;align-items:center;justify-content:center;font-family:inherit;font-size:7px;font-weight:850;line-height:1;font-variant-numeric:tabular-nums;box-shadow:inset 0 0 0 1px rgba(255,255,255,.5);transition:transform .12s ease,box-shadow .12s ease}}
.house-cart-tile:hover,.house-cart-tile:focus{{z-index:4;transform:scale(1.3);box-shadow:0 0 0 2px #FFFFFF,0 0 0 4px rgba(139,92,246,.65);outline:none}}
.house-cart-tile.is-selected{{z-index:3;box-shadow:0 0 0 2px #FFFFFF,0 0 0 4px #8B5CF6}}
.cartogram-note{{margin-top:16px;padding-top:10px;border-top:1px solid #DDE3EC;color:#66758E;font-size:10px}}
.map-toolbar-controls{{display:flex;align-items:center;gap:12px;flex-wrap:wrap}}
.map-toolbar-subtitle{{margin-top:5px;color:#66758E;font-size:10px;font-weight:600}}
.geographic-note{{padding:9px 12px 11px;color:#66758E;font-size:10px;border-top:1px solid #DDE3EC;background:#F8FAFC}}
.house-map-visual>[hidden]{{display:none!important}}
@media(max-width:980px){{.house-cartogram{{grid-template-columns:repeat(3,minmax(0,1fr))}}}}
@media(max-width:680px){{.house-cartogram{{grid-template-columns:repeat(2,minmax(0,1fr));gap:15px}}.house-cartogram-shell{{padding:15px 12px}}.map-toolbar-controls{{align-items:flex-start;flex-direction:column}}}}
.house-svg{{display:block;width:100%;height:auto;aspect-ratio:960/510;min-height:0;background:#F7F9FC}}
.house-district{{stroke:rgba(255,255,255,.94);stroke-width:.55;fill-rule:evenodd;vector-effect:non-scaling-stroke;cursor:pointer;transition:filter .12s ease,stroke-width .12s ease}}
.house-state-outline{{fill:none;stroke:#334155;stroke-width:.82;stroke-linejoin:round;vector-effect:non-scaling-stroke;pointer-events:none}}
.house-inset-frame{{fill:#FFFFFF;stroke:#CBD5E1;stroke-width:.8;stroke-dasharray:3 3;vector-effect:non-scaling-stroke;pointer-events:none}}
.house-inset-label{{pointer-events:none}}.house-inset-label rect{{fill:rgba(255,255,255,.94);stroke:#D6DEE9;stroke-width:.45}}.house-inset-label text{{font-size:7px;font-weight:850;letter-spacing:.08em;fill:#64748B}}
.house-district:hover,.house-district:focus{{filter:brightness(1.08) drop-shadow(0 0 3px rgba(139,92,246,.55));stroke:#FFFFFF;stroke-width:1.25;outline:none}}
.house-district.is-selected{{stroke:#8B5CF6;stroke-width:1.8;filter:drop-shadow(0 0 3px rgba(139,92,246,.55))}}
.water-mask{{fill:#F3F6FA;stroke:#CBD5E1;stroke-width:.8;vector-effect:non-scaling-stroke;pointer-events:all;cursor:default}}
.map-tooltip{{position:absolute;z-index:15;pointer-events:none;display:none;min-width:210px;max-width:280px;padding:12px 14px;border:1px solid rgba(148,163,184,.45);border-radius:13px;background:rgba(255,255,255,.97);box-shadow:0 18px 44px rgba(15,23,42,.18);font-size:11px;line-height:1.5;color:#334155}}
.map-tooltip b{{font-size:14px;color:#111827}}
.map-selection{{margin-top:12px;padding:12px 14px;border-radius:12px;background:#F8FAFC;border:1px solid #E2E8F0;color:#50617C;font-size:12px}}
.state-tile{{font-family:inherit}}
.state-monitored{{outline:3px solid rgba(139,92,246,.42)!important;outline-offset:1px}}
.state-watch-toss{{background:linear-gradient(145deg,#F8DF82,#D9A91B);color:#3B2B00}}
.senate-map-stage{{overflow:visible}}
.senate-desk-heading{{margin-top:38px}}
.house-audit{{margin-bottom:18px;overflow:hidden}}
.house-audit>summary{{font-size:17px;font-weight:850;color:#111827}}
.wide-audit{{max-width:100%;overflow:hidden}}
.wide-audit .table-shell{{max-height:430px;overflow:auto}}
.wide-audit .data-table{{min-width:1500px}}
.house-race-row td{{transition:background .12s ease}}
.house-race-row:hover td{{background:#F8FAFC!important}}
.district-cell{{position:relative;padding-left:17px!important}}
.row-rail{{position:absolute;left:0;top:5px;bottom:5px;width:5px;border-radius:5px;background:#94A3B8}}
.rating-row-safe-d .row-rail{{background:#073B75}}.rating-row-likely-d .row-rail{{background:#1769AA}}.rating-row-lean-d .row-rail{{background:#5B9BD5}}.rating-row-tilt-d .row-rail{{background:#A8D5EC}}.rating-row-toss-up .row-rail{{background:#F4D35E}}.rating-row-tilt-r .row-rail{{background:#F2B5B9}}.rating-row-lean-r .row-rail{{background:#DF6670}}.rating-row-likely-r .row-rail{{background:#C1121F}}.rating-row-safe-r .row-rail{{background:#7F0000}}
.rating-chip{{display:inline-flex;padding:5px 8px;border-radius:999px;font-size:10px;font-weight:800;white-space:nowrap;background:#EEF2F7;color:#475569}}
.rating-safe-d{{background:#E2EEF8;color:#073B75}}.rating-likely-d{{background:#D9ECF7;color:#1769AA}}.rating-lean-d{{background:#E6F2FA;color:#317DB7}}.rating-tilt-d{{background:#EEF7FB;color:#3C789D}}.rating-toss-up,.rating-tossup{{background:#FFF4C7;color:#755700}}.rating-tilt-r{{background:#FCEAEC;color:#A43B45}}.rating-lean-r{{background:#F8DDE0;color:#A82733}}.rating-likely-r{{background:#F4D2D5;color:#8E101A}}.rating-safe-r{{background:#EFD4D6;color:#7F0000}}
.probability-cell{{min-width:108px}}.probability-cell>b{{font-size:11px}}.probability-cell>span{{display:block;height:5px;border-radius:4px;background:#E7ECF2;margin-top:5px;overflow:hidden}}.probability-cell i{{display:block;height:100%;border-radius:4px}}.probability-cell-dem b{{color:#0B5CAB}}.probability-cell-dem i{{background:#1769AA}}.probability-cell-rep b{{color:#C1121F}}.probability-cell-rep i{{background:#C1121F}}
@media(max-width:760px){{.rating-row{{grid-template-columns:1fr}}.rating-legend{{grid-column:1}}.forecast-poster{{padding:20px 15px}}.map-toolbar{{align-items:flex-start;flex-direction:column}}.house-svg{{min-height:360px}}}}

@media(max-width:1050px){{
.nav{{display:none;}}
.quick-strip,.hero,.mini-grid,.prob-grid{{grid-template-columns:repeat(2,1fr);}}
.power-grid,.two{{grid-template-columns:1fr;}}
}}

@media(max-width:680px){{
.topbar{{height:54px;padding:0 16px;}}
.header{{padding-top:48px;}}
.quick-strip,.hero,.mini-grid,.prob-grid{{grid-template-columns:1fr;}}
.section-heading{{display:block;}}
.section-copy{{text-align:left;margin-top:9px;}}
.panel{{padding:15px;}}
.section-title{{font-size:27px;}}
.title{{font-size:46px;}}
.scroll-hint{{display:block;}}
}}
</style>
</head>

<body>

<div class="topbar">
  <div class="brand">
    {brand_mark_html}
    2026 Forecast Desk
  </div>
  <div class="nav">
    <a href="#overview">Overview</a>
    <a href="#house">House</a>
    <a href="#house-races">435 Races</a>
    <a href="#senate">Senate</a>
    <a href="#probability">Probability</a>
    <a href="#simulation">Simulation</a>
    <a href="#context">Context</a>
    <a href="#validation">Validation</a>
  </div>
</div>

<header class="header" id="overview">
  <div class="kicker-row">
    <div class="kicker">Observatorio de los Estados Unidos · CIEP-UCR</div>
    <div class="live-chip"><span class="live-dot"></span> Model snapshot</div>
  </div>
  <div class="title">2026 MIDTERM FORECAST</div>
  <div class="subtitle">
    Updated {last_updated}<br>
    {SIMS:,} Monte Carlo simulations · audited v17 forecast outputs
  </div>
  <div class="author">Forecast model by <b>Juan Ignacio Garbanzo Fallas</b></div>
</header>

<div class="quick-strip">
  <div class="quick-item">
    <div class="quick-label">House control</div>
    <div class="quick-value" style="color:{house_control_color};">{HOUSE_CONTROL}</div>
  </div>
  <div class="quick-item">
    <div class="quick-label">Senate control</div>
    <div class="quick-value" style="color:{senate_control_color};">{SENATE_CONTROL}</div>
  </div>
  <div class="quick-item">
    <div class="quick-label">Model stability</div>
    <div class="quick-value purple">{OVERALL_STABILITY}</div>
  </div>
  <div class="quick-item">
    <div class="quick-label">Senate 50–50 probability</div>
    <div class="quick-value">{SENATE_5050:.1f}%</div>
  </div>
</div>

<div class="hero">
  <div class="card card-vote">
    <div class="card-label">Popular Vote</div>
    <div class="big"><span class="dem">D {DPP:.2f}%</span></div>
    <div class="note"><span class="rep">R {RPP:.2f}%</span> &nbsp;•&nbsp; Other {OTHER:.2f}%</div>
  </div>

  <div class="card {'card-rep' if HOUSE_CONTROL == 'Republican' else 'card-dem'}">
    <div class="card-label">House</div>
    <div class="big"><span class="dem">D {D_HOUSE}</span> <span class="muted">·</span> <span class="rep">R {R_HOUSE}</span></div>
    <div class="note">{HOUSE_CONTROL} control · Democratic margin {HOUSE_MARGIN}</div>
  </div>

  <div class="card {'card-dem' if SENATE_CONTROL == 'Democratic' else 'card-rep'}">
    <div class="card-label">Senate</div>
    <div class="big"><span class="dem">D {D_SENATE}</span> <span class="muted">·</span> <span class="rep">R {R_SENATE}</span></div>
    <div class="note">{SENATE_CONTROL} control · Democratic margin {SENATE_MARGIN}</div>
  </div>

  <div class="card card-purple">
    <div class="card-label">Diagnostic Stability</div>
    <div class="big purple">{OVERALL_STABILITY}</div>
    <div class="note">Heuristic sensitivity index · not a probability</div>
  </div>
</div>

<section class="section">
  <div class="section-heading">
    <div>
      <div class="section-kicker">Balance of power</div>
      <h2 class="section-title">Congress at a glance</h2>
    </div>
    <div class="section-copy">A presentation-only summary of the district-level House total and the unchanged national/Senate architecture.</div>
  </div>

  <div class="power-grid">
    <div class="power-card">
      <div class="power-top">
        <div class="power-title">House of Representatives</div>
        <div class="control-badge">
          <span class="control-dot" style="background:{house_control_color};"></span>
          {HOUSE_CONTROL} control
        </div>
      </div>
      <div class="balance-numbers">
        <div>
          <div class="balance-number dem">D {D_HOUSE}</div>
          <div class="balance-caption">Democratic seats</div>
        </div>
        <div style="text-align:right;">
          <div class="balance-number rep">R {R_HOUSE}</div>
          <div class="balance-caption">Republican seats</div>
        </div>
      </div>
      <div class="balance-track">
        <div class="balance-dem" style="width:{house_dem_width:.4f}%"></div>
        <div class="balance-rep" style="width:{house_rep_width:.4f}%"></div>
        <div class="balance-mid"></div>
      </div>
      <div class="balance-foot"><span>Democratic</span><span>Majority line</span><span>Republican</span></div>
    </div>

    <div class="power-card">
      <div class="power-top">
        <div class="power-title">United States Senate</div>
        <div class="control-badge">
          <span class="control-dot" style="background:{senate_control_color};"></span>
          {SENATE_CONTROL} control
        </div>
      </div>
      <div class="balance-numbers">
        <div>
          <div class="balance-number dem">D {D_SENATE}</div>
          <div class="balance-caption">Democratic seats</div>
        </div>
        <div style="text-align:right;">
          <div class="balance-number rep">R {R_SENATE}</div>
          <div class="balance-caption">Republican seats</div>
        </div>
      </div>
      <div class="balance-track">
        <div class="balance-dem" style="width:{senate_dem_width:.4f}%"></div>
        <div class="balance-rep" style="width:{senate_rep_width:.4f}%"></div>
        <div class="balance-mid"></div>
      </div>
      <div class="balance-foot"><span>Democratic</span><span>50-seat midpoint</span><span>Republican</span></div>
    </div>
  </div>
</section>


<section class="section" id="house">
  <div class="section-heading">
    <div><div class="section-kicker">District forecast engine</div><h2 class="section-title">The House, district by district</h2></div>
    <div class="section-copy">All 435 districts receive a probability. Expected seats, district leaders and the simulation median are reported separately; the frozen national output remains a comparator and never becomes a quota.</div>
  </div>
  {house_forecast_panel_html}
  <div class="forecast-callout" style="grid-template-columns:repeat(4,1fr)">
    <div class="mini-card" style="border-top:5px solid {house_control_color};"><div class="mini-label">Simulation median</div><div class="mini-value"><span class="dem">D {D_HOUSE}</span> · <span class="rep">R {R_HOUSE}</span></div></div>
    <div class="mini-card" style="border-top:5px solid #14B8A6;"><div class="mini-label">Expected seats</div><div class="mini-value"><span class="dem">D {HOUSE_EXPECTED_D_DISPLAY:.1f}</span> · <span class="rep">R {HOUSE_EXPECTED_R_DISPLAY:.1f}</span></div></div>
    <div class="mini-card" style="border-top:5px solid #111827;"><div class="mini-label">District leaders</div><div class="mini-value"><span class="dem">D {HOUSE_LEADER_D_DISPLAY}</span> · <span class="rep">R {HOUSE_LEADER_R_DISPLAY}</span></div></div>
    <div class="mini-card" style="border-top:5px solid #8B5CF6;"><div class="mini-label">Frozen national comparator</div><div class="mini-value"><span class="dem">D {HOUSE_NATIONAL_D_SEAT_PRIOR}</span> · <span class="rep">R {HOUSE_NATIONAL_R_SEAT_PRIOR}</span></div></div>
  </div>
  <div class="panel house-map-wrap">
    {house_map_html}
    <div class="distribution-copy">Official U.S. Census Bureau 2026 legislative geography for the 120th Congress · 435 voting districts · hover for the selected metric and click to open the district record.</div>
    <noscript><p class="map-fallback">JavaScript is required for map hover. The complete 435-row Race Desk remains available below.</p></noscript>
  </div>
</section>

<section class="section" id="house-races">
  <div class="section-heading"><div><div class="section-kicker">Interactive Race Desk</div><h2 class="section-title">Explore all 435 forecasts</h2></div><div class="section-copy">Search and filter the summary, then click any district to inspect every stored input, candidate-history field, source rating and model output.</div></div>
  <div class="panel">
    <div class="race-toolbar"><input id="houseSearch" type="search" placeholder="Search district, candidate, incumbent, PVI…"><select id="houseRating"><option value="">All ratings</option><option>Toss-Up</option><option>Tilt D</option><option>Tilt R</option><option>Lean D</option><option>Lean R</option><option>Likely D</option><option>Likely R</option><option>Safe D</option><option>Safe R</option></select><b id="houseCount">435 races</b></div>
    <div class="house-browser"><table class="data-table" id="houseTable"><thead><tr><th>District</th><th>Model</th><th>Consensus</th><th>D−R</th><th>D win</th><th>R win</th><th>Incumbent</th><th>Party</th><th>D candidate</th><th>R candidate</th><th>PVI</th><th>Hold / flip</th><th>Open</th><th>Switch</th></tr></thead><tbody>{house_rows_html}</tbody></table></div>
    <div class="inspector" id="houseInspector"><div class="inspector-title">Select a district to inspect all variables</div><div class="inspector-grid"></div></div>
    <details class="diagnostic-details"><summary>Show all competitive districts</summary>{house_competitive_html}</details>
  </div>
</section>

<section class="section" id="senate">
  <div class="section-heading">
    <div><div class="section-kicker">Senate battlefield</div><h2 class="section-title">The Senate, race by race</h2></div>
    <div class="section-copy">All 35 regular and special 2026 elections are visible. Monitored races use strong outlines; grey states have no election.</div>
  </div>
  {senate_forecast_panel_html}
  <div class="panel">
    <div class="plot-stage senate-map-stage">{senate_map_html}</div>
  </div>
  <div class="section-heading senate-desk-heading">
    <div><div class="section-kicker">Interactive Race Desk</div><h2 class="section-title">Senate Race Forecasts</h2></div>
    <div class="section-copy">Model outputs, both parties’ probabilities, normalized polling margins and validated historical polling-error corrections.</div>
  </div>
  <div class="panel senate-race-panel">
    <p class="distribution-copy"><b>Raw Poll Margin</b> is DEM MEAN minus REP MEAN before undecided and third-party shares are removed. <b>Poll Margin 2P</b> normalizes that poll to a two-party vote. <b>Error Correction</b> is learned from historical result-minus-poll errors using only pre-election information and nested election-year validation. <b>Projected Margin</b> is Poll Margin 2P plus the validated correction. Probabilities use out-of-cycle calibrated uncertainty; no national flip quota is applied.</p>
    {race_forecast_table_html}
    <details class="diagnostic-details">
      <summary>Show full diagnostic table ({diagnostic_field_count} fields)</summary>
      <p class="diagnostic-note"><b>Input and consensus ratings</b> are retained here for audit only. Columns labeled <b>(Model)</b> are forecast outputs used by the dashboard, map, flips and seat totals.</p>
      {race_diagnostics_html}
    </details>
  </div>
</section>

<section class="section" id="probability">
  <div class="section-heading">
    <div>
      <div class="section-kicker">Control odds</div>
      <h2 class="section-title">Control Probability</h2>
    </div>
    <div class="section-copy">Monte Carlo control probabilities, with Democratic values in blue and Republican values in red.</div>
  </div>
  <div class="panel">{probability_cards_html}</div>
</section>

<section class="section" id="simulation">
  <div class="section-heading">
    <div>
      <div class="section-kicker">Simulation</div>
      <h2 class="section-title">Monte Carlo Seat Distribution</h2>
    </div>
    <div class="section-copy">The same {SIMS:,} v17 simulations, presented as interactive seat-distribution plots.</div>
  </div>

  <div class="panel">
    <p class="distribution-copy">Each dot groups simulations with the same seat outcome. Blue outcomes produce Democratic control; red outcomes produce Republican control. Blue numbers on the lower scale are Democratic seats; red numbers on the upper scale are the corresponding Republican seats.</p>
    <div class="plot-stage">{mc_house_html}</div>
    <div class="distribution-copy">≈ {house_dot_full_run:,} full-run simulations per complete dot · {SIMS:,} simulations total · House uncertainty uses a continuous, robust predictive distribution fitted to aggregate LOOCV seat error.</div>
  </div>

  <div class="panel">
    <div class="plot-stage">{mc_senate_html}</div>
    <div class="distribution-copy">≈ {senate_dot_full_run:,} full-run simulations per complete dot · A 50–50 Senate is colored Republican because the 2026 vice presidency is Republican.</div>
  </div>
</section>

<section class="section" id="context">
  <div class="section-heading">
    <div>
      <div class="section-kicker">National environment</div>
      <h2 class="section-title">National Context Indicators</h2>
    </div>
    <div class="section-copy">Political and economic conditions read dynamically from the canonical 2026 model row.</div>
  </div>
  <div class="mini-grid">{context_cards_html}</div>
</section>

<section class="section">
  <div class="section-heading">
    <div>
      <div class="section-kicker">Projected change</div>
      <h2 class="section-title">Senate Flips</h2>
    </div>
    <div class="section-copy">Seats where the model-assigned winner changes the party holding the seat.</div>
  </div>
  <div class="panel">{flips_table_html}</div>
</section>

<section class="section" id="validation">
  <div class="section-heading">
    <div>
      <div class="section-kicker">Audit & validation</div>
      <h2 class="section-title">Time-Machine Historical Validation</h2>
    </div>
    <div class="section-copy">Sealed outer-election tests, nested model selection and leave-cycle-out sensitivity diagnostics.</div>
  </div>

  <details class="panel diagnostic-details house-audit">
    <summary>Five sealed House elections · district-layer validation</summary>
    <p class="distribution-copy">Each outer election is excluded from regularization selection and fitting. Brier score, log loss, competitive-race calibration and aggregate seat error are measured out of sample.</p>
    <h3>Five sealed House elections</h3>
    <div class="wide-audit">{house_validation_html}</div>
    <h3 style="margin-top:24px;">Feature and aggregation contract</h3>
    <div class="wide-audit">{house_contract_html}</div>
  </details>

  <div class="panel">
    <p class="distribution-copy">
      Each completed midterm election is treated as the future once. The outer test cycle is excluded from model fitting, hyperparameter selection and strategy choice; the remaining four cycles are training data. Across five tests this creates 170 learned-target forecasts plus 40 deterministic constraint outcomes. These are repeated comparisons from five elections, not 210 independent elections.
    </p>

    <div class="mini-grid">
      <div class="mini-card" style="border-top:5px solid #7C3AED;">
        <div class="mini-label">Historical outer tests</div>
        <div class="mini-value purple">5</div>
      </div>
      <div class="mini-card" style="border-top:5px solid #111827;">
        <div class="mini-label">Popular margin MAE</div>
        <div class="mini-value">{TM_MARGIN_MAE:.2f} pp</div>
      </div>
      <div class="mini-card" style="border-top:5px solid {DEM};">
        <div class="mini-label">Democratic House MAE</div>
        <div class="mini-value dem">{TM_HOUSE_MAE:.2f}</div>
      </div>
      <div class="mini-card" style="border-top:5px solid {REP};">
        <div class="mini-label">Senate prior MAE</div>
        <div class="mini-value rep">{TM_SENATE_MAE:.2f}</div>
      </div>
    </div>

    <div class="plot-stage">{time_machine_chart_html}</div>

    <h3 style="margin-top:28px;">Popular-vote model selection</h3>
    <p class="distribution-copy">The national popular vote remains one of the 42 forecast targets, but its production method is selected on the raw D-R margin—the headline quantity shown above. The direct EDPP/ERPP national expectation, a guarded historical-error correction, Ridge, the tree ensemble and the historical mean all face sealed election-year tests. The correction may be used only when it improves inner out-of-sample margin error; it is never added mechanically.</p>
    {popular_vote_methods_html}
    {popular_vote_bridge_html}

    <details class="diagnostic-details">
      <summary>Show national-to-Senate isolation tests</summary>
      {module_isolation_html}
    </details>

    <h3 style="margin-top:28px;">Senate polling-error backtest</h3>
    <p class="distribution-copy">The Senate state model predicts the historical polling miss—not the final margin in isolation. Every bar is an election-year outer test: the displayed cycle was excluded from error-model fitting, correction-size selection and probability calibration. A lower purple bar means the validated correction beat the poll-only anchor for that cycle.</p>
    <div class="plot-stage">{senate_error_chart_html}</div>

    <details class="diagnostic-details">
      <summary>Show Senate error-validation table</summary>
      {senate_error_table_html}
    </details>

    {time_machine_scorecard_html}

    <h3 style="margin-top:28px;">Did refinement improve the model?</h3>
    <p class="distribution-copy">The first exam occurs only inside the four-cycle training sample and selects the strategy and hyperparameters. The second exam is the sealed outer election. Improvement is credited only when the refined nested selection lowers error on that untouched outer election relative to a fixed baseline. Repeating the same test after reading its answers is deliberately excluded.</p>
    {second_exam_improvement_html}

    <details class="diagnostic-details">
      <summary>Show all 15 inner-to-outer exam transitions</summary>
      {nested_exam_progression_html}
    </details>
  </div>

  <div class="panel">
    <h3>2026 leave-cycle-out sensitivity</h3>
    <p class="distribution-copy">Five partial models forecast 2026 after omitting one historical election at a time. Their mean and spread are a jackknife-style sensitivity diagnostic. The production forecast is still refitted on all five completed elections; the partial-model average is not substituted automatically and is not a cross-validation accuracy score.</p>
    {time_machine_2026_folds_html}
    <details class="diagnostic-details">
      <summary>Show jackknife sensitivity summary</summary>
      {time_machine_2026_summary_html}
    </details>
  </div>
</section>

<section class="section two">
  <div class="panel">
    <div class="section-kicker">Diagnostics</div>
    <h2 class="section-title">Model Quality</h2>
    <div style="height:14px;"></div>
    {quality_table_html}
  </div>

  <div class="panel">
    <div class="section-kicker">Diagnostics</div>
    <h2 class="section-title">Electoral Risk</h2>
    <div style="height:14px;"></div>
    {risk_table_html}
  </div>
</section>

<section class="section">
  <div class="panel">
    <div class="section-kicker">Uncertainty</div>
    <h2 class="section-title">Final Uncertainty Snapshot</h2>
    <div style="height:14px;"></div>
    {uncertainty_table_html}
  </div>
</section>

<footer class="footer">
  Generated automatically from the audited Election Model<br>
  <b>Forecast by Juan Ignacio Garbanzo Fallas</b><br>
  Observatorio de los Estados Unidos · CIEP-UCR
</footer>


<script id="houseData" type="application/json">{house_records_json}</script>
<script>
(() => {{
  const records = JSON.parse(document.getElementById('houseData').textContent);
  const rows = Array.from(document.querySelectorAll('#houseTable tbody tr'));
  const search = document.getElementById('houseSearch');
  const rating = document.getElementById('houseRating');
  const count = document.getElementById('houseCount');
  const inspector = document.getElementById('houseInspector');
  const esc = value => String(value ?? '—').replace(/[&<>"']/g, c => ({{'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}}[c]));
  const render = index => {{
    const record = records[index];
    rows.forEach(row => row.classList.toggle('is-selected', Number(row.dataset.index) === index));
    inspector.innerHTML = `<div class="inspector-title">${{esc(record['District ID'])}} · all available variables</div><div class="inspector-grid">${{Object.entries(record).map(([key,value]) => `<div class="inspector-item"><div class="inspector-key">${{esc(key)}}</div><div class="inspector-value">${{esc(value)}}</div></div>`).join('')}}</div>`;
  }};
  const filterRows = () => {{
    const q = search.value.trim().toLowerCase();
    const wanted = rating.value;
    let shown = 0;
    rows.forEach(row => {{
      const visible = (!q || row.dataset.search.includes(q)) && (!wanted || row.dataset.rating === wanted);
      row.style.display = visible ? '' : 'none';
      shown += visible ? 1 : 0;
    }});
    count.textContent = `${{shown}} races`;
  }};
  rows.forEach(row => row.addEventListener('click', () => render(Number(row.dataset.index))));

  const selectHouseDistrict = index => {{
    render(index);
    document.querySelectorAll('.house-district,.house-cart-tile').forEach(mark => mark.classList.toggle('is-selected', Number(mark.dataset.index) === index));
  }};
  window.selectHouseDistrict = selectHouseDistrict;

  const houseMap = document.getElementById('houseDistrictMap');
  const houseVisual = document.getElementById('houseMapVisual');
  const houseMapTitle = document.getElementById('houseMapTitle');
  const houseMapDescription = document.getElementById('houseMapDescription');
  const houseCartogram = document.getElementById('houseCartogram');
  const houseGeographicMap = document.getElementById('houseGeographicMap');
  const houseLayout = document.getElementById('houseMapLayout');
  const houseMetric = document.getElementById('houseMapMetric');
  const houseLegend = document.getElementById('houseMapLegend');
  const houseTooltip = document.getElementById('houseMapTooltip');
  const housePaths = houseMap ? Array.from(houseMap.querySelectorAll('.house-district')) : [];
  const houseTiles = Array.from(document.querySelectorAll('.house-cart-tile'));
  const houseMarks = [...houseTiles, ...housePaths];
  const colorForRating = ratingValue => ({{
    'Safe D':'#073B75','Likely D':'#1769AA','Lean D':'#5B9BD5','Tilt D':'#A8D5EC',
    'Toss-Up':'#F4D35E','Tossup':'#F4D35E','Tilt R':'#F2B5B9','Lean R':'#DF6670',
    'Likely R':'#C1121F','Safe R':'#7F0000'
  }}[ratingValue] || '#CBD5E1');
  const gradientColor = value => {{
    const p = Math.max(0, Math.min(100, Number(value)));
    if (p >= 95) return '#073B75'; if (p >= 80) return '#1769AA';
    if (p >= 65) return '#5B9BD5'; if (p >= 55) return '#A8D5EC';
    if (p > 45) return '#F4D35E'; if (p > 35) return '#F2B5B9';
    if (p > 20) return '#DF6670'; if (p > 5) return '#C1121F'; return '#7F0000';
  }};
  const partyMargin = value => {{
    const numeric = Number(value);
    if (!Number.isFinite(numeric)) return '—';
    const magnitude = Math.abs(numeric);
    const decimals = magnitude < 1 ? 2 : 1;
    if (numeric > 0) return `D+${{magnitude.toFixed(decimals)}}`;
    if (numeric < 0) return `R+${{magnitude.toFixed(decimals)}}`;
    return 'EVEN 0.00';
  }};
  const marginColor = value => {{
    const margin = Number(value);
    if (!Number.isFinite(margin)) return '#D8DEE5';
    const magnitude = Math.abs(margin);
    if (magnitude < 0.005) return '#F4D35E';
    if (margin > 0) {{
      if (magnitude < 1) return '#C7E4F2';
      if (magnitude < 3) return '#9CCCE4';
      if (magnitude < 7) return '#5B9BD5';
      if (magnitude < 15) return '#1769AA';
      return '#073B75';
    }}
    if (magnitude < 1) return '#F5C6CA';
    if (magnitude < 3) return '#EC9FA6';
    if (magnitude < 7) return '#DF6670';
    if (magnitude < 15) return '#C1121F';
    return '#7F0000';
  }};
  const readableTileText = (tile, mode) => {{
    if (tile.dataset.tier === 'None') return '#667085';
    if (mode === 'flips' && tile.dataset.flip) return '#111827';
    if (mode === 'margin' && Math.abs(Number(tile.dataset.margin)) < 7) return '#111827';
    if (mode === 'probability' && Number(tile.dataset.prob) > 35 && Number(tile.dataset.prob) < 65) return '#111827';
    if (mode === 'ratings' && ['Toss-Up','Tilt D','Tilt R'].includes(tile.dataset.ratingKey)) return '#111827';
    return '#FFFFFF';
  }};
  const houseLegends = {{
    probability:'<span><i style="background:#7F0000"></i>R win</span><span><i style="background:#F4D35E"></i>Toss-up</span><span><i style="background:#073B75"></i>D win</span>',
    rating:'<span><i style="background:#073B75"></i>Safe D</span><span><i style="background:#5B9BD5"></i>Lean D</span><span><i style="background:#F4D35E"></i>Toss-up</span><span><i style="background:#DF6670"></i>Lean R</span><span><i style="background:#7F0000"></i>Safe R</span>',
    consensus:'<span><i style="background:#5B9BD5"></i>D source consensus</span><span><i style="background:#F4D35E"></i>Mixed sources</span><span><i style="background:#DF6670"></i>R source consensus</span>',
    margin:'<span><i style="background:#073B75"></i>D+25 or more</span><span><i style="background:#F4D35E"></i>Even</span><span><i style="background:#7F0000"></i>R+25 or more</span>',
    flips:'<span><i style="background:#1769AA"></i>D hold</span><span><i style="background:repeating-linear-gradient(45deg,#1769AA 0 4px,#D9ECF7 4px 8px)"></i>D flip</span><span><i style="background:repeating-linear-gradient(45deg,#C1121F 0 4px,#F8D5D8 4px 8px)"></i>R flip</span><span><i style="background:#C1121F"></i>R hold</span>'
  }};
  const houseMarkColor = (mark, mode, svgMark) => {{
    if (mode === 'rating') return colorForRating(mark.dataset.rating);
    if (mode === 'consensus') return colorForRating(mark.dataset.consensus);
    if (mode === 'margin') return marginColor(mark.dataset.margin);
    if (mode === 'flips') {{
      if (mark.dataset.flip === 'D→R') return svgMark ? 'url(#houseFlipR)' : 'repeating-linear-gradient(45deg,#C1121F 0 5px,#F8D5D8 5px 10px)';
      if (mark.dataset.flip === 'R→D') return svgMark ? 'url(#houseFlipD)' : 'repeating-linear-gradient(45deg,#1769AA 0 5px,#D9ECF7 5px 10px)';
      return mark.dataset.winner === 'D' ? '#1769AA' : '#C1121F';
    }}
    return gradientColor(mark.dataset.prob);
  }};
  const houseTileText = (tile, mode) => {{
    if (mode === 'flips' && ['D→R','R→D'].includes(tile.dataset.flip)) return '#111827';
    if (mode === 'margin' && Math.abs(Number(tile.dataset.margin)) < 7) return '#111827';
    if (mode === 'probability' && Number(tile.dataset.prob) > 35 && Number(tile.dataset.prob) < 65) return '#111827';
    if ((mode === 'rating' && ['Toss-Up','Tilt D','Tilt R'].includes(tile.dataset.rating)) ||
        (mode === 'consensus' && ['Toss-Up','Tilt D','Tilt R'].includes(tile.dataset.consensus))) return '#111827';
    return '#FFFFFF';
  }};
  const updateHouseMap = () => {{
    const mode = houseMetric.value;
    housePaths.forEach(path => path.setAttribute('fill', houseMarkColor(path, mode, true)));
    houseTiles.forEach(tile => {{
      tile.style.background = houseMarkColor(tile, mode, false);
      tile.style.color = houseTileText(tile, mode);
    }});
    houseLegend.innerHTML = houseLegends[mode];
  }};
  const updateHouseLayout = () => {{
    const cartogramActive = houseLayout.value === 'cartogram';
    houseCartogram.hidden = !cartogramActive;
    houseGeographicMap.hidden = cartogramActive;
    houseMapTitle.textContent = cartogramActive ? 'House district cartogram' : 'House geographic forecast map';
    houseMapDescription.textContent = cartogramActive
      ? 'One square represents one voting district.'
      : 'Official CD120 districts · Census cartographic shorelines · standard composite Albers USA.';
    houseTooltip.style.display = 'none';
  }};
  const houseModeDetail = mark => {{
    const mode = houseMetric.value;
    const dprob = Number(mark.dataset.prob);
    const rprob = Number(mark.dataset.rprob);
    const base = `<b>${{esc(mark.dataset.district)}}</b>`;
    if (mode === 'probability') return `${{base}}<br><b style="color:#0B5CAB">D win ${{dprob.toFixed(1)}}%</b> · <b style="color:#C1121F">R win ${{rprob.toFixed(1)}}%</b><br>${{esc(mark.dataset.rating)}} · ${{partyMargin(mark.dataset.margin)}}`;
    if (mode === 'rating') return `${{base}}<br><b>Model rating: ${{esc(mark.dataset.rating)}}</b><br>D ${{dprob.toFixed(1)}}% · R ${{rprob.toFixed(1)}}% · ${{partyMargin(mark.dataset.margin)}}`;
    if (mode === 'consensus') return `${{base}}<br><b>Source consensus: ${{esc(mark.dataset.consensus)}}</b><br>${{esc(mark.dataset.sources)}} sources · Model: ${{esc(mark.dataset.rating)}}`;
    if (mode === 'margin') return `${{base}}<br><b>Projected margin: ${{partyMargin(mark.dataset.margin)}}</b><br>D ${{dprob.toFixed(1)}}% · R ${{rprob.toFixed(1)}}% · PVI ${{esc(mark.dataset.pvi)}}`;
    return `${{base}}<br><b>${{esc(mark.dataset.flip)}}</b> · projected ${{esc(mark.dataset.winner)}}<br>Incumbent: ${{esc(mark.dataset.incumbent)}} · PVI ${{esc(mark.dataset.pvi)}}`;
  }};
  const showHouseTooltip = (event, mark) => {{
    houseTooltip.innerHTML = houseModeDetail(mark);
    const shell = houseVisual.getBoundingClientRect();
    const tooltipWidth = 280;
    houseTooltip.style.left = `${{Math.max(8, Math.min(event.clientX - shell.left + 14, shell.width - tooltipWidth - 8))}}px`;
    houseTooltip.style.top = `${{Math.max(event.clientY - shell.top - 30, 8)}}px`;
    houseTooltip.style.display = 'block';
  }};
  houseMarks.forEach(mark => {{
    mark.addEventListener('mousemove', event => showHouseTooltip(event, mark));
    mark.addEventListener('mouseenter', event => showHouseTooltip(event, mark));
    mark.addEventListener('mouseleave', () => houseTooltip.style.display = 'none');
    mark.addEventListener('click', () => {{
      selectHouseDistrict(Number(mark.dataset.index));
      document.getElementById('house-races').scrollIntoView({{behavior:'smooth', block:'start'}});
    }});
    mark.addEventListener('keydown', event => {{ if ((event.key === 'Enter' || event.key === ' ') && mark.tagName.toLowerCase() === 'path') mark.click(); }});
  }});
  if (houseMetric) houseMetric.addEventListener('change', updateHouseMap);
  if (houseLayout) houseLayout.addEventListener('change', updateHouseLayout);

  const senateMetric = document.getElementById('senateMapMetric');
  const senateLegend = document.getElementById('senateMapLegend');
  const senateTooltip = document.getElementById('senateMapTooltip');
  const senateSelection = document.getElementById('senateMapSelection');
  const senateTiles = Array.from(document.querySelectorAll('.state-tile'));
  const senateOutcomeParty = tile => {{
    const outcome = tile.dataset.outcome.toLowerCase();
    if (outcome.includes('democratic')) return 'D';
    if (outcome.includes('republican')) return 'R';
    return Number(tile.dataset.prob) >= 50 ? 'D' : 'R';
  }};
  const senateColor = (tile, mode) => {{
    if (tile.dataset.tier === 'None') return '#D8DEE5';
    const probability = Number(tile.dataset.prob);
    const party = senateOutcomeParty(tile);
    if (mode === 'forecast') return party === 'D' ? '#1769AA' : '#C1121F';
    if (mode === 'probability') return gradientColor(probability);
    if (mode === 'ratings') return colorForRating(tile.dataset.ratingKey);
    if (mode === 'margin') return marginColor(tile.dataset.margin);
    if (tile.dataset.flip === 'D') return 'repeating-linear-gradient(45deg,#1769AA 0 6px,#D9ECF7 6px 12px)';
    if (tile.dataset.flip === 'R') return 'repeating-linear-gradient(45deg,#C1121F 0 6px,#F8D5D8 6px 12px)';
    return party === 'D' ? '#073B75' : '#7F0000';
  }};
  const senateTileLabel = (tile, mode) => {{
    if (tile.dataset.tier === 'None') return '—';
    const probability = Number(tile.dataset.prob);
    const party = senateOutcomeParty(tile);
    if (mode === 'forecast') return `${{party}} ${{tile.dataset.flip ? 'FLIP' : 'HOLD'}}`;
    if (mode === 'probability') return `${{party}} ${{Math.max(probability,100-probability).toFixed(0)}}%`;
    if (mode === 'ratings') return tile.dataset.ratingKey.replace(' D','').replace(' R','').toUpperCase();
    if (mode === 'margin') return partyMargin(tile.dataset.margin);
    return tile.dataset.flip ? `${{tile.dataset.flip}} FLIP` : `${{party}} HOLD`;
  }};
  const senateLegends = {{
    forecast:'<span><i style="background:#1769AA"></i>Democratic forecast</span><span><i style="background:#C1121F"></i>Republican forecast</span><span><i class="outline-key"></i>Model monitored</span>',
    probability:'<span><i style="background:#7F0000"></i>R win probability</span><span><i style="background:#F4D35E"></i>Near 50–50</span><span><i style="background:#073B75"></i>D win probability</span>',
    ratings:'<span><i style="background:#073B75"></i>Safe D</span><span><i style="background:#5B9BD5"></i>Lean D</span><span><i style="background:#F4D35E"></i>Toss-up</span><span><i style="background:#DF6670"></i>Lean R</span><span><i style="background:#7F0000"></i>Safe R</span>',
    margin:'<span><i style="background:#073B75"></i>D+15 or more</span><span><i style="background:#5B9BD5"></i>D+3 to D+6.9</span><span><i style="background:#C7E4F2"></i>D+0.01 to D+2.9</span><span><i style="background:#F4D35E"></i>Exactly even</span><span><i style="background:#F5C6CA"></i>R+0.01 to R+2.9</span><span><i style="background:#DF6670"></i>R+3 to R+6.9</span><span><i style="background:#7F0000"></i>R+15 or more</span>',
    flips:'<span><i style="background:#073B75"></i>D hold</span><span><i style="background:repeating-linear-gradient(45deg,#1769AA 0 4px,#D9ECF7 4px 8px)"></i>D flip</span><span><i style="background:repeating-linear-gradient(45deg,#C1121F 0 4px,#F8D5D8 4px 8px)"></i>R flip</span><span><i style="background:#7F0000"></i>R hold</span>'
  }};
  const updateSenateMap = () => {{
    const mode = senateMetric.value;
    senateTiles.forEach(tile => {{
      tile.style.background = senateColor(tile, mode);
      tile.style.color = readableTileText(tile, mode);
      const status = tile.querySelector('.state-status');
      if (status) status.textContent = senateTileLabel(tile, mode);
      const star = tile.querySelector('.state-star');
      if (star) star.style.color = tile.style.color;
    }});
    senateLegend.innerHTML = senateLegends[mode];
    senateSelection.innerHTML = 'Hover or click a state to inspect the selected view.';
  }};
  const senatePartyColor = party => party === 'D' ? '#0B5CAB' : '#C1121F';
  const senateMarginMarkup = value => {{
    const numeric = Number(value);
    const party = numeric >= 0 ? 'D' : 'R';
    return `<span style="color:${{senatePartyColor(party)}};font-weight:850">${{partyMargin(value)}}</span>`;
  }};
  const senateOutcomeMarkup = tile => {{
    const party = senateOutcomeParty(tile);
    return `<b style="color:${{senatePartyColor(party)}}">${{esc(tile.dataset.outcome)}}</b>`;
  }};
  const senateRatingMarkup = tile => {{
    const rating = tile.dataset.ratingKey;
    const lower = rating.toLowerCase();
    const color = lower.includes('d') ? '#0B5CAB' : lower.includes('r') ? '#C1121F' : '#B18400';
    return `<span style="color:${{color}};font-weight:850">${{esc(rating)}}</span>`;
  }};
  const senateVoteShareMarkup = tile => {{
    const dshare = Number(tile.dataset.dshare), rshare = Number(tile.dataset.rshare);
    if (!Number.isFinite(dshare) || !Number.isFinite(rshare)) return '';
    return `<b style="color:#0B5CAB">Democratic ${{dshare.toFixed(2)}}%</b> · <b style="color:#C1121F">Republican ${{rshare.toFixed(2)}}%</b><br><span style="color:#64748B;font-size:11px">Projected two-party vote share</span>`;
  }};
  const senateDetail = (tile, mode) => {{
    const base = `<b>${{esc(tile.dataset.state)}} (${{esc(tile.dataset.abbr)}})</b>`;
    if (tile.dataset.tier === 'None') return `${{base}}<br>No 2026 Senate election`;
    const dprob = Number(tile.dataset.prob), rprob = 100 - dprob;
    if (mode === 'forecast') return `${{base}}<br>${{senateOutcomeMarkup(tile)}}<br>${{senateRatingMarkup(tile)}} · ${{senateMarginMarkup(tile.dataset.margin)}}<br>${{senateVoteShareMarkup(tile)}}`;
    if (mode === 'probability') return `${{base}}<br><b style="color:#0B5CAB">D win ${{dprob.toFixed(1)}}%</b> · <b style="color:#C1121F">R win ${{rprob.toFixed(1)}}%</b><br>${{esc(tile.dataset.outcome)}}`;
    if (mode === 'ratings') return `${{base}}<br><b>Model rating: ${{senateRatingMarkup(tile)}}</b><br><b style="color:#0B5CAB">D win ${{dprob.toFixed(1)}}%</b> · <b style="color:#C1121F">R win ${{rprob.toFixed(1)}}%</b>`;
    if (mode === 'margin') return `${{base}}<br><b>Projected margin: ${{senateMarginMarkup(tile.dataset.margin)}}</b><br>${{senateVoteShareMarkup(tile)}}`;
    return `${{base}}<br>${{senateOutcomeMarkup(tile)}}<br>${{senateRatingMarkup(tile)}} · ${{senateMarginMarkup(tile.dataset.margin)}}`;
  }};
  senateTiles.forEach(tile => {{
    tile.addEventListener('mousemove', event => {{
      senateTooltip.innerHTML = senateDetail(tile, senateMetric.value);
      const shell = tile.closest('.senate-map-shell').getBoundingClientRect();
      senateTooltip.style.left = `${{Math.min(event.clientX - shell.left + 14, shell.width - 290)}}px`;
      senateTooltip.style.top = `${{Math.max(event.clientY - shell.top - 25, 8)}}px`;
      senateTooltip.style.display = 'block';
    }});
    tile.addEventListener('mouseleave', () => senateTooltip.style.display = 'none');
    tile.addEventListener('click', () => {{ senateSelection.innerHTML = senateDetail(tile, senateMetric.value); }});
  }});
  if (senateMetric) senateMetric.addEventListener('change', updateSenateMap);
  updateHouseMap();
  updateSenateMap();

  search.addEventListener('input', filterRows);
  rating.addEventListener('change', filterRows);
  const firstCompetitive = records.findIndex(record => record.Competitive === true);
  render(firstCompetitive >= 0 ? firstCompetitive : 0);
}})();
</script>

</body>
</html>
"""

required_dashboard_sections = [
    "Congress at a glance", "House, district by district", "Explore all 435 forecasts",
    "Senate forecast map", "Senate Race Forecasts", "Control Probability",
    "Monte Carlo Seat Distribution", "National Context Indicators", "Senate Flips",
    "Time-Machine Historical Validation", "Model Quality", "Electoral Risk",
    "Final Uncertainty Snapshot",
]
missing_dashboard_sections = [title for title in required_dashboard_sections if title not in html]
if missing_dashboard_sections:
    raise AssertionError("El dashboard perdió secciones: " + ", ".join(missing_dashboard_sections))
house_html_row_count = html.count('class="house-race-row')
if house_html_row_count != 435:
    raise AssertionError(
        f"El Race Desk House contiene {house_html_row_count} filas interactivas; se esperaban 435."
    )
if html.count('data-rprob=') != 870 or '<th>R win</th>' not in html:
    raise AssertionError("El dashboard no expone ambas probabilidades House en mapa y Race Desk.")
if html.count('class="water-mask"') != 0:
    raise AssertionError("La revisión House no debe usar máscaras cartográficas manuales.")
if html.count('class="house-state-outline"') != 50:
    raise AssertionError("El mapa House no contiene los 50 contornos estatales oficiales.")
if html.count('class="house-cart-tile"') != 435:
    raise AssertionError("El cartograma House no expone exactamente 435 distritos.")
if html.count('class="cart-state"') != 50:
    raise AssertionError("El cartograma House no expone los 50 estados.")
if 'id="houseCartogram"' not in html or 'id="houseMapLayout"' not in html:
    raise AssertionError("Falta la vista principal de cartograma House.")
if '!important;color:#3B2B00!important' in html:
    raise AssertionError("Un estilo antiguo sigue bloqueando los modos Senate.")
if html.index('id="house-races"') > html.index('id="senate"'):
    raise AssertionError("El Race Desk House debe aparecer inmediatamente después del mapa House.")
if 'data-rating-key=' not in html or 'senateTileLabel' not in html:
    raise AssertionError("Los modos interactivos del mapa Senate no están completos.")
if len(house_svg_paths) != 435 or html.count('class="house-district"') != 435:
    raise AssertionError("El HTML House no contiene 435 polígonos SVG interactivos.")
if 'id="houseMapMetric"' not in html or 'id="senateMapMetric"' not in html:
    raise AssertionError("Faltan selectores interactivos de mapas.")
if 'data-dshare=' not in html or 'Projected two-party vote share' not in html:
    raise AssertionError("Los hovers del Senado no incluyen el vote share proyectado.")
if 'Midterms 2026 model logo' not in html:
    raise AssertionError("El logo Núcleo 42 no quedó integrado en el dashboard.")
if 'url(#houseFlipD)' not in html or 'url(#houseFlipR)' not in html:
    raise AssertionError("Faltan patrones de flips House.")

if len(house_map_geometry["districts"]) != 435 or house_detail["GEOID4"].nunique() != 435:
    raise AssertionError("El mapa House no contiene 435 geometrías y 435 ubicaciones.")
if len(senate_map.loc[senate_map["Tier"].ne("None")]) != 35:
    raise AssertionError("El mapa Senate no contiene las 35 elecciones de 2026.")

# ------------------------------------------------------------
# 15. EXPORTAR
# ------------------------------------------------------------
with open(output_html, "w", encoding="utf-8") as f:
    f.write(html)

print()
print("="*70)
print("PLOTLY VISUAL A — DASHBOARD GENERATED")
print("="*70)
print(output_html)
print("="*70)
